In [ ]:
!pip install -q fastapi 'uvicorn[standard]' python-multipart bcrypt pyjwt pandas numpy scikit-learn scipy matplotlib langgraph vaderSentiment pdfplumber requests pyngrok


In [ ]:
import os
os.makedirs('/content/app/static', exist_ok=True)
os.chdir('/content/app')


In [ ]:
%%writefile /content/app/alts.py
"""Alternatives shown in the detail panels: other sea routes for a shipment, nearby lower-congestion ports for a port, nearby outlets that can transfer stock to an outlet."""
import json, numpy as np, pandas as pd
import blocks as B
from blocks import Q, TEAL, BLUE, AMBER, RED, VIOLET
COLORS = [TEAL, BLUE, AMBER, VIOLET, "#d6409f", "#8b8d98"]
def hav(la1, lo1, la2, lo2, R=6371.0):
    p = np.pi / 180; a = np.sin((np.asarray(la2) - la1) * p / 2) ** 2 + np.cos(la1 * p) * np.cos(np.asarray(la2) * p) * np.sin((np.asarray(lo2) - lo1) * p / 2) ** 2; return 2 * R * np.arcsin(np.sqrt(a))
def usd(x): return "${:,.0f}".format(x)
VARIANTS = [((), "Default"), (("suez",), "Avoid Suez"), (("panama",), "Avoid Panama"), (("malacca",), "Avoid Malacca"), (("hormuz",), "Avoid Hormuz"), (("storm",), "Avoid storms")]
def route_alternatives(dom, o_xy, d_xy, tonnes=20.0, interp=None, rate=0.012, day_cost=120.0, dwell=0.0):
    """-> list of options sorted by total cost; each {label, name, nm, days, freight, delay_cost, total, wind, chain, avoid, flags}"""
    seen, out = {}, []
    for av, lab in VARIANTS:
        try: opts = dom.route_options(o_xy, d_xy, avoid=av)
        except Exception: continue
        if not opts: continue
        o = opts[0]; key = o["name"]
        if key in seen: continue
        seen[key] = 1; ch = o["chain"]; wind = 0.0
        if interp:
            try: wind = max(dom.weather_at(*interp(ch, f), allow_live=False)["wind"] for f in (.15, .35, .55, .75, .95))
            except Exception: pass
        days = o["nm"] / (14 * 24) + dwell; fr = o["nm"] * tonnes * rate; dl = days * day_cost; risk_cost = max(0, wind - 15) * 0.02 * days * day_cost * 3
        out.append({"label": lab, "name": o["name"], "nm": o["nm"], "days": round(days, 1), "freight": fr, "delay_cost": dl + risk_cost, "total": fr + dl + risk_cost, "wind": round(wind, 1), "chain": ch, "avoid": list(av)})
    if not out: return out
    out.sort(key=lambda r: r["total"]); fast = min(out, key=lambda r: r["days"]); short = min(out, key=lambda r: r["nm"])
    for r in out: r["tag"] = ", ".join(t for t, c in (("Recommended", r is out[0]), ("Fastest", r is fast), ("Shortest", r is short)) if c)
    return out
def route_blocks(opts, sid=None, title="Route alternatives", cur=None):
    if not opts: return [B.callout("No alternative sea route found for this lane.", "info")]
    lines = [{"pts": r["chain"], "color": COLORS[i % len(COLORS)], "w": 4 if i == 0 else 2.5, "dash": None if i == 0 else "6 6"} for i, r in enumerate(opts)]
    rows = [{"option": f"{i + 1}. {r['label']}", "route": r["name"], "nm": r["nm"], "days": r["days"], "freight": r["freight"], "delay": r["delay_cost"], "total": r["total"], "wind": r["wind"], "tag": r["tag"]} for i, r in enumerate(opts)]
    best, worst = opts[0], opts[-1]; pts = [p for r in opts for p in r["chain"]]; la = [p[0] for p in pts]; lo = [p[1] for p in pts]
    view = {"bounds": [[min(la) - 4, min(lo) - 6], [max(la) + 4, max(lo) + 6]]}
    ends = [{"id": "o", "lat": opts[0]["chain"][0][0], "lon": opts[0]["chain"][0][1], "color": "#1f2430", "r": 6, "tip": "start"}, {"id": "d", "lat": opts[0]["chain"][-1][0], "lon": opts[0]["chain"][-1][1], "color": RED, "r": 6, "tip": "destination"}]
    bl = [B.card(title, B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "alt_ends", "pts": ends}], h=300, view=view, key="alt_map", legend=[(str(rows[i]["option"]) + (" (solid = recommended)" if i == 0 else " (dashed)"), COLORS[i % len(COLORS)]) for i in range(min(len(rows), len(COLORS)))] + [("Start", "#1f2430"), ("Destination", RED)]), B.table(rows, ["option", "route", B.num_col("nm", "nm", "int"), B.num_col("days", "days", "num"), B.num_col("freight", "freight cost", "money"), B.num_col("delay", "time + weather cost", "money"), B.num_col("total", "total", "money"), B.num_col("wind", "max wind m/s", "num"), "tag"], h=min(60 + 40 * len(rows), 330), key="alt_t"),
                       sub=f"Best: {best['label']} ({best['name']}), {best['days']} days, {usd(best['total'])}. Worst: {usd(worst['total'])}. Cost = $0.012 per tonne-nm + $120 per day + weather penalty above 15 m/s.")]
    return bl
def port_alternatives(port, k=6):
    p = Q("SELECT port_name, country, region, lat, lon, congestion_index ci, avg_dwell_days dw FROM ports"); me = p[p.port_name == port]
    if not len(me): return []
    me = me.iloc[0]; p["km"] = hav(me.lat, me.lon, p.lat.values, p.lon.values); c = p[(p.port_name != port) & (p.km < 1800)].copy()
    c["better"] = (me.ci - c.ci); c["score"] = c.ci * 12 + c.dw * 4 + c.km / 100; c = c[c.ci < me.ci].sort_values("score").head(k)
    rows = [{"port": r.port_name, "country": r.country, "km": r.km, "ci": r.ci, "dwell": r.dw, "days_saved": max(0, me.dw - r.dw), "haul_days": r.km / 600 / 1 if r.country == me.country else r.km / 450} for r in c.itertuples()]
    if not rows: return [B.callout(f"No less-congested port within 1,800 km of {port}.", "info")]
    return [B.card("Alternative ports nearby", B.table(rows, ["port", "country", B.num_col("km", "km away", "int"), B.num_col("ci", "congestion", "num"), B.num_col("dwell", "dwell days", "num"), B.num_col("days_saved", "dwell saved", "num"), B.num_col("haul_days", "extra haul days (approx.)", "num")], h=min(70 + 40 * len(rows), 340), open={"kind": "port", "col": "port"}, key="alt_p"),
                    sub="Cargo for the port could be diverted here; haul by road/rail is estimated at 600 km/day domestic, 450 km/day cross-border. Check customs and carrier agreements before diverting.")]
def outlet_alternatives(oid, max_km=400):
    g = Q("SELECT outlet_id, lat, lon FROM outlet_geo"); me = g[g.outlet_id == oid]
    if not len(me): return []
    me = me.iloc[0]; g["km"] = hav(me.lat, me.lon, g.lat.values, g.lon.values); near = g[(g.outlet_id != oid) & (g.km <= max_km)].sort_values("km").head(12)
    need = Q("SELECT sku_name, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5 ORDER BY current_stock*1.0/MAX(weekly_demand,1) LIMIT 6", (oid,))
    if not len(need): return [B.callout("This outlet has no SKUs at or near the reorder level (within 150%), so no transfers are suggested.", "info")]
    if not len(near): return [B.callout(f"No outlet within {max_km} km to transfer stock from.", "info")]
    inv = Q(f"SELECT outlet_id, sku_name, current_stock st, reorder_threshold thr, weekly_demand wk FROM inventory WHERE outlet_id IN ({','.join('?' * len(near))}) AND sku_name IN ({','.join('?' * len(need))})", tuple(near.outlet_id) + tuple(need.sku_name))
    inv["spare"] = inv.st - (inv.thr + inv.wk); inv = inv[inv.spare > 0].merge(near[["outlet_id", "km"]], on="outlet_id"); rows = []
    for r in need.itertuples():
        want = max(1, int(r.thr + r.wk - r.st)); c = inv[inv.sku_name == r.sku_name].sort_values("km")
        if len(c):
            s = c.iloc[0]; q = int(min(want, s.spare)); days = max(.5, s.km * 1.3 / 320); rows.append({"sku": r.sku_name, "short_by": want, "from": s.outlet_id, "km": s.km, "units": q, "days": days, "vs_supplier": r.lead - days, "cost": s.km * 1.3 * 14})
        else: rows.append({"sku": r.sku_name, "short_by": want, "from": "none nearby has spare", "km": None, "units": 0, "days": None, "vs_supplier": None, "cost": None})
    d = pd.DataFrame(rows); mp = [{"type": "lines", "lines": [{"pts": [[float(me.lat), float(me.lon)], [float(g[g.outlet_id == r["from"]].lat.iloc[0]), float(g[g.outlet_id == r["from"]].lon.iloc[0])]], "color": COLORS[i % 6], "w": 3} for i, r in enumerate(rows) if r["units"]]},
                                {"type": "points", "id": "alt_o", "pts": [{"id": oid, "lat": float(me.lat), "lon": float(me.lon), "color": "#1f2430", "r": 8, "tip": oid}] + [{"id": r["from"], "lat": float(g[g.outlet_id == r["from"]].lat.iloc[0]), "lon": float(g[g.outlet_id == r["from"]].lon.iloc[0]), "color": TEAL, "r": 6, "tip": r["from"]} for r in rows if r["units"]]}]
    return [B.card("Suggested stock transfers from nearby outlets", B.map_(mp, h=240, key="alt_om", legend=[("This outlet (needs stock)", "#1f2430"), ("Outlet that can send stock", TEAL), ("Transfer line (colour per supplier outlet)", BLUE)]) if any(r["units"] for r in rows) else B.md("No nearby outlet has spare stock for these SKUs."), B.table(d, ["sku", B.num_col("short_by", "short by", "int"), "from", B.num_col("km", "km (straight line)", "int"), B.num_col("units", "transfer units", "int"), B.num_col("days", "transfer days (approx.)", "num"), B.num_col("vs_supplier", "days faster than supplier", "num"), B.num_col("cost", "est. cost ₹", "money")], h=min(80 + 40 * len(d), 330), open={"kind": "outlet", "col": "from"}, key="alt_ot"),
                    sub="Spare = stock above reorder level plus one week of demand. Distance is straight-line; road distance ≈ 1.3×. Cost ₹14 per road km. Compare with the supplier lead time before moving stock.")]


In [ ]:
%%writefile /content/app/analyst.py
"""Question understanding for the Copilot, ahead of the generic table query (qa.py). Each handler computes the answer from live data and returns
{"answer": lead text (markdown), "df": optional table, "sql": how it was computed, "extra": more markdown}. Nothing here asks the language model for a number."""
import re, json, datetime
import numpy as np, pandas as pd
from blocks import Q
import qa as QA, alts
CITY_STATE = {"Mumbai": "Maharashtra", "Pune": "Maharashtra", "Nashik": "Maharashtra", "Nagpur": "Maharashtra", "Delhi": "Delhi", "Bengaluru": "Karnataka", "Mysuru": "Karnataka", "Hyderabad": "Telangana", "Chennai": "Tamil Nadu", "Coimbatore": "Tamil Nadu", "Madurai": "Tamil Nadu",
              "Kolkata": "West Bengal", "Ahmedabad": "Gujarat", "Surat": "Gujarat", "Vadodara": "Gujarat", "Jaipur": "Rajasthan", "Lucknow": "Uttar Pradesh", "Kochi": "Kerala", "Thiruvananthapuram": "Kerala", "Chandigarh": "Chandigarh", "Bhopal": "Madhya Pradesh", "Indore": "Madhya Pradesh",
              "Patna": "Bihar", "Ranchi": "Jharkhand", "Raipur": "Chhattisgarh", "Bhubaneswar": "Odisha", "Guwahati": "Assam", "Goa": "Goa", "Visakhapatnam": "Andhra Pradesh", "Vijayawada": "Andhra Pradesh", "Kanpur": "Uttar Pradesh", "Dehradun": "Uttarakhand", "Amritsar": "Punjab", "Ludhiana": "Punjab"}
CHOKE = {"suez": ("Suez Canal", 30.5, 32.3), "panama": ("Panama Canal", 9.1, -79.7), "malacca": ("Strait of Malacca", 2.5, 101.5), "hormuz": ("Strait of Hormuz", 26.5, 56.5), "bosphorus": ("Bosphorus", 41.1, 29.0), "cape": ("Cape of Good Hope", -34.4, 18.5), "red sea": ("Red Sea", 18.0, 40.0)}
def _w(q, t): return re.search(r"(?<![a-z0-9])" + re.escape(t) + r"(?![a-z0-9])", q) is not None
def _md(df, n=10): return QA.md(df.head(n)) if df is not None and len(df) else ""
def _health(dec): return (((dec or {}).get("ev") or {}).get("health")) or {}
def _agent_scores(e):
    import entities as EN; out = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(e)
        if v is not None: out[a] = round(float(v), 1)
    return out
NEXT = {"staffing_shortage": "add a cover shift or move staff from a nearby outlet", "health_alert": "review the lowest agent score below and fix that first", "reorder": "place the reorder or transfer stock from a nearby outlet", "stockout": "reorder now or transfer stock",
        "corrective_action": "complete the corrective plan before the re-audit", "anomaly_review": "check the POS feed and the day's sales for this outlet", "campaign_review": "pause or re-target the campaign", "retention": "talk to the staff at risk and review overtime"}
# ------------------------------------------------------------------------------------------------ Franchise
def explain_outlet(e, dec):
    o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
    if not len(o): return None
    r = o.iloc[0]; h = _health(dec).get(e); sc = _agent_scores(e); tk = Q("SELECT action_id id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT 8", (e,))
    avg = Q("SELECT AVG(revenue) rv, AVG(customer_satisfaction) cs FROM outlets").iloc[0]; low = Q("SELECT COUNT(*) n FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5", (e,)).n[0]
    att = Q("SELECT AVG(predicted_attrition_prob) p, COUNT(*) n FROM staff WHERE outlet_id=?", (e,)).iloc[0]
    lines = []
    if h is not None: lines.append(f"Health score **{h:.1f}**/100 ({'Critical' if h < 60 else 'Needs attention' if h < 80 else 'Healthy'}).")
    if sc: worst = sorted(sc.items(), key=lambda kv: kv[1])[:2]; lines.append("Weakest areas: " + ", ".join(f"**{a}** {v}" for a, v in worst) + " (each agent scores 0-100, higher is better).")
    lines.append(f"Revenue {r.revenue:,.0f} ({(r.revenue / avg.rv - 1):+.0%} vs chain average), satisfaction {r.customer_satisfaction:.2f} (chain {avg.cs:.2f}), {int(att.n)} staff with average attrition risk {att.p:.0%}, {int(low)} SKUs at or near reorder level.")
    why = "\n".join(f"- {t.type} ({t.severity}): {t.detail}" + (f" → {NEXT[t.type]}" if t.type in NEXT else "") for t in tk.itertuples()) or "- no open tickets"
    return {"answer": (f"**{e}** ({r.outlet_name}, {r.location}) is flagged because:\n" if (h is None or h < 80 or len(tk)) else f"**{e}** ({r.outlet_name}, {r.location}) is not flagged right now; here is its status:\n") + "\n".join("- " + l for l in lines) + "\n\n**Open tickets:**\n" + why, "df": pd.DataFrame([{"agent": a, "score": v} for a, v in sc.items()]) if sc else None, "sql": f"decision health + agent per_outlet scores + agent_actions for {e}"}
def risk_rank(q, dec, scope, n=5):
    hl = _health(dec)
    if not hl: return None
    s = pd.Series(hl).astype(float); worst = bool(re.search(r"\b(highest risk|most at risk|worst|lowest|riskiest|critical|weakest|needs? attention|problem|bottom)\b", q)); best = bool(re.search(r"\b(best|top|highest health|strongest|healthiest)\b", q)) and not re.search(r"highest risk", q)
    both = re.search(r"best.*worst|worst.*best", q) is not None
    if scope: s = s[s.index == scope]
    o = Q("SELECT outlet_id, outlet_name, location, revenue, customer_satisfaction csat FROM outlets").set_index("outlet_id")
    def frame(ix):
        rows = []
        for e in ix:
            sc = _agent_scores(e); wk = min(sc, key=sc.get) if sc else "-"; ct = int(Q("SELECT COUNT(*) n FROM agent_actions WHERE target=? AND status IN ('open','proposed') AND severity='Critical'", (e,)).n[0])
            rows.append({"outlet": e, "name": o.name if False else (o.outlet_name.get(e, "") if e in o.index else ""), "location": o.location.get(e, ""), "health": round(float(s[e]), 1), "weakest_area": f"{wk} {sc.get(wk, '')}", "critical_tickets": ct, "revenue": o.revenue.get(e), "csat": o.csat.get(e)})
        return pd.DataFrame(rows)
    low = s.sort_values().head(n).index; hi = s.sort_values(ascending=False).head(n).index
    if both or (best and worst):
        a, b = frame(hi), frame(low); return {"answer": f"Best outlet: **{a.outlet[0]}** (health {a.health[0]}). Worst outlet: **{b.outlet[0]}** (health {b.health[0]}). Health combines outlet, inventory, staff, marketing, audit and feedback agent scores.", "df": None, "sql": "decision health per outlet", "extra": "**Top " + str(n) + " by health**\n" + QA.md(a) + "\n\n**Bottom " + str(n) + " by health**\n" + QA.md(b)}
    if best and not worst: d = frame(hi); return {"answer": f"Healthiest outlet: **{d.outlet[0]}** (health {d.health[0]}, {d.location[0]}). Top {n} below.", "df": d, "sql": "decision health per outlet, descending"}
    d = frame(low); crit = int((s < 60).sum()); return {"answer": f"Highest risk: **{d.outlet[0]}** (health {d.health[0]}, weakest area {d.weakest_area[0]}). {crit} outlets are below 60 (critical). Lowest {n} below; ask \"why is {d.outlet[0]} flagged\" for the reasons.", "df": d, "sql": "decision health per outlet, ascending"}
def inventory_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name, category, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead, stockout_risk_prob p FROM inventory" + w, p)
    if not len(d): return None
    d["cover_d"] = d.st / (d.wk / 7).replace(0, np.nan); below = d[d.st <= d.thr]; wk1 = d[d.cover_d < 7]; lt = d[d.cover_d < d.lead]
    po = Q("SELECT status, COUNT(*) n FROM purchase_orders" + w + " GROUP BY status", p); cat = d.groupby("category").agg(skus=("sku_name", "count"), below_reorder=("st", lambda s: int((d.loc[s.index, "st"] <= d.loc[s.index, "thr"]).sum())), avg_cover_days=("cover_d", "mean"), avg_risk=("p", "mean")).reset_index().sort_values("avg_risk", ascending=False)
    worst = d.sort_values("cover_d").head(8)[["outlet_id", "sku_name", "st", "cover_d", "lead", "p"]].rename(columns={"st": "stock", "cover_d": "days_of_cover", "lead": "lead_days", "p": "stockout_risk"})
    return {"answer": f"Inventory: **{len(d):,}** SKU records; **{len(below):,}** at or below reorder level, **{len(wk1):,}** have under 7 days of cover, **{len(lt):,}** will run out before a new order can arrive (cover shorter than lead time). Average stock-out risk {d.p.mean():.0%}." + (" Open purchase orders: " + ", ".join(f"{r.status} {r.n}" for r in po.itertuples()) + "." if len(po) else ""), "df": worst, "sql": "inventory: stock vs reorder threshold, cover = stock ÷ daily demand", "extra": "**By category**\n" + QA.md(cat.round(2))}
def sales_trend(scope):
    w = " AND outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q(f"SELECT substr(ts,1,10) day, SUM(amount) amount, SUM(tx_count) tx FROM sales WHERE ts IS NOT NULL{w} GROUP BY day ORDER BY day DESC LIMIT 14", p)
    if len(d) < 2: return None
    d = d.iloc[::-1].reset_index(drop=True); last7, prev7 = d.tail(7).amount.sum(), d.iloc[:-7].tail(7).amount.sum() if len(d) > 7 else np.nan
    ch = f" ({(last7 / prev7 - 1):+.1%} vs the 7 days before)" if prev7 == prev7 and prev7 else ""
    return {"answer": f"Sales over the last 7 recorded days: **{last7:,.0f}**{ch}. Best day {d.loc[d.amount.idxmax(), 'day']} ({d.amount.max():,.0f}), weakest {d.loc[d.amount.idxmin(), 'day']} ({d.amount.min():,.0f}).", "df": d.tail(10), "sql": "sales grouped by day"}

def _place(ql):
    """Find a state or city named in the question. Returns ("state"|"city", name) or None."""
    flat = re.sub(r"[^a-z]", "", ql); cities = Q("SELECT DISTINCT location FROM outlets").location.dropna().tolist()
    for st in sorted(set(CITY_STATE.values()), key=len, reverse=True):
        if re.sub(r"[^a-z]", "", st.lower()) in flat: return "state", st
    for ct in sorted(cities, key=len, reverse=True):
        if re.search(r"\b" + re.escape(ct.lower()) + r"\b", ql): return "city", ct
    return None
def outlets_in_place(ql, scope=None):
    pl = _place(ql)
    if not pl: return None
    kind, name = pl; o = Q("SELECT outlet_id, outlet_name name, location, tier, revenue, customer_satisfaction csat, staff_headcount staff FROM outlets")
    o["state"] = o.location.map(CITY_STATE).fillna("Other"); d = o[o.state == name] if kind == "state" else o[o.location == name]
    if not len(d): return {"answer": f"No outlets found in {name}.", "df": None, "sql": "outlets filtered by place"}
    by = d.groupby("location").size().sort_values(ascending=False); txt = ", ".join(f"{k} {v}" for k, v in by.items())
    return {"answer": f"**{len(d)}** outlets in {name}" + (f": {txt}." if kind == "state" else "."), "df": d.drop(columns=["state"]).sort_values(["location", "outlet_id"]), "sql": f"outlets where {'state (from city)' if kind == 'state' else 'location'} = {name}"}
def staff_by_place(ql, scope=None):
    s = Q("SELECT s.staff_id, s.outlet_id, o.location FROM staff s LEFT JOIN outlets o USING(outlet_id)")
    if not len(s): return None
    s["state"] = s.location.map(CITY_STATE).fillna("Other"); pl = _place(ql)
    if pl:
        kind, name = pl; d = s[s.state == name] if kind == "state" else s[s.location == name]; g = d.groupby("location").size().sort_values(ascending=False).reset_index(name="employees")
        return {"answer": f"**{len(d):,}** employees in {name}" + (f"; most in **{g.location.iloc[0]}** ({int(g.employees.iloc[0]):,})." if len(g) else "."), "df": g, "sql": f"staff joined to outlets, place = {name}"}
    by_state = bool(re.search(r"\bstates?\b", ql)); by_out = bool(re.search(r"\boutlets?\b", ql)); col = "outlet_id" if by_out else ("state" if by_state else "location")
    g = s.groupby(col).size().sort_values(ascending=False).head(15).reset_index(name="employees"); low = bool(re.search(r"\b(fewest|least|lowest|less|low number)\b", ql)); r = (s.groupby(col).size().sort_values().reset_index(name="employees").iloc[0] if low else g.iloc[0])
    return {"answer": f"**{r[col]}** has the {'fewest' if low else 'most'} employees: **{int(r.employees):,}** (of {len(s):,} in total, by {'outlet' if by_out else 'state' if by_state else 'city'}).", "df": g, "sql": f"staff joined to outlets, grouped by {col}"}
def by_state(q):
    o = Q("SELECT outlet_id, location, revenue FROM outlets"); o["state"] = o.location.map(CITY_STATE).fillna("Other"); g = o.groupby("state").agg(outlets=("outlet_id", "count"), revenue=("revenue", "sum")).reset_index().sort_values("outlets", ascending=False)
    low = bool(re.search(r"\b(fewest|least|lowest)\b", q)); r = g.iloc[-1] if low else g.iloc[0]
    return {"answer": f"**{r.state}** has the {'fewest' if low else 'most'} outlets: **{int(r.outlets)}**. Your outlets table stores the city, so states are derived from the city name.", "df": g, "sql": "outlets grouped by state (city → state mapping)"}
def staff_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    s = Q("SELECT outlet_id, role, overtime_hrs ot, predicted_attrition_prob pr FROM staff" + w, p); at = Q("SELECT status, COUNT(*) n FROM attendance" + (" WHERE outlet_id=?" if scope else "") + " GROUP BY status", p)
    if not len(s): return None
    hi = s[s.pr > .6]; g = s.groupby("outlet_id").pr.mean().sort_values(ascending=False).head(8).reset_index().rename(columns={"pr": "avg_attrition_risk"})
    return {"answer": f"Workforce: **{len(s):,}** staff, **{len(hi):,}** at high attrition risk (>60%), average overtime {s.ot.mean():.1f} h" + (". Attendance: " + ", ".join(f"{r.status} {r.n:,}" for r in at.itertuples()) if len(at) else "") + ".", "df": g, "sql": "staff + attendance", }
# ------------------------------------------------------------------------------------------------ Freight
STOP = {"long", "port", "new", "fort", "saint", "south", "north", "east", "west", "city", "bay", "cape", "grand", "great", "royal", "santa", "puerto", "king", "bandar"}
def _ports_in(q):
    p = Q("SELECT port_name FROM ports").port_name.tolist(); spans = []; first = {}
    for name in p:
        core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and len(core[0]) >= 4 and core[0] not in STOP: first.setdefault(core[0], []).append(name)
    for name in p:
        als = set(QA._alias(name)); core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and core[0] not in STOP and len(first.get(core[0], [])) == 1 and len(core[0]) >= 4: als.add(core[0])
        for al in sorted(als, key=len, reverse=True):
            m = re.search(r"(?<![a-z0-9])" + re.escape(al) + r"(?![a-z0-9])", q)
            if m: spans.append((m.start(), m.end(), len(al), name)); break
    spans.sort(key=lambda x: (-x[2], x[0])); kept = []
    for s_ in spans:
        if not any(s_[0] < k[1] and k[0] < s_[1] for k in kept): kept.append(s_)
    kept.sort(); return [k[3] for k in kept], kept
def plan_lane(q, dom):
    ps, hits = _ports_in(re.sub(r"how long", "how_long", q.lower()))
    if len(ps) < 2: return None
    pre = q.lower(); o, d = ps[0], ps[1]; m = re.search(r"from\s+(.+?)\s+to\s+", pre)
    cargo = next((k for k in dom.CARGO if k.lower() in pre or k.lower().rstrip("s") in pre), "General"); wm = re.search(r"(\d+(?:\.\d+)?)\s*(t|tons?|tonnes?|mt|kg|kgs)\b", pre); wt = 20.0
    if wm: wt = float(wm.group(1)) / (1000 if wm.group(2).startswith("kg") else 1)
    po, pd_ = dom.port_xy(o), dom.port_xy(d)
    if not po or not pd_: return None
    cust = Q("SELECT customer_id FROM customers LIMIT 1").customer_id[0]; req = {"origin": o, "dest": d, "cargo": cargo, "weight_t": wt, "hs_code": "HS-8501", "customer_id": cust, "docs_provided": None}
    pq = dom.preview_quote(req); opts = alts.route_alternatives(dom, (po[0], po[1]), (pd_[0], pd_[1]), tonnes=wt, interp=dom.interp, dwell=(po[3] + pd_[3]) * .5)
    if not pq and not opts: return None
    rows = [{"option": f"{i + 1}. {x['label']}", "route": x["name"], "nm": x["nm"], "days": round(x["days"], 1), "freight_cost": round(x["freight"]), "time_cost": round(x["delay_cost"]), "total": round(x["total"]), "tag": x["tag"]} for i, x in enumerate(opts)]
    head = f"**{o} → {d}**, {wt:g} t of {cargo}: "
    if pq: head += f"recommended route **{pq['route']}**, about **{pq['days']} days** at sea and in port, quoted price **${pq['price']:,.0f}** (margin {pq['margin_pct']}%), weather risk {pq['weather']}, decision: {pq['decision']}."
    elif opts: head += f"best route **{opts[0]['name']}**, about **{opts[0]['days']} days**."
    ex = (pq["narrative"] if pq and pq.get("narrative") else "") + ("\n\nOnly one sea route exists for this lane in the routing model (it does not pass a closable canal or strait), so the alternatives are commercial: another carrier, or diverting to a nearby less-congested port (open the port in the map to see candidates)." if len(opts) == 1 else "")
    return {"answer": head, "df": pd.DataFrame(rows) if rows else None, "sql": "dom.preview_quote (route, weather, ML price, customs, margin) + sea-route alternatives; nothing was saved", "extra": ex, "map": opts}
def choke_ports(q):
    k = next((k for k in CHOKE if _w(q, k)), None)
    if not k: return None
    name, la, lo = CHOKE[k]; p = Q("SELECT port_name, country, congestion_index ci, avg_dwell_days dw, lat, lon FROM ports"); p["km"] = alts.hav(la, lo, p.lat.values, p.lon.values); near = p[p.km <= 1800].sort_values("ci", ascending=False)
    v = Q("SELECT COUNT(*) n FROM vessels WHERE route_name LIKE ?", (f"%{name.split()[0]}%",)).n[0]; sh = Q("SELECT COUNT(*) n FROM shipments s JOIN vessels v USING(shipment_id) WHERE v.route_name LIKE ? AND s.status LIKE '%Delay%'", (f"%{name.split()[0]}%",)).n[0]
    if not len(near): return None
    return {"answer": f"Ports within 1,800 km of the **{name}**: **{len(near)}**; **{int((near.ci >= 4).sum())}** are congested (index ≥ 4). Most congested: **{near.port_name.iloc[0]}** ({near.ci.iloc[0]:.1f}, dwell {near.dw.iloc[0]:.1f} days). {int(v)} tracked vessels are routed via this passage, {int(sh)} of them delayed.", "df": near[["port_name", "country", "ci", "dw", "km"]].rename(columns={"ci": "congestion", "dw": "dwell_days", "km": "km_from_passage"}), "sql": f"ports within 1,800 km of {name}"}
def freight_summary(q):
    if re.search(r"\bcustoms\b", q) and re.search(r"\b(hold|held|status|summary|how many|why|docs?|documents?)\b", q):
        d = Q("SELECT dest_port, COUNT(*) n FROM shipments WHERE status='Customs Hold' GROUP BY dest_port ORDER BY n DESC LIMIT 8"); t = int(Q("SELECT COUNT(*) n FROM shipments WHERE status='Customs Hold'").n[0]); tk = Q("SELECT detail FROM agent_actions WHERE type='customs_docs' AND status IN ('open','proposed') LIMIT 400")
        miss = pd.Series([x for dd in tk.detail for x in re.findall(r"missing ([^:]+?)(?:\.|$)", dd)[0].split(", ")] if len(tk) and all("missing" in x for x in tk.detail) else []).value_counts().head(6)
        return {"answer": f"**{t}** shipments are on customs hold. Most held at **{d.dest_port.iloc[0] if len(d) else '-'}**." + (" Most often missing: " + ", ".join(f"{k} ({v})" for k, v in miss.items()) + "." if len(miss) else ""), "df": d.rename(columns={"n": "held"}), "sql": "shipments with status Customs Hold; customs_docs tickets"}
    return None
def greeting(q, kind):
    ql = re.sub(r"[^a-z0-9 ?']", " ", q.lower()).strip()
    hi = re.match(r"^(hi+|hello+|hey+|hola|namaste|good (morning|afternoon|evening)|greetings|yo|sup|howdy)\b( there| team| copilot| buddy)?$", ql)
    thanks = re.match(r"^(thanks?|thank you|thx|ok(ay)?|cool|great|nice|got it|awesome)( a lot| so much)?$", ql)
    who = re.search(r"\b(who are you|what are you|what can you do|what do you do|help me|how can you help|your capabilities|what is this)\b", ql) or ql in ("help", "?")
    how = re.match(r"^how are you", ql)
    if not (hi or thanks or who or how): return None
    ex = (["Which stock needs immediate refill?", "Which outlets are at highest risk and why?", "Top 5 outlets by revenue", "Why is OUT-007 flagged?", "Which state has the most outlets?"] if kind == "Franchise" else
          ["Mumbai to Singapore 50 t electronics: days, cost and best route", "Which ports are most congested?", "How many shipments are on customs hold?", "Which quotes have the lowest margin?"])
    what = ("outlets, stock, staff, marketing, audits, customer feedback and anomalies" if kind == "Franchise" else "routes, ports, weather, customs, quotes and carriers")
    if thanks: return {"answer": "You're welcome. Ask me anything else about " + what + ".", "df": None, "sql": ""}
    head = "Hello! " if hi else ("I'm doing well, thank you. " if how else "")
    return {"answer": head + f"I'm your operations copilot. I answer from your live data on {what}, and I can explain *why* something is flagged, suggest actions and send them to the approval queue. I understand 37 languages.\n\nTry one of these:\n" + "\n".join(f"- {e}" for e in ex), "df": None, "sql": ""}

def refill(scope, n=10):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name sku, category, current_stock stock, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory" + w, p)
    if not len(d): return None
    d["daily"] = d.wk / 7.0; d["cover"] = d.stock / d.daily.replace(0, np.nan); d["cover"] = d.cover.fillna(999)
    d["order_qty"] = np.maximum(d.thr * 3, np.ceil(d.daily * (d.lead + 7)) - d.stock).astype(int)
    d["gap"] = d.cover - d.lead
    d["urgency"] = np.where(d.cover < d.lead, "Critical: runs out before delivery", np.where(d.stock <= d.thr, "High: below reorder level", np.where(d.cover < 7, "Medium: under 7 days", "OK")))
    d["rank"] = d.urgency.str[:4].map({"Crit": 0, "High": 1, "Medi": 2}).fillna(3); need = d[d.urgency != "OK"].sort_values(["rank", "gap", "cover"])
    if not len(need): return {"answer": "No SKU needs an urgent refill right now: all have stock above the reorder level and cover longer than the supplier lead time.", "df": None, "sql": "inventory: cover = stock / daily demand vs lead time"}
    crit = need[need.urgency.str.startswith("Critical")]; po = Q("SELECT outlet_id, sku_name FROM purchase_orders WHERE status IN ('open','ordered','pending','Pending','Ordered','Open')" + (" AND outlet_id=?" if scope else ""), p) if "purchase_orders" in [r[0] for r in Q("SELECT name FROM sqlite_master WHERE type='table'").values] else None
    top = need.head(n)[["outlet_id", "sku", "stock", "thr", "cover", "lead", "order_qty", "urgency"]].rename(columns={"thr": "reorder_level", "cover": "days_of_cover", "lead": "lead_days"}); top["days_of_cover"] = top.days_of_cover.round(1)
    pool = crit if len(crit) else need[need.urgency.str.startswith("High")] if (need.urgency.str.startswith("High")).any() else need; by = pool.groupby("outlet_id").size().sort_values(ascending=False).head(5)
    outs = ", ".join(f"{k} ({v} SKUs)" for k, v in by.items()) if len(by) else "none"
    return {"answer": f"**{len(crit):,}** SKU records will run out *before a new order can arrive* (days of cover < supplier lead time) and **{len(need) - len(crit):,}** more are low. Most urgent outlets: {outs}. The table lists the {len(top)} most urgent SKUs, with the quantity to order.",
            "df": top, "sql": "inventory: days_of_cover = stock / (weekly_demand/7); urgent when cover < lead_time_days; order_qty = max(3 x reorder level, daily demand x (lead + 7 days) - stock)",
            "extra": "**How to read it:** *days_of_cover* is how long current stock lasts at current demand; *lead_days* is how long the supplier takes. When cover is shorter than lead time, stock-out is certain unless you order now (or transfer stock from a nearby outlet). The matching *reorder* tickets are raised by the agent."}

# ------------------------------------------------------------------------------------------------ router
def answer(kind, q, scope=None, dec=None, dom=None):
    r = _answer(kind, q, scope, dec, dom)
    if r: r["complete"] = True
    return r
def _answer(kind, q, scope=None, dec=None, dom=None):
    g = greeting(q, kind)
    if g: return g
    ql = q.lower().strip(); ids = re.findall(r"\b(OUT-\d+|STF-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+)\b", q, re.I)
    try:
        if kind == "Franchise":
            if ids and ids[0].upper().startswith("OUT") and re.search(r"\b(why|reason|flag\w*|wrong|problem|issue|explain|risk)\b", ql):
                if scope and ids[0].upper() != scope: return {"answer": f"{ids[0].upper()} is outside your outlet scope ({scope}).", "df": None, "sql": ""}
                return explain_outlet(ids[0].upper(), dec)
            _pl = (bool(re.search(r"\b(where|which|list|all|show|located|locations?|states?|cities|city|how many|number)\b", ql)) or bool(_place(ql))) and not re.search(r"\b(risk\w*|health\w*|revenue|sales|csat|satisfaction|inventory|stock|attrition|salary|overtime|best|worst|weakest|critical)\b", ql)
            if _pl and re.search(r"\b(employees?|staff|workers?|people|headcount|manpower|workforce)\b", ql) and (_place(ql) or re.search(r"\b(where|which|most|highest|more|largest|big|fewest|least|lowest|number)\b", ql)): return staff_by_place(ql, scope)
            if _pl and re.search(r"\b(outlets?|stores?|branch\w*|franchises?)\b", ql) and _place(ql): return outlets_in_place(ql, scope)
            if re.search(r"\b(state|states)\b", ql) and re.search(r"\b(outlets?|stores?|branch\w*)\b", ql): return by_state(ql)
            if re.search(r"\b(outlets?|stores?|branch\w*)\b", ql) and re.search(r"\b(risk\w*|health\w*|best|worst|weakest|strongest|critical|attention|problem\w*|performing|performance)\b", ql) and not re.search(r"\b(revenue|sales|csat|satisfaction|headcount)\b", ql): return risk_rank(ql, dec, scope)
            if re.search(r"\b(inventory|stock|stocks|stocked|sku|skus|items?|products?|ingredients?)\b", ql) and re.search(r"\b(refill\w*|restock\w*|re-?order\w*|replenish\w*|immediate\w*|urgent\w*|attention|run(ning)? (out|low)|shortage\w*|low|critical|need\w*|order now)\b", ql) and not re.search(r"\b(how many|count|average|total)\b", ql): return refill(scope)
            if re.search(r"\b(inventory|stock)\b", ql) and re.search(r"\b(how|status|holding|summary|overview|situation|look|doing|week|today|health)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return inventory_status(scope)
            if re.search(r"\b(sales|revenue|takings)\b", ql) and re.search(r"\b(trend|this week|last week|today|yesterday|last 7|past week|recent|how (is|are|did))\b", ql): return sales_trend(scope)
            if re.search(r"\b(staff|workforce|employees?|team)\b", ql) and re.search(r"\b(how|status|summary|overview|situation|doing|look)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return staff_status(scope)
        else:
            if dom is not None and re.search(r"\b(days?|time|transit|cost|price|quote|route|how long|how much|eta)\b", ql):
                r = plan_lane(q, dom)
                if r: return r
            if re.search(r"\b(congest\w*|delay\w*|busy|ports?)\b", ql) and any(_w(ql, k) for k in CHOKE): return choke_ports(ql)
            r = freight_summary(ql)
            if r: return r
    except Exception as e:
        import sys; print("analyst failed:", repr(e), file=sys.stderr)
    return None


# ================================================================================================================================
# COPILOT ENHANCEMENT: multi-query analyst handlers (compare outlets, chain / city / outlet overview) with metric-level role filtering.
# Every statement goes through QA.run_sql (single read-only SELECT, whitelisted tables, bound values).
# ================================================================================================================================
def _ids_in(q):
    """All outlet ids mentioned in a question: OUT-007, out 7, outlet 12 ... in order of appearance."""
    out = []
    for m in re.finditer(r"\bout-?\s?(\d{1,3})\b|\boutlet\s*(?:no\.?|number|#)?\s*(\d{1,3})\b", q, re.I):
        n = int(m.group(1) or m.group(2)); i = f"OUT-{n:03d}"
        if i not in out: out.append(i)
    return out

def _run(sqls, sql, binds=()):
    df = QA.run_sql(sql, binds); sqls.append(QA.render_sql(sql, binds)); return df

def _metric_ok(role, tables=(), tags=()):
    pol = QA.policy_for(role)
    if pol["tables"] != QA.ALL and any(t not in pol["tables"] for t in tables): return False
    return not (set(tags) & pol["deny"])

def compare_outlets(q, role="Admin", scope=None):
    """'Compare OUT-007 and OUT-012' -> side-by-side metrics. Needs two outlet ids and a comparison word."""
    ids = _ids_in(q)
    if len(ids) < 2 or not re.search(r"\b(compare|comparison|versus|vs\.?|against|difference|differ|better|between)\b", q.lower()): return None
    ids = ids[:2]
    if scope and any(i != scope for i in ids): return {"answer": f"You can only see {scope}, so a comparison with another outlet is not available for your role.", "df": None, "sql": "", "rows": 0, "status": "denied"}
    ph = ",".join("?" * len(ids)); sqls = []; rows = {}; hidden = False
    base = _run(sqls, f"SELECT o.outlet_id, o.outlet_name, o.location, o.tier, o.revenue, o.operating_costs, o.customer_satisfaction, o.staff_headcount FROM outlets o WHERE o.outlet_id IN ({ph}) LIMIT 5", ids).set_index("outlet_id")
    miss = [i for i in ids if i not in base.index]
    if miss: return {"answer": f"I could not find {', '.join(miss)} in the outlets table.", "df": None, "sql": sqls[0], "rows": 0, "status": "empty"}
    rows["City"] = {i: base.location[i] for i in ids}; rows["Tier"] = {i: base.tier[i] for i in ids}
    if _metric_ok(role, ["outlets"], ["revenue", "operating_costs", "net_margin"]):
        rows["Revenue"] = {i: f"{base.revenue[i]:,.0f}" for i in ids}; rows["Operating costs"] = {i: f"{base.operating_costs[i]:,.0f}" for i in ids}
        rows["Net margin"] = {i: f"{(base.revenue[i] - base.operating_costs[i]) * 100 / base.revenue[i]:.1f}%" for i in ids}
    else: hidden = True
    rows["Customer satisfaction"] = {i: f"{base.customer_satisfaction[i]:.2f}" for i in ids}
    if _metric_ok(role, ["outlets"], ["headcount"]): rows["Headcount"] = {i: int(base.staff_headcount[i]) for i in ids}
    else: hidden = True
    if _metric_ok(role, ["staff"], ["attrition"]):
        s = _run(sqls, f"SELECT s.outlet_id, ROUND(AVG(s.predicted_attrition_prob),3) AS attr, ROUND(AVG(s.overtime_hrs),1) AS ot FROM staff s WHERE s.outlet_id IN ({ph}) GROUP BY s.outlet_id", ids).set_index("outlet_id")
        rows["Avg attrition risk"] = {i: (f"{s.attr[i]:.0%}" if i in s.index else "-") for i in ids}; rows["Avg overtime (h)"] = {i: (s.ot[i] if i in s.index else "-") for i in ids}
    else: hidden = True
    if _metric_ok(role, ["inventory"]):
        v = _run(sqls, f"SELECT i.outlet_id, SUM(CASE WHEN i.current_stock <= i.reorder_threshold THEN 1 ELSE 0 END) AS below FROM inventory i WHERE i.outlet_id IN ({ph}) GROUP BY i.outlet_id", ids).set_index("outlet_id")
        rows["SKUs below reorder"] = {i: (int(v.below[i]) if i in v.index else 0) for i in ids}
    else: hidden = True
    if _metric_ok(role, ["audits"]):
        a = _run(sqls, f"SELECT a.outlet_id, ROUND(AVG(a.score),1) AS sc, SUM(a.violations) AS vi FROM audits a WHERE a.outlet_id IN ({ph}) GROUP BY a.outlet_id", ids).set_index("outlet_id")
        rows["Avg audit score"] = {i: (a.sc[i] if i in a.index else "-") for i in ids}; rows["Audit violations"] = {i: (int(a.vi[i]) if i in a.index else 0) for i in ids}
    else: hidden = True
    if _metric_ok(role, ["agent_actions"]):
        try:
            t = _run(sqls, f"SELECT t.target AS outlet_id, COUNT(*) AS n FROM agent_actions t WHERE t.status IN ('open','proposed') AND t.target IN ({ph}) GROUP BY t.target", ids).set_index("outlet_id")
            rows["Open tickets"] = {i: (int(t.n[i]) if i in t.index else 0) for i in ids}
        except Exception: pass
    else: hidden = True
    df = pd.DataFrame(rows).T.reset_index().rename(columns={"index": "metric"}); a, b = ids
    bits = [f"**{a}** ({base.location[a]}) vs **{b}** ({base.location[b]})."]
    if "Revenue" in rows: bits.append(f"Revenue: {a if base.revenue[a] >= base.revenue[b] else b} is ahead.")
    bits.append(f"Customer satisfaction: {a if base.customer_satisfaction[a] >= base.customer_satisfaction[b] else b} is ahead.")
    return {"answer": " ".join(bits) + (" Some metrics are hidden for your role." if hidden else ""), "df": df, "sql": "\n\n".join(sqls), "rows": int(len(df)), "status": "ok", "entity": a}

def overview(kind, value, role="Admin", scope=None):
    """Headline numbers for the chain, a city or one outlet. Each block is skipped when the role may not see it."""
    cond, binds = {"chain": ("", []), "city": ("o.location = ?", [value]), "outlet": ("o.outlet_id = ?", [value])}[kind]
    if scope: cond = (cond + " AND " if cond else "") + "o.outlet_id = ?"; binds = binds + [scope]
    W = (" WHERE " + cond) if cond else ""; A = (" AND " + cond) if cond else ""; sqls = []; rows = []; hidden = False
    def J(a, col="outlet_id"): return f" JOIN outlets o ON o.outlet_id = {a}.{col}"
    o = _run(sqls, f"SELECT COUNT(*) AS outlets, ROUND(SUM(o.revenue),0) AS revenue, ROUND(SUM(o.operating_costs),0) AS costs, ROUND(AVG(o.customer_satisfaction),2) AS csat, SUM(o.staff_headcount) AS headcount FROM outlets o{W}", binds).iloc[0]
    if not o.outlets: return {"answer": f"No outlets found for {value}.", "df": None, "sql": sqls[0], "rows": 0, "status": "empty"}
    rows.append(("Outlets", int(o.outlets)))
    if _metric_ok(role, ["outlets"], ["revenue", "operating_costs", "net_margin"]):
        rows += [("Total revenue", f"{o.revenue:,.0f}"), ("Operating costs", f"{o.costs:,.0f}"), ("Net margin", f"{(o.revenue - o.costs) * 100 / o.revenue:.1f}%")]
    else: hidden = True
    rows.append(("Average customer satisfaction", f"{o.csat:.2f}"))
    if _metric_ok(role, ["staff"], ["attrition"]):
        s = _run(sqls, f"SELECT COUNT(*) AS staff, ROUND(AVG(s.predicted_attrition_prob),3) AS attr, ROUND(AVG(s.overtime_hrs),1) AS ot FROM staff s{J('s')}{W}", binds).iloc[0]
        rows += [("Staff", int(s.staff)), ("Average attrition risk", f"{s.attr:.0%}" if s.attr == s.attr else "-"), ("Average overtime (h)", s.ot)]
    else: hidden = True
    if _metric_ok(role, ["inventory"]):
        v = _run(sqls, f"SELECT COUNT(*) AS skus, SUM(CASE WHEN i.current_stock <= i.reorder_threshold THEN 1 ELSE 0 END) AS below FROM inventory i{J('i')}{W}", binds).iloc[0]
        rows.append(("SKU records below reorder level", f"{int(v.below or 0)} of {int(v.skus)}"))
    else: hidden = True
    if _metric_ok(role, ["audits"]):
        a = _run(sqls, f"SELECT COUNT(*) AS n, ROUND(AVG(a.score),1) AS sc, SUM(CASE WHEN a.status = 'Action Required' THEN 1 ELSE 0 END) AS act FROM audits a{J('a')}{W}", binds).iloc[0]
        rows += [("Average audit score", a.sc), ("Audits needing action", int(a.act or 0))]
    else: hidden = True
    if _metric_ok(role, ["marketing"], ["budget"]):
        m = _run(sqls, f"SELECT COUNT(*) AS n, ROUND(AVG(m.actual_roi),2) AS roi FROM marketing m{J('m')}{W}", binds).iloc[0]
        rows.append(("Average campaign ROI", f"{m.roi}x" if m.roi == m.roi else "-"))
    else: hidden = True
    if _metric_ok(role, ["agent_actions"]):
        try:
            t = _run(sqls, f"SELECT COUNT(*) AS n, SUM(CASE WHEN t.severity = 'Critical' THEN 1 ELSE 0 END) AS crit FROM agent_actions t{J('t', 'target')} WHERE t.status IN ('open','proposed'){A}", binds).iloc[0]
            rows.append(("Open tickets (critical)", f"{int(t.n)} ({int(t.crit or 0)})"))
        except Exception: pass
    else: hidden = True
    df = pd.DataFrame(rows, columns=["metric", "value"]); label = {"chain": "the whole chain", "city": value, "outlet": value}[kind]
    return {"answer": f"Snapshot for **{label}**: {int(o.outlets)} outlet{'s' if o.outlets != 1 else ''}, customer satisfaction {o.csat:.2f}." + (" Some figures are hidden for your role." if hidden else ""), "df": df, "sql": "\n\n".join(sqls), "rows": int(len(df)), "status": "ok", "entity": value if kind == "outlet" else None}

_DOMAIN_WORDS = re.compile(r"\b(staff|inventory|stock|marketing|campaigns?|audits?|feedback|sales|revenue|salary|attrition|tickets?|outlets? (with|by|that|which)|margin|profit|ratings?|orders?)\b")
def smart_answer(kind, q, role="Admin", scope=None, dec=None, dom=None, qn=None):
    """New multi-query handlers first, then the original router. `q` keeps the user's ids; `qn` is the normalised text."""
    if kind != "Franchise": return answer(kind, q, scope, dec, dom)
    ql = (qn or q).lower()
    try:
        r = compare_outlets(q, role, scope)
        if r: return r
        ids = _ids_in(q); p = QA.extract_params(qn or q)
        asks_overview = bool(re.search(r"\b(overview|summary|snapshot|kpis?|dashboard|big picture|status of|tell me about|about|profile|how is|how are|how's|doing|performance of|performing)\b", ql))
        if ids and asks_overview and not re.search(r"\b(why|flag\w*|reason|wrong|issue|problem)\b", ql): return overview("outlet", ids[0], role, scope)
        if p.get("city") and asks_overview and not _DOMAIN_WORDS.search(ql): return overview("city", p["city"], role, scope)
        if not ids and not p.get("city") and re.search(r"\b(overview|snapshot|kpis?|dashboard|big picture|business summary|overall summary|summary of (the )?(business|chain)|how is (the )?(business|chain)|how are we doing)\b", ql) and not _DOMAIN_WORDS.search(ql):
            return overview("chain", None, role, scope)
    except QA.SQLRejected as ex:
        return {"answer": f"That query was blocked by the SQL safety check ({ex}).", "df": None, "sql": "", "rows": 0, "status": "error"}
    except Exception as ex:
        import sys; print("analyst enhancement failed:", repr(ex), file=sys.stderr)
    return answer(kind, q, scope, dec, dom)


In [ ]:
%%writefile /content/app/blocks.py
"""Server-driven UI blocks. Pages are Python functions returning JSON blocks; the single-page front end renders them and live-updates only what changed.
Charts are ECharts option dicts built here with one consistent palette."""
import math, json, datetime
import numpy as np, pandas as pd
import live_engine as LE

TEAL, BLUE, AMBER, RED, VIOLET, GREEN, PINK, GREY = "#12a594", "#3e63dd", "#f5a524", "#e5484d", "#8e4ec6", "#30a46c", "#d6409f", "#8b8d98"
PRI = "#5b5bd6"
PAL = [PRI, TEAL, AMBER, VIOLET, RED, GREEN, PINK, BLUE, GREY]
TONE = {"Critical": RED, "High": RED, "Warning": AMBER, "Medium": AMBER, "Info": BLUE, "Low": TEAL, "Pass": TEAL, "Fail": RED}

def Q(sql, p=()):
    with LE._DB_LOCK:
        return pd.read_sql(sql, LE.conn(), params=p)

def clean(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating, float)): return None if (v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))) or (isinstance(v, np.floating) and (np.isnan(v) or np.isinf(v)))) else round(float(v), 4)
    if isinstance(v, (np.bool_,)): return bool(v)
    if isinstance(v, (pd.Timestamp, datetime.datetime, datetime.date)): return str(v)
    if isinstance(v, (list, tuple)): return [clean(x) for x in v]
    if isinstance(v, dict): return {str(k): clean(x) for k, x in v.items()}
    if v is pd.NA or v is pd.NaT: return None
    return v
def L(x): return [clean(v) for v in (x.tolist() if hasattr(x, "tolist") else list(x))]

# ------------------------------------------------------------------ charts (ECharts options)
def _base(extra=None, legend=False, grid=None):
    o = {"animation": True, "animationDuration": 350, "color": PAL, "tooltip": {"trigger": "axis", "confine": True}, "grid": grid or {"left": 8, "right": 14, "top": 28 if legend else 14, "bottom": 6, "containLabel": True}}
    if legend: o["legend"] = {"top": 0, "type": "scroll"}
    if extra: o.update(extra)
    return o
def bar(cats, series, horizontal=False, stack=False, colors=None, fmt=None, max_=None, pct=False, itemcolors=None, sort=False):
    """series: {name: values} or a list of values."""
    series = {"": series} if not isinstance(series, dict) else series; cats = L(cats)
    if sort and len(series) == 1:
        v = L(list(series.values())[0]); order = sorted(range(len(v)), key=lambda i: (v[i] is None, v[i] or 0)); cats = [cats[i] for i in order]; series = {k: [L(x)[i] for i in order] for k, x in series.items()}
    ser = []
    for i, (n, v) in enumerate(series.items()):
        d = L(v); s = {"type": "bar", "name": n, "data": d, "barMaxWidth": 22, "itemStyle": {"borderRadius": [0, 4, 4, 0] if horizontal else [4, 4, 0, 0], "color": (colors or PAL)[i % len(colors or PAL)]}, "emphasis": {"focus": "series"}}
        if itemcolors and i == 0: s["data"] = [{"value": x, "itemStyle": {"color": c}} for x, c in zip(d, itemcolors)]
        if stack: s["stack"] = "s"
        ser.append(s)
    cat = {"type": "category", "data": cats, "axisLabel": {"hideOverlap": True}}; val = {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if max_: val["max"] = max_
    if pct: val["axisLabel"] = {"formatter": "{value}%"}
    o = _base(legend=len(series) > 1 and any(series)); o.update({"xAxis": val if horizontal else cat, "yAxis": cat if horizontal else val, "series": ser})
    return o
def line(x, series, area=True, band=None, colors=None, dashed=(), ymin=None, ymax=None, step=False, markers=False):
    series = {"": series} if not isinstance(series, dict) else series; ser = []
    for i, (n, v) in enumerate(series.items()):
        c = (colors or PAL)[i % len(colors or PAL)]; s = {"type": "line", "name": n, "data": L(v), "smooth": not step, "showSymbol": markers, "symbolSize": 5, "lineStyle": {"width": 2, "color": c, "type": "dashed" if n in dashed else "solid"}, "itemStyle": {"color": c}}
        if step: s["step"] = "end"
        if area and i == 0 and len(series) <= 2: s["areaStyle"] = {"color": {"type": "linear", "x": 0, "y": 0, "x2": 0, "y2": 1, "colorStops": [{"offset": 0, "color": c + "55"}, {"offset": 1, "color": c + "00"}]}}
        ser.append(s)
    if band:
        lo, hi = L(band[0]), L(band[1]); ser += [{"type": "line", "data": lo, "stack": "b", "lineStyle": {"opacity": 0}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}, {"type": "line", "data": [None if (a is None or b is None) else b - a for a, b in zip(lo, hi)], "stack": "b", "lineStyle": {"opacity": 0}, "areaStyle": {"color": "rgba(251,191,36,.16)"}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}]
    y = {"type": "value", "scale": True, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if ymin is not None: y["min"] = ymin
    if ymax is not None: y["max"] = ymax
    o = _base(legend=len([s for s in series if s]) > 1); o.update({"xAxis": {"type": "category", "data": L(x), "boundaryGap": False, "axisLabel": {"hideOverlap": True}}, "yAxis": y, "series": ser}); return o
def pie(items, donut=True, colors=None):
    """items: {name: value} or list of (name, value, color?)"""
    its = list(items.items()) if isinstance(items, dict) else items; data = []
    for i, it in enumerate(its):
        d = {"name": str(it[0]), "value": clean(it[1])}
        if (colors and str(it[0]) in colors) or len(it) > 2: d["itemStyle"] = {"color": (colors or {}).get(str(it[0])) or it[2]}
        data.append(d)
    return {"animation": True, "color": PAL, "tooltip": {"trigger": "item", "confine": True}, "legend": {"bottom": 0, "type": "scroll"}, "series": [{"type": "pie", "radius": ["52%", "78%"] if donut else "75%", "center": ["50%", "44%"], "avoidLabelOverlap": True, "label": {"show": False}, "itemStyle": {"borderColor": "#ffffff", "borderWidth": 2, "borderRadius": 4}, "data": data}]}
def scatter(groups, xname="", yname="", colors=None, sized=False):
    """groups: {name: [ {x,y,label,size?} ]}. Points are [x, y, label, size]; the client resolves the {"__fn": ...} formatters."""
    ser = []
    for i, (n, pts) in enumerate(groups.items()):
        ser.append({"type": "scatter", "name": n, "symbolSize": {"__fn": "size3"} if sized else 10, "data": [[clean(p["x"]), clean(p["y"]), p.get("label", ""), clean(p.get("size", 10))] for p in pts],
                    "itemStyle": {"color": (colors or {}).get(n, PAL[i % len(PAL)]), "opacity": .85}, "emphasis": {"focus": "series"}})
    o = _base(legend=len(groups) > 1); o["tooltip"] = {"trigger": "item", "confine": True, "formatter": {"__fn": "scatterTip"}}
    ax = lambda n: {"type": "value", "name": n, "scale": True, "nameTextStyle": {"color": "#6b7080"}, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    o.update({"xAxis": ax(xname), "yAxis": ax(yname), "series": ser}); return o
def hist(values, bins=24, color=TEAL, pct=False):
    v = np.asarray([x for x in values if x is not None and not (isinstance(x, float) and math.isnan(x))], dtype=float)
    if not len(v): return bar([], [])
    h, e = np.histogram(v, bins=bins); labels = [f"{(e[i] * (100 if pct else 1)):.{0 if pct else 1}f}" + ("%" if pct else "") for i in range(len(h))]
    return bar(labels, [int(x) for x in h], colors=[color])
def waterfall(labels, deltas, total_label="Total"):
    base, run, up = [], 0.0, []
    for d in deltas: base.append(run if d >= 0 else run + d); up.append(abs(d)); run += d
    cats = list(labels) + [total_label]; base.append(0); up.append(run)
    o = _base(); o.update({"xAxis": {"type": "category", "data": cats}, "yAxis": {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}},
                           "series": [{"type": "bar", "stack": "w", "data": [clean(x) for x in base], "itemStyle": {"color": "transparent"}, "tooltip": {"show": False}}, {"type": "bar", "stack": "w", "data": [{"value": clean(u), "itemStyle": {"color": TEAL if i == len(up) - 1 else (BLUE if deltas[i] >= 0 else RED)}} for i, u in enumerate(up)], "barMaxWidth": 34, "itemStyle": {"borderRadius": 4}}]}); return o
def radar(names, series, mx=100):
    o = {"animation": True, "color": PAL, "tooltip": {"confine": True}, "legend": {"bottom": 0}, "radar": {"indicator": [{"name": n, "max": mx} for n in names], "splitArea": {"show": False}, "axisName": {"color": "#6b7080"}}, "series": [{"type": "radar", "data": [{"name": k, "value": L(v), "areaStyle": {"opacity": .18}} for k, v in series.items()]}]}; return o
def gauge(value, name="", mx=100):
    c = RED if value < 60 else AMBER if value < 80 else TEAL
    return {"series": [{"type": "gauge", "startAngle": 210, "endAngle": -30, "min": 0, "max": mx, "progress": {"show": True, "width": 14, "itemStyle": {"color": c}}, "axisLine": {"lineStyle": {"width": 14, "color": [[1, "rgba(148,163,184,.15)"]]}}, "axisTick": {"show": False}, "splitLine": {"show": False}, "axisLabel": {"show": False}, "pointer": {"show": False}, "anchor": {"show": False},
                         "detail": {"valueAnimation": True, "fontSize": 30, "fontWeight": 700, "offsetCenter": [0, "8%"], "color": "#1c2024", "formatter": "{value}"}, "title": {"offsetCenter": [0, "62%"], "color": "#6b7080", "fontSize": 12}, "data": [{"value": clean(round(value, 1)), "name": name}]}]}
def graph(nodes, links, h=520):
    """nodes: [{id,name,kind,size,color,tip}] ; links: [(a,b)]"""
    cat = sorted({n.get("kind", "x") for n in nodes}); return {"animation": True, "tooltip": {"confine": True, "formatter": {"__fn": "nodeTip"}}, "legend": {"data": cat, "bottom": 0, "textStyle": {"color": "#6b7080"}}, "color": PAL,
        "series": [{"type": "graph", "layout": "force", "roam": True, "draggable": True, "force": {"repulsion": 260, "edgeLength": [40, 120], "gravity": .08}, "label": {"show": True, "position": "right", "color": "#454a58", "fontSize": 10, "formatter": {"__fn": "nodeLabel"}},
                    "categories": [{"name": c} for c in cat], "lineStyle": {"color": "#cfd3e3", "width": 1}, "emphasis": {"focus": "adjacency", "lineStyle": {"width": 2}},
                    "data": [{"id": n["id"], "name": n["id"], "category": cat.index(n.get("kind", "x")), "symbolSize": n.get("size", 12), "itemStyle": {"color": n.get("color")} if n.get("color") else {}, "tip": n.get("tip", ""), "showLabel": n.get("label", False), "open": n.get("open")} for n in nodes],
                    "links": [{"source": a, "target": b} for a, b in links]}]}

# ------------------------------------------------------------------ blocks
def kpis(*items):
    out = []
    for it in items:
        d = {"label": it[0], "value": clean(it[1])}
        if len(it) > 2 and it[2] is not None: d["sub"] = str(it[2])
        if len(it) > 3 and it[3]: d["tone"] = it[3]
        out.append(d)
    return {"t": "kpis", "items": out}
def chart(opt, title=None, h=290, open=None, sub=None, key=None): return {"t": "chart", "opt": opt, "title": title, "h": h, "open": open, "sub": sub, "key": key or title}
def table(df, cols=None, title=None, h=None, open=None, key=None, sub=None, search=False, page=None):
    """cols: list of names or dicts {k,label,type: text|num|int|pct|pill|bar|money|time|id, max}. open = {"kind":"outlet","col":"outlet_id"} makes rows clickable."""
    if isinstance(df, list): df = pd.DataFrame(df)
    cols = cols or list(df.columns); cs = [{"k": c, "label": c.replace("_", " ")} if isinstance(c, str) else c for c in cols]
    rows = [{c["k"]: clean(r.get(c["k"])) for c in cs} for r in df.to_dict("records")] if len(df) else []
    return {"t": "table", "cols": cs, "rows": rows, "title": title, "h": h, "open": open, "key": key or title, "sub": sub, "search": search}
def callout(text, tone="info", title=None): return {"t": "callout", "text": text, "tone": tone, "title": title}
def md(text): return {"t": "md", "text": text}
def row(*cols, ratio=None): return {"t": "row", "cols": list(cols), "ratio": ratio}
def col(*blocks): return {"t": "col", "blocks": list(blocks)}
def card(title, *blocks, sub=None): return {"t": "card", "title": title, "sub": sub, "blocks": list(blocks)}
def form(id, fields, submit="Submit", url=None, title=None, tone="primary", result=True, sub=None):
    """fields: {name,label,type: select|number|text|textarea|range|checks|multi|file|date, options, value, min,max,step}"""
    return {"t": "form", "id": id, "fields": fields, "submit": submit, "url": url, "title": title, "tone": tone, "sub": sub}
def button(id, label, url, tone="primary", confirm=None, payload=None): return {"t": "button", "id": id, "label": label, "url": url, "tone": tone, "confirm": confirm, "payload": payload or {}}
def map_(layers, h=470, view=None, open=None, key="map", legend=None, live=None): return {"t": "map", "layers": layers, "h": h, "view": view, "open": open, "key": key, "legend": legend, "live": live}
def tickets(items, key="tickets", title=None, sub=None): return {"t": "tickets", "items": items, "key": key, "title": title, "sub": sub}
def hero(agent, title, blurb, r, can_run=True):
    return {"t": "hero", "agent": agent, "title": title, "blurb": blurb, "can_run": can_run, "score": None if not r else clean(r["score"]), "risk": None if not r else r["risk"], "summary": None if not r else r["summary"], "ts": None if not r else r["ts"], "ms": None if not r else r["ms"]}
def tabs(items): return {"t": "tabs", "tabs": items}
def page(title, sub, blocks, tabs=None, live=8, actions=None): return {"title": title, "sub": sub, "blocks": blocks, "tabs": tabs, "live": live, "actions": actions or []}
def pill_col(k, label=None): return {"k": k, "label": label or k.replace("_", " "), "type": "pill"}
def num_col(k, label=None, fmt="num", mx=None): return {"k": k, "label": label or k.replace("_", " "), "type": fmt, "max": mx}


In [ ]:
%%writefile /content/app/config.py
"""Shared configuration. APP_KIND (Franchise | Freight) is set by the launcher. No Google Drive prompts: data lives in DATA_DIR (set PERSIST_DIR to keep it between sessions)."""
import os
KIND = os.environ.get("APP_KIND", "Franchise")
APP_DIR = os.path.dirname(os.path.abspath(__file__))
_default = "/content/app_data" if os.path.isdir("/content") else os.path.join(APP_DIR, "runtime_data")
DATA_DIR = os.environ.get("PERSIST_DIR") or os.environ.get("APP_DATA_DIR") or os.environ.get("FRANCHISEOPS_DATA_DIR") or os.environ.get("FREIGHTQUOTE_DATA_DIR") or _default
os.makedirs(DATA_DIR, exist_ok=True)
DB_PATH = os.path.join(DATA_DIR, "franchise_database.db" if KIND == "Franchise" else "freight_database.db")
def _secret(key, default=""):
    try:
        from google.colab import userdata
        v = userdata.get(key)
        if v: return str(v).strip()
    except Exception: pass
    return str(os.environ.get(key, default) or "").strip()
HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGINGFACE_TOKEN")
def _persisted_secret():
    """No JWT_SECRET_KEY given: generate a random one once and keep it beside the database, so sessions survive restarts and are never signed with a public default."""
    import secrets as _s
    p = os.path.join(DATA_DIR, ".jwt_secret")
    try:
        if os.path.exists(p): return open(p).read().strip()
        v = _s.token_urlsafe(48); fd = os.open(p, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600); os.write(fd, v.encode()); os.close(fd); return v
    except Exception: return _s.token_urlsafe(48)
JWT_SECRET_KEY = _secret("JWT_SECRET_KEY") or _persisted_secret()
ADMIN_EMAIL_ID = _secret("ADMIN_EMAIL_ID", "admin@infosys.com"); DEMO_MODE = os.environ.get("DEMO_MODE", "1") != "0"     # 1 = seeded demo accounts shown on the login page (Colab demo). 0 = production: only the admin exists, with ADMIN_PASSWORD (or a random one printed once in the server log)
def _admin_pw():
    v = _secret("ADMIN_PASSWORD")
    if v: return v
    if DEMO_MODE: return "Admin@123"
    import secrets as _s; p = os.path.join(DATA_DIR, ".admin_initial");
    if os.path.exists(p): return open(p).read().strip()
    v = _s.token_urlsafe(9) + "1aA"; open(p, "w").write(v); print("\n*** DEMO_MODE=0: initial admin password:", v, "(stored in", p, ") ***\n"); return v
ADMIN_PASSWORD = _admin_pw()
EMAIL_ID = _secret("EMAIL_ID"); EMAIL_PASSWORD = _secret("EMAIL_PASSWORD").replace(" ", "")
MODEL_NAME = os.environ.get("LLM_MODEL", "")
NLLB_MODEL = os.environ.get("NLLB_MODEL", "facebook/nllb-200-distilled-600M")


In [ ]:
%%writefile /content/app/db.py
import sqlite3, os
import pandas as pd
from config import DB_PATH

def get_conn():
    """Establishes and returns a connection to the SQLite database with WAL enabled."""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH, timeout=30)
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    return conn

def save_chat_message(username, role, message):
    """Saves a user or assistant chat message to the history ledger."""
    try:
        with get_conn() as conn:
            conn.execute("INSERT INTO chat_history (username, role, message) VALUES (?, ?, ?);", (username, role, message))
            conn.commit()
    except Exception:
        pass

def load_chat_history(username=None, limit=100):
    """Loads chat history for a specific user or the global system up to the specified limit."""
    try:
        with get_conn() as conn:
            if username:
                df = pd.read_sql("SELECT role, message FROM chat_history WHERE username=? ORDER BY id ASC LIMIT ?;", conn, params=(username, limit))
                if df.empty:
                    df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))
            else:
                df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))

            res = []
            for _, r in df.iterrows():
                content_val = str(r.get("message") or r.get("content") or "")
                res.append({
                    "role": str(r.get("role", "assistant")),
                    "content": content_val,
                    "message": content_val
                })
            return res
    except Exception:
        return []

def clear_chat_history(username=None):
    """Clears stored chat history entries."""
    try:
        with get_conn() as conn:
            if username:
                conn.execute("DELETE FROM chat_history WHERE username=?;", (username,))
            else:
                conn.execute("DELETE FROM chat_history;")
            conn.commit()
    except Exception:
        pass

def init_db():
    """Initializes all database tables and executes migrations for existing database schemas."""
    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT,
            role TEXT,
            message TEXT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            role TEXT DEFAULT 'Staff',
            outlet_id TEXT,
            security_question TEXT,
            security_answer TEXT,
            profile_picture TEXT,
            failed_attempts INTEGER DEFAULT 0,
            lock_until DATETIME DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            last_login DATETIME DEFAULT NULL,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)

        # Safe migration for DBs created before these columns existed
        existing_cols = {r[1] for r in conn.execute("PRAGMA table_info(users);").fetchall()}
        migrations = {
            "username": "ALTER TABLE users ADD COLUMN username TEXT;",
            "security_question": "ALTER TABLE users ADD COLUMN security_question TEXT;",
            "security_answer": "ALTER TABLE users ADD COLUMN security_answer TEXT;",
            "profile_picture": "ALTER TABLE users ADD COLUMN profile_picture TEXT;",
            "failed_attempts": "ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0;",
            "lock_until": "ALTER TABLE users ADD COLUMN lock_until DATETIME DEFAULT NULL;",
            "account_status": "ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active';",
            "last_login": "ALTER TABLE users ADD COLUMN last_login DATETIME DEFAULT NULL;",
            "outlet_id": "ALTER TABLE users ADD COLUMN outlet_id TEXT;",
        }
        for col, ddl in migrations.items():
            if col not in existing_cols:
                try:
                    conn.execute(ddl)
                except Exception:
                    pass


        conn.execute('''
        CREATE TABLE IF NOT EXISTS user_outlet_assignments (
            username TEXT PRIMARY KEY,
            outlet_id TEXT NOT NULL,
            assigned_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        ''')

        conn.execute("""
        CREATE TABLE IF NOT EXISTS otp_requests (
            email TEXT PRIMARY KEY COLLATE NOCASE,
            attempts INTEGER NOT NULL DEFAULT 0,
            window_start REAL NOT NULL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS otp_attempts (
            email TEXT PRIMARY KEY COLLATE NOCASE,
            attempts INTEGER NOT NULL DEFAULT 0,
            last_attempt REAL NOT NULL
        );
        """)


        conn.execute("""
        CREATE TABLE IF NOT EXISTS alerts (
            alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
            shipment_id TEXT,
            outlet_id TEXT,
            severity TEXT,
            category TEXT,
            message TEXT,
            date TEXT,
            resolved INT DEFAULT 0
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS shipments (
            shipment_id TEXT PRIMARY KEY,
            origin_port TEXT,
            dest_port TEXT,
            carrier TEXT,
            status TEXT,
            eta TEXT,
            weight_kg REAL,
            hs_code TEXT,
            predicted_delay_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS freight_quotes (
            quote_id TEXT PRIMARY KEY,
            shipment_id TEXT,
            customer_id TEXT,
            carrier TEXT,
            base_cost REAL,
            insurance REAL,
            customs_fee REAL,
            fuel_surcharge REAL,
            final_price REAL,
            margin_pct REAL,
            status TEXT,
            created_at TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ports (
            port_id TEXT PRIMARY KEY,
            port_name TEXT,
            country TEXT,
            congestion_index REAL,
            avg_dwell_days REAL,
            lat REAL,
            lon REAL,
            region TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS carriers (
            carrier_id TEXT PRIMARY KEY,
            name TEXT,
            rating REAL,
            on_time_pct REAL,
            avg_cost_index REAL,
            risk_level TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id TEXT PRIMARY KEY,
            name TEXT,
            industry TEXT,
            priority_tier TEXT,
            credit_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ml_metrics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            module TEXT,
            model_name TEXT,
            metric_name TEXT,
            metric_value REAL,
            trained_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY,
            outlet_name TEXT,
            location TEXT,
            tier TEXT,
            revenue REAL,
            operating_costs REAL,
            customer_satisfaction REAL,
            staff_headcount INT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            name TEXT,
            role TEXT,
            salary REAL,
            overtime_hrs REAL,
            job_satisfaction INT,
            age INT,
            tenure_years INT,
            work_life_balance INT,
            predicted_attrition_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS inventory (
            record_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            sku_name TEXT,
            category TEXT,
            current_stock INT,
            reorder_threshold INT,
            weekly_demand REAL,
            lead_time_days INT,
            stockout_risk_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS marketing (
            campaign_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            campaign_name TEXT,
            channel TEXT,
            budget REAL,
            actual_roi REAL,
            reach INT,
            conversions INT,
            start_date TEXT,
            end_date TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS feedback (
            feedback_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            rating INT,
            comment TEXT,
            date TEXT,
            sentiment_score REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS audits (
            audit_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            audit_date TEXT,
            score REAL,
            violations INT,
            category TEXT,
            status TEXT,
            notes TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS weather_risks (
            port_name TEXT PRIMARY KEY,
            current_severity INT,
            forecast TEXT,
            wind_speed REAL,
            wave_height REAL,
            temperature REAL
        );
        """)
        conn.commit()

# ------------------------------------------------------------------ User DB Helpers (Milestone 1 Authentication)
def get_user_by_email(email):
    """Retrieves a user by email address (case-insensitive)."""
    if not email:
        return None
    try:
        with get_conn() as conn:
            conn.row_factory = sqlite3.Row
            r = conn.execute("SELECT * FROM users WHERE LOWER(email)=LOWER(?);", (str(email).strip(),)).fetchone()
            return dict(r) if r else None
    except Exception:
        return None

def get_user_by_username(username):
    """Retrieves a user by username (case-insensitive)."""
    if not username:
        return None
    try:
        with get_conn() as conn:
            conn.row_factory = sqlite3.Row
            r = conn.execute("SELECT * FROM users WHERE LOWER(username)=LOWER(?);", (str(username).strip(),)).fetchone()
            return dict(r) if r else None
    except Exception:
        return None

def get_user_by_identifier(identifier):
    """Retrieves a user matching email or username."""
    if not identifier:
        return None
    ident = str(identifier).strip()
    try:
        with get_conn() as conn:
            conn.row_factory = sqlite3.Row
            r = conn.execute("SELECT * FROM users WHERE LOWER(email)=LOWER(?) OR LOWER(username)=LOWER(?);", (ident, ident)).fetchone()
            return dict(r) if r else None
    except Exception:
        return None

def create_user(username, email, password_hash, role="Customer", security_question=None, security_answer=None, profile_picture=None):
    """Creates a new user account in SQLite."""
    try:
        with get_conn() as conn:
            cur = conn.cursor()
            cur.execute("""
                INSERT INTO users (username, email, password_hash, role, security_question, security_answer, profile_picture, failed_attempts, lock_until, account_status)
                VALUES (?, ?, ?, ?, ?, ?, ?, 0, NULL, 'active');
            """, (username, email, password_hash, role, security_question, security_answer, profile_picture))
            conn.commit()
            return cur.lastrowid
    except Exception as e:
        print("create_user error:", e)
        return None

def is_user_locked(user_or_ident):
    """Checks whether user account is locked (5 failed attempts with active cooldown)."""
    user = user_or_ident if isinstance(user_or_ident, dict) else get_user_by_identifier(user_or_ident)
    if not user:
        return False
    lock_until = user.get("lock_until")
    failed = user.get("failed_attempts") or 0
    if failed >= 5 and lock_until:
        try:
            import datetime as _dt
            lock_clean = str(lock_until).split(".")[0].replace("Z", "")
            lock_dt = _dt.datetime.fromisoformat(lock_clean)
            if _dt.datetime.utcnow() < lock_dt:
                return True
            else:
                # Cooldown expired: reset lockout automatically
                unlock_user(user.get("email") or user.get("username"))
                return False
        except Exception:
            return False
    return False

def record_login_failure(identifier, max_attempts=5, lockout_seconds=900):
    """Increments failed login attempts and applies 5-attempt lockout (15 min cooldown)."""
    user = get_user_by_identifier(identifier)
    if not user:
        return {"failed_attempts": 0, "locked": False}
    attempts = (user.get("failed_attempts") or 0) + 1
    locked = False
    lock_until = None
    status = user.get("account_status", "active")
    if attempts >= max_attempts:
        locked = True
        status = "locked"
        import datetime as _dt
        lock_until = (_dt.datetime.utcnow() + _dt.timedelta(seconds=lockout_seconds)).isoformat()
    try:
        with get_conn() as conn:
            conn.execute("UPDATE users SET failed_attempts=?, lock_until=?, account_status=? WHERE id=?;",
                         (attempts, lock_until, status, user["id"]))
            conn.commit()
    except Exception:
        pass
    return {"failed_attempts": attempts, "locked": locked, "lock_until": lock_until}

def record_login_success(identifier):
    """Resets failed login attempts and updates last_login timestamp."""
    user = get_user_by_identifier(identifier)
    if not user:
        return
    import datetime as _dt
    now_iso = _dt.datetime.utcnow().isoformat()
    try:
        with get_conn() as conn:
            conn.execute("UPDATE users SET failed_attempts=0, lock_until=NULL, account_status='active', last_login=? WHERE id=?;",
                         (now_iso, user["id"]))
            conn.commit()
    except Exception:
        pass

def unlock_user(identifier):
    """Resets lockout parameters for an account."""
    user = get_user_by_identifier(identifier)
    if not user:
        return
    try:
        with get_conn() as conn:
            conn.execute("UPDATE users SET failed_attempts=0, lock_until=NULL, account_status='active' WHERE id=?;",
                         (user["id"],))
            conn.commit()
    except Exception:
        pass

def update_user_password(identifier, new_password_hash):
    """Updates password hash and unlocks account."""
    user = get_user_by_identifier(identifier)
    if not user:
        return
    try:
        with get_conn() as conn:
            conn.execute("UPDATE users SET password_hash=?, failed_attempts=0, lock_until=NULL, account_status='active' WHERE id=?;",
                         (new_password_hash, user["id"]))
            conn.commit()
    except Exception:
        pass



# -----------------------------------------------------------------------------
# Milestone 1: OTP Rate-Limiting & Verification Attempts
# -----------------------------------------------------------------------------
def get_otp_attempts(email: str) -> int:
    email = str(email or "").lower().strip()
    try:
        with get_conn() as conn:
            row = conn.execute("SELECT attempts FROM otp_attempts WHERE email = ?;", (email,)).fetchone()
            return row["attempts"] if row else 0
    except Exception:
        return 0

def record_failed_otp_attempt(email: str):
    email = str(email or "").lower().strip()
    import time
    try:
        with get_conn() as conn:
            row = conn.execute("SELECT attempts FROM otp_attempts WHERE email = ?;", (email,)).fetchone()
            attempts = (row["attempts"] if row else 0) + 1
            conn.execute("""
                INSERT INTO otp_attempts (email, attempts, last_attempt)
                VALUES (?, ?, ?)
                ON CONFLICT(email) DO UPDATE SET
                    attempts = excluded.attempts,
                    last_attempt = excluded.last_attempt;
            """, (email, attempts, time.time()))
            conn.commit()
    except Exception:
        pass

def clear_otp_attempts(email: str):
    email = str(email or "").lower().strip()
    try:
        with get_conn() as conn:
            conn.execute("DELETE FROM otp_attempts WHERE email = ?;", (email,))
            conn.commit()
    except Exception:
        pass

def can_request_otp(email: str, max_requests: int = 10, window_seconds: int = 300):
    email = str(email or "").lower().strip()
    import time
    now = time.time()
    try:
        with get_conn() as conn:
            row = conn.execute("SELECT attempts, window_start FROM otp_requests WHERE email = ?;", (email,)).fetchone()
            if not row:
                return True, ""
            if now - row["window_start"] > window_seconds:
                conn.execute("DELETE FROM otp_requests WHERE email = ?;", (email,))
                conn.commit()
                return True, ""
            if row["attempts"] >= max_requests:
                rem = int(window_seconds - (now - row["window_start"]))
                return False, f"Too many verification requests. Please wait {rem} seconds."
            return True, ""
    except Exception:
        return True, ""

def record_otp_request(email: str, window_seconds: int = 300):
    email = str(email or "").lower().strip()
    import time
    now = time.time()
    try:
        with get_conn() as conn:
            row = conn.execute("SELECT attempts, window_start FROM otp_requests WHERE email = ?;", (email,)).fetchone()
            if not row or (now - row["window_start"] > window_seconds):
                conn.execute("INSERT OR REPLACE INTO otp_requests (email, attempts, window_start) VALUES (?, 1, ?);", (email, now))
            else:
                conn.execute("UPDATE otp_requests SET attempts = attempts + 1 WHERE email = ?;", (email,))
            conn.commit()
    except Exception:
        pass


# ------------------------------------------------------------------------------
# RBAC Persisted Outlet Assignment & User Management
# ------------------------------------------------------------------------------
def assign_user_outlet(username, outlet_id):
    """Assign a Store Manager to one outlet; the assignment is persisted in SQLite."""
    with get_conn() as conn:
        user = conn.execute("SELECT role FROM users WHERE username=?", (username,)).fetchone()
        outlet = conn.execute("SELECT 1 FROM outlets WHERE outlet_id=?", (outlet_id,)).fetchone()
        if not user:
            raise ValueError("Unknown username")
        if not outlet:
            raise ValueError("Unknown outlet_id")
        if (user[0] or "").strip().lower() != "store manager":
            raise ValueError("Outlet assignment is only supported for Store Manager role")
        conn.execute("DELETE FROM user_outlet_assignments WHERE username=?", (username,))
        conn.execute("INSERT INTO user_outlet_assignments(username,outlet_id) VALUES(?,?)", (username, outlet_id))
        conn.execute("UPDATE users SET outlet_id=? WHERE username=?", (outlet_id, username))
        conn.commit()
    return True


def list_users():
    """All accounts with role and assigned outlet (password hashes and security answers are never returned)."""
    with get_conn() as conn:
        rows = conn.execute("SELECT username,email,role,outlet_id,account_status FROM users ORDER BY id").fetchall()
    return [{"username": r[0], "email": r[1], "role": r[2] or "Staff", "outlet_id": r[3], "account_status": r[4] or "active"} for r in rows]


DEMO_ROLE_USERS = [
    ("admin", "admin@infosys.com", "Admin"),
    ("owner", "owner@infosys.com", "Franchise Owner"),
    ("manager", "manager@infosys.com", "Store Manager"),
    ("staff", "staff@infosys.com", "Staff"),
    ("auditor", "auditor@infosys.com", "Auditor"),
]

def seed_role_users():
    """Create demo accounts for each of the five roles if not existing yet (idempotent).
    The demo Store Manager is assigned the first outlet."""
    with get_conn() as conn:
        for username, email, role in DEMO_ROLE_USERS:
            conn.execute("INSERT OR IGNORE INTO users(username,email,password_hash,role,account_status) VALUES(?,?,'!',?,'active')", (username, email, role))
        conn.commit()
        first = conn.execute("SELECT outlet_id FROM outlets ORDER BY outlet_id LIMIT 1").fetchone()
        mgr = conn.execute("SELECT username,outlet_id FROM users WHERE email='manager@infosys.com' AND role='Store Manager'").fetchone()
    if first and mgr and not mgr[1]:
        assign_user_outlet(mgr[0], first[0])


In [ ]:
%%writefile /content/app/rbac.py
"""Role-based access control for FranchiseOps AI.

Roles are enforced on the backend; hiding a menu item is not a security boundary.
localuser.py is a notebook demo identity only. In a deployed application, login middleware
must set request.state.user to the authenticated DB user (see server.user_of).

Public API
    ROLES, ROLE_MENU, ROLE_CAPS     the five roles, the pages each may open, and what each may do
    menu_for(user)                  menu ids a user (dict or role string) may open
    has_capability(ctx, cap)        True / False
    need(ctx, cap)                  raise HTTP 403 unless the capability is held
    scope_for(user)                 None = all outlets | "OUT-001" = that outlet only | NO_OUTLET = nothing
    enforce_outlet(ctx, outlet_id)  raise HTTP 403 if a Store Manager touches an outlet that is not theirs
    set_denial_hook(fn)             fn(user, what) is called for every denial (the server writes it to the audit log)
"""
from fastapi import HTTPException

ROLES = ("Admin", "Franchise Owner", "Store Manager", "Staff", "Auditor")

# Pages each role may open (ids match views_franchise.MENU).
ROLE_MENU = {
    "Admin": {"admin", "profile", "copilot", "workforce", "outlets", "inventory", "marketing", "sentiment", "audit"},
    "Franchise Owner": {"profile", "copilot", "workforce", "outlets", "inventory", "marketing", "sentiment", "audit"},
    "Store Manager": {"profile", "copilot", "workforce", "outlets", "inventory", "sentiment", "audit"},
    "Staff": {"profile", "copilot"},
    "Auditor": {"profile", "copilot", "workforce", "outlets", "inventory", "marketing", "sentiment", "audit"},
}

# Actions each role may perform. Every protected route calls need(ctx, <capability>).
ROLE_CAPS = {
    "Admin": {"view", "run", "review", "audit_write", "resolve", "quote", "approve", "admin", "manage_users", "users", "assign_outlet", "export", "weather_refresh", "attach_doc", "inject_storm"},
    "Franchise Owner": {"view", "run", "review", "audit_write", "resolve", "quote", "approve", "export", "weather_refresh", "attach_doc"},
    "Store Manager": {"view", "review", "resolve"},
    "Staff": {"view", "review"},
    "Auditor": {"view", "export"},  # explicitly read-only
}

# Every capability that changes data or starts work. The Auditor must never hold any of these.
WRITE_CAPS = frozenset({"run", "review", "audit_write", "resolve", "quote", "approve", "admin", "manage_users",
                        "assign_outlet", "weather_refresh", "attach_doc", "inject_storm"})
if ROLE_CAPS["Auditor"] & WRITE_CAPS:                      # fail at import time, not in production
    raise RuntimeError("Auditor must stay read-only: " + ", ".join(sorted(ROLE_CAPS["Auditor"] & WRITE_CAPS)))
if set(ROLES) != set(ROLE_MENU) or set(ROLES) != set(ROLE_CAPS):
    raise RuntimeError("ROLES, ROLE_MENU and ROLE_CAPS must list the same five roles")

# Returned by scope_for() for a Store Manager who has no outlet assigned yet. It matches no real outlet, so
# every scoped query returns nothing (fail closed) instead of falling back to "all outlets".
NO_OUTLET = "__no_outlet_assigned__"

ROLE_ALIASES = {"administrator": "Admin", "owner": "Franchise Owner", "manager": "Store Manager"}
_BY_LOWER = {r.lower(): r for r in ROLES}

_denial_hook = None
def set_denial_hook(fn):
    """Register fn(user_dict, what) to be called on every denied request."""
    global _denial_hook
    _denial_hook = fn

def _user(ctx_or_user):
    return ctx_or_user.get("user", ctx_or_user) if isinstance(ctx_or_user, dict) else {}

def _deny(ctx_or_user, what, detail):
    if _denial_hook:
        try: _denial_hook(_user(ctx_or_user), what)
        except Exception: pass                              # logging must never turn a 403 into a 500
    raise HTTPException(status_code=403, detail=detail)

def normalize_role(role):
    value = str(role or "Staff").strip()
    low = value.lower()
    return ROLE_ALIASES.get(low) or _BY_LOWER.get(low) or "Staff"

def has_capability(user_or_ctx, capability):
    return capability in ROLE_CAPS[normalize_role(_user(user_or_ctx).get("role"))]

def menu_for(user):
    """Return permitted menu IDs for a user dictionary or a role string."""
    role = normalize_role(user.get("role") if isinstance(user, dict) else user)
    return sorted(ROLE_MENU[role])

def need(ctx, capability):
    """Raise HTTP 403 unless the current context has the requested capability."""
    if not has_capability(ctx, capability):
        _deny(ctx, capability, f"Role is not allowed to perform: {capability}")
    return True

def scope_for(user):
    """Outlet scope of a user: None (every outlet) for all roles except Store Manager.
    A Store Manager is limited to the assigned outlet; with no assignment the scope is NO_OUTLET (sees nothing)."""
    user = user if isinstance(user, dict) else {"role": user}
    if normalize_role(user.get("role")) != "Store Manager":
        return None
    return str(user.get("outlet_id") or "").strip() or NO_OUTLET

def enforce_outlet(ctx, outlet_id):
    """Store Managers can only access their assigned outlet; every other role is not restricted here."""
    user = _user(ctx)
    if normalize_role(user.get("role")) == "Store Manager":
        assigned = str(user.get("outlet_id") or "").strip() or (str(ctx.get("scope") or "").strip() if isinstance(ctx, dict) else "")
        if not assigned or assigned == NO_OUTLET or str(outlet_id).strip() != assigned:
            _deny(ctx, f"outlet:{outlet_id}", "Store Managers may access only their assigned outlet")
    return True


In [ ]:
%%writefile /content/app/demo_auth.py
"""Demo sign-in for the five roles (enable with DEMO_AUTH=1).

This is the login middleware that rbac.py / server.user_of expect: it only decides WHO the caller is
(request.state.user = {"email": ...}). Role, outlet and account status are always re-read from the users
table by server.user_of, so the browser can never choose its own role or outlet.

    GET  /api/demo/roles    the five roles with their menu + capabilities (login screen and permission matrix)
    POST /api/demo/login    {"role": "Store Manager"} -> sets a signed cookie for that role's demo account
    POST /api/demo/logout   clears the cookie
    GET  /api/admin/outlets Admin-only: outlet ids for the assignment dropdown

The cookie is HMAC-signed and HttpOnly. The demo accounts have no password ('!' hash), so this must not be
used on a public deployment: replace it with real authentication that sets request.state.user.
"""
import os, hmac, hashlib, secrets
from fastapi import Request
from fastapi.responses import JSONResponse
import rbac

COOKIE = "fo_demo"
_SECRET = (os.environ.get("DEMO_AUTH_SECRET") or secrets.token_hex(32)).encode()
ENABLED = os.environ.get("DEMO_AUTH") == "1"

BLURB = {
    "Admin": "Full control: users, outlet assignment, every agent and every action.",
    "Franchise Owner": "All outlets and agents. Runs agents, approves and resolves. No user management.",
    "Store Manager": "Only their own outlet. Reviews feedback and resolves tickets for that outlet.",
    "Staff": "Sentiment page only. Can submit customer feedback reviews.",
    "Auditor": "Read-only across outlets, plus export. Cannot change or run anything.",
}
WRITE_NOTE = "read-only"

def _sign(email):
    return email + "|" + hmac.new(_SECRET, email.encode(), hashlib.sha256).hexdigest()

def _verify(cookie):
    email, _, sig = (cookie or "").rpartition("|")
    if email and hmac.compare_digest(sig, hmac.new(_SECRET, email.encode(), hashlib.sha256).hexdigest()):
        return email
    return None

def _accounts():
    from db import DEMO_ROLE_USERS
    return {role: (username, email) for username, email, role in DEMO_ROLE_USERS}

def install(app, user_of, ctx, need):
    if not ENABLED:
        return
    OPEN = ("/api/public", "/api/health", "/api/demo/")

    @app.middleware("http")
    async def demo_login(request: Request, call_next):
        email = _verify(request.cookies.get(COOKIE))
        if email:
            request.state.user = {"email": email}
        elif request.url.path.startswith("/api/") and not request.url.path.startswith(OPEN):
            return JSONResponse({"ok": False, "msg": "Sign in to continue", "detail": "Authentication required"}, status_code=401)
        return await call_next(request)

    @app.get("/api/demo/roles")
    def roles():
        acc = _accounts()
        return {"roles": [{"role": r, "email": acc[r][1], "blurb": BLURB[r],
                           "menu": rbac.menu_for(r), "caps": sorted(rbac.ROLE_CAPS[r]),
                           "read_only": not (rbac.ROLE_CAPS[r] & rbac.WRITE_CAPS),
                           "outlet_scoped": r == "Store Manager"} for r in rbac.ROLES],
                "all_caps": sorted(set().union(*rbac.ROLE_CAPS.values())),
                "write_caps": sorted(rbac.WRITE_CAPS)}

    @app.post("/api/demo/login")
    async def login(request: Request):
        d = await request.json()
        role = str(d.get("role", ""))
        if role not in rbac.ROLES:
            return JSONResponse({"ok": False, "msg": "Choose one of the five roles"}, status_code=400)
        email = _accounts()[role][1]
        r = JSONResponse({"ok": True, "msg": "Signed in as " + role, "role": role})
        r.set_cookie(COOKIE, _sign(email), httponly=True, samesite="lax", path="/", max_age=8 * 3600)
        return r

    @app.post("/api/demo/logout")
    def logout():
        r = JSONResponse({"ok": True, "msg": "Signed out"})
        r.delete_cookie(COOKIE, path="/")
        return r

    @app.get("/api/admin/outlets")
    def outlets(request: Request):
        need(ctx(user_of(request)), "assign_outlet")
        import blocks as B
        df = B.Q("SELECT outlet_id FROM outlets ORDER BY outlet_id")
        return {"ok": True, "outlets": df.outlet_id.tolist() if len(df) else []}


def get_roles_data():
    acc = _accounts()
    return {"roles": [{"role": r, "email": acc[r][1], "blurb": BLURB[r],
                       "menu": rbac.menu_for(r), "caps": sorted(rbac.ROLE_CAPS[r]),
                       "read_only": not (rbac.ROLE_CAPS[r] & rbac.WRITE_CAPS),
                       "outlet_scoped": r == "Store Manager"} for r in rbac.ROLES],
            "all_caps": sorted(set().union(*rbac.ROLE_CAPS.values())),
            "write_caps": sorted(rbac.WRITE_CAPS)}


In [ ]:
%%writefile /content/app/domain_franchise.py
"""FranchiseOps AI live domain: simulator, 5 agents, intelligence engine, action handlers."""
import os, re, json, math, random, time, datetime, uuid
import numpy as np, pandas as pd, requests
from live_engine import AgentResult, risk_label, conn, now, _DB_LOCK, set_state, get_state, emit

KIND = "Franchise"; NAME = "FranchiseOps AI"; ENTITY_COL = "outlet_id"; SMS = True; ESCALATE_TO = "VP Operations"
OWNER = {"reorder": "Store Manager", "pause_promo": "Marketing Manager", "corrective_action": "Regional Manager", "retention_plan": "HR Lead",
         "outlet_review": "Regional Manager", "staffing_shortage": "Store Manager", "reallocate_budget": "Marketing Manager"}
ACTION_AGENT = {"reorder": "inventory", "pause_promo": "inventory", "corrective_action": "audit", "retention_plan": "staff",
                "outlet_review": "outlet", "staffing_shortage": "staff", "reallocate_budget": "marketing"}
SIM_DAY_SEC = 60          # one simulated day = 60 s of wall-clock (5 s ticks -> 12 ticks per day)
TICKS_PER_DAY = 12
def Q(sql, p=()):
    with _DB_LOCK, conn() as c: return pd.read_sql(sql, c, params=p)

# ---------------------------------------------------------------- bootstrap / simulator
def bootstrap():
    from scale_data import scale_franchise; scale_franchise()
    with _DB_LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS sales(sale_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT, ts TEXT, amount REAL, tx_count INT, source TEXT);
        CREATE TABLE IF NOT EXISTS attendance(att_id INTEGER PRIMARY KEY AUTOINCREMENT, staff_id TEXT, outlet_id TEXT, ts TEXT, status TEXT);
        CREATE TABLE IF NOT EXISTS purchase_orders(po_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT, sku_name TEXT, qty INT, ts TEXT, eta_ts TEXT, status TEXT);
        CREATE TABLE IF NOT EXISTS ml_models(model_id INTEGER PRIMARY KEY AUTOINCREMENT, model_name TEXT, metrics TEXT, trained_at TEXT, is_active INT);
        CREATE INDEX IF NOT EXISTS ix_sales ON sales(outlet_id, ts);
        """)
        if c.execute("SELECT COUNT(*) FROM sales").fetchone()[0] == 0:       # 90 days of seeded history so baselines exist
            rng = np.random.default_rng(7); today = datetime.date.today()
            for oid, rev in c.execute("SELECT outlet_id, revenue FROM outlets").fetchall():
                base = (rev or 100000) / 30
                for d in range(90, 0, -1):
                    dt = today - datetime.timedelta(days=d)
                    c.execute("INSERT INTO sales(outlet_id,ts,amount,tx_count,source) VALUES(?,?,?,?,'seed')", (oid, dt.isoformat(), float(base * rng.normal(1, .07) * (1.15 if dt.weekday() >= 5 else 1)), int(base / 250)))
        active = c.execute("SELECT COUNT(*) FROM marketing WHERE end_date>=date('now')").fetchone()[0]
        if active == 0:                                                       # launch a live promo set (seed campaigns are all expired)
            ids = [r[0] for r in c.execute("SELECT campaign_id FROM marketing ORDER BY RANDOM() LIMIT 14").fetchall()]
            for i in ids: c.execute("UPDATE marketing SET start_date=date('now','-3 day'), end_date=date('now','+14 day') WHERE campaign_id=?", (i,))
        c.execute("UPDATE inventory SET weekly_demand=MAX(weekly_demand,1)")
        for rid, st_, wk, lead in c.execute("SELECT record_id, current_stock, weekly_demand, lead_time_days FROM inventory").fetchall():   # (re)start from a healthy, varied stock position
            if st_ / max(wk / 7, 1) < lead + 1.5: c.execute("UPDATE inventory SET current_stock=? WHERE record_id=?", (int(wk / 7 * (lead + random.uniform(1.5, 14))), rid))

_incident = {}; _tickn = [0]
def tick():
    reasons = []; _tickn[0] += 1; n = _tickn[0]; mult = get_state("demand_mult", 1.0) or 1.0
    ts = now()
    with _DB_LOCK, conn() as c:
        outlets = c.execute("SELECT outlet_id, revenue, operating_costs, customer_satisfaction FROM outlets").fetchall()
        real = set()
        for oid, rev, cost, csat in outlets:
            if oid in real: continue
            if oid not in _incident and random.random() < 0.002: _incident[oid] = random.randint(8, 14)
            f = 0.35 if oid in _incident else 1.0
            if oid in _incident:
                _incident[oid] -= 1
                if _incident[oid] <= 0: _incident.pop(oid)
            base = 3000000 / 30 / TICKS_PER_DAY if not rev else (rev / 30 / TICKS_PER_DAY)
            amt = float(base * random.uniform(.8, 1.2) * mult * f)
            c.execute("INSERT INTO sales(outlet_id,ts,amount,tx_count,source) VALUES(?,?,?,?,'sim')", (oid, ts, amt, max(1, int(amt / 250))))
            c.execute("UPDATE outlets SET revenue=revenue+?, operating_costs=operating_costs+? WHERE outlet_id=?", (amt, amt * (cost / rev if rev else .65), oid))
        # stock drawdown + deliveries
        for rid, oid, sku, stock, thr, wk, lead in c.execute("SELECT record_id,outlet_id,sku_name,current_stock,reorder_threshold,weekly_demand,lead_time_days FROM inventory").fetchall():
            use = max(0, int(round(wk / 7 / TICKS_PER_DAY * mult * random.uniform(.6, 1.6))))
            new = max(0, stock - use); cover = new / max(wk / 7, 1)
            risk = float(min(1, max(0, 1 - cover / (lead + 3))))
            c.execute("UPDATE inventory SET current_stock=?, stockout_risk_prob=? WHERE record_id=?", (new, round(risk, 2), rid))
            if stock / max(wk / 7, 1) >= lead + 2 > cover: reasons.append(f"stock-low:{oid}")
        for po, oid, sku, qty in c.execute("SELECT po_id,outlet_id,sku_name,qty FROM purchase_orders WHERE status='ordered' AND eta_ts<=?", (ts,)).fetchall():
            c.execute("UPDATE inventory SET current_stock=current_stock+? WHERE outlet_id=? AND sku_name=?", (qty, oid, sku))
            c.execute("UPDATE purchase_orders SET status='delivered' WHERE po_id=?", (po,)); emit("po_delivered", oid, {"sku": sku, "qty": qty})
        # staff: attendance, overtime drift, attrition recompute (same heuristic the app uses)
        for sid, oid, sal, sat, ot in c.execute("SELECT staff_id,outlet_id,salary,job_satisfaction,overtime_hrs FROM staff ORDER BY RANDOM() LIMIT 40").fetchall():
            st = random.choices(["Present", "Late", "Absent"], [.86, .09, .05 + (.25 if oid in _incident else 0)])[0]
            c.execute("INSERT INTO attendance(staff_id,outlet_id,ts,status) VALUES(?,?,?,?)", (sid, oid, ts, st))
            ot2 = float(min(45, max(0, ot + random.uniform(-1.5, 2) + (1.5 if st == "Absent" else 0))))
            c.execute("UPDATE staff SET overtime_hrs=? WHERE staff_id=?", (round(ot2, 1), sid))
        # feedback + csat moving average
        for oid, csat in random.sample([(o[0], o[3]) for o in outlets], 6):
            r = int(min(5, max(1, round(random.gauss(csat if csat else 4, .9) - (1.5 if oid in _incident else 0)))))
            c.execute("INSERT INTO feedback(feedback_id,outlet_id,rating,comment,date,sentiment_score) VALUES(?,?,?,?,?,?)",
                      ("FB-L"+uuid.uuid4().hex[:10], oid, r, "Live feedback", datetime.date.today().isoformat(), (r - 3) / 2))
            c.execute("UPDATE outlets SET customer_satisfaction=ROUND(0.95*customer_satisfaction+0.05*?,3) WHERE outlet_id=?", (r, oid))
        if random.random() < 0.05:
            oid = random.choice(outlets)[0]; sc = float(np.clip(random.gauss(84, 12), 40, 100))
            c.execute("INSERT INTO audits(audit_id,outlet_id,audit_date,score,violations,category,status,notes) VALUES(?,?,?,?,?,?,?,?)",
                      ("AUD-L"+uuid.uuid4().hex[:10], oid, datetime.date.today().isoformat(), sc, int((100 - sc) / 6), "Food Safety", "Pass" if sc >= 85 else "Conditional Pass" if sc >= 70 else "Fail", "Live inspection"))
            if sc < 70: reasons.append(f"audit-fail:{oid}")
    return list(dict.fromkeys(reasons))[:5]

def process(): pass
def feed(): _external()
DEFAULT_POLICY = {"pause_promo": "approval"}

def _external():
    """Real feeds: Open-Meteo weather (city list from the app), Nager.Date public holidays. Falls back silently when offline."""
    mult, feeds, rain = 1.0, {}, []
    try:
        from weather_context import CITY_COORDS
        for city, (la, lo) in list(CITY_COORDS.items())[:5]:
            r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={la}&longitude={lo}&current=temperature_2m,precipitation,wind_speed_10m", timeout=4).json()["current"]
            rain.append(r.get("precipitation", 0) or 0); feeds[city] = {"temp": r["temperature_2m"], "rain": r["precipitation"], "wind": r["wind_speed_10m"]}
        if np.mean(rain) > 1.5: mult *= .92
        feeds_ok = True
    except Exception: feeds_ok = False
    hol = False
    try:
        hol = requests.get("https://date.nager.at/api/v3/IsTodayPublicHoliday/IN", timeout=4).status_code == 200
        if hol: mult *= 1.15
    except Exception: pass
    set_state("demand_mult", round(mult, 3)); set_state("weather", feeds); set_state("feeds", {"weather": feeds_ok, "holiday": hol, "ts": now()})

def feed_status():
    f = get_state("feeds", {}) or {}
    return [("Open-Meteo weather", "Real · live" if f.get("weather") else "Offline · no demand adjustment"),
            ("Public holidays (Nager.Date)", "Real · today is a holiday" if f.get("holiday") else "Real · no holiday today" if f else "Waiting"),
            ("Outlet POS, stock, attendance, feedback", "Simulated feed (replace via Data Feed Center)")]

# ---------------------------------------------------------------- agents (PRS v3: 7 agents + Intelligence Engine)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, silhouette_score
TIER_BASE = {"Excellent": 95, "Good": 80, "Needs Attention": 58, "Critical": 35}
def _reg_model(name, metrics, active=1):
    with _DB_LOCK, conn() as c:
        c.execute("UPDATE ml_models SET is_active=0 WHERE model_name=?", (name,))
        c.execute("INSERT INTO ml_models(model_name,metrics,trained_at,is_active) VALUES(?,?,?,?)", (name, json.dumps(metrics), now(), active))

def outlet_agent(s):
    """Agent 1: KMeans (k=4) on 30-day revenue/orders/customers, ranked into Excellent/Good/Needs Attention/Critical."""
    d = Q("SELECT outlet_id, source, AVG(amount) rev, AVG(tx_count) orders FROM sales GROUP BY outlet_id, source")
    d.loc[d.source == "sim", ["rev", "orders"]] *= TICKS_PER_DAY
    d = d.sort_values("source").drop_duplicates("outlet_id", keep="first")            # sim rows preferred once present ('sim' < 'seed')
    d["cust"] = d.orders * .8; X = StandardScaler().fit_transform(d[["rev", "orders", "cust"]])
    km = KMeans(4, n_init=10, random_state=42).fit(X); d["cl"] = km.labels_
    order = d.groupby("cl").rev.mean().sort_values(ascending=False).index.tolist(); tmap = dict(zip(order, ["Excellent", "Good", "Needs Attention", "Critical"]))
    d["tier"] = d.cl.map(tmap); dist = np.linalg.norm(X - km.cluster_centers_[km.labels_], axis=1); d["conf"] = (1 / (1 + dist)).round(2)
    d["score"] = d.tier.map(TIER_BASE) + d.groupby("tier").rev.rank(pct=True) * 8 - 4
    sil = float(silhouette_score(X, km.labels_)) if len(d) > 4 else 0
    _reg_model("kmeans_outlet_clusters", {"silhouette": round(sil, 3), "n": len(d)})
    crit = d[d.tier == "Critical"]
    return AgentResult("outlet", round(float(d.score.mean()), 1), risk_label(d.score.mean()), f"KMeans tiers: " + ", ".join(f"{t} {int((d.tier == t).sum())}" for t in TIER_BASE),
        {"per_outlet": d.set_index("outlet_id").score.round(1).to_dict(), "tiers": d.set_index("outlet_id").tier.to_dict(), "silhouette": round(sil, 3), "weak": crit.outlet_id.tolist()},
        [{"type": "outlet_review", "target": r.outlet_id, "detail": f"KMeans tier Critical (revenue {r.rev:,.0f}/day, confidence {r.conf})", "severity": "Warning"} for r in crit.itertuples()])

def forecast_series(outlet, horizon=14, method=None):
    """Demand forecast for one outlet. Prophet when installed and USE_PROPHET != 0 (daily + weekly seasonality), else linear trend x weekday-seasonal index."""
    h = Q("SELECT substr(ts,1,10) d, SUM(amount) a FROM sales WHERE outlet_id=? AND source='seed' GROUP BY d ORDER BY d", (outlet,))
    if len(h) < 21: return h.assign(kind="actual"), pd.DataFrame(), "insufficient history"
    h["d"] = pd.to_datetime(h.d); fut_d = pd.date_range(h.d.iloc[-1] + pd.Timedelta(days=1), periods=horizon)
    if (method or os.environ.get("USE_PROPHET", "1")) not in ("0", "trend"):
        try:
            from prophet import Prophet
            m = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=False); m.fit(h.rename(columns={"d": "ds", "a": "y"}))
            f = m.predict(pd.DataFrame({"ds": fut_d})); return h, pd.DataFrame({"d": f.ds, "a": f.yhat, "lo": f.yhat_lower, "hi": f.yhat_upper}), "Prophet"
        except Exception: pass
    y = h.a.values; x = np.arange(len(y)); k, b = np.polyfit(x, y, 1); trend = k * x + b; ratio = pd.Series(y / np.maximum(trend, 1)).groupby(h.d.dt.weekday.values).mean()
    xf = np.arange(len(y), len(y) + horizon); fa = (k * xf + b) * pd.Series(fut_d.weekday).map(ratio).values; sd = float(np.std(y - trend * h.d.dt.weekday.map(ratio).values))
    return h, pd.DataFrame({"d": fut_d, "a": fa, "lo": fa - 1.28 * sd, "hi": fa + 1.28 * sd}), "trend x weekday seasonality"

def _forecast_ratio(outlet):
    """Agent 2 forecast: Prophet when installed (USE_PROPHET=1), otherwise weekday-seasonal linear trend. Returns next-7-day / last-7-day demand ratio."""
    h, f, _ = forecast_series(outlet, 7, method="trend")                 # the fleet-wide loop stays on the fast method; Prophet is used on the Inventory page per outlet
    if not len(f): return 1.0
    return float(np.clip(f.a.mean() / max(h.a.iloc[-7:].mean(), 1), .8, 1.3))

def inventory_agent(s):
    mult = get_state("demand_mult", 1.0) or 1.0
    d = Q("SELECT outlet_id, sku_name, current_stock, reorder_threshold mn, weekly_demand, lead_time_days lead FROM inventory")
    fr = get_state("forecast_ratio", {}) or {}
    if not fr or now()[:13] != (get_state("forecast_hour") or ""):
        fr = {o: _forecast_ratio(o) for o in d.outlet_id.unique()}; set_state("forecast_ratio", fr); set_state("forecast_hour", now()[:13])
    d["daily"] = d.weekly_demand.clip(lower=1) / 7 * mult * d.outlet_id.map(fr).fillna(1.0); d["cover"] = d.current_stock / d.daily
    po = Q("SELECT outlet_id, sku_name FROM purchase_orders WHERE status='ordered'"); open_po = set(zip(po.outlet_id, po.sku_name))
    d["short"] = (d.current_stock <= d.mn) | (d.cover < d.lead + 2)
    per = (100 - d.groupby("outlet_id").short.mean() * 200).clip(0, 100).round(1)
    acts = []
    for r in d[d.short].sort_values("cover").itertuples():
        if (r.outlet_id, r.sku_name) in open_po: continue
        qty = int(max(r.mn * 3, math.ceil(r.daily * (r.lead + 7)) - r.current_stock))               # PRS: reorder_qty = min_stock x 3 (or forecast need if larger)
        acts.append({"type": "reorder", "target": r.outlet_id, "detail": f"{r.sku_name}: {r.cover:.1f} days cover, order {qty} units", "severity": "Critical" if r.cover < r.lead else "Warning", "sku": r.sku_name, "qty": qty, "lead": int(r.lead)})
    n = int(d.short.sum())
    return AgentResult("inventory", round(float(per.mean()), 1), risk_label(per.mean()), f"{n} SKUs flagged of {len(d)}, demand x{mult}", {"per_outlet": per.to_dict(), "outlets": d[d.short].outlet_id.unique().tolist(), "demand_mult": mult, "forecast": "weekday-seasonal trend (Prophet optional)"}, acts[:40])

_LR = {}
def _attrition_model():
    """Agent 3 ML: LogisticRegression. Uses a fed HR dataset (table hr_training, e.g. IBM HR Analytics) when present; otherwise a bootstrap model on the staff table labelled by the rule score."""
    d = Q("SELECT st.staff_id, st.outlet_id, st.salary, st.overtime_hrs, st.job_satisfaction sat, st.tenure_years ten, st.work_life_balance wlb, st.age, st.predicted_attrition_prob p FROM staff st")
    att = Q("SELECT staff_id, AVG(status='Present') pres FROM attendance GROUP BY staff_id").set_index("staff_id").pres
    d["att"] = d.staff_id.map(att).fillna(.9) * 100; feats = ["salary", "overtime_hrs", "sat", "ten", "wlb", "age", "att"]
    src = "bootstrap (labels from rule score)"
    try:
        h = Q("SELECT * FROM hr_training")
        if len(h) >= 300 and "attrition" in h.columns:
            src = "fed HR dataset"; tr = h.rename(columns={"job_satisfaction": "sat", "tenure_years": "ten", "work_life_balance": "wlb", "monthly_income": "salary", "overtime_hrs": "overtime_hrs"})
            for f in feats:
                if f not in tr: tr[f] = d[f].median()
            X, y = tr[feats].fillna(0), tr.attrition.astype(int)
    except Exception: pass
    if src.startswith("boot"): X, y = d[feats], (d.p + np.random.default_rng(1).normal(0, .05, len(d)) > .5).astype(int)
    sc = StandardScaler().fit(X); Xtr, Xte, ytr, yte = train_test_split(sc.transform(X), y, test_size=.2, random_state=42, stratify=y if y.nunique() > 1 else None)
    lr = LogisticRegression(max_iter=500).fit(Xtr, ytr)
    m = {"accuracy": round(accuracy_score(yte, lr.predict(Xte)), 3), "auc": round(roc_auc_score(yte, lr.predict_proba(Xte)[:, 1]), 3) if yte.nunique() > 1 else None, "source": src}
    _reg_model("logreg_attrition", m); _LR.update(model=lr, sc=sc, feats=feats, metrics=m, trained=time.time()); return d

def _staff_frame():
    d = Q("SELECT st.staff_id, st.outlet_id, st.name, st.role, st.salary, st.overtime_hrs, st.job_satisfaction sat, st.tenure_years ten, st.work_life_balance wlb, st.age, st.predicted_attrition_prob p FROM staff st")
    att = Q("SELECT staff_id, AVG(status='Present') pres FROM attendance GROUP BY staff_id").set_index("staff_id").pres
    d["att"] = d.staff_id.map(att).fillna(.9) * 100; return d

def attrition_table(scope=None, n=25):
    """Per-staff attrition probability with the single biggest driver (coefficient x standardised value): what the Workforce page shows."""
    if not _LR: return pd.DataFrame()
    d = _staff_frame()
    if scope: d = d[d.outlet_id == scope]
    if not len(d): return d
    X = _LR["sc"].transform(d[_LR["feats"]]); d["prob"] = _LR["model"].predict_proba(X)[:, 1].round(3)
    contrib = X * _LR["model"].coef_[0]; d["driver"] = [_LR["feats"][i] for i in contrib.argmax(1)]
    return d.sort_values("prob", ascending=False).head(n)
def attrition_importance():
    if not _LR: return {}
    return dict(zip(_LR["feats"], _LR["model"].coef_[0].round(3).tolist()))

def staff_agent(s):
    if not _LR or time.time() - _LR["trained"] > 900: _attrition_model()
    d = _staff_frame()
    d["prob"] = _LR["model"].predict_proba(_LR["sc"].transform(d[_LR["feats"]]))[:, 1].round(3)
    with _DB_LOCK, conn() as c: c.executemany("UPDATE staff SET predicted_attrition_prob=? WHERE staff_id=?", list(zip(d.prob.tolist(), d.staff_id)))
    d["perf"] = d.sat * 20; hi = d[d.prob > .6]; low = d[(d.perf < 60) | (d.att < 75)]
    per = (100 - (d.groupby("outlet_id").prob.mean() * 100 * .6 + (100 - d.groupby("outlet_id").att.mean()) * .4)).clip(0, 100).round(1)
    a = [{"type": "retention_plan", "target": o, "detail": f"{n} staff above 60% predicted attrition (LogReg {_LR['metrics']['source']})", "severity": "Warning"} for o, n in hi.groupby("outlet_id").size().items() if n >= 4]
    a += [{"type": "staffing_shortage", "target": o, "detail": f"{n} staff below 75% attendance or performance 60", "severity": "Warning"} for o, n in low.groupby("outlet_id").size().items() if n >= 6]
    return AgentResult("staff", round(float(per.mean()), 1), risk_label(per.mean()), f"{len(hi)} high attrition risk, {len(low)} low performers or attendance (attrition model AUC {_LR['metrics'].get('auc', 0):.2f})", {"per_outlet": per.to_dict(), "short_staffed": [x["target"] for x in a if x["type"] == "staffing_shortage"], "model": _LR["metrics"]}, a[:25])

def marketing_agent(s):
    """Agent 4: ROI = (revenue-budget)/budget; KMeans (k=3) customer segments: High-Value, Frequent, Churned."""
    m = Q("SELECT outlet_id, campaign_name, channel, actual_roi, end_date FROM marketing"); m["roi_pct"] = (m.actual_roi - 1) * 100
    cu = Q("SELECT outlet_id, total_spend_90d sp, visit_freq vf, avg_order_value aov, days_since_last_visit dl FROM customers_fr")
    churn = {}; seg_counts = {}
    if len(cu) > 30:
        X = StandardScaler().fit_transform(cu[["sp", "vf", "aov", "dl"]]); km = KMeans(3, n_init=10, random_state=42).fit(X); cu["cl"] = km.labels_
        g = cu.groupby("cl").agg(sp=("sp", "mean"), dl=("dl", "mean")); names = {g.sp.idxmax(): "High-Value", g.dl.idxmax(): "Churned"}
        cu["seg"] = cu.cl.map(lambda k: names.get(k, "Frequent")); churn = cu.assign(ch=cu.seg == "Churned").groupby("outlet_id").ch.mean().to_dict(); seg_counts = cu.seg.value_counts().to_dict()
    per = (m.groupby("outlet_id").roi_pct.mean().clip(0, 300) / 3).round(1); best = m.groupby("channel").actual_roi.mean().idxmax()
    live = m[m.end_date >= datetime.date.today().isoformat()]; low = live[live.actual_roi < 2.0]
    acts = [{"type": "reallocate_budget", "target": r.outlet_id, "detail": f"{r.campaign_name}: ROI {r.roi_pct:.0f}% below 100%; shift budget to {best}", "severity": "Info"} for r in low.head(10).itertuples()]
    acts += [{"type": "win_back_campaign", "target": o, "detail": f"{v*100:.0f}% of customers in Churned segment", "severity": "Warning"} for o, v in churn.items() if v > .5][:10]
    return AgentResult("marketing", round(float(per.mean()), 1), risk_label(per.mean()), f"{len(live)} live campaigns, best channel {best}, {sum(v > .5 for v in churn.values())} outlets with high churn", {"per_outlet": per.to_dict(), "active_outlets": live.outlet_id.unique().tolist(), "best_channel": best, "segments": seg_counts, "churn_share": {k: round(v, 3) for k, v in churn.items()}}, acts)

def audit_agent(s):
    """Agent 5: latest audit per outlet. Pass >= 80, Warning >= 60, Fail < 60. SOP hint from the FAISS/keyword RAG when available."""
    d = Q("SELECT outlet_id, score, category FROM audits ORDER BY audit_date DESC, rowid DESC").drop_duplicates("outlet_id")
    per = d.set_index("outlet_id").score.clip(0, 100).round(1); fail = d[d.score < 60]; warn = d[(d.score >= 60) & (d.score < 80)]
    hint = ""
    try:
        from rag_engine import answer_with_citation
        ctx, _ = answer_with_citation("food safety corrective action procedure"); hint = (str(ctx)[:140] + "...") if ctx else ""
    except Exception: pass
    acts = [{"type": "corrective_action", "target": r.outlet_id, "detail": f"Audit FAIL {r.score:.0f} ({r.category}): corrective plan within 72 h. {hint}", "severity": "Critical", "sla_h": 72} for r in fail.itertuples()]
    acts += [{"type": "corrective_action", "target": r.outlet_id, "detail": f"Audit WARNING {r.score:.0f} ({r.category}): re-check in 14 days", "severity": "Warning", "sla_h": 336} for r in warn.itertuples()][:15]
    return AgentResult("audit", round(float(per.mean()), 1), risk_label(per.mean()), f"Pass {int((d.score >= 80).sum())}, Warning {len(warn)}, Fail {len(fail)}", {"per_outlet": per.to_dict(), "failed": fail.outlet_id.tolist()}, acts)

POS = {"great", "good", "clean", "tasty", "friendly", "fast", "love", "excellent", "value", "fresh"}; NEG = {"slow", "rude", "dirty", "cold", "bad", "long", "wait", "stale", "expensive", "late"}
ISSUES = {"food quality": ("cold", "stale", "taste", "food"), "staff behavior": ("rude", "staff"), "wait time": ("slow", "wait", "long", "late"), "hygiene": ("dirty", "clean", "hygiene"), "value": ("expensive", "value", "price")}
def feedback_agent(s):
    """Agent 7: VADER when installed, else lexicon + rating blend; keyword issue categorisation."""
    f = Q("SELECT outlet_id, rating, comment FROM feedback ORDER BY rowid DESC LIMIT 6000")
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer; v = SentimentIntensityAnalyzer(); f["sent"] = f.comment.map(lambda t: v.polarity_scores(str(t))["compound"]); method = "VADER"
    except Exception:
        f["sent"] = f.comment.map(lambda t: (sum(w in POS for w in str(t).lower().split()) - sum(w in NEG for w in str(t).lower().split())) / 2); method = "lexicon"
    f["sent"] = (.6 * (f.rating - 3) / 2 + .4 * f.sent.clip(-1, 1)); per = ((f.groupby("outlet_id").sent.mean() + 1) / 2 * 100).round(1)
    cat = {k: int(f.comment.str.lower().str.contains("|".join(v)).sum()) for k, v in ISSUES.items()}
    low = per[per < 45]
    return AgentResult("feedback", round(float(per.mean()), 1), risk_label(per.mean()), f"{len(low)} outlets with poor sentiment ({method}); top issue {max(cat, key=cat.get)}", {"per_outlet": per.to_dict(), "issues": cat},
        [{"type": "service_recovery", "target": o, "detail": f"Sentiment score {v:.0f}/100; main issue: {max(cat, key=cat.get)}", "severity": "Warning"} for o, v in low.items()][:15])

def digest_agent(s):
    """Agent 6: weekly digest + immediate alerts. Runs on the weekly trigger or on request; stores the digest and emails the owner when SMTP is set."""
    r = s["results"]; per = r.get("outlet").findings["per_outlet"] if "outlet" in r else {}
    top = sorted(per, key=per.get, reverse=True)[:3]; bot = sorted(per, key=per.get)[:3]
    txt = f"Weekly digest {now()[:10]}: top {top}, bottom {bot}. " + " ".join(f"{k}: {v.summary}." for k, v in r.items() if k != "digest")
    with _DB_LOCK, conn() as c:
        c.execute("CREATE TABLE IF NOT EXISTS digests(digest_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, body TEXT)"); c.execute("INSERT INTO digests(ts,body) VALUES(?,?)", (now(), txt))
    return AgentResult("digest", 100, "Low", "weekly digest generated", {"digest": txt}, [{"type": "digest_sent", "target": "OWNER", "detail": txt[:300], "severity": "Info"}])


def anomaly_agent(s):
    """Live anomaly detection: IsolationForest over each outlet's recent sales-vs-baseline ratio, volatility and feedback sentiment."""
    from sklearn.ensemble import IsolationForest
    sl = Q("SELECT outlet_id, amount FROM sales WHERE source='sim' ORDER BY sale_id DESC LIMIT 4000")
    if len(sl) < 600: return AgentResult("anomaly", 100, "Low", "collecting live data (needs a few minutes of ticks)", {"flagged": []})
    base = Q("SELECT outlet_id, AVG(amount)/? b FROM sales WHERE source='seed' GROUP BY outlet_id", (TICKS_PER_DAY,)).set_index("outlet_id").b
    g = sl.groupby("outlet_id").amount; d = pd.DataFrame({"ratio": g.agg(lambda x: x.iloc[:5].mean()) / base, "vol": g.std() / g.mean()}).dropna()
    if len(d) < 10: return AgentResult("anomaly", 100, "Low", "waiting for enough outlets with both live and baseline sales", {"flagged": []})
    fb = Q("SELECT outlet_id, AVG(sentiment_score) s FROM (SELECT * FROM feedback ORDER BY rowid DESC LIMIT 1500) GROUP BY outlet_id").set_index("outlet_id").s
    d["sent"] = fb.reindex(d.index).fillna(0); iso = IsolationForest(contamination=.05, random_state=42).fit(d); d["flag"] = iso.predict(d) == -1
    bad = d[d.flag | (d.ratio < .6)]; sc = 100 - len(bad) / max(len(d), 1) * 300
    return AgentResult("anomaly", round(max(sc, 0), 1), risk_label(sc), f"{len(bad)} outlets with anomalous sales/feedback pattern (IsolationForest)", {"flagged": bad.index.tolist(), "detail": bad.round(2).reset_index().to_dict("records")[:30], "all": d.round(3).reset_index().to_dict("records")},
        [{"type": "anomaly_review", "target": o, "detail": f"Unusual pattern: sales at {r.ratio*100:.0f}% of baseline, volatility {r.vol:.2f}, sentiment {r.sent:.2f}", "severity": "Critical" if r.ratio < .6 else "Warning"} for o, r in bad.iterrows()][:20])

AGENTS = {"outlet": outlet_agent, "inventory": inventory_agent, "staff": staff_agent, "marketing": marketing_agent, "audit": audit_agent, "feedback": feedback_agent, "anomaly": anomaly_agent, "digest": digest_agent}
REPLAN = {"inventory": lambda r: ["marketing"] if r.risk != "Low" else [], "audit": lambda r: ["staff", "feedback"] if r.risk == "High" else [], "outlet": lambda r: ["inventory", "staff"] if r.risk != "Low" else []}
KW = {"outlet": ["outlet", "store", "revenue", "sales", "performance", "health"], "inventory": ["stock", "inventory", "reorder", "demand"], "staff": ["staff", "attrition", "workforce", "attendance", "employee"],
      "marketing": ["campaign", "roi", "marketing", "promo", "customer"], "audit": ["audit", "compliance", "violation", "sop"], "feedback": ["feedback", "sentiment", "review", "complaint"], "digest": ["digest", "weekly report"], "anomaly": ["anomal", "unusual", "spike", "fraud", "outlier"]}
ACTION_AGENT.update({"digest_sent": "digest", "anomaly_review": "anomaly", "win_back_campaign": "marketing", "service_recovery": "feedback", "health_alert": "outlet"}); OWNER.update({"anomaly_review": "Regional Manager", "win_back_campaign": "Marketing Manager", "service_recovery": "Store Manager", "health_alert": "Regional Manager", "digest_sent": "Franchise Owner"})
W = {"outlet": .30, "inventory": .20, "staff": .20, "marketing": .15, "audit": .15}     # PRS v3 weighted health score
def plan(req):
    q = (req or "").lower(); picked = [a for a, ks in KW.items() if any(re.search(r"\b" + k, q) for k in ks)]
    base = [a for a in AGENTS if a != "digest"]
    return (picked + (["digest"] if "digest" in picked and len(picked) == 1 else [])) if picked else base

def engine(s):
    r = s["results"]; used = [k for k in W if k in r and "per_outlet" in r[k].findings]
    outlets = set().union(*[set(r[k].findings.get("per_outlet", {})) for k in used]) if used else set()
    health = {o: round(sum(W[k] * r[k].findings["per_outlet"].get(o, r[k].score) for k in used) / sum(W[k] for k in used), 1) for o in outlets}
    score = float(np.mean(list(health.values()))) if health else 0.0
    actions = [a for res in r.values() for a in res.actions]
    for o, h in health.items():
        if h < 60: actions.append({"type": "health_alert", "target": o, "detail": f"Outlet health score {h} below 60 (perf, stock, staff, ROI, audit)", "severity": "Critical"})
    if "inventory" in r and "marketing" in r:
        for o in set(r["inventory"].findings.get("outlets", [])) & set(r["marketing"].findings.get("active_outlets", [])):
            actions.append({"type": "pause_promo", "target": o, "detail": "Stock shortage during active campaign: promo paused", "severity": "Critical"})
    if "staff" in r and "inventory" in r:
        for o in set(r["staff"].findings.get("short_staffed", [])) & set(r["inventory"].findings.get("outlets", [])):
            actions.append({"type": "staffing_shortage", "target": o, "detail": "Short-staffed while stock is low: add cover shift", "severity": "Critical"})
    order = {"Critical": 0, "Warning": 1, "Info": 2}; actions.sort(key=lambda a: order.get(a.get("severity"), 3))
    ev = {"franchise_health": round(score, 1), "outlets": len(health), "outlets_below_60": sum(h < 60 for h in health.values()), "agents": {k: {"score": v.score, "risk": v.risk, "summary": v.summary} for k, v in r.items()},
          "critical": sum(a.get("severity") == "Critical" for a in actions), "total_actions": len(actions)}
    return {"score": round(score, 1), "risk": risk_label(score), "narrative": _narrate(ev, actions), "actions": actions, "agents": ev["agents"], "health": health, "evidence": ev}

def _narrate(ev, actions):
    tmpl = f"Franchise health {ev['franchise_health']}/100 across {ev['outlets']} outlets, {ev['outlets_below_60']} below 60. " + " ".join(f"{k.title()}: {v['summary']}." for k, v in ev["agents"].items()) + f" {ev['critical']} critical of {ev['total_actions']} actions."
    try:
        from llm_engine import is_llm_loaded, generate_text
        if not is_llm_loaded(): return tmpl
        txt = generate_text([{"role": "system", "content": "Write a 3-line executive recommendation for a franchise owner. Use only numbers present in the JSON."}, {"role": "user", "content": json.dumps(ev)}], max_new_tokens=160, temperature=0.1)
        ok = all(n in json.dumps(ev) for n in re.findall(r"\d+(?:\.\d+)?", txt or ""))
        return txt.strip() if txt and ok else tmpl
    except Exception: return tmpl

def apply(a, c):
    if a["type"] == "reorder":
        eta = (datetime.datetime.now() + datetime.timedelta(seconds=SIM_DAY_SEC * a.get("lead", 3))).isoformat(timespec="seconds")
        cur = c.execute("INSERT INTO purchase_orders(outlet_id,sku_name,qty,ts,eta_ts,status) VALUES(?,?,?,?,?,'ordered')", (a["target"], a["sku"], a["qty"], now(), eta))
        return f"PO-{cur.lastrowid} created, ETA {eta[11:19]}"
    if a["type"] == "pause_promo":
        n = c.execute("UPDATE marketing SET end_date=date('now','-1 day') WHERE outlet_id=? AND end_date>=date('now')", (a["target"],)).rowcount
        return f"{n} campaign(s) paused"
    return ""

# ---------------------------------------------------------------- UI hooks
def kpi_extra():
    s = Q("SELECT COALESCE(SUM(amount),0) v FROM sales WHERE source='sim'").v[0]; po = Q("SELECT COUNT(*) n FROM purchase_orders WHERE status='ordered'").n[0]
    return [("Live sales since start", f"Rs {s:,.0f}"), ("Purchase orders in transit", int(po))]


In [ ]:
%%writefile /content/app/entities.py
"""One description of any entity (outlet, staff, quote, shipment, vessel, port): master data, relations, and everything flagged on it. Feeds the drawer, the knowledge graph and the Copilot."""
import json
from blocks import Q, clean

def _tickets(e, n=30):
    return Q("SELECT action_id id, type, severity, status, detail, owner, assignee, due FROM agent_actions WHERE target=? ORDER BY action_id DESC LIMIT ?", (e, n))
def latest_out(agent):
    r = Q("SELECT output FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,))
    try: return json.loads(r.output[0]) if len(r) else {}
    except Exception: return {}
def kind_of(e):
    e = str(e)
    if e.startswith("OUT-"): return "outlet"
    if e.startswith("QTE"): return "quote"
    if e.startswith("SHP"): return "shipment"
    if e.startswith("EMP") or e.startswith("STF"): return "staff"
    return "port"

def describe(kind, dom, e):
    """-> {"kind","info":{label:value},"rel":[(name,kind,tip)], "tickets": DataFrame}"""
    k = kind_of(e); info, rel = {}, []
    if kind == "Franchise" and k == "outlet":
        o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
        if len(o):
            r = o.iloc[0]; info = {"Outlet": r.outlet_name, "Location": r.location, "Revenue": f"{r.revenue:,.0f}", "Customer satisfaction": round(float(r.customer_satisfaction), 2), "Headcount": int(r.staff_headcount)}
            for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
                v = latest_out(a).get("per_outlet", {}).get(e)
                if v is not None: rel.append((f"{a} agent: {v:.0f}", "agent", f"{a} score for {e}"))
            for s in Q("SELECT staff_id, predicted_attrition_prob p FROM staff WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.staff_id, "staff", f"attrition {s.p:.0%}"))
            for s in Q("SELECT sku_name, stockout_risk_prob p FROM inventory WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.sku_name, "sku", f"stock-out risk {s.p:.0%}"))
            for s in Q("SELECT campaign_name, channel FROM marketing WHERE outlet_id=? AND end_date>=date('now')", (e,)).itertuples(): rel.append((s.campaign_name, "campaign", f"live via {s.channel}"))
    elif kind == "Freight" and k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if len(q):
            r = q.iloc[0]; info = {"Quote": e, "Price": f"${r.final_price:,.0f}", "Margin": f"{r.margin_pct}%", "Status": r.status}
            for n, tip in ((r.shipment_id, "shipment"), (r.customer_id, "customer"), (r.carrier, "carrier")): rel.append((n, tip, tip))
    elif kind == "Freight" and k == "shipment":
        s = Q("SELECT * FROM shipments WHERE shipment_id=?", (e,))
        if len(s):
            r = s.iloc[0]; info = {"Shipment": e, "Route": f"{r.origin_port} to {r.dest_port}", "Carrier": r.carrier, "Status": r.status, "Delay risk": r.predicted_delay_risk}
            for n, tip in ((r.origin_port, "port"), (r.dest_port, "port"), (r.carrier, "carrier")): rel.append((n, tip, tip))
            v = Q("SELECT lat, lon, route_name, progress FROM vessels WHERE shipment_id=?", (e,))
            if len(v): info["Vessel"] = f"{v.lat[0]:.1f}, {v.lon[0]:.1f} · {v.route_name[0]} · {v.progress[0]:.0%}"
    elif kind == "Freight":
        p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
        if len(p):
            r = p.iloc[0]; info = {"Port": e, "Country": r.country, "Congestion index": round(float(r.congestion_index), 2), "Dwell days": r.avg_dwell_days}
            for s in Q("SELECT shipment_id FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 6", (e, e)).itertuples(): rel.append((s.shipment_id, "shipment", "via port"))
    return {"kind": k, "info": {a: clean(b) for a, b in info.items()}, "rel": rel, "tickets": _tickets(e)}

def facts(kind, dom, text):
    import re
    out = []
    for e in dict.fromkeys(m.upper() for m in re.findall(r"\b(OUT-\d+|QTE-L\d+|SHP-[A-Z0-9-]+)\b", text, re.I)):
        d = describe(kind, dom, e); t = d["tickets"]; t = t[t.status.isin(["open", "proposed"])].head(8)
        if d["info"] or len(t): out.append(f"{e}: " + "; ".join(f"{k}={v}" for k, v in d["info"].items()) + (f". Related: {', '.join(n for n, _, _ in d['rel'][:8])}." if d["rel"] else "") + (" Open tickets: " + " | ".join(f"{r.type} ({r.severity}): {r.detail}" for r in t.itertuples()) if len(t) else " No open tickets."))
    return out


In [ ]:
%%writefile /content/app/gpu.py
"""Minimal stub (no language model in this version). Kept so hardening.py can report health."""
def status(): return {"cuda": False, "gpu": None, "models": {}, "stats": {}}


In [ ]:
%%writefile /content/app/hardening.py
"""Production hardening for the FastAPI app: request ids + JSON access log, security headers (CSP, frame, permissions), per-IP rate limits on auth routes,
a global JSON error handler that never leaks tracebacks, and Prometheus-style metrics. Install with hardening.install(app)."""
import os, time, json, uuid, threading, logging, traceback, collections
from logging.handlers import RotatingFileHandler
from fastapi import Request
from fastapi.responses import JSONResponse, PlainTextResponse
import config

_LOG = logging.getLogger("access"); _LOG.setLevel(logging.INFO); _LOG.propagate = False
try:
    _h = RotatingFileHandler(os.path.join(config.DATA_DIR, "access.log"), maxBytes=2_000_000, backupCount=3); _h.setFormatter(logging.Formatter("%(message)s")); _LOG.addHandler(_h)
except Exception: pass
_ERR = logging.getLogger("errors"); _ERR.propagate = False
try:
    _e = RotatingFileHandler(os.path.join(config.DATA_DIR, "errors.log"), maxBytes=1_000_000, backupCount=2); _e.setFormatter(logging.Formatter("%(asctime)s %(message)s")); _ERR.addHandler(_e)
except Exception: pass

M = {"requests": collections.Counter(), "status": collections.Counter(), "lat_sum": collections.Counter(), "lat_n": collections.Counter(), "errors": 0, "limited": 0, "started": time.time()}
_L = threading.Lock(); _BUCKET = {}
CSP = "default-src 'self'; img-src 'self' data: blob:; style-src 'self' 'unsafe-inline'; script-src 'self' 'unsafe-inline'; connect-src 'self'; font-src 'self' data:; frame-ancestors 'none'; base-uri 'self'; form-action 'self'"
# path prefix -> (max requests, per seconds). Everything else: 900 per minute per IP.
LIMITS = {"/api/login": (10, 60), "/api/register": (6, 600), "/api/otp": (5, 300), "/api/reset": (8, 600), "/api/profile/password": (8, 600), "/api/copilot": (40, 60), "/api/translate": (60, 60)}
def _ip(r):
    f = r.headers.get("x-forwarded-for"); return (f.split(",")[0].strip() if f else (r.client.host if r.client else "?"))
def _allow(ip, path):
    key, (n, per) = next(((p, v) for p, v in LIMITS.items() if path.startswith(p)), ("*", (900, 60))); k = (ip, key); now = time.time()
    with _L:
        q = _BUCKET.setdefault(k, collections.deque())
        while q and now - q[0] > per: q.popleft()
        if len(q) >= n: return False, int(per - (now - q[0])) + 1
        q.append(now)
        if len(_BUCKET) > 20000: _BUCKET.clear()
    return True, 0
def _route(path):
    parts = path.split("/")
    return "/".join(parts[:3]) if path.startswith("/api/") else "static"

def install(app, kind="app"):
    @app.middleware("http")
    async def guard(request: Request, call_next):
        rid = request.headers.get("x-request-id") or uuid.uuid4().hex[:12]; t0 = time.time(); path = request.url.path
        if os.environ.get("RATE_LIMIT_OFF") != "1" and (path.startswith("/api/") and request.method != "GET" or path.startswith("/api/login")):
            ok, wait = _allow(_ip(request), path)
            if not ok:
                M["limited"] += 1; return JSONResponse({"ok": False, "msg": f"Too many requests. Try again in {wait}s.", "request_id": rid}, status_code=429, headers={"Retry-After": str(wait), "X-Request-ID": rid})
        try: r = await call_next(request)
        except Exception as e:
            M["errors"] += 1; _ERR.error("%s %s %s\n%s", rid, request.method, path, traceback.format_exc())
            return JSONResponse({"ok": False, "msg": "Something went wrong on the server. Quote this id to the admin: " + rid, "request_id": rid}, status_code=500, headers={"X-Request-ID": rid})
        ms = (time.time() - t0) * 1000; rt = _route(path)
        with _L: M["requests"][rt] += 1; M["status"][r.status_code // 100] += 1; M["lat_sum"][rt] += ms; M["lat_n"][rt] += 1
        h = r.headers; h["X-Request-ID"] = rid; h["X-Frame-Options"] = "DENY"; h["Content-Security-Policy"] = CSP; h["Permissions-Policy"] = "microphone=(self), camera=(), geolocation=()"; h["Cross-Origin-Opener-Policy"] = "same-origin"
        if request.headers.get("x-forwarded-proto") == "https": h["Strict-Transport-Security"] = "max-age=31536000"
        if path.startswith("/api/") and path not in ("/api/stream", "/api/pulse"): _LOG.info(json.dumps({"t": round(t0), "id": rid, "ip": _ip(request), "m": request.method, "p": path, "s": r.status_code, "ms": round(ms)}))
        return r

    @app.get("/api/ready")
    def ready(): return {"ok": True, "kind": kind, "uptime_s": round(time.time() - M["started"])}

    @app.get("/api/metrics")
    def metrics(request: Request):
        """Prometheus text format. Set METRICS_TOKEN to require ?token=..., otherwise only logged-in admins (cookie) may read it."""
        tok = os.environ.get("METRICS_TOKEN")
        if tok and request.query_params.get("token") != tok: return PlainTextResponse("forbidden", status_code=403)
        if not tok:
            try:
                u = {"role": "Admin"}
                if not u or u.get("role") != "Admin": return PlainTextResponse("forbidden", status_code=403)
            except Exception: return PlainTextResponse("forbidden", status_code=403)
        L = ["# TYPE app_requests_total counter"] + [f'app_requests_total{{route="{k}"}} {v}' for k, v in M["requests"].items()] + ["# TYPE app_responses_total counter"] + [f'app_responses_total{{class="{k}xx"}} {v}' for k, v in M["status"].items()]
        L += ["# TYPE app_latency_ms_avg gauge"] + [f'app_latency_ms_avg{{route="{k}"}} {M["lat_sum"][k] / max(1, M["lat_n"][k]):.1f}' for k in M["lat_n"]] + [f"app_errors_total {M['errors']}", f"app_rate_limited_total {M['limited']}", f"app_uptime_seconds {time.time() - M['started']:.0f}"]
        try:
            import live_engine as LE; c = LE.conn(); L += [f"app_open_tickets {c.execute('SELECT COUNT(*) FROM agent_actions WHERE status=?', ('open',)).fetchone()[0]}"]
        except Exception: pass
        try:
            import gpu; g = gpu.status(); L += [f'app_model_ready{{model="{k}"}} {1 if v["state"] == "ready" else 0}' for k, v in g["models"].items()]
        except Exception: pass
        return PlainTextResponse("\n".join(L) + "\n")


In [ ]:
%%writefile /content/app/intent_router.py
import re as _re_live
import pandas as pd
import re, math
from db import get_conn

def df_to_markdown_safe(df):
    """Safely converts a pandas DataFrame to Markdown format."""
    if df is None or df.empty:
        return ""
    try:
        return df.to_markdown(index=False)
    except Exception:
        headers = list(df.columns)
        lines = ["| " + " | ".join([str(h) for h in headers]) + " |"]
        lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
        for _, row in df.iterrows():
            vals = [str(v) if v is not None else "" for v in row.values]
            lines.append("| " + " | ".join(vals) + " |")
        return "\n".join(lines)

# ================================================================================================================================
# COPILOT ENHANCEMENT: intent detection + routing for the AI Copilot.
#   classify_detailed()  domain + confidence + question type, tolerant of spelling mistakes and alternative wording
#   respond()            one entry point used by server.py: safety -> follow-up -> whitelisted SQL -> analyst -> generic SQL -> polite fallback
# ================================================================================================================================
import time, difflib
import qa as QA, analyst as AN

# Meaningful phrases / whole words only (tiny substrings such as "ad" or "item" must never hijack unrelated questions).
INTENT_MAP = {
    "staff": ["staff", "employee", "employees", "attrition", "salary", "workforce", "headcount", "hiring", "hire", "overtime", "attendance", "absent", "role", "roles", "job satisfaction", "resign"],
    "outlet": ["outlet", "outlets", "store", "branch", "revenue", "margin", "sales", "profit", "csat", "customer satisfaction", "city", "tier", "loss making", "net margin"],
    "inventory": ["inventory", "stock", "stockout", "sku", "reorder", "demand", "supply", "safety stock", "supplier", "lead time", "purchase order", "purchase orders", "low stock", "category"],
    "marketing": ["marketing", "campaign", "campaigns", "roi", "conversion", "promotion", "budget", "channel", "advertising", "advertisement", "ads", "spend"],
    "audit": ["audit", "audits", "compliance", "violation", "violations", "inspection", "policy", "standard", "fssai", "food safety", "hygiene"],
    "feedback": ["feedback", "review", "reviews", "sentiment", "rating", "ratings", "complaint", "complaints", "customer review", "negative feedback"],
    "ops": ["risk", "flagged", "health", "critical", "ticket", "tickets", "alert", "alerts", "attention", "anomaly", "severity", "urgent", "action plan", "overview", "summary", "snapshot", "kpi"],
}
_VOCAB = sorted({w for v in INTENT_MAP.values() for k in v for w in k.split()})

def _contains_term(q, term):
    return bool(re.search(r"(?<!\w)" + re.escape(term) + r"(?!\w)", q))

def _prep(query):
    """Spelling-corrected, synonym-normalised, lower-case text (falls back to plain lower-casing if the data layer is unavailable)."""
    try: return QA.normalize(query)[0].lower()
    except Exception: return str(query).lower().strip()

QTYPES = [("compare", r"\b(compare|versus|vs\.?|difference|against)\b"), ("explain", r"\b(why|reason|explain|flagged|what is wrong|what's wrong)\b"), ("threshold", r"\b(below|under|above|over|more than|less than|at least|at most|exceeding)\s*\d"),
          ("count", r"\b(how many|number of|count)\b"), ("average", r"\b(average|mean|avg)\b"), ("total", r"\b(total|sum|overall|combined)\b"), ("trend", r"\b(trend|daily|weekly|last \d+ days|this week|last week|last month|over time)\b"),
          ("rank", r"\b(highest|lowest|most|least|top|bottom|best|worst|rank\w*)\b"), ("status", r"\b(status|overview|summary|snapshot|how is|how are|doing|profile|tell me about)\b"), ("list", r"\b(show|list|give|which|what|who)\b")]

def classify_detailed(query):
    """-> {"domain", "confidence", "scores", "qtype", "text"}. Whole-word and phrase matches (phrases count double), plus a fuzzy pass for words that
    are one or two letters off, so 'revnue', 'inventry' or 'complaintz' still land in the right domain."""
    q = _prep(query); toks = re.findall(r"[a-z]+", q); scores = {}
    for intent, kws in INTENT_MAP.items():
        sc = 0
        for kw in kws:
            if _contains_term(q, kw): sc += 2 if " " in kw else 1
        for t in toks:                                        # fuzzy: a near-miss of a single-word keyword counts half
            if len(t) >= 5 and not any(t == k for k in kws):
                m = difflib.get_close_matches(t, [k for k in kws if " " not in k], n=1, cutoff=0.84)
                if m: sc += 0.5
        scores[intent] = sc
    best = max(scores, key=scores.get); top = scores[best]; rest = sorted(scores.values(), reverse=True)[1]
    conf = round(top / (top + rest + 1e-9), 2) if top > 0 else 0.0
    qt = next((n for n, rx in QTYPES if re.search(rx, q)), "other")
    return {"domain": best if top > 0 else "general", "confidence": conf, "scores": scores, "qtype": qt, "text": q}

def classify_intent(query):
    """Primary domain (kept for older callers)."""
    return classify_detailed(query)["domain"]

# ------------------------------------------------------------------ safety and scope checks
WRITE_RE = re.compile(r"\b(delete|drop|truncate|insert into|alter table|erase|wipe|remove all)\b|\b(update|change|modify|edit|set|overwrite|increase|decrease|raise|cut|reduce)\b.{0,30}\b(salary|salaries|revenue|stock|inventory|data|records?|tables?|ratings?|scores?|passwords?|roles?|budget|price)\b|;\s*(select|delete|drop|update|insert)", re.I)
CRED_RE = re.compile(r"\b(passwords?|password hash|secrets?|tokens?|jwt|otp|api keys?|credentials?|login details)\b", re.I)
OFFTOPIC_RE = re.compile(r"\b(weather|joke|poem|song|movie|recipe|cricket|football|bitcoin|crypto|horoscope|prime minister|president of|capital of|who (is|was) the ceo|translate|write (me )?(a|an|some) (email|essay|code|story|letter)|python|javascript|hello world|meaning of life)\b", re.I)
GREET_RE = re.compile(r"^\W*(hi+|hello+|hey+|hola|namaste|good (morning|afternoon|evening)|yo|howdy|greetings)\b[\w\s!.,]*$", re.I)
THANKS_RE = re.compile(r"^\W*(thanks?|thank you|thx|ok(ay)?|cool|great|nice|got it|awesome|perfect)\b[\w\s!.,]*$", re.I)
HELP_RE = re.compile(r"\b(help|what can you (do|answer)|what (do|can) i ask|who are you|what are you|capabilities|examples?|what questions)\b", re.I)

AREAS = [("outlets and revenue", "revenue by city"), ("staff and attrition", "which staff are at high risk of leaving"), ("inventory and orders", "show pending purchase orders"),
         ("marketing campaigns", "which marketing channel works best"), ("customer feedback", "show recent negative feedback"), ("audits and compliance", "which audits failed"), ("tickets and alerts", "show the open critical tickets")]

def _help(role, scope):
    ex = QA.allowed_examples(role)
    pol = QA.policy_for(role)
    pick = [ex[i] for i in range(0, len(ex), max(1, len(ex) // 6))][:6] if ex else []
    return pick

def sanitize_state(st):
    """The follow-up state round-trips through the browser, so it is never trusted: keep only known keys with safe types and ranges."""
    if not isinstance(st, dict): return {}
    out = {}
    try:
        if st.get("key") in QA.WL_BY_KEY: out["key"] = st["key"]
        else: out["key"] = None
        p = st.get("params") if isinstance(st.get("params"), dict) else {}; cp = {}
        for k in ("city", "tier"):
            if isinstance(p.get(k), str) and 0 < len(p[k]) <= 40: cp[k] = p[k]
        if isinstance(p.get("outlet"), str) and re.fullmatch(r"OUT-\d{1,4}", p["outlet"]): cp["outlet"] = p["outlet"]
        if isinstance(p.get("n"), (int, float)) and not isinstance(p.get("n"), bool): cp["n"] = max(1, min(50, int(p["n"])))
        if isinstance(p.get("days"), (int, float)) and not isinstance(p.get("days"), bool): cp["days"] = max(1, min(90, int(p["days"])))
        if isinstance(p.get("thr"), (int, float)) and not isinstance(p.get("thr"), bool): cp["thr"] = float(p["thr"])
        if p.get("cmp") in ("<", ">"): cp["cmp"] = p["cmp"]
        if p.get("dir") in ("high", "low"): cp["dir"] = p["dir"]
        for k in ("worst", "best"):
            if p.get(k) is True: cp[k] = True
        out["params"] = cp
        if isinstance(st.get("limit"), (int, float)) and not isinstance(st.get("limit"), bool): out["limit"] = max(1, min(50, int(st["limit"])))
        if st.get("dir") in ("ASC", "DESC"): out["dir"] = st["dir"]
        if isinstance(st.get("entity"), str) and re.fullmatch(r"OUT-\d{1,4}", st["entity"]): out["entity"] = st["entity"]
        if isinstance(st.get("q"), str): out["q"] = st["q"][:300]
        if isinstance(st.get("title"), str): out["title"] = st["title"][:80]
    except Exception: return {}
    return out

def _entity_of(res):
    """An outlet id to remember for 'it' / 'that outlet' follow-ups."""
    if res.get("entity"): return res["entity"]
    p = res.get("params") or {}
    if p.get("outlet"): return p["outlet"]
    df = res.get("df")
    if df is not None and len(df) and "outlet_id" in df.columns: return str(df.iloc[0]["outlet_id"])
    m = re.search(r"\bOUT-\d+\b", str(res.get("answer", "")))
    return m.group(0) if m else None

# ------------------------------------------------------------------ follow-ups
MORE_RE = re.compile(r"^(?:ok(?:ay)?[, ]*)?(show |give |list )?(me )?(some )?(more|next|another \d+|the rest|further)\b|\bshow more\b|\bmore (results|rows|please)\b", re.I)
REV_RE = re.compile(r"\b(reverse|opposite|flip|the other way|invert|ascending instead|descending instead|worst instead|best instead|and the (worst|best|lowest|highest|bottom|top))\b", re.I)
ALL_RE = re.compile(r"\b(all cities|every city|all tiers|everywhere|overall|chain[- ]?wide|remove (the )?filter|without (the )?filter|clear filter|all outlets|whole chain)\b", re.I)
FU_START = re.compile(r"^(and|also|now|then|ok|okay|so|but|what about|how about|what if|same (for|but|with|in)|only|just|instead|for|in|at|show|give|list|top|bottom|more|next)\b", re.I)
PRONOUN_RE = re.compile(r"\b(that outlet|this outlet|that store|this store|that one|it|its|there|them)\b", re.I)

def _followup(qn, q, state):
    """Elliptical follow-ups ("what about Mumbai", "top 3", "show more", "reverse it", "why is it flagged") -> ("rerun", key, params) | ("rewrite", new_question) | None."""
    if not state: return None
    words = qn.split(); p = QA.extract_params(qn); ent = state.get("entity")
    full_new_q = QA.match_whitelist(qn)[1] > 0 and len(words) > 5
    if ent and PRONOUN_RE.search(qn) and not (MORE_RE.search(qn) or REV_RE.search(qn) or ALL_RE.search(qn)) and not re.search(r"\bOUT-\d+\b", q, re.I) and len(words) <= 10 and not QA.match_whitelist(qn)[1]:
        return ("rewrite", PRONOUN_RE.sub(ent, q, count=1))
    meaningful = any(k in p for k in ("city", "tier", "outlet", "n", "thr", "days")) or (p.get("dir") and len(words) <= 4)
    cue = bool(MORE_RE.search(qn) or REV_RE.search(qn) or ALL_RE.search(qn) or FU_START.match(qn))
    if full_new_q or len(words) > 9 or not (meaningful or cue): return None
    if not (meaningful or MORE_RE.search(qn) or REV_RE.search(qn) or ALL_RE.search(qn)): return None
    key = state.get("key")
    if key and key in QA.WL_BY_KEY:
        np_ = dict(state.get("params") or {})
        for k in ("city", "tier", "outlet", "n", "thr", "cmp", "days"):
            if k in p: np_[k] = p[k]
        if ALL_RE.search(qn):
            for k in ("city", "tier", "outlet"): np_.pop(k, None)
        if MORE_RE.search(qn): np_["n"] = min(50, int(state.get("limit") or np_.get("n") or 10) + 10)
        if REV_RE.search(qn):
            cur = state.get("dir") or "DESC"; np_["dir"] = "low" if cur == "DESC" else "high"; np_.pop("worst", None); np_.pop("best", None)
        elif "dir" in p: np_["dir"] = p["dir"]; np_.pop("worst", None); np_.pop("best", None)
        if "worst" in p: np_["worst"] = True; np_.pop("best", None)
        if "best" in p: np_["best"] = True; np_.pop("worst", None)
        return ("rerun", key, np_)
    base = state.get("q")
    if base and meaningful:
        extra = []
        if p.get("city"): extra.append("in " + p["city"])
        if p.get("tier"): extra.append("tier " + p["tier"])
        if p.get("outlet"): extra.append("for " + p["outlet"])
        if p.get("n"): base = re.sub(r"\b(top|bottom|first)\s+\d+\b", "", base).strip() + f" top {p['n']}"
        return ("rewrite", (base + " " + " ".join(extra)).strip())
    return None

# ------------------------------------------------------------------ the entry point
def _fmt_table(df, limit=50): return QA.md(df.head(limit)) if df is not None and len(df) else ""
def _bul(items): return "\n".join("- " + x for x in items)

def respond(kind, q, role="Admin", scope=None, state=None, dec=None, dom=None):
    """Answer one Copilot question. Returns a dict the server streams to the browser:
    text, sql, rows, status (ok|empty|denied|unsupported|refused|greeting|error), intent{domain,confidence,qtype,key,followup},
    suggestions (valid questions to try), followups (what to ask next), corrections, state (send back with the next question), validated."""
    t0 = time.time(); q = str(q or "").strip()[:800]; role = QA.canon_role(role); pol = QA.policy_for(role)
    state = sanitize_state(state)
    out = {"text": "", "sql": "", "rows": None, "status": "ok", "intent": {}, "suggestions": [], "followups": [], "corrections": [], "state": state or {}, "validated": False, "role": role, "normalized": ""}
    def done(**kw):
        out.update(kw); out["ms"] = round((time.time() - t0) * 1000); return out
    if not q: return done(status="unsupported", text="Please type a question about your franchise data, for example:\n" + _bul(QA.suggest_questions("", role)), suggestions=QA.suggest_questions("", role))
    if GREET_RE.match(q) or THANKS_RE.match(q) or (HELP_RE.search(q) and len(q.split()) <= 8 and not QA.match_whitelist(_prep(q))[1]):
        ex = _help(role, scope); about = pol["about"]
        if THANKS_RE.match(q) and not GREET_RE.match(q): txt = "You're welcome! Ask me anything else about your data."
        else: txt = ("Hello! " if GREET_RE.match(q) else "") + f"I'm your franchise data copilot. Every answer comes from a read-only database query, and I show you the SQL and the number of rows it returned. For your role ({role}) I can cover: {about}."
        if ex and not (THANKS_RE.match(q) and not GREET_RE.match(q)): txt += "\n\nTry one of these:\n" + _bul(ex)
        return done(status="greeting", text=txt, suggestions=ex, intent={"domain": "chat", "qtype": "greeting"})
    # ---- safety: this assistant only ever reads
    if CRED_RE.search(q): return done(status="refused", text="I can't help with passwords, tokens or other credentials, and I never have access to them. I can help with business questions instead:\n" + _bul(QA.suggest_questions(q, role)), suggestions=QA.suggest_questions(q, role), intent={"domain": "security", "qtype": "refused"})
    if WRITE_RE.search(q): return done(status="refused", text="I'm read-only, so I can't change, add or delete any data. I can look things up for you though. For example:\n" + _bul(QA.suggest_questions(q, role)), suggestions=QA.suggest_questions(q, role), intent={"domain": "write", "qtype": "refused"})
    qn, fixes = QA.normalize(q); out["normalized"] = qn; out["corrections"] = [[a, b] for a, b, k in fixes if k == "spell"]
    if kind != "Franchise": # the SQL-enhancement catalogue is for the franchise database; other deployments keep the original engine
        r = AN.answer(kind, q, scope, dec, dom) or QA.answer(kind, q, scope, None)
        return done(text=(r["answer"] + ("\n\n" + _fmt_table(r.get("df")) if r and r.get("df") is not None else "")) if r else "I could not compute an exact answer to that.", sql=(r or {}).get("sql", ""), rows=(r or {}).get("rows"), status="ok" if r else "unsupported")
    info = classify_detailed(q); intent = {"domain": info["domain"], "confidence": info["confidence"], "qtype": info["qtype"]}; out["intent"] = intent
    if pol["scoped"] and not scope:
        return done(status="denied", text=f"Your {role} account is not linked to an outlet yet, so I can't show outlet data. Please ask an administrator to assign your outlet.", intent=intent)
    if scope:                                  # a single-outlet role may only ask about its own outlet
        other = [i for i in AN._ids_in(q) if i != scope]
        if other: return done(status="denied", text=f"{other[0]} is outside your outlet scope ({scope}), so I can't show it. Here is what you can ask about {scope}:\n" + _bul(QA.suggest_questions(qn, role)), suggestions=QA.suggest_questions(qn, role), intent=intent)
    note = ""; res = None; fu = None
    # ---- follow-ups
    try: fu = _followup(qn, q, state)
    except Exception: fu = None
    if fu and fu[0] == "rerun":
        e = QA.WL_BY_KEY[fu[1]]; res = QA.run_entry(e, fu[2], role, scope); note = f"*Follow-up on “{e['title']}”.*\n\n"; intent["followup"] = True; intent["key"] = e["key"]
    elif fu and fu[0] == "rewrite":
        q = fu[1]; qn, fixes = QA.normalize(q); note = f"*Follow-up, understood as: “{q}”.*\n\n"; intent["followup"] = True
    # ---- off-topic
    if res is None and OFFTOPIC_RE.search(q) and not QA.match_whitelist(qn)[1]:
        return done(status="unsupported", text="I'm sorry, that's outside what I can help with. I only answer questions about your franchise data (outlets, staff, inventory, marketing, feedback and audits). Here are three you could try:\n" + _bul(QA.suggest_questions(qn, role)), suggestions=QA.suggest_questions(qn, role), intent=intent)
    # ---- whitelisted SQL
    denied_reason = None
    if res is None:
        e, sc = QA.match_whitelist(qn)
        if e:
            res = QA.run_entry(e, QA.extract_params(qn), role, scope); intent["key"] = e["key"]
    # ---- analyst (multi-query handlers + original agentic handlers), guarded by the role's restrictions
    if res is None:
        why = QA.role_blocks(qn, role)
        if why:
            sug = QA.suggest_questions(qn, role); return done(status="denied", text=f"I'm sorry, {why[0].lower() + why[1:]}. Here are three questions you can ask instead:\n" + _bul(sug), suggestions=sug, intent=intent)
        try: res = AN.smart_answer(kind, q, role, scope, dec, dom, qn)
        except Exception as ex:
            import sys; print("copilot analyst failed:", repr(ex), file=sys.stderr); res = None
        if res and res.get("answer"): intent["key"] = intent.get("key") or "analyst"
        else: res = None
    # ---- generic SQL builder over whitelisted tables
    if res is None and re.search(r"\b(at risk|highest risk|riskiest|health|flagged|needs? attention)\b", qn) and not re.search(r"\b(attrition|stockout|stock out|delay|credit)\b", qn):
        sug = QA.suggest_questions(qn, role)       # risk / health scores come from the agents; never guess them from another table
        return done(status="empty", text="I don't have agent risk scores to rank outlets with yet. They appear once the live agents have completed a cycle, so please try again in a minute. In the meantime you could ask:\n" + _bul(sug), suggestions=sug, intent=intent)
    if res is None:
        allowed = None if pol["tables"] == QA.ALL else set(pol["tables"])
        try: res = QA.answer(kind, qn, scope, mask=("salary" in pol["deny"] or "personal" in pol["deny"]), allowed=allowed)
        except Exception as ex:
            import sys; print("copilot sql failed:", repr(ex), file=sys.stderr); res = None
        if res: intent["key"] = intent.get("key") or "generic_sql"
    # ---- polite fallback
    if not res or not res.get("answer"):
        sug = QA.suggest_questions(qn, role)
        return done(status="unsupported", text="I'm sorry, I can't answer that one yet. I can only answer questions that map to your franchise database (outlets, staff, inventory, marketing, feedback, audits and tickets). Here are three questions you could try:\n" + _bul(sug), suggestions=sug, intent=intent)
    status = res.get("status") or ("denied" if res.get("denied") else "empty" if (res.get("empty") or res.get("rows") == 0 and res.get("df") is None and "SELECT" in str(res.get("sql", "")).upper()) else "ok")
    df = res.get("df"); text = note + res["answer"]; tbl = _fmt_table(df); extra = res.get("extra")
    if tbl: text += "\n\n" + tbl
    if extra: text += "\n\n" + extra
    rows = res.get("rows"); rows = (int(len(df)) if df is not None else 0) if rows is None else rows
    if status == "ok" and df is not None and res.get("limit") and rows >= res["limit"]: text += f"\n\n*Showing the first {rows} rows (limit {res['limit']}). Say “show more” to see more.*"
    sug = []; fups = []
    if status in ("empty", "denied"):
        sug = QA.suggest_questions(qn, role); text += "\n\nYou could try:\n" + _bul(sug)
    else:
        fups = [x for x in (res.get("more") or []) if _allowed_q(x, role)][:2]
        city = df.iloc[0].get("city") if df is not None and len(df) and "city" in df.columns else None
        p = res.get("params") or {}
        if res.get("key") in QA.WL_BY_KEY:
            if city and not p.get("city") and "city" in QA.WL_BY_KEY[res["key"]]["filters"]: fups.append(f"What about {city}?")
            fups.append("Show more" if rows >= (res.get("limit") or 10) else "Reverse the order")
        elif _entity_of(res): fups.append(f"Why is {_entity_of(res)} flagged?")
        fups = list(dict.fromkeys(fups))[:3]
    sql = res.get("sql") or ""; ent = _entity_of(res) or (state or {}).get("entity")
    is_sql = sql.strip().upper().startswith(("SELECT", "WITH"))
    if not is_sql and df is None: rows = None
    st = {"key": res.get("key") if res.get("key") in QA.WL_BY_KEY else None, "params": res.get("params") or {}, "limit": res.get("limit"), "dir": None, "entity": ent, "q": qn, "title": res.get("title")}
    if st["key"]:
        try: st["dir"] = "ASC" if re.search(r"ORDER BY [^;]*\bASC\b", sql.split("GROUP BY")[-1], re.I) and not re.search(r"ORDER BY [^;]*\bDESC\b", sql.split("GROUP BY")[-1], re.I) else "DESC"
        except Exception: st["dir"] = "DESC"
    return done(text=text, sql=sql, rows=rows, status=status, suggestions=sug, followups=fups, state=st, validated=bool(sql and sql.strip().upper().startswith(("SELECT", "WITH"))), intent=intent)

def _allowed_q(question, role):
    qn = _prep(question); e, s = QA.match_whitelist(qn)
    return True if not e else QA.entry_access(e, role)[0]

def handle_franchise_intent(query):
    """Return only data for the most relevant franchise domain."""
    q_low = query.lower()
    intent = classify_intent(query)

    try:
        with get_conn() as conn:
            if intent == "outlet":
                total = conn.execute("SELECT COUNT(*) FROM outlets;").fetchone()[0]
                revenue = conn.execute("SELECT SUM(revenue) FROM outlets;").fetchone()[0] or 0
                csat = conn.execute("SELECT AVG(customer_satisfaction) FROM outlets;").fetchone()[0] or 0

                if any(_contains_term(q_low, k) for k in ["margin", "profit"]):
                    sql = """SELECT outlet_id, outlet_name, location, tier,
                             ROUND(revenue, 0) AS revenue_rs,
                             ROUND(operating_costs, 0) AS costs_rs,
                             ROUND(((revenue - operating_costs)/NULLIF(revenue,0))*100, 2) AS net_margin_pct,
                             customer_satisfaction
                             FROM outlets ORDER BY net_margin_pct DESC LIMIT 10;"""
                else:
                    sql = """SELECT outlet_id, outlet_name, location, tier,
                             ROUND(revenue, 0) AS revenue_rs,
                             ROUND(((revenue - operating_costs)/NULLIF(revenue,0))*100, 2) AS net_margin_pct,
                             customer_satisfaction
                             FROM outlets ORDER BY revenue_rs DESC LIMIT 10;"""
                df = pd.read_sql(sql, conn)
                return f"### Franchise Outlet Data\nActive outlets: **{total}** | Total revenue: **Rs. {revenue:,.0f}** | Average CSAT: **{csat:.2f}/5**\n\n{df_to_markdown_safe(df)}", "Outlets DB"

            if intent == "staff":
                total = conn.execute("SELECT COUNT(*) FROM staff;").fetchone()[0]
                high_risk = conn.execute("SELECT COUNT(*) FROM staff WHERE predicted_attrition_prob > 0.6;").fetchone()[0]
                df = pd.read_sql("""SELECT outlet_id, role, COUNT(*) AS staff_count,
                                    ROUND(AVG(salary), 0) AS avg_salary_rs,
                                    ROUND(AVG(predicted_attrition_prob), 2) AS avg_attrition_risk
                                    FROM staff GROUP BY outlet_id, role
                                    ORDER BY avg_attrition_risk DESC LIMIT 10;""", conn)
                return f"### Workforce Data\nTotal staff: **{total}** | High attrition risk (>60%): **{high_risk}**\n\n{df_to_markdown_safe(df)}", "Staff DB"

            if intent == "inventory":
                total_skus = conn.execute("SELECT COUNT(*) FROM inventory;").fetchone()[0]
                risk_items = conn.execute("SELECT COUNT(*) FROM inventory WHERE stockout_risk_prob > 0.7;").fetchone()[0]
                df = pd.read_sql("""SELECT outlet_id, sku_name, category, current_stock,
                                    reorder_threshold, stockout_risk_prob
                                    FROM inventory ORDER BY stockout_risk_prob DESC LIMIT 12;""", conn)
                return f"### Inventory Data\nTracked SKUs: **{total_skus}** | SKUs above 70% stockout risk: **{risk_items}**\n\n{df_to_markdown_safe(df)}", "Inventory DB"

            if intent == "marketing":
                avg_roi = conn.execute("SELECT AVG(actual_roi) FROM marketing;").fetchone()[0] or 0
                df = pd.read_sql("""SELECT channel, COUNT(*) AS campaigns,
                                    ROUND(AVG(actual_roi), 2) AS avg_roi,
                                    SUM(conversions) AS conversions
                                    FROM marketing GROUP BY channel ORDER BY avg_roi DESC;""", conn)
                return f"### Marketing Data\nAverage campaign ROI: **{avg_roi:.2f}x**\n\n{df_to_markdown_safe(df)}", "Marketing DB"

            if intent == "feedback":
                avg_rating = conn.execute("SELECT AVG(rating) FROM feedback;").fetchone()[0] or 0
                df = pd.read_sql("""SELECT outlet_id, rating, comment, sentiment_score, date
                                    FROM feedback ORDER BY sentiment_score ASC LIMIT 10;""", conn)
                return f"### Customer Feedback Data\nAverage rating: **{avg_rating:.2f}/5**\n\n{df_to_markdown_safe(df)}", "Feedback DB"

            if intent == "audit":
                avg_score = conn.execute("SELECT AVG(score) FROM audits;").fetchone()[0] or 0
                open_items = conn.execute("SELECT COUNT(*) FROM audits WHERE status != 'Pass';").fetchone()[0]
                df = pd.read_sql("""SELECT outlet_id, audit_date, score, violations, category, status
                                    FROM audits ORDER BY score ASC LIMIT 10;""", conn)
                return f"### Audit & Compliance Data\nAverage score: **{avg_score:.1f}/100** | Non-pass audits: **{open_items}**\n\n{df_to_markdown_safe(df)}", "Audits DB"

    except Exception:
        return None

    return None

def handle_port_intent(query):
    try:
        with get_conn() as conn:
            tot_ports = conn.execute("SELECT COUNT(*) FROM ports;").fetchone()[0]
            df_ports = pd.read_sql("""SELECT port_name AS 'Port Name', country AS 'Country',
                                      region AS 'Region',
                                      congestion_index AS 'Congestion (1-5)',
                                      avg_dwell_days AS 'Avg Dwell Days'
                                      FROM ports ORDER BY congestion_index ASC LIMIT 10;""", conn)
        return f"### Global Ports Data ({tot_ports} monitored)\n\n{df_to_markdown_safe(df_ports)}", f"Ports Ledger DB ({tot_ports} Hubs)"
    except Exception:
        return None, None


_AGENTIC_Q = _re_live.compile(r"(need[s]? attention|what should i do|action plan|prioriti[sz]e|overall (status|health)|run (the )?agents?|full analysis|full status|why is .{0,40}(low|down|failing|underperform|risky)|what is going wrong)")
def _agentic_query(query):
    """Open-ended 'what should I do / what needs attention' questions run the whole agent graph instead of a single table lookup."""
    if not _AGENTIC_Q.search(query.lower()): return None
    try:
        from live_engine import run_cycle
        import domain_franchise as _dom
        dec = run_cycle(_dom, "copilot", query)
        top = "\n".join(f"- **{a['severity']}** · {a['type']} · {a['target']}: {a['detail']}" for a in dec["actions"][:8])
        return f"### Agent orchestration result\n\n{dec['narrative']}\n\n**Top actions**\n{top}", "Agent Orchestrator (live)"
    except Exception:
        return None

def _run_centralized_orig(query):
    """Retrieve one relevant evidence set. Do not mix unrelated data sources."""
    q_low = query.lower()

    franchise = handle_franchise_intent(query)
    if franchise:
        return franchise

    if any(_contains_term(q_low, k) for k in ["port", "ports", "harbor", "terminal", "congestion"]):
        return handle_port_intent(query)

    if any(_contains_term(q_low, k) for k in ["shipment", "shipments", "carrier"]):
        try:
            with get_conn() as conn:
                tot_shipments = conn.execute("SELECT COUNT(*) FROM shipments;").fetchone()[0]
                df_ship = pd.read_sql("""SELECT shipment_id, origin_port, dest_port, carrier,
                                         status, predicted_delay_risk
                                         FROM shipments ORDER BY predicted_delay_risk DESC LIMIT 10;""", conn)
            return f"### Active Shipments ({tot_shipments})\n\n{df_to_markdown_safe(df_ship)}", "Shipments Ledger DB"
        except Exception:
            pass

    # Only invoke RAG for questions that clearly look like document/SOP/policy questions.
    rag_terms = ["sop", "document", "manual", "policy", "procedure", "fssai", "hygiene", "food safety", "guideline"]
    if any(_contains_term(q_low, k) for k in rag_terms):
        try:
            from rag_engine import answer_with_citation
            ctx, src = answer_with_citation(query)
            if ctx and str(ctx).strip():
                return ctx, src
        except Exception:
            pass

    return None, None

def run_grounded_query(query):
    return run_centralized_brain_query(query)

def text_to_sql(query):
    return run_centralized_brain_query(query)

def run_centralized_brain_query(query):
    _ag = _agentic_query(query)
    return _ag if _ag else _run_centralized_orig(query)


In [ ]:
%%writefile /content/app/live_engine.py
"""Shared live agent runtime (FranchiseOps AI + FreightQuote AI).
Domain modules (domain_franchise.py / domain_freight.py) plug in agents, a simulator and action handlers."""
import os, re, json, time, sqlite3, threading, datetime, smtplib, ssl
from dataclasses import dataclass, field
from email.message import EmailMessage
import requests
from config import DB_PATH

_DB_LOCK = threading.RLock()
_TL = threading.local()
def conn():
    """One SQLite connection per thread: nested helpers (emit/set_state) share the caller's transaction instead of deadlocking."""
    c = getattr(_TL, "c", None)
    if c is None:
        c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("PRAGMA busy_timeout=30000;"); _TL.c = c
    return c
def now():
    return datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=5, minutes=30))).isoformat(timespec="seconds")

@dataclass
class AgentResult:
    agent: str
    score: float = 100.0
    risk: str = "Low"
    summary: str = ""
    findings: dict = field(default_factory=dict)
    actions: list = field(default_factory=list)   # {"type","target","detail","severity","owner"?,"sla_h"?}
    confidence: float = 0.8

def risk_label(score): return "High" if score < 60 else "Medium" if score < 80 else "Low"

def init_live_tables():
    with _DB_LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS agent_runs(run_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, agent TEXT, risk TEXT, score REAL, latency_ms INT, summary TEXT, output TEXT);
        CREATE TABLE IF NOT EXISTS agent_actions(action_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, type TEXT, target TEXT, detail TEXT, severity TEXT, owner TEXT, status TEXT DEFAULT 'open', due TEXT, resolved_ts TEXT, source_cycle TEXT, note TEXT);
        CREATE TABLE IF NOT EXISTS live_events(event_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, etype TEXT, entity TEXT, payload TEXT);
        CREATE TABLE IF NOT EXISTS live_state(k TEXT PRIMARY KEY, v TEXT, ts TEXT);
        CREATE TABLE IF NOT EXISTS decisions(decision_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, score REAL, risk TEXT, narrative TEXT, evidence TEXT);
        CREATE TABLE IF NOT EXISTS audit_log(audit_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, actor TEXT, event TEXT, target TEXT, detail TEXT);
        CREATE TABLE IF NOT EXISTS ticket_comments(comment_id INTEGER PRIMARY KEY AUTOINCREMENT, action_id INT, ts TEXT, author TEXT, body TEXT);
        CREATE INDEX IF NOT EXISTS ix_act_status ON agent_actions(status, type, target);
        CREATE INDEX IF NOT EXISTS ix_runs_agent ON agent_runs(agent, run_id);
        CREATE INDEX IF NOT EXISTS ix_runs_cycle ON agent_runs(cycle_id);
        CREATE INDEX IF NOT EXISTS ix_cm_act ON ticket_comments(action_id);
        CREATE INDEX IF NOT EXISTS ix_audit_ts ON audit_log(audit_id);
        """)
        for col in ("assignee TEXT", "approval TEXT"):
            try: c.execute(f"ALTER TABLE agent_actions ADD COLUMN {col}")
            except sqlite3.OperationalError: pass
        for t, cols in (("sales", "outlet_id, ts"), ("attendance", "outlet_id, ts"), ("feedback", "outlet_id, date")):
            try: c.execute(f"CREATE INDEX IF NOT EXISTS ix_{t}_o ON {t}({cols})")
            except sqlite3.OperationalError: pass

def audit(actor, event, target="", detail=""):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO audit_log(ts,actor,event,target,detail) VALUES(?,?,?,?,?)", (now(), str(actor), event, str(target), str(detail)[:500]))

RETAIN = {"sales": ("ts", 6000), "attendance": ("ts", 3000), "feedback": ("date", 2000), "agent_runs": ("ts", 4000), "decisions": ("ts", 800), "live_events": ("ts", 2000), "audit_log": ("ts", 20000)}
def maintenance():
    """Keep the live DB bounded: prune to the newest N rows per table (never touches seed/master data)."""
    n = 0
    with _DB_LOCK, conn() as c:
        for t, (col, keep) in RETAIN.items():
            try:
                pk = c.execute(f"PRAGMA table_info({t})").fetchall()
                if not pk: continue
                if t in ("sales", "attendance"):
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 4: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep * 2,)).rowcount
                else:
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 2: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep,)).rowcount
            except sqlite3.OperationalError: pass
    return n

def set_state(k, v):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT OR REPLACE INTO live_state(k,v,ts) VALUES(?,?,?)", (k, json.dumps(v, default=str), now()))
def get_state(k, default=None):
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT v FROM live_state WHERE k=?", (k,)).fetchone()
    return json.loads(r[0]) if r else default
def emit(etype, entity, payload=None):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO live_events(ts,etype,entity,payload) VALUES(?,?,?,?)", (now(), etype, str(entity), json.dumps(payload or {}, default=str)))

# ---------------- notifications (real channels, all optional via env) ----------------
_SENT = {}
def _nk(s): return re.sub(r"\d+(?:\.\d+)?", "#", str(s))
def notify(dom, target, severity, category, message):
    with _DB_LOCK, conn() as c:
        c.execute(f"INSERT INTO alerts ({dom.ENTITY_COL}, severity, category, message, date) VALUES (?,?,?,?,?)",
                  (target, severity, category, message, datetime.datetime.now().strftime("%Y-%m-%d %H:%M")))
    if severity not in ("Critical", "High"): return
    key = (target, category, _nk(message))
    if time.time() - _SENT.get(key, 0) < 3600: return          # no alert spam
    _SENT[key] = time.time()
    text = f"[{dom.NAME}] {severity} · {target} · {category}: {message}"
    for var in ("SLACK_WEBHOOK_URL", "TEAMS_WEBHOOK_URL"):
        if os.environ.get(var):
            try: requests.post(os.environ[var], json={"text": text}, timeout=5)
            except Exception: pass
    em, pw, to = os.environ.get("EMAIL_ID"), (os.environ.get("EMAIL_APP_PASSWORD") or os.environ.get("EMAIL_PASSWORD") or "").replace(" ", ""), os.environ.get("ALERT_EMAIL_TO")
    if em and pw and to:
        try:
            m = EmailMessage(); m["Subject"] = text[:90]; m["From"] = em; m["To"] = to; m.set_content(text)
            with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=15) as s:
                s.login(em, pw); s.send_message(m)
        except Exception: pass
    def _mail_subscribers():
        try:
            import mailer
            if not mailer.configured(): return
            with conn() as cc: rows = [r[0] for r in cc.execute("SELECT email FROM user_prefs WHERE email_alerts=1")]
            for e in rows: mailer.send(e, text[:90], text)
        except Exception: pass
    threading.Thread(target=_mail_subscribers, daemon=True).start()
    sid, tok, frm, tph = (os.environ.get(k) for k in ("TWILIO_SID", "TWILIO_TOKEN", "TWILIO_FROM", "ALERT_SMS_TO"))
    if sid and tok and frm and tph and getattr(dom, "SMS", False):
        try: requests.post(f"https://api.twilio.com/2010-04-01/Accounts/{sid}/Messages.json", auth=(sid, tok),
                           data={"From": frm, "To": tph, "Body": text[:300]}, timeout=8)
        except Exception: pass

# ---------------- action layer ----------------
SLA_H = {"Critical": 24, "Warning": 72, "Info": 168}
def policy(dom):
    """Autonomy policy per action type: 'auto' (agent executes) or 'approval' (human approves first). Admin-editable."""
    return {}      # no approval queue in this version: agents act on their own
def set_policy(t, mode, actor="admin"):
    p = get_state("policy", {}) or {}; p[t] = mode; set_state("policy", p); audit(actor, "policy", t, mode)
# ---------------------------------------------------------------- multi-level approval chains
# Each ticket that needs a human gets an ordered chain of steps (table approval_steps). Every step names a role; the next step opens only after the previous one is approved.
# Four-eyes rule: nobody can approve two steps of the same ticket, and nobody can approve a request they raised. Any step rejection rejects the whole ticket.
LADDER = ["Operations Manager", "Franchise Owner", "Admin"]
def _role_ok(role, need): return role == "Admin" or role == need or (need == "Operations Manager" and role == "Franchise Owner")
def chain_for(severity, typ, levels=None):
    cfg = get_state("approval_chain", {}) or {}
    if levels: return LADDER[:1] + (["Admin"] if int(levels) >= 2 else [])
    if typ in cfg and cfg[typ]: return list(cfg[typ])
    return ["Operations Manager", "Admin"] if severity == "Critical" else ["Operations Manager"]
def _steps_ddl(c): c.execute("CREATE TABLE IF NOT EXISTS approval_steps(action_id INT, step INT, role TEXT, status TEXT DEFAULT 'pending', by TEXT, ts TEXT, note TEXT, PRIMARY KEY(action_id, step))")
def _ensure_chain(c, aid, severity, typ):
    _steps_ddl(c)
    if not c.execute("SELECT 1 FROM approval_steps WHERE action_id=?", (aid,)).fetchone():
        for i, r in enumerate(chain_for(severity, typ)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
def chains(ids):
    """{action_id: [{step, role, status, by}]} for the ticket list (creates the default chain for tickets that have none yet)."""
    out = {}
    if not ids: return out
    with _DB_LOCK, conn() as c:
        _steps_ddl(c)
        for aid, sev, typ in c.execute("SELECT action_id, severity, type FROM agent_actions WHERE action_id IN (%s) AND (status='proposed' OR type='quote_approval')" % ",".join("?" * len(ids)), list(ids)).fetchall(): _ensure_chain(c, aid, sev, typ)
        for r in c.execute("SELECT action_id, step, role, status, by FROM approval_steps WHERE action_id IN (%s) ORDER BY action_id, step" % ",".join("?" * len(ids)), list(ids)).fetchall(): out.setdefault(r[0], []).append({"step": r[1], "role": r[2], "status": r[3], "by": r[4] or ""})
    return out
def next_role(chain):
    for s in chain or []:
        if s["status"] == "pending": return s["role"]
    return None
def advance(aid, actor, role, approve=True, note=""):
    """Record one decision. Returns ('final'|'partial'|'rejected'|'denied'|'missing', message). Does not execute anything."""
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT severity, type, note FROM agent_actions WHERE action_id=? AND (status='proposed' OR type='quote_approval')", (aid,)).fetchone()
        if not r: return "missing", "nothing to approve on this ticket"
        _ensure_chain(c, aid, r[0], r[1]); steps = c.execute("SELECT step, role, status, by FROM approval_steps WHERE action_id=? ORDER BY step", (aid,)).fetchall()
        cur = next((s for s in steps if s[2] == "pending"), None)
        if not cur: return "missing", "this ticket has no pending approval step"
        if not _role_ok(role, cur[1]): return "denied", f"step {cur[0]} needs a {cur[1]} (you are {role})"
        if approve:
            if actor in [s[3] for s in steps if s[3]]: return "denied", "four-eyes rule: you already approved an earlier step of this ticket"
            if f"requested by {actor}" in (r[2] or ""): return "denied", "four-eyes rule: you cannot approve your own request"
        c.execute("UPDATE approval_steps SET status=?, by=?, ts=?, note=? WHERE action_id=? AND step=?", ("approved" if approve else "rejected", actor, now(), note, aid, cur[0]))
        left = [s for s in steps if s[2] == "pending" and s[0] != cur[0]]
        if not approve: return "rejected", f"rejected at step {cur[0]} ({cur[1]})"
        if left:
            c.execute("UPDATE agent_actions SET note=COALESCE(note,'')||? WHERE action_id=?", (f" step {cur[0]}/{len(steps)} approved by {actor};", aid)); audit(actor, "approve_step", aid, f"step {cur[0]}/{len(steps)} {note}".strip())
            return "partial", f"step {cur[0]} of {len(steps)} approved; waiting for {left[0][1]}"
    return "final", "all steps approved"
def request_approval(dom, typ, target, detail, severity, requester, levels=None):
    """Anyone with access can ask for a decision: creates a proposed ticket with its own approval chain."""
    a = {"type": "manual_request", "target": target, "detail": detail, "request": typ}
    with _DB_LOCK, conn() as c:
        cur = c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,status,note,approval) VALUES(?,?,?,?,?,?,'proposed',?,?)", (now(), "manual_request", target, f"{typ}: {detail}", severity, requester, f"requested by {requester};", json.dumps(a)))
        aid = cur.lastrowid; _steps_ddl(c)
        for i, r in enumerate(chain_for(severity, "manual_request", levels)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
    audit(requester, "request_approval", f"{typ}:{target}", detail[:120]); return aid
def approve_action(dom, aid, actor="user", approve=True, note="", role="Admin"):
    """Human-in-the-loop: runs the approval chain; the action executes only when the last step is approved. Returns (state, message)."""
    st, msg = advance(aid, actor, role, approve, note)
    if st in ("denied", "missing", "partial"): return st, msg
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT type,target,detail,severity,approval FROM agent_actions WHERE action_id=?", (aid,)).fetchone()
        if st == "final":
            a = json.loads(r[4] or "{}") or {"type": r[0], "target": r[1], "detail": r[2]}
            try: res = dom.apply(a, c) or ""
            except Exception as ex: res = f"apply failed: {ex}"
            if r[0] == "manual_request": c.execute("UPDATE agent_actions SET status='resolved', resolved_ts=?, note=COALESCE(note,'')||?, approval='approved' WHERE action_id=?", (now(), f" approved by {actor}; request granted", aid))
            else: c.execute("UPDATE agent_actions SET status='open', note=COALESCE(note,'')||?, approval=? WHERE action_id=?", (f" approved by {actor}; {res}", "approved", aid))
        else:
            c.execute("UPDATE agent_actions SET status='rejected', resolved_ts=?, note=COALESCE(note,'')||?, approval='rejected' WHERE action_id=?", (now(), f" rejected by {actor} {note}", aid))
    audit(actor, "approve" if approve else "reject", f"{r[0]}:{r[1]}", note); return st, msg
def assign_action(aid, assignee, actor="user"):
    with _DB_LOCK, conn() as c: c.execute("UPDATE agent_actions SET assignee=? WHERE action_id=?", (assignee, aid))
    audit(actor, "assign", aid, assignee)
def comment_action(aid, author, body):
    with _DB_LOCK, conn() as c: c.execute("INSERT INTO ticket_comments(action_id,ts,author,body) VALUES(?,?,?,?)", (now(), author, body) if False else (aid, now(), author, body))
    audit(author, "comment", aid, body[:80])
def act(dom, decision, cid, results):
    raised = set(); sent = 0; extra = 0; pol = policy(dom)
    for a in decision.get("actions", []):
        key = (a["type"], str(a["target"]), _nk(a["detail"])); raised.add(key)
        sev = a.get("severity", "Warning")
        due = (datetime.datetime.now() + datetime.timedelta(hours=a.get("sla_h", SLA_H.get(sev, 72)))).isoformat(timespec="seconds")
        with _DB_LOCK, conn() as c:
            hit = next((r[0] for r in c.execute("SELECT action_id, detail FROM agent_actions WHERE type=? AND target=? AND status IN ('open','proposed')", key[:2]).fetchall() if _nk(r[1].replace("OVERDUE: ", "")) == key[2]), None)
            if hit:
                c.execute("UPDATE agent_actions SET detail=CASE WHEN detail LIKE 'OVERDUE:%' THEN 'OVERDUE: '||? ELSE ? END WHERE action_id=?", (a["detail"], a["detail"], hit)); continue
            note = ""; status = "open"; appr = None
            if pol.get(a["type"]) == "approval": status = "proposed"; note = "awaiting human approval"; appr = json.dumps(a, default=str)
            else:
                try: note = dom.apply(a, c) or ""
                except Exception as ex: note = f"apply failed: {ex}"
            c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,due,source_cycle,note,status,approval) VALUES(?,?,?,?,?,?,?,?,?,?,?)",
                      (now(), a["type"], str(a["target"]), a["detail"], sev, a.get("owner") or dom.OWNER.get(a["type"], "Operations"), due, cid, note, status, appr))
        if sev == "Critical":
            if sent < 8: notify(dom, a["target"], "Critical", a["type"], a["detail"]); sent += 1
            else: extra += 1
    if extra: notify(dom, "MULTIPLE", "Critical", "Summary", f"{extra} more critical actions raised this cycle; see Command Center")
    with _DB_LOCK, conn() as c:      # auto-close: the owning agent re-ran and no longer raises it
        for aid, t, tg, d in c.execute("SELECT action_id,type,target,detail FROM agent_actions WHERE status IN ('open','proposed')").fetchall():
            if dom.ACTION_AGENT.get(t) in results and (t, tg, _nk(d.replace("OVERDUE: ", ""))) not in raised:
                c.execute("UPDATE agent_actions SET status='auto-closed', resolved_ts=? WHERE action_id=?", (now(), aid))

def escalate_overdue(dom):
    n = 0
    with _DB_LOCK, conn() as c:
        rows = c.execute("SELECT action_id,target,detail,type FROM agent_actions WHERE status='open' AND due<? AND detail NOT LIKE 'OVERDUE:%'", (now(),)).fetchall()
        for aid, tg, d, t in rows:
            nd = (datetime.datetime.now() + datetime.timedelta(hours=24)).isoformat(timespec="seconds")
            c.execute("UPDATE agent_actions SET severity='Critical', owner=?, detail=?, due=? WHERE action_id=?", (dom.ESCALATE_TO, "OVERDUE: " + d, nd, aid)); n += 1
    for aid, tg, d, t in rows: notify(dom, tg, "Critical", "Escalation", "OVERDUE: " + d)
    return n

def resolve_action(aid, status="resolved", note="", actor="user"):
    audit(actor, status, aid, note)
    with _DB_LOCK, conn() as c:
        c.execute("UPDATE agent_actions SET status=?, resolved_ts=?, note=COALESCE(note,'')||? WHERE action_id=?", (status, now(), (" " + note) if note else "", aid))

# ---------------- orchestration (LangGraph when installed, plain loop otherwise) ----------------
def _log_run(cid, trig, res, ms):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO agent_runs(ts,cycle_id,trigger_src,agent,risk,score,latency_ms,summary,output) VALUES(?,?,?,?,?,?,?,?,?)",
                  (now(), cid, trig, res.agent, res.risk, res.score, ms, res.summary, json.dumps(res.findings, default=str)[:30000]))

def make_nodes(dom):
    def planner(s): return {"queue": dom.plan(s["request"]), "results": {}}
    def agent(s):
        name = s["queue"][0]; t = time.time()
        try: res = dom.AGENTS[name](s)
        except Exception as e: res = AgentResult(name, 50.0, "Medium", f"agent error: {e}")
        _log_run(s["cycle_id"], s["trigger"], res, int((time.time() - t) * 1000))
        new = [n for n in dom.REPLAN.get(name, lambda r: [])(res) if n not in s["results"] and n not in s["queue"]]
        if new: _log_run(s["cycle_id"], s["trigger"], AgentResult("re-plan", 100, "Low", f"{name} risk {res.risk}: adding {', '.join(new)}"), 0)
        return {"results": {**s["results"], name: res}, "queue": s["queue"][1:] + new}
    def engine(s):
        d = dom.engine(s)
        with _DB_LOCK, conn() as c:
            c.execute("INSERT INTO decisions(ts,cycle_id,trigger_src,score,risk,narrative,evidence) VALUES(?,?,?,?,?,?,?)",
                      (now(), s["cycle_id"], s["trigger"], d["score"], d["risk"], d["narrative"], json.dumps({k: v for k, v in d.items() if k not in ("narrative", "actions")}, default=str)))
        return {"decision": d}
    def act_node(s): act(dom, s["decision"], s["cycle_id"], s["results"]); return {}
    return planner, agent, engine, act_node

def _compile(dom):
    from typing import TypedDict
    from langgraph.graph import StateGraph, END
    class S(TypedDict, total=False):
        cycle_id: str; trigger: str; request: str; queue: list; results: dict; decision: dict
    planner, agent, engine, act_node = make_nodes(dom)
    g = StateGraph(S)
    for n, f in (("planner", planner), ("agent", agent), ("engine", engine), ("act", act_node)): g.add_node(n, f)
    g.set_entry_point("planner"); g.add_edge("planner", "agent")
    g.add_conditional_edges("agent", lambda s: "agent" if s["queue"] else "engine")
    g.add_edge("engine", "act"); g.add_edge("act", END)
    return g.compile()

_GRAPHS = {}
def run_cycle(dom, trigger="manual", request=""):
    init_live_tables()
    cid = datetime.datetime.now().strftime("%m%d-%H%M%S") + "-" + trigger[:4]
    s = {"cycle_id": cid, "trigger": trigger, "request": request, "queue": [], "results": {}, "decision": {}}
    try:
        if dom.NAME not in _GRAPHS: _GRAPHS[dom.NAME] = _compile(dom)
        out = _GRAPHS[dom.NAME].invoke(s)
    except ImportError:
        planner, agent, engine, act_node = make_nodes(dom)
        s.update(planner(s))
        while s["queue"]: s.update(agent(s))
        s.update(engine(s)); act_node(s); out = s
    return out["decision"]

# ---------------- background runtime ----------------
class Runtime:
    def __init__(self, dom, tick=5, cycle=60, min_gap=12):
        self.dom, self.tick, self.cycle, self.min_gap = dom, tick, cycle, min_gap
        self.stop = threading.Event(); self.ticks = 0; self.cycles = 0; self.last_cycle = 0; self.last_err = ""
        self.th = threading.Thread(target=self._loop, daemon=True); self.th.start()
        if hasattr(dom, "feed"): self.fth = threading.Thread(target=self._feed_loop, daemon=True); self.fth.start()
    def _feed_loop(self):          # network polling lives here so slow/offline APIs never stall the tick
        while not self.stop.is_set():
            try: self.dom.feed()
            except Exception as e: self.last_err = f"feed: {e}"
            self.stop.wait(int(os.environ.get("FEED_SEC", 60)))
    def _loop(self):
        init_live_tables(); set_state("boot", {"stage": "bootstrap", "ts": now()})
        try: self.dom.bootstrap()
        except Exception as e: self.last_err = f"bootstrap: {e}"
        set_state("boot", {"stage": "first_cycle", "ts": now()})
        while not self.stop.is_set():
            try:
                reasons = self.dom.tick() or []; self.ticks += 1
                if self.ticks % 120 == 0: maintenance()
                reasons = reasons + (self.dom.process() or [])
                due = time.time() - self.last_cycle
                if (reasons and due >= self.min_gap) or due >= self.cycle:
                    run_cycle(self.dom, "event" if reasons else "sched", ""); escalate_overdue(self.dom)
                    self.cycles += 1; self.last_cycle = time.time()
                    set_state("runtime", {"ticks": self.ticks, "cycles": self.cycles, "last_cycle": now(), "reasons": reasons[:3], "err": self.last_err})
                    if self.cycles == 1: set_state("boot", {"stage": "ready", "ts": now()})
                elif self.ticks % 6 == 0: set_state("runtime", {**(get_state("runtime", {}) or {}), "ticks": self.ticks})
            except Exception as e: self.last_err = str(e)
            self.stop.wait(self.tick)

_RT = None; _RT_LOCK = threading.Lock()
def start_runtime(dom):
    global _RT
    with _RT_LOCK:
        if _RT is None:
            _RT = Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    return _RT


In [ ]:
%%writefile /content/app/localuser.py
"""Bridge localuser module for backwards compatibility."""
try:
    from auth import user_of, USER, hash_password, check_password
except Exception:
    USER = {"email": "admin@infosys.com", "username": "Admin", "role": "Admin"}
    def user_of(request):
        return USER


In [ ]:
%%writefile /content/app/mailer.py
"""One place that sends e-mail (OTP codes, reports, alerts). Gmail SMTP with an app password: set EMAIL_ID + EMAIL_PASSWORD (Colab secrets or env)."""
import os, ssl, smtplib
from email.message import EmailMessage
def creds(): return os.environ.get("EMAIL_ID") or "", (os.environ.get("EMAIL_PASSWORD") or os.environ.get("EMAIL_APP_PASSWORD") or "").replace(" ", "")
def configured(): a, b = creds(); return bool(a and b)
def send(to, subject, body, attachments=(), html=None):
    """attachments: [(filename, bytes, mimetype)]. Returns (ok, message)."""
    em, pw = creds()
    if not (em and pw): return False, "E-mail is not configured (set EMAIL_ID and EMAIL_PASSWORD)."
    try:
        m = EmailMessage(); m["Subject"] = subject; m["From"] = em; m["To"] = to; m.set_content(body)
        if html: m.add_alternative(html, subtype="html")
        for fn, data, mt in attachments: m.add_attachment(data, maintype=mt.split("/")[0], subtype=mt.split("/")[1], filename=fn)
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=20) as s: s.login(em, pw); s.send_message(m)
        return True, f"Sent to {to}"
    except Exception as e: return False, f"Could not send: {e}"


In [ ]:
%%writefile /content/app/qa.py
"""Exact-answer engine for the Copilot. Turns a plain-English question into a read-only, parameterised SQL query over the live tables
(counts, top/bottom N, averages, totals, group-by, filters, ID lookups) and returns the real rows. The LLM only explains afterwards;
it never has to guess a number. Table and column names come from a fixed whitelist, values are always bound parameters."""
import re, time, sqlite3, threading, difflib
from blocks import Q as _Q_unchecked

TABLES = {
    "Franchise": {
        "outlets": (["outlet", "store", "branch", "franchise"], "outlet_id", ["outlet_id", "outlet_name", "location", "tier", "revenue", "customer_satisfaction", "staff_headcount"], "revenue"),
        "staff": (["staff", "employee", "worker", "workforce", "people"], "staff_id", ["staff_id", "name", "role", "outlet_id", "salary", "overtime_hrs", "predicted_attrition_prob"], "predicted_attrition_prob"),
        "inventory": (["inventory", "sku", "item", "product", "stock level"], "record_id", ["outlet_id", "sku_name", "category", "current_stock", "reorder_threshold", "stockout_risk_prob"], "stockout_risk_prob"),
        "marketing": (["campaign", "marketing", "promotion", "advert"], "campaign_id", ["campaign_id", "campaign_name", "outlet_id", "channel", "budget", "actual_roi", "conversions"], "actual_roi"),
        "feedback": (["feedback", "review", "complaint", "comment"], "feedback_id", ["feedback_id", "outlet_id", "rating", "sentiment_score", "date", "comment"], "rating"),
        "audits": (["audit", "inspection", "compliance"], "audit_id", ["audit_id", "outlet_id", "audit_date", "score", "violations", "category", "status"], "score"),
        "sales": (["sale", "transaction", "takings"], "sale_id", ["outlet_id", "ts", "amount", "tx_count", "source"], "amount"),
        "purchase_orders": (["purchase order", "reorder order", "po"], "po_id", ["po_id", "outlet_id", "sku_name", "qty", "eta_ts", "status"], "qty"),
        "attendance": (["attendance", "absent", "absence", "present"], "att_id", ["staff_id", "outlet_id", "ts", "status"], None),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
    "Freight": {
        "shipments": (["shipment", "cargo", "container", "consignment"], "shipment_id", ["shipment_id", "origin_port", "dest_port", "carrier", "status", "eta", "weight_kg", "predicted_delay_risk"], "predicted_delay_risk"),
        "freight_quotes": (["quote", "quotation", "price", "pricing"], "quote_id", ["quote_id", "shipment_id", "customer_id", "carrier", "final_price", "margin_pct", "status"], "final_price"),
        "ports": (["port", "terminal", "harbour", "harbor"], "port_id", ["port_name", "country", "region", "congestion_index", "avg_dwell_days"], "congestion_index"),
        "carriers": (["carrier", "shipping line", "liner", "line"], "carrier_id", ["name", "rating", "on_time_pct", "avg_cost_index", "risk_level"], "on_time_pct"),
        "customers": (["customer", "client", "shipper"], "customer_id", ["customer_id", "name", "industry", "priority_tier", "credit_risk"], "credit_risk"),
        "weather_risks": (["weather", "storm", "wave", "wind", "forecast"], "port_name", ["port_name", "current_severity", "forecast", "wind_speed", "wave_height"], "wave_height"),
        "vessels": (["vessel", "ship ", "ais", "voyage"], "shipment_id", ["shipment_id", "route_name", "lat", "lon", "progress", "speed_kn"], "progress"),
        "quote_requests": (["request", "enquiry", "enquiries", "inquiry"], "req_id", ["req_id", "ts", "origin", "dest", "cargo", "weight_t", "status", "quote_id"], None),
        "customs_rules": (["customs", "duty", "hs code", "tariff"], "chapter", ["chapter", "goods", "duty_pct", "docs", "restricted"], "duty_pct"),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
}
CSYN = {"revenue": ["revenue", "turnover"], "customer_satisfaction": ["satisfaction", "csat"], "staff_headcount": ["headcount"], "salary": ["salary", "pay", "wage"], "overtime_hrs": ["overtime"],
        "job_satisfaction": ["job satisfaction"], "tenure_years": ["tenure"], "predicted_attrition_prob": ["attrition", "churn", "likely to leave", "quit"], "current_stock": ["current stock", "units in stock"],
        "reorder_threshold": ["reorder level", "threshold"], "weekly_demand": ["demand"], "lead_time_days": ["lead time"], "stockout_risk_prob": ["stockout", "stock-out", "stock out", "shortage", "run out"],
        "actual_roi": ["roi", "return on"], "sentiment_score": ["sentiment"], "score": ["audit score", "score"], "violations": ["violation"], "amount": ["amount", "sales value"], "qty": ["quantity", "qty"],
        "congestion_index": ["congestion", "congested", "crowded"], "avg_dwell_days": ["dwell"], "on_time_pct": ["on-time", "on time", "punctual", "punctuality", "reliab"], "avg_cost_index": ["cost index"],
        "predicted_delay_risk": ["delay risk", "delay", "delayed"], "weight_kg": ["weight"], "final_price": ["final price", "price", "quote value", "worth"], "margin_pct": ["margin"], "base_cost": ["base cost"],
        "insurance": ["insurance"], "customs_fee": ["customs fee"], "fuel_surcharge": ["fuel"], "credit_risk": ["credit risk", "credit"], "duty_pct": ["duty"], "wave_height": ["wave"], "wind_speed": ["wind"],
        "speed_kn": ["speed"], "progress": ["progress", "completed"], "operating_costs": ["operating cost", "costs"], "rating": ["rating", "rated", "stars"], "budget": ["budget", "spend"], "reach": ["reach"], "conversions": ["conversion"]}
BAD = {"predicted_attrition_prob", "stockout_risk_prob", "violations", "congestion_index", "avg_dwell_days", "predicted_delay_risk", "credit_risk", "operating_costs", "overtime_hrs", "lead_time_days", "avg_cost_index", "wave_height", "wind_speed"}
PROB = {"predicted_attrition_prob", "stockout_risk_prob", "predicted_delay_risk", "credit_risk", "progress", "sentiment_score"}
LABEL = {"outlets": "outlet_name", "staff": "name", "carriers": "name", "customers": "name", "ports": "port_name", "inventory": "sku_name", "marketing": "campaign_name"}
GENERIC = {"port", "line", "terminal", "cargo", "ocean", "shipping", "the", "of"}

def _w(q, term): return re.search(r"(?<![a-z0-9])" + re.escape(term.strip()) + r"(?:s|es)?(?![a-z0-9])", q) is not None
def _num(s, unit):
    v = float(s.replace(",", "")); u = (unit or "").lower()
    return v * {"k": 1e3, "m": 1e6, "lakh": 1e5, "lakhs": 1e5, "cr": 1e7, "crore": 1e7, "crores": 1e7}.get(u, 1)
_NUM = {}
def _cols(t):
    df = Q(f"SELECT * FROM {t} LIMIT 20"); _NUM[t] = {c for c in df.columns if str(df[c].dtype)[:3] in ("flo", "int") }
    return list(df.columns)
PL = {"inventory": "inventory records", "staff": "staff", "marketing": "campaigns", "feedback": "feedback entries", "audits": "audits", "sales": "sales records", "attendance": "attendance records", "freight_quotes": "quotes", "ports": "ports", "carriers": "carriers", "customs_rules": "customs rules", "weather_risks": "port weather entries", "agent_actions": "tickets", "purchase_orders": "purchase orders", "quote_requests": "quote requests", "outlets": "outlets", "shipments": "shipments", "customers": "customers", "vessels": "vessels"}
def pct(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns and c in PROB: df[c] = df[c].map(lambda x: f"{x:.0%}" if isinstance(x, (int, float)) and abs(x) <= 1 else x)
    return df
def _alias(v):
    v = str(v); out = {v.lower()}; base = re.sub(r"\(.*?\)", "", v).strip().lower(); out.add(base)
    out.update(m.lower() for m in re.findall(r"\((.*?)\)", v))
    core = " ".join(w for w in base.split() if w not in GENERIC)
    if len(core) >= 3: out.add(core)
    return {a for a in out if len(a) >= 3}
def _fmt(v):
    if isinstance(v, float): return f"{v:,.2f}" if abs(v) < 1000 else f"{v:,.0f}"
    if isinstance(v, int): return f"{v:,}" if abs(v) >= 10000 else str(v)
    return str(v)
def md(df):
    if df is None or not len(df): return ""
    h = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return h + "\n".join("| " + " | ".join(_fmt(x).replace("|", "/").replace("\n", " ")[:90] for x in r) + " |" for r in df.itertuples(index=False))

def lookup(kind, q, scope=None):
    """ID in the question -> the exact row(s) plus its open tickets."""
    ids = list(dict.fromkeys(re.findall(r"\b(OUT-\d+|STF-\d+|EMP-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+|PORT-\d+|CAR-\d+|CUST-\d+|CAMP-\d+|AUD-\d+|FB-\d+)\b", q, re.I)))
    ids = [i.upper() for i in ids]
    if not ids: return None
    parts, sqls = [], []
    for e in ids[:4]:
        if scope and e != scope and not e.startswith("PORT") and not e.startswith("CAR"): parts.append(f"{e} is outside your outlet scope ({scope})."); continue
        for t, (_, idc, cols, _m) in TABLES[kind].items():
            if t == "agent_actions": continue
            try: df = Q(f"SELECT * FROM {t} WHERE {idc}=? LIMIT 1", (e,))
            except Exception: continue
            if len(df):
                r = df.iloc[0]; parts.append(f"**{e}** ({t}): " + "; ".join(f"{c}={_fmt(r[c])}" for c in df.columns if str(r[c]) not in ("None", "nan"))); sqls.append(f"SELECT * FROM {t} WHERE {idc}='{e}'"); break
        tk = Q("SELECT action_id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY action_id DESC LIMIT 5", (e,))
        if len(tk): parts.append(f"Open tickets on {e}:\n" + md(tk)); sqls.append(f"SELECT … FROM agent_actions WHERE target='{e}' AND status IN ('open','proposed')")
    return {"answer": "\n\n".join(parts), "df": None, "sql": "; ".join(sqls)} if parts else None

def _answer_core(kind, q, scope=None, mask=False, allowed=None):
    """-> dict(answer, sql, df, table) when the question maps to the data, else None."""
    ql = q.lower().strip()
    if not ql: return None
    d = lookup(kind, q, scope)
    if d: return d
    T = TABLES[kind]; sc = {}
    for t, (syn, _, _, _) in T.items(): sc[t] = sum(3 for s in syn if _w(ql, s))
    cols_of = {}
    for t in T:
        try: cols_of[t] = _cols(t)
        except Exception: cols_of[t] = []
    hits = {c: any(_w(ql, s) for s in (CSYN.get(c, []) + [c.replace("_", " ")])) for c in {c for v in cols_of.values() for c in v}}
    for t, cs in cols_of.items(): sc[t] += sum(2 for c in cs if hits.get(c) and c not in ("status", "type", "date", "name"))
    if not sc or max(sc.values()) <= 0: return None
    order = list(T); t = max(order, key=lambda x: (sc[x], -order.index(x)))
    if allowed is not None and t not in allowed: return {"answer": f"Sorry, {PL.get(t, t)} data is not available for your role.", "sql": "", "df": None, "rows": 0, "denied": True}
    syn, idc, show, dmetric = T[t]; cs = cols_of[t]
    # aggregation
    agg = "list"
    if re.search(r"\b(how many|number of|count)\b", ql): agg = "count"
    elif re.search(r"\b(average|avg|mean)\b", ql): agg = "avg"
    elif re.search(r"\b(total|sum|overall|combined)\b", ql): agg = "sum"
    elif re.search(r"\b(highest|most|top|best|largest|biggest|maximum|max|greatest|riskiest|worst|lowest|least|smallest|minimum|min|bottom|weakest|cheapest|fewest|poorest)\b", ql): agg = "rank"
    # metric
    mc = [c for c in cs if hits.get(c) and c in _NUM.get(t, ()) and c not in ("lat", "lon")]
    metric = mc[0] if mc else dmetric
    if metric and metric not in cs: metric = None
    # filters
    where, params, desc = [], [], []
    def val_filter(tab, col, sub=False):
        try: vals = Q(f"SELECT DISTINCT {col} v FROM {tab} WHERE {col} IS NOT NULL LIMIT 600").v.tolist()
        except Exception: return
        found = []
        for v in vals:
            if isinstance(v, (int, float)): continue
            for a in _alias(v):
                if _w(ql, a): found.append((len(a), str(v), a)); break
        found.sort(reverse=True); used, taken = [], ql
        for _, v, a in found:
            if a in taken and not any(a in u or u in a for u in used):
                used.append(a)
                if sub: where.append(f"outlet_id IN (SELECT outlet_id FROM {tab} WHERE {col}=?)")
                else:
                    where.append(f"{col}=?")
                params.append(v); desc.append(f"{col} = {v}")
    for c in cs:
        if c in ("status", "category", "channel", "role", "tier", "severity", "type", "location", "origin_port", "dest_port", "carrier", "region", "country", "risk_level", "industry", "priority_tier", "source", "sku_name", "name", "port_name", "customer_id", "forecast", "route_name", "origin", "dest", "cargo"):
            if c in ("origin_port", "dest_port", "origin", "dest"): continue
            val_filter(t, c)
    if t in ("shipments", "quote_requests"):
        oc, dc = ("origin_port", "dest_port") if t == "shipments" else ("origin", "dest")
        try: pv = Q("SELECT DISTINCT port_name v FROM ports").v.tolist()
        except Exception: pv = []
        for pn in pv:
            for a_ in sorted(_alias(pn), key=len, reverse=True):
                mm = re.search(r"(?<![a-z0-9])" + re.escape(a_) + r"(?![a-z0-9])", ql)
                if not mm: continue
                pre = ql[max(0, mm.start() - 22):mm.start()].strip()
                frm = re.search(r"(from|origin|out of|leaving|departing|ex)$", pre); to = re.search(r"(to|towards|into|destination|dest|bound for|arriving at|arriving in|reaching)$", pre)
                if t == "quote_requests": pn_ = a_.split(" (")[0]; pat = pn_
                else: pat = pn
                if frm: where.append(f"{oc}=?"); params.append(pat); desc.append(f"{oc} = {pn}")
                elif to: where.append(f"{dc}=?"); params.append(pat); desc.append(f"{dc} = {pn}")
                else: where.append(f"({oc}=? OR {dc}=?)"); params += [pat, pat]; desc.append(f"{pn} is origin or destination")
                break
    if "outlet_id" in cs and t != "outlets":
        try: val_filter("outlets", "location", sub=True); val_filter("outlets", "tier", sub=True)
        except Exception: pass
    m = re.search(r"\b(outlet|store|branch)\s*(?:no\.?|number|#)?\s*(\d{1,3})\b", ql)
    if m and "outlet_id" in cs: where.append("outlet_id=?"); params.append(f"OUT-{int(m.group(2)):03d}"); desc.append(f"outlet_id = OUT-{int(m.group(2)):03d}")
    for pat, op in ((r"(?:below|under|less than|lower than|smaller than|<)", "<"), (r"(?:above|over|more than|greater than|higher than|exceed\w*|>)", ">"), (r"(?:at least|minimum of|>=)", ">="), (r"(?:at most|no more than|<=)", "<=")):
        mm = re.search(pat + r"\s*(?:rs\.?|₹|\$|inr|usd)?\s*([\d,]*\.?\d+)\s*(%|k|m|lakhs?|cr(?:ores?)?)?", ql)
        if mm and metric:
            v = _num(mm.group(1), mm.group(2) if mm.group(2) != "%" else "");
            if metric in PROB and v > 1: v /= 100
            where.append(f"{metric}{op}?"); params.append(v); desc.append(f"{metric} {op} {v:g}"); break
    if re.search(r"\b(open|pending|unresolved)\b", ql) and "status" in cs and not any("status" in x for x in desc) and t in ("agent_actions", "purchase_orders"):
        where.append("status IN ('open','proposed')" if t == "agent_actions" else "status NOT IN ('delivered','received','closed')"); desc.append("status = open")
    if re.search(r"\b(critical|high severity|urgent)\b", ql) and "severity" in cs and not any("severity" in x for x in desc): where.append("severity IN ('critical','high')"); desc.append("severity = critical/high")
    if re.search(r"\b(delayed|late)\b", ql) and t == "shipments" and not any("status" in x for x in desc): where.append("(status LIKE '%delay%' OR predicted_delay_risk>=0.5)"); desc.append("delayed or delay risk ≥ 0.5")
    if re.search(r"\b(low stock|running low|below reorder|need reorder|reorder)\b", ql) and t == "inventory": where.append("current_stock<=reorder_threshold"); desc.append("stock ≤ reorder threshold")
    if re.search(r"\b(high risk|at risk)\b", ql) and metric in PROB and not any(metric in x for x in desc): where.append(f"{metric}>=0.5"); desc.append(f"{metric} ≥ 0.5")
    if scope and t != "agent_actions":
        if "outlet_id" in cs: where.append("outlet_id=?"); params.append(scope); desc.append(f"outlet_id = {scope}")
        else: return {"answer": "That data is not available for your outlet role.", "sql": "", "df": None}
    elif scope: return {"answer": "Ticket lists are limited to your outlet; open the Actions page.", "sql": "", "df": None}
    # an unfiltered, unaggregated, unranked question is too vague to answer exactly
    if agg == "list" and not where and not re.search(r"\b(show|list|which|what|who|give)\b", ql): return None
    W = (" WHERE " + " AND ".join(where)) if where else ""
    nm = re.search(r"\b(?:top|bottom|first|best|worst|lowest|highest)\s+(\d{1,2})\b", ql); n = int(nm.group(1)) if nm else (5 if agg == "rank" else 10)
    tdesc = PL.get(t, syn[0] + "s"); fdesc = (" where " + ", ".join(desc)) if desc else ""
    grp = re.search(r"\b(?:by|per|for each|across|grouped by|each)\s+(location|city|tier|role|category|channel|carrier|region|country|status|type|severity|industry|outlet|port|origin|destination|sku|customer)\b", ql)
    gcol = {"city": "location", "outlet": "outlet_id", "port": "port_name" if "port_name" in cs else "origin_port", "origin": "origin_port", "destination": "dest_port", "sku": "sku_name", "customer": "customer_id"}.get(grp.group(1), grp.group(1)) if grp else None
    if gcol and gcol not in cs and "outlet_id" in cs and gcol in ("location", "tier"):
        j = f" JOIN outlets o ON o.outlet_id={t}.outlet_id"; gq = f"o.{gcol}"
    elif gcol in cs: j, gq = "", gcol
    else: j, gq = "", None
    if gq:
        fn = {"count": "COUNT(*)", "avg": f"AVG({metric})", "sum": f"SUM({metric})"}.get(agg, f"AVG({metric})" if metric else "COUNT(*)")
        if agg == "rank" and re.search(r"\b(lowest|least|smallest|minimum|min|bottom|worst|fewest|weakest|poorest|cheapest)\b", ql) and not re.search(r"\b(worst|riskiest)\b", ql) or (re.search(r"\b(worst)\b", ql) and metric not in BAD): order_dir = "ASC"
        else: order_dir = "DESC"
        if re.search(r"\bhow many|number of|count\b", ql): fn = "COUNT(*)"
        sql = f"SELECT {gq} AS grp, {fn} AS value, COUNT(*) AS rows_n FROM {t}{j}{W} GROUP BY {gq} ORDER BY value {order_dir} LIMIT 25"
        df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None, "rows": 0, "empty": True}
        lab = {"COUNT(*)": "count"}.get(fn, fn.lower().replace("(", " of ").replace(")", ""))
        top = df.iloc[0]; df = df.rename(columns={"grp": gcol, "value": lab});
        return {"answer": f"{lab.capitalize()} of {tdesc} by {gcol}{fdesc}. Highest: **{top.iloc[0]}** ({_fmt(top.iloc[1])}); lowest: **{df.iloc[-1, 0]}** ({_fmt(df.iloc[-1, 1])}).", "sql": sql, "df": df}
    if agg == "count":
        sql = f"SELECT COUNT(*) AS n FROM {t}{W}"; n_ = int(Q(sql, tuple(params)).n[0])
        return {"answer": f"**{n_:,}** {tdesc}{fdesc}.", "sql": sql, "df": None, "rows": 1}
    if agg in ("avg", "sum") and metric:
        fn = "AVG" if agg == "avg" else "SUM"; sql = f"SELECT {fn}({metric}) AS v, COUNT(*) AS n, MIN({metric}) AS lo, MAX({metric}) AS hi FROM {t}{W}"; r = Q(sql, tuple(params)).iloc[0]
        if not r.n: return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None, "rows": 0, "empty": True}
        pv = f"{r.v:.1%}" if metric in PROB and abs(r.v) <= 1 else _fmt(float(r.v))
        return {"answer": f"{'Average' if agg == 'avg' else 'Total'} {metric.replace('_', ' ')} of {tdesc}{fdesc}: **{pv}** across {int(r.n):,} rows (min {_fmt(float(r.lo))}, max {_fmt(float(r.hi))}).", "sql": sql, "df": None, "rows": 1}
    if mask and t == "staff" and (metric in ("salary", "age") or re.search(r"\b(salary|salaries|pay|wage|age)\b", ql)): return {"answer": "Salary and personal details are visible to Admin, Owner and Operations roles only.", "sql": "", "df": None}
    sel = list(dict.fromkeys([c for c in show if c in cs] + ([metric] if metric else [])))
    if mask: sel = [c for c in sel if not (t == "staff" and c in ("name", "salary", "age"))]
    if t == "inventory" and "outlet_id" in cs and "sku_name" not in sel: sel.append("sku_name")
    if agg == "rank" and metric:
        low = re.search(r"\b(lowest|least|smallest|minimum|min|bottom|fewest|weakest|poorest|cheapest)\b", ql) is not None; worst = re.search(r"\b(worst|riskiest)\b", ql) is not None
        asc = (low and not worst) or (worst and metric not in BAD) or (re.search(r"\b(best)\b", ql) is not None and metric in BAD)
        sql = f"SELECT {', '.join(sel)} FROM {t}{W} ORDER BY {metric} {'ASC' if asc else 'DESC'} LIMIT {n}"; df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None, "rows": 0, "empty": True}
        r0 = df.iloc[0]; lab = r0[LABEL[t]] if t in LABEL and LABEL[t] in df.columns else r0[idc] if idc in df.columns else r0.iloc[0]; pv = f"{r0[metric]:.1%}" if metric in PROB and abs(r0[metric]) <= 1 else _fmt(r0[metric])
        return {"answer": f"{'Lowest' if asc else 'Highest'} {metric.replace('_', ' ')} among {tdesc}{fdesc}: **{lab}** at **{pv}**. Top {len(df)} below.", "sql": sql, "df": pct(df, sel)}
    sql = f"SELECT {', '.join(sel)} FROM {t}{W}" + (f" ORDER BY {metric} {'ASC' if metric in BAD and False else 'DESC'}" if metric else "") + f" LIMIT {max(n, 10) if not nm else n}"
    tot = int(Q(f"SELECT COUNT(*) n FROM {t}{W}", tuple(params)).n[0]); df = Q(sql, tuple(params))
    if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None, "rows": 0, "empty": True}
    return {"answer": f"**{tot:,}** {tdesc}{fdesc}; showing {len(df)}" + (f", sorted by {metric.replace('_', ' ')}" if metric else "") + ".", "sql": sql, "df": pct(df, sel)}


# ================================================================================================================================
# COPILOT ENHANCEMENT (Milestone 2 - AI Copilot Enhancement owner)
#   1. SQL validation + read-only execution       4. Spelling / alternative-wording normaliser
#   2. Role-based data policy                     5. Parameter extraction (city, tier, outlet, top-N, threshold, days)
#   3. Whitelisted query catalogue (25 queries)   6. Suggestions for unsupported questions
# Everything the Copilot runs goes through run_sql(): a single read-only SELECT over whitelisted tables, values bound as parameters.
# ================================================================================================================================
from config import DB_PATH

# ------------------------------------------------------------------ 1. SQL validation and read-only execution
ALLOWED_TABLES = {t for k in TABLES.values() for t in k} | {"outlets"}
_FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|detach|pragma|replace|vacuum|reindex|truncate|exec|execute|load_extension|grant|revoke|begin|commit|rollback)\b", re.I)
_TABLE_REF = re.compile(r"\b(?:from|join)\s+([A-Za-z_][A-Za-z0-9_]*)", re.I)
_AGG = re.compile(r"\b(count|sum|avg|min|max)\s*\(", re.I)
_LOG = threading.local()
MAX_ROWS = 1000

class SQLRejected(Exception):
    """Raised when a statement is not a safe, read-only, whitelisted SELECT."""

def validate_sql(sql, allowed=None):
    """Return the table names used if `sql` is a single read-only SELECT over whitelisted tables; otherwise raise SQLRejected."""
    if not isinstance(sql, str) or not sql.strip(): raise SQLRejected("empty statement")
    s = sql.strip().rstrip(";").strip()
    if ";" in s: raise SQLRejected("multiple statements are not allowed")
    if "--" in s or "/*" in s or "*/" in s: raise SQLRejected("comments are not allowed")
    if not re.match(r"^(select|with)\b", s, re.I): raise SQLRejected("only SELECT statements are allowed")
    bad = _FORBIDDEN.search(re.sub(r"'[^']*'", "''", s))
    if bad: raise SQLRejected(f"keyword '{bad.group(1).upper()}' is not allowed")
    tabs = {t.lower() for t in _TABLE_REF.findall(s)}
    ok = {t.lower() for t in (allowed or ALLOWED_TABLES)}
    if not tabs: raise SQLRejected("no table referenced")
    extra = tabs - ok
    if extra: raise SQLRejected("table not whitelisted: " + ", ".join(sorted(extra)))
    if not re.search(r"\blimit\s+(\?|\d+)", s, re.I) and not _AGG.search(s): raise SQLRejected("a LIMIT or an aggregate is required")
    m = re.search(r"\blimit\s+(\d+)", s, re.I)
    if m and int(m.group(1)) > MAX_ROWS: raise SQLRejected(f"LIMIT above {MAX_ROWS}")
    return tabs

def _ro_conn():
    c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA query_only=ON;"); c.execute("PRAGMA busy_timeout=30000;"); return c

def run_sql(sql, params=(), allowed=None):
    """Validate, then run on a throw-away connection with query_only=ON (a second line of defence). Returns a DataFrame."""
    import pandas as pd
    validate_sql(sql, allowed)
    c = _ro_conn()
    try: df = pd.read_sql(sql, c, params=tuple(params))
    finally: c.close()
    if not hasattr(_LOG, "d"): _LOG.d = {}
    _LOG.d[sql] = tuple(params)
    return df

def Q(sql, p=()):
    """Drop-in replacement for blocks.Q used by this module: every statement is validated and read-only."""
    return run_sql(sql, p)

def render_sql(sql, params=()):
    """Display form of a parameterised statement: placeholders replaced by their literal values (for reading only, never executed)."""
    it = iter(params or ())
    def lit(_):
        try: v = next(it)
        except StopIteration: return "?"
        return str(v) if isinstance(v, (int, float)) else "'" + str(v).replace("'", "''") + "'"
    return re.sub(r"\?", lit, " ".join(sql.split()))

# ------------------------------------------------------------------ 2. role-based data policy
ALL = "*"
_PUBLIC_DENY = {"revenue", "operating_costs", "net_margin", "headcount", "salary", "personal", "attrition", "budget"}
ROLE_POLICY = {
    "Admin":             {"tables": ALL, "deny": set(), "scoped": False, "about": "full access"},
    "Franchise Owner":   {"tables": ALL, "deny": set(), "scoped": False, "about": "all outlets, including salaries"},
    "VP Operations":     {"tables": ALL, "deny": set(), "scoped": False, "about": "all outlets"},
    "Regional Manager":  {"tables": ALL, "deny": set(), "scoped": False, "about": "all outlets"},
    "HR Lead":           {"tables": {"outlets", "staff", "attendance"}, "deny": {"revenue", "operating_costs", "net_margin", "budget"}, "scoped": False, "about": "workforce data"},
    "Marketing Manager": {"tables": {"outlets", "marketing", "feedback", "sales"}, "deny": {"salary", "personal", "attrition", "operating_costs", "net_margin"}, "scoped": False, "about": "marketing, feedback and sales"},
    "Store Manager":     {"tables": {"outlets", "staff", "inventory", "purchase_orders", "sales", "feedback", "audits", "attendance", "agent_actions", "marketing"}, "deny": {"salary", "personal"}, "scoped": True, "about": "your own outlet"},
    "Staff":             {"tables": {"outlets", "inventory", "purchase_orders", "attendance"}, "deny": _PUBLIC_DENY | {"revenue"}, "scoped": False, "about": "permitted operational information"},
    "Auditor":           {"tables": ALL, "deny": {"salary", "personal"}, "scoped": False, "about": "read-only compliance access across all outlets"},
    "Customer":          {"tables": {"outlets", "feedback"}, "deny": _PUBLIC_DENY, "scoped": False, "about": "public outlet information and ratings"},
}
_ROLE_ALIAS = {"owner": "Franchise Owner", "franchiseowner": "Franchise Owner", "manager": "Store Manager", "storemanager": "Store Manager", "admin": "Admin", "administrator": "Admin",
               "vpoperations": "VP Operations", "regionalmanager": "Regional Manager", "hrlead": "HR Lead", "hr": "HR Lead", "marketingmanager": "Marketing Manager", "staff": "Staff", "employee": "Staff", "customer": "Customer"}
def canon_role(role):
    k = re.sub(r"[^a-z]", "", str(role or "").lower())
    for r in ROLE_POLICY:
        if re.sub(r"[^a-z]", "", r.lower()) == k: return r
    return _ROLE_ALIAS.get(k, "Customer")          # unknown role: least privilege
def policy_for(role): return ROLE_POLICY[canon_role(role)]

# data topics mentioned in a question -> the restriction tags they touch (used for questions that do not hit a whitelisted query)
TAG_WORDS = {"salary": r"\b(salary|salaries|wage|wages|pay|payroll|earn\w*|ctc)\b", "attrition": r"\b(attrition|resign\w*|quit\w*|churn)\b", "revenue": r"\b(revenue|turnover|profit\w*|margin\w*|loss\w*|operating cost\w*)\b",
             "budget": r"\b(budget|marketing spend|ad spend)\b", "personal": r"\b(age of|ages of|employee names?|staff names?|date of birth)\b"}
TABLE_WORDS = {"staff": r"\b(staff|employees?|workers?|workforce|attendance|absent|absence|headcount|overtime|salary|salaries)\b", "inventory": r"\b(inventory|stock\w*|sku|skus|reorder|supplier|lead time)\b",
               "marketing": r"\b(campaigns?|marketing|roi|channels?|promotions?)\b", "audits": r"\b(audits?|compliance|violations?|inspection)\b", "feedback": r"\b(feedback|reviews?|complaints?|ratings?)\b",
               "purchase_orders": r"\b(purchase orders?|incoming stock)\b", "agent_actions": r"\b(tickets?|alerts?|flagged|anomal\w+|critical issues?)\b", "sales": r"\b(daily sales|sales trend|transactions?)\b"}
def question_tags(ql): return {t for t, rx in TAG_WORDS.items() if re.search(rx, ql)}
def question_tables(ql): return {t for t, rx in TABLE_WORDS.items() if re.search(rx, ql)}
def role_blocks(ql, role):
    """None if the question is fine for this role, else a polite reason."""
    pol = policy_for(role); r = canon_role(role)
    if pol["tables"] != ALL:
        no = sorted(t for t in question_tables(ql) if t not in pol["tables"])
        if no: return f"{PL.get(no[0], no[0]).capitalize()} data is not available to the {r} role"
    no = sorted(question_tags(ql) & pol["deny"])
    if no: return f"{ {'salary': 'Salary', 'attrition': 'Attrition', 'revenue': 'Revenue and profit', 'budget': 'Marketing budget', 'personal': 'Personal staff'}[no[0]] } data is not available to the {r} role"
    return None

# ------------------------------------------------------------------ 3. spelling and alternative-wording normaliser
_VOCAB_BASE = """outlet outlets store stores branch branches revenue sales profit profits profitable margin margins loss losses cost costs satisfaction customer customers rating ratings feedback complaint complaints review reviews
staff employee employees worker workers salary salaries overtime attrition role roles tier tiers city cities location locations inventory stock stockout reorder supplier suppliers category categories lead time delivery
order orders purchase pending open campaign campaigns marketing channel channels budget spend spending roi conversion conversions audit audits compliance violation violations hygiene safety critical ticket tickets alert alerts severity
attendance absent absence highest lowest best worst average total number count many most least top bottom below above under over trend daily weekly monthly week month today yesterday active running current status
performance performing profile summary compare comparison health risk flagged each every across overview satisfied unhappy fail failed failing weakest strongest rated expensive cheapest items products unprofitable negative likely leave leaving resign resigning quit quitting""".split()
_STOP = set("""what which where when whose whom how why who the and for are was were you your can could would should will with from that this these those have has had not but about into than then them they their there here also just give show tell list find get need want please
me my our any all some more less than very much each every both only same other another been being does did doing done its it's i'm im ive is isn't of in on at to by as an or if so be do we us a one two three four five six seven eight nine ten first last next previous recent latest might may will would data table
making working having going being taking looking doing rows row result results record records name names details detail info information report reports year years days day hours hour minutes minute now ever never always again still yet maybe like look looking""".split())
_TYPOS = {"tim": "time", "mny": "many", "whch": "which", "wich": "which", "wat": "what", "hw": "how", "shw": "show", "mst": "most", "wher": "where", "wht": "what", "waht": "what", "teir": "tier", "tiar": "tier", "tire": "tier", "stor": "store", "stoer": "store", "ctiy": "city", "ciy": "city", "staf": "staff", "satff": "staff", "wrst": "worst", "bst": "best", "mnth": "month", "wek": "week", "avrg": "average", "avg": "average", "csat": "customer satisfaction"}
_PHRASES = [(r"\blikely to (leave|quit|resign)\b", "attrition risk"), (r"\bgoing to (leave|quit|resign)\b", "attrition risk"), (r"\bflight risk\b", "attrition risk"), (r"\bwho (might|may|will) (leave|quit|resign)\b", "attrition risk staff"),
            (r"\bmaking (a )?loss\b", "loss making"), (r"\bin the red\b", "loss making"), (r"\bnot making (any )?money\b", "loss making"), (r"\bmost money\b", "highest revenue"), (r"\bearn(s|ing)? (the )?most\b", "highest revenue"),
            (r"\bbest performing\b", "best"), (r"\bout of stock\b", "stockout"), (r"\brunning (low|out)\b", "low stock"), (r"\bpo\b", "purchase order"), (r"\bpos\b", "purchase orders"),
            (r"\bstock up\b", "reorder"), (r"\bhow much (money )?did (we|the chain) (make|earn)\b", "total revenue"), (r"\bwhat'?s? ?up with\b", "status of"), (r"\bhow('?s| is| are) (my|the|our) business\b", "overview")]
_SYN = {"shops": "outlets", "shop": "outlet", "stores": "outlets", "store": "outlet", "branches": "outlets", "branch": "outlet", "franchises": "outlets", "franchise": "outlet",
        "earnings": "revenue", "income": "revenue", "turnover": "revenue", "takings": "revenue", "employees": "staff", "employee": "staff", "workers": "staff", "worker": "staff", "personnel": "staff", "manpower": "staff",
        "wages": "salary", "wage": "salary", "salaries": "salary", "pay": "salary", "complaints": "complaints", "unprofitable": "loss making", "profitability": "profit margin", "cities": "city", "towns": "city", "town": "city",
        "tiers": "tier", "category": "category", "categories": "category", "channels": "channel", "ads": "campaigns", "adverts": "campaigns", "advertisements": "campaigns", "promotions": "campaigns", "promos": "campaigns",
        "inspections": "audits", "inspection": "audit", "vendors": "suppliers", "vendor": "supplier", "bad": "negative", "poor": "negative", "unhappy": "negative", "dissatisfied": "negative",
        "greatest": "highest", "biggest": "highest", "largest": "highest", "maximum": "highest", "max": "highest", "smallest": "lowest", "minimum": "lowest", "min": "lowest", "weakest": "worst", "strongest": "best", "fewest": "lowest"}
ALIASES_CITY = {"bangalore": "Bengaluru", "bengaluru": "Bengaluru", "bombay": "Mumbai", "calcutta": "Kolkata", "cochin": "Kochi", "madras": "Chennai", "new delhi": "Delhi", "poona": "Pune", "vizag": "Visakhapatnam"}
_DYN = {"t": 0, "vals": {}, "vocab": []}
def _dyn():
    """Cities, tiers, categories, channels, roles and statuses read from the live tables (refreshed every 5 minutes)."""
    if time.time() - _DYN["t"] < 300 and _DYN["vals"]: return _DYN
    vals = {}
    for key, tab, col in (("city", "outlets", "location"), ("tier", "outlets", "tier"), ("inv_cat", "inventory", "category"), ("channel", "marketing", "channel"), ("role", "staff", "role"), ("audit_cat", "audits", "category")):
        try: vals[key] = [str(v) for v in Q(f"SELECT DISTINCT {col} v FROM {tab} WHERE {col} IS NOT NULL LIMIT 200").v.tolist()]
        except Exception: vals[key] = []
    words = set(_VOCAB_BASE)
    for v in vals.values():
        for x in v: words.update(w for w in re.findall(r"[a-z]{3,}", x.lower()))
    words.update(ALIASES_CITY)
    _DYN.update(t=time.time(), vals=vals, vocab=sorted(words)); return _DYN

def normalize(q):
    """-> (normalised_text, [(original, corrected, "spell"|"wording"), ...]). Fixes spelling (fuzzy match against the app's own vocabulary and live values), rewrites
    alternative wording to the canonical terms the query catalogue uses, and leaves IDs such as OUT-007 untouched."""
    d = _dyn(); vocab = d["vocab"]; vset = set(vocab); fixes = []
    t = re.sub(r"[^\w\s\-\.%₹$<>?#/]", " ", str(q)).strip(); t = re.sub(r"\s+", " ", t)
    out = []
    for tok in t.split(" "):
        pre, core, tail = re.match(r"^([?.,!:;]*)(.*?)([?.,!:;]*)$", tok).groups(); low = core.lower()
        if not core: out.append(tok); continue
        if re.search(r"\d", core): out.append(pre + core + tail); continue          # IDs and numbers stay exactly as typed
        if low in _STOP: out.append(pre + low + tail); continue
        new = low
        if low in _TYPOS: new = _TYPOS[low]
        elif low not in vset and low.isalpha() and len(low) >= 4:
            m = difflib.get_close_matches(low, vocab, n=1, cutoff=0.8 if len(low) >= 5 else 0.88)
            if m: new = m[0]
        if new != low: fixes.append((core, new, "spell"))
        out.append(pre + new + tail)
    t = " ".join(out)
    for rx, rep in _PHRASES:
        n = re.sub(rx, rep, t, flags=re.I)
        if n != t: fixes.append((re.search(rx, t, re.I).group(0), rep, "wording")); t = n
    for w, c in ((k, v) for k, v in _SYN.items() if k != v):
        n = re.sub(r"(?<![a-z0-9])" + re.escape(w) + r"(?![a-z0-9])", c, t)
        if n != t: fixes.append((w, c, "wording")); t = n
    seen = set(); fx = []
    for a, b, k in fixes:
        if (a.lower(), b) not in seen and a.lower() != b.lower(): seen.add((a.lower(), b)); fx.append((a, b, k))
    return t.strip(), fx

# ------------------------------------------------------------------ 5. parameter extraction
_HI = re.compile(r"\b(highest|most|top|best|largest|more|higher|greater)\b"); _LO = re.compile(r"\b(lowest|least|bottom|worst|fewest|smallest|less|lower)\b")
def extract_params(qn):
    """Pull city, tier, outlet id, top-N, threshold, comparison, days and direction out of a normalised question."""
    ql = qn.lower(); p = {}; d = _dyn()
    m = re.search(r"\bout-?\s?(\d{1,3})\b", qn, re.I) or re.search(r"\boutlet\s*(?:no\.?|number|#)?\s*(\d{1,3})\b", ql)
    if m: p["outlet"] = f"OUT-{int(m.group(1)):03d}"
    for a, c in sorted(ALIASES_CITY.items(), key=lambda kv: -len(kv[0])):
        if re.search(r"(?<![a-z])" + re.escape(a) + r"(?![a-z])", ql): p["city"] = c; break
    if "city" not in p:
        for c in sorted(d["vals"].get("city", []), key=len, reverse=True):
            if re.search(r"(?<![a-z])" + re.escape(c.lower()) + r"(?![a-z])", ql): p["city"] = c; break
    for tname in sorted(d["vals"].get("tier", []), key=len, reverse=True):
        low = tname.lower(); first = low.split()[0]
        if re.search(r"(?<![a-z])" + re.escape(low) + r"(?![a-z])", ql) or (len(low.split()) > 1 and re.search(r"(?<![a-z])" + re.escape(first) + r"(?![a-z])", ql)): p["tier"] = tname; break
    m = re.search(r"\b(?:top|bottom|first|best|worst|lowest|highest|show|give|list)\s+(?:me\s+)?(?:the\s+)?(\d{1,2})\b", ql) or re.search(r"\b(\d{1,2})\s+(?:outlets?|campaigns?|items?|skus?|staff|orders?|audits?|tickets?|complaints?|reviews?|results?|rows?|cities|tiers?|categories|roles?)\b", ql)
    if m: p["n"] = max(1, min(int(m.group(1)), 50))
    m = re.search(r"\b(below|under|less than|lower than|beneath|<|above|over|more than|greater than|higher than|exceeding|exceeds?|>|at least|minimum of|at most|no more than)\s*(?:rs\.?|₹|\$|inr|usd)?\s*([\d,]*\.?\d+)\s*(%|k|m|lakhs?|cr(?:ores?)?)?", ql)
    if m:
        cmp_ = "<" if re.match(r"(below|under|less|lower|beneath|<|at most|no more)", m.group(1)) else ">"; v = _num(m.group(2), (m.group(3) or "").replace("%", ""))
        if m.group(3) == "%" or (v > 1 and re.search(r"\b(attrition|risk|probability)\b", ql) and v <= 100): v = v / 100 if v > 1 or m.group(3) == "%" else v
        p["thr"] = v; p["cmp"] = cmp_
    m = re.search(r"\b(?:last|past|previous)\s+(\d{1,3})\s*-?\s*days?\b", ql) or re.search(r"\b(\d{1,3})\s*-?\s*days?\b", ql)
    if m: p["days"] = max(1, min(int(m.group(1)), 90))
    elif re.search(r"\b(this|last|past) week\b|\bweekly\b", ql): p["days"] = 7
    elif re.search(r"\b(this|last|past) (month|30)\b|\bmonthly\b", ql): p["days"] = 30
    elif re.search(r"\btoday\b", ql): p["days"] = 1
    hi, lo = bool(_HI.search(ql)), bool(_LO.search(ql))
    if hi and not lo: p["dir"] = "high"
    elif lo and not hi: p["dir"] = "low"
    if re.search(r"\bworst\b", ql): p["worst"] = True
    if re.search(r"\bbest\b", ql): p["best"] = True
    return p

# ------------------------------------------------------------------ 4. whitelisted query catalogue
# Each entry is one pre-written, read-only statement. {WHERE} and {DIR} are filled only from fixed fragments; values are always bound parameters.
#   tables: tables used (role policy checks them)   uses: restriction tags (salary, revenue, ...)   chain: only meaningful across outlets (denied to single-outlet roles)
#   where: [(condition, (param, default[, transform]) or None)]   dflt: default order (DESC/ASC)   bad: a high value is bad (so "worst" means DESC)
OJ = " JOIN outlets o ON o.outlet_id = {a}.outlet_id"
WL = []
def _e(key, title, domain, examples, pats, sql, tables, lead, uses=(), chain=False, where=(), dflt="DESC", bad=False, n=10, col_tags=None, empty="", filters=("city", "tier", "outlet"), more=None):
    WL.append(dict(key=key, title=title, domain=domain, examples=examples, pats=[re.compile(x, re.I) for x in pats], sql=" ".join(sql.split()), tables=set(tables), uses=set(uses), chain=chain, where=list(where),
                   dflt=dflt, bad=bad, n=n, lead=lead, col_tags=col_tags or {}, empty=empty, filters=set(filters), more=more or []))
def _f(v, d=0): return f"{float(v):,.{d}f}"

# ---- outlets / revenue
_e("revenue_by_tier", "Revenue by outlet tier", "outlet", ["Revenue by tier", "Compare outlet tiers by revenue"],
   [r"\b(revenue|sales)\b.*\b(by|per|across|each|every|for each|wise)\b.*\btier\b", r"\btier\b.*\b(revenue|sales)\b", r"\bcompare\b.*\btiers?\b", r"\btier\s?wise\b", r"\bwhich tier\b.*\b(highest|best|lowest|worst|earns?|revenue)\b"],
   "SELECT o.tier, COUNT(*) AS outlets, ROUND(SUM(o.revenue),0) AS total_revenue, ROUND(AVG(o.revenue),0) AS avg_revenue, ROUND(AVG(o.customer_satisfaction),2) AS avg_csat FROM outlets o {WHERE} GROUP BY o.tier ORDER BY total_revenue {DIR} LIMIT ?",
   ["outlets"], lambda d, p: f"**{d.iloc[0]['tier']}** outlets bring in the {'least' if p['_dir'] == 'ASC' else 'most'} revenue overall (**{_f(d.iloc[0]['total_revenue'])}**, average {_f(d.iloc[0]['avg_revenue'])} per outlet).",
   uses=["revenue"], chain=True, filters=("city",), more=["Revenue by city", "How many outlets are in each tier?"])
_e("revenue_by_city", "Revenue by city", "outlet", ["Revenue by city", "Which city earns the most revenue?"],
   [r"\b(revenue|sales)\b.*\b(by|per|across|each|every|for each|wise)\b.*\bcity\b", r"\bcity\s?wise\b.*\b(revenue|sales)\b", r"\bwhich city\b.*\b(revenue|earns?|sales|highest|best|lowest|worst)\b", r"\bcity\b.*\b(highest|lowest|best|worst) revenue\b"],
   "SELECT o.location AS city, COUNT(*) AS outlets, ROUND(SUM(o.revenue),0) AS total_revenue, ROUND(AVG(o.revenue),0) AS avg_revenue FROM outlets o {WHERE} GROUP BY o.location ORDER BY total_revenue {DIR} LIMIT ?",
   ["outlets"], lambda d, p: f"**{d.iloc[0]['city']}** has the {'lowest' if p['_dir'] == 'ASC' else 'highest'} total revenue: **{_f(d.iloc[0]['total_revenue'])}** across {int(d.iloc[0]['outlets'])} outlets.",
   uses=["revenue"], chain=True, filters=("tier",), more=["Revenue by tier", "How many outlets are in each city?"])
_e("outlets_per_tier", "Outlets per tier", "outlet", ["How many outlets are in each tier?", "Outlet count by tier"],
   [r"\bhow many\b.*\boutlets?\b.*\b(each|every|per|by)\b.*\btier\b", r"\boutlets?\b.*\b(per|by|each|wise)\b.*\btier\b", r"\btier (distribution|breakdown|split|mix)\b", r"\bnumber of outlets\b.*\btier\b"],
   "SELECT o.tier, COUNT(*) AS outlets, ROUND(AVG(o.customer_satisfaction),2) AS avg_csat FROM outlets o {WHERE} GROUP BY o.tier ORDER BY outlets {DIR} LIMIT ?",
   ["outlets"], lambda d, p: f"The chain has **{int(d.outlets.sum()):,}** outlets across {len(d)} tiers; the biggest tier is **{d.iloc[0]['tier']}** ({int(d.iloc[0]['outlets'])}).", chain=True, filters=("city",), more=["How many outlets are in each city?", "Revenue by tier"])
_e("outlets_per_city", "Outlets per city", "outlet", ["How many outlets are in each city?", "Outlet count by city"],
   [r"\bhow many\b.*\boutlets?\b.*\b(each|every|per|by|in each)\b.*\bcity\b", r"\boutlets?\b.*\b(per|by|each)\b.*\bcity\b", r"\bcity\s?wise\b.*\boutlets?\b", r"\boutlets?\b.*\bcity\s?wise\b", r"\bwhere are (our|the) outlets\b"],
   "SELECT o.location AS city, COUNT(*) AS outlets, ROUND(AVG(o.customer_satisfaction),2) AS avg_csat FROM outlets o {WHERE} GROUP BY o.location ORDER BY outlets {DIR} LIMIT ?",
   ["outlets"], lambda d, p: f"Outlets are spread over **{len(d)}** cities; **{d.iloc[0]['city']}** has {'the fewest' if p['_dir'] == 'ASC' else 'the most'} (**{int(d.iloc[0]['outlets'])}**).", chain=True, filters=("tier",), more=["How many outlets are in each tier?", "Revenue by city"])
_e("net_margin", "Net margin by outlet", "outlet", ["Which outlets have the highest net margin?", "Most profitable outlets"],
   [r"\b(net|profit)\s?margins?\b", r"\bprofit(able|ability)?\b", r"\bmost profitable\b", r"\bleast profitable\b", r"\bhighest (profit|margin)\b", r"\blowest (profit|margin)\b", r"\bmargin\b.*\boutlets?\b"],
   "SELECT o.outlet_id, o.outlet_name, o.location AS city, o.tier, ROUND(o.revenue,0) AS revenue, ROUND(o.operating_costs,0) AS operating_costs, ROUND((o.revenue-o.operating_costs)*100.0/NULLIF(o.revenue,0),1) AS net_margin_pct FROM outlets o {WHERE} ORDER BY net_margin_pct {DIR} LIMIT ?",
   ["outlets"], lambda d, p: f"{'Lowest' if p['_dir'] == 'ASC' else 'Highest'} net margin: **{d.iloc[0]['outlet_id']}** ({d.iloc[0]['city']}) at **{d.iloc[0]['net_margin_pct']}%** (revenue {_f(d.iloc[0]['revenue'])}, costs {_f(d.iloc[0]['operating_costs'])}).",
   uses=["revenue", "operating_costs", "net_margin"], more=["Which outlets are loss making?", "Revenue by tier"])
_e("loss_making", "Loss-making outlets", "outlet", ["Which outlets are loss making?", "Which outlets have costs above revenue?"],
   [r"\bloss making\b", r"\bunprofitable\b", r"\bcosts?\b.*\b(exceed|above|more than|higher than|over)\b.*\brevenue\b", r"\boperating at a loss\b", r"\bnegative (margin|profit)\b", r"\b(outlets?|stores?)\b.*\bloss\b"],
   "SELECT o.outlet_id, o.outlet_name, o.location AS city, o.tier, ROUND(o.revenue,0) AS revenue, ROUND(o.operating_costs,0) AS operating_costs, ROUND(o.revenue-o.operating_costs,0) AS profit_or_loss FROM outlets o {WHERE} ORDER BY profit_or_loss ASC LIMIT ?",
   ["outlets"], lambda d, p: f"**{len(d)}** outlets spend more than they earn; the biggest loss is **{d.iloc[0]['outlet_id']}** ({_f(d.iloc[0]['profit_or_loss'])}).",
   uses=["revenue", "operating_costs", "net_margin"], where=[("o.operating_costs > o.revenue", None)], empty="Good news: no outlet currently has operating costs above its revenue.", more=["Which outlets have the highest net margin?", "Which outlets have the lowest net margin?"])
_e("csat_threshold", "Outlets by customer satisfaction threshold", "outlet", ["Which outlets have customer satisfaction below 3.5?", "Outlets with satisfaction above 4.2"],
   [r"\b(customer )?satisfaction\b.*\b(below|under|less than|above|over|more than|greater than|higher than|beneath|exceeding|at least)\s*\d", r"\b(below|under|less than|above|over|more than|greater than|higher than)\s*\d(\.\d+)?\b.*\b(customer )?satisfaction\b", r"\b(csat|customer satisfaction)\b.*\b(below|under|above|over)\b.*\bscore\b"],
   "SELECT o.outlet_id, o.outlet_name, o.location AS city, o.tier, ROUND(o.customer_satisfaction,2) AS customer_satisfaction FROM outlets o {WHERE} ORDER BY o.customer_satisfaction {DIR} LIMIT ?",
   ["outlets"], lambda d, p: f"**{len(d)}** outlets {'score below' if p['_cmp'] == '<' else 'score above'} **{p['_thr']:g}** customer satisfaction (showing up to {p['_n']}); {'lowest' if p['_dir'] == 'ASC' else 'highest'} is **{d.iloc[0]['outlet_id']}** at {d.iloc[0]['customer_satisfaction']}.",
   where=[("o.customer_satisfaction {CMP} ?", ("thr", 3.5))], dflt="ASC", empty="No outlet matches that satisfaction threshold. Try a different number.", more=["Which outlets have customer satisfaction above 4.2?", "Which outlets have the best customer ratings?"])
# ---- staff
_e("attrition_staff", "Staff at high attrition risk", "staff", ["Which staff are at high risk of leaving?", "Who has attrition risk above 70%?"],
   [r"\battrition risk\b", r"\battrition\b.*\b(staff|employees?|people|crew)\b", r"\b(staff|employees?)\b.*\battrition\b", r"\bhigh (risk|attrition)\b.*\bstaff\b", r"\bstaff\b.*\b(high|highest) risk\b", r"\bwho\b.*\battrition\b"],
   "SELECT s.staff_id, s.name, s.role, s.outlet_id, o.location AS city, ROUND(s.predicted_attrition_prob,2) AS attrition_risk, ROUND(s.overtime_hrs,1) AS overtime_hrs, s.tenure_years FROM staff s" + OJ.format(a="s") + " {WHERE} ORDER BY s.predicted_attrition_prob {DIR} LIMIT ?",
   ["staff", "outlets"], lambda d, p: f"**{len(d)}** staff members have attrition risk {'above' if p['_cmp'] == '>' else 'below'} {p['_thr']:.0%} (showing up to {p['_n']}). Highest: **{d.iloc[0].get('name') or d.iloc[0]['staff_id']}** ({d.iloc[0]['role']}, {d.iloc[0]['outlet_id']}) at {d.iloc[0]['attrition_risk']:.0%}.",
   uses=["attrition"], where=[("s.predicted_attrition_prob {CMP} ?", ("thr", 0.6))], bad=True, col_tags={"name": "personal"}, empty="No staff members are above that attrition-risk level right now.", more=["Which outlets have the most overtime?", "How many staff are there per role?"])
_e("overtime_outlets", "Overtime by outlet", "staff", ["Which outlets have the most overtime?", "Where do staff work the longest overtime?"],
   [r"\bovertime\b", r"\boverworked\b", r"\bworking too (many|long)\b", r"\blong hours\b"],
   "SELECT o.outlet_id, o.location AS city, COUNT(*) AS staff, ROUND(AVG(s.overtime_hrs),1) AS avg_overtime_hrs, ROUND(MAX(s.overtime_hrs),1) AS max_overtime_hrs FROM staff s" + OJ.format(a="s") + " {WHERE} GROUP BY o.outlet_id, o.location ORDER BY avg_overtime_hrs {DIR} LIMIT ?",
   ["staff", "outlets"], lambda d, p: f"Highest average overtime: **{d.iloc[0]['outlet_id']}** ({d.iloc[0]['city']}) with **{d.iloc[0]['avg_overtime_hrs']} h** per staff member.", bad=True, more=["Which staff are at high risk of leaving?", "Which outlets have the lowest job satisfaction among staff?"])
_e("salary_by_role", "Average salary by role", "staff", ["What is the average salary by role?", "Compare pay across roles"],
   [r"\bsalary\b.*\b(by|per|across|each|every|for each|wise)\b.*\brole\b", r"\brole\s?wise\b.*\bsalary\b", r"\bsalary\b.*\broles?\b", r"\broles?\b.*\bsalary\b", r"\bhow much\b.*\b(chef|cashier|crew|shift lead|manager)\b.*\b(earn|paid|salary)\b"],
   "SELECT s.role, COUNT(*) AS staff, ROUND(AVG(s.salary),0) AS avg_salary, ROUND(MIN(s.salary),0) AS min_salary, ROUND(MAX(s.salary),0) AS max_salary FROM staff s" + OJ.format(a="s") + " {WHERE} GROUP BY s.role ORDER BY avg_salary {DIR} LIMIT ?",
   ["staff", "outlets"], lambda d, p: f"**{d.iloc[0]['role']}** is the {'lowest' if p['_dir'] == 'ASC' else 'highest'}-paid role at an average of **{_f(d.iloc[0]['avg_salary'])}** ({int(d.iloc[0]['staff']):,} staff).", uses=["salary"], more=["How many staff are there per role?", "Which outlets have the most overtime?"])
_e("staff_per_role", "Staff count per role", "staff", ["How many staff are there per role?", "Team composition by role"],
   [r"\bhow many\b.*\bstaff\b.*\b(per|by|each|every)\b.*\brole\b", r"\bstaff\b.*\b(per|by|each)\b.*\brole\b", r"\brole\s?wise\b.*\bstaff\b", r"\b(team|staff) (composition|mix|breakdown|split)\b", r"\bhow many\b.*\b(chefs?|cashiers?|crew|shift leads?)\b"],
   "SELECT s.role, COUNT(*) AS staff, ROUND(AVG(s.tenure_years),1) AS avg_tenure_years, ROUND(AVG(s.job_satisfaction),2) AS avg_job_satisfaction FROM staff s" + OJ.format(a="s") + " {WHERE} GROUP BY s.role ORDER BY staff {DIR} LIMIT ?",
   ["staff", "outlets"], lambda d, p: f"There are **{int(d.staff.sum()):,}** staff in {len(d)} roles; the largest group is **{d.iloc[0]['role']}** ({int(d.iloc[0]['staff']):,}).", more=["What is the average salary by role?", "Which staff are at high risk of leaving?"])
_e("staff_satisfaction", "Job satisfaction by outlet", "staff", ["Which outlets have the lowest job satisfaction among staff?", "Where are employees least satisfied?"],
   [r"\bjob satisfaction\b", r"\bwork[- ]?life balance\b", r"\bstaff\b.*\bsatisf\w*", r"\b(staff|employees?)\b.*\b(satisfaction|morale|happiness|engagement)\b", r"\b(satisfaction|morale)\b.*\b(staff|employees?)\b"],
   "SELECT o.outlet_id, o.location AS city, COUNT(*) AS staff, ROUND(AVG(s.job_satisfaction),2) AS avg_job_satisfaction, ROUND(AVG(s.work_life_balance),2) AS avg_work_life_balance FROM staff s" + OJ.format(a="s") + " {WHERE} GROUP BY o.outlet_id, o.location ORDER BY avg_job_satisfaction {DIR} LIMIT ?",
   ["staff", "outlets"], lambda d, p: f"{'Lowest' if p['_dir'] == 'ASC' else 'Highest'} staff job satisfaction: **{d.iloc[0]['outlet_id']}** ({d.iloc[0]['city']}) at **{d.iloc[0]['avg_job_satisfaction']}**.", dflt="ASC", more=["Which outlets have the most overtime?", "Which staff are at high risk of leaving?"])
_e("attendance_summary", "Attendance summary", "staff", ["Show the attendance summary", "How many staff were absent?"],
   [r"\battendance\b", r"\babsent\b", r"\babsence\b", r"\bpresent today\b", r"\bturned up\b"],
   "SELECT a.status, COUNT(*) AS records, COUNT(DISTINCT a.staff_id) AS staff FROM attendance a" + OJ.format(a="a") + " {WHERE} GROUP BY a.status ORDER BY records {DIR} LIMIT ?",
   ["attendance", "outlets"], lambda d, p: f"Attendance records: " + ", ".join(f"**{r.status}** {int(r.records):,}" for r in d.itertuples()) + ".", empty="No attendance has been recorded yet. The live simulator logs it as the day progresses, so check again shortly.", more=["Which outlets have the most overtime?", "How many staff are there per role?"])
# ---- inventory
_e("low_stock_by_outlet", "Outlets with items below reorder level", "inventory", ["Which outlets have the most items below reorder level?", "Show low stock items by outlet"],
   [r"\b(below|under|beneath|lower than|at or below)\b.*\breorder\b", r"\blow stock\b.*\b(outlet|by|per|each)\b", r"\boutlets?\b.*\blow stock\b", r"\bstock\b.*\b(below|under)\b.*\b(level|threshold)\b", r"\breorder (level|point|threshold)\b"],
   "SELECT o.outlet_id, o.location AS city, COUNT(*) AS skus_below_reorder, ROUND(AVG(i.stockout_risk_prob),2) AS avg_stockout_risk FROM inventory i" + OJ.format(a="i") + " WHERE i.current_stock <= i.reorder_threshold {AND} GROUP BY o.outlet_id, o.location ORDER BY skus_below_reorder {DIR} LIMIT ?",
   ["inventory", "outlets"], lambda d, p: f"**{int(d.skus_below_reorder.sum()):,}** SKU records are at or below their reorder level across {len(d)} outlets; the most is at **{d.iloc[0]['outlet_id']}** ({d.iloc[0]['city']}, {int(d.iloc[0]['skus_below_reorder'])} SKUs).",
   bad=True, empty="Every SKU is above its reorder level right now.", more=["Which item category has the highest stockout risk?", "Which items have the longest supplier lead time?"])
_e("stockout_by_category", "Stock-out risk by category", "inventory", ["Which item category has the highest stockout risk?", "Stock risk by category"],
   [r"\b(stockout|stock out|stock risk)\b.*\bcategor", r"\bcategor\w*\b.*\b(stockout|stock out|risk|low stock|reorder)\b", r"\b(food|beverage|packaging|consumable)\b.*\b(stockout|risk|low stock|stock)\b", r"\bstock\b.*\b(by|per|each|across)\b.*\bcategory\b"],
   "SELECT i.category, COUNT(*) AS sku_records, ROUND(AVG(i.stockout_risk_prob),3) AS avg_stockout_risk, SUM(CASE WHEN i.current_stock <= i.reorder_threshold THEN 1 ELSE 0 END) AS below_reorder FROM inventory i" + OJ.format(a="i") + " {WHERE} GROUP BY i.category ORDER BY avg_stockout_risk {DIR} LIMIT ?",
   ["inventory", "outlets"], lambda d, p: f"**{d.iloc[0]['category']}** carries the {'lowest' if p['_dir'] == 'ASC' else 'highest'} average stock-out risk (**{d.iloc[0]['avg_stockout_risk']:.1%}**), with {int(d.iloc[0]['below_reorder'])} records below reorder level.", bad=True, more=["Which outlets have the most items below reorder level?", "Which items have the longest supplier lead time?"])
_e("longest_lead_time", "Longest supplier lead times", "inventory", ["Which items have the longest supplier lead time?", "Slowest items to arrive"],
   [r"\blead time\b", r"\bsupplier\b.*\b(slow|delay|late|longest)\b", r"\b(slowest|longest)\b.*\b(deliver|arrive|supply|supplier)\w*\b", r"\btake\b.*\b(longest|most time)\b.*\b(arrive|deliver)\b", r"\bdelivery (time|days)\b", r"\b(slowest|longest)\b.*\b(items?|products?|skus?)\b", r"\bto arrive\b"],
   "SELECT i.sku_name, i.category, ROUND(AVG(i.lead_time_days),1) AS avg_lead_days, MAX(i.lead_time_days) AS max_lead_days, COUNT(*) AS records FROM inventory i" + OJ.format(a="i") + " {WHERE} GROUP BY i.sku_name, i.category ORDER BY avg_lead_days {DIR} LIMIT ?",
   ["inventory", "outlets"], lambda d, p: f"**{d.iloc[0]['sku_name']}** ({d.iloc[0]['category']}) has the {'shortest' if p['_dir'] == 'ASC' else 'longest'} supplier lead time: **{d.iloc[0]['avg_lead_days']} days** on average.", bad=True, more=["Which outlets have the most items below reorder level?", "Show pending purchase orders"])
_e("open_purchase_orders", "Pending purchase orders", "inventory", ["Show pending purchase orders", "Which orders are on the way?"],
   [r"\bpurchase orders?\b", r"\bincoming (stock|orders?|deliveries)\b", r"\borders?\b.*\b(on the way|in transit|arriving|coming)\b", r"\borders? (pending|open|outstanding)\b", r"\b(pending|open|outstanding)\b.*\borders?\b"],
   "SELECT po.po_id, po.outlet_id, o.location AS city, po.sku_name, po.qty, po.eta_ts AS eta, po.status FROM purchase_orders po" + OJ.format(a="po") + " WHERE po.status NOT IN ('delivered','received','closed','cancelled') {AND} ORDER BY po.eta_ts ASC LIMIT ?",
   ["purchase_orders", "outlets"], lambda d, p: f"**{len(d)}** purchase orders are still open (showing up to {p['_n']}); next to arrive: **{d.iloc[0]['sku_name']}** for {d.iloc[0]['outlet_id']} ({int(d.iloc[0]['qty'])} units).", empty="There are no open purchase orders right now.", more=["Which outlets have the most items below reorder level?", "Which items have the longest supplier lead time?"])
# ---- marketing
_e("campaign_roi", "Campaigns ranked by ROI", "marketing", ["Which campaigns have the best ROI?", "Show the worst performing campaigns"],
   [r"\bcampaigns?\b.*\b(roi|return|perform\w*|best|worst|top|bottom|highest|lowest)\b", r"\b(best|worst|top|bottom)\b.*\bcampaigns?\b", r"\broi\b.*\bcampaigns?\b", r"\bcampaigns?\b.*\broi\b"],
   "SELECT m.campaign_id, m.campaign_name, m.channel, o.location AS city, ROUND(m.budget,0) AS budget, ROUND(m.actual_roi,2) AS roi, m.conversions FROM marketing m" + OJ.format(a="m") + " {WHERE} ORDER BY m.actual_roi {DIR} LIMIT ?",
   ["marketing", "outlets"], lambda d, p: f"{'Lowest' if p['_dir'] == 'ASC' else 'Highest'} ROI: **{d.iloc[0]['campaign_name']}** ({d.iloc[0]['channel']}, {d.iloc[0]['city']}) at **{d.iloc[0]['roi']}x**.", more=["Which marketing channel works best?", "Which campaigns are running now?"])
_e("channel_performance", "ROI by marketing channel", "marketing", ["Which marketing channel works best?", "Compare ROI by channel"],
   [r"\b(channel|channels)\b.*\b(roi|best|worst|perform\w*|works?|compare|return)\b", r"\b(roi|return)\b.*\b(by|per|each|across)\b.*\bchannel\b", r"\bbest (marketing )?channel\b", r"\bchannel\s?wise\b", r"\bcompare channels\b", r"\bwhich channel\b"],
   "SELECT m.channel, COUNT(*) AS campaigns, ROUND(AVG(m.actual_roi),2) AS avg_roi, ROUND(SUM(m.budget),0) AS total_budget, SUM(m.conversions) AS conversions FROM marketing m" + OJ.format(a="m") + " {WHERE} GROUP BY m.channel ORDER BY avg_roi {DIR} LIMIT ?",
   ["marketing", "outlets"], lambda d, p: f"**{d.iloc[0]['channel']}** has the {'lowest' if p['_dir'] == 'ASC' else 'highest'} average ROI (**{d.iloc[0]['avg_roi']}x** over {int(d.iloc[0]['campaigns'])} campaigns).", more=["Which campaigns have the best ROI?", "What is the marketing spend by channel?"])
_e("active_campaigns", "Campaigns running now", "marketing", ["Which campaigns are running now?", "Show active promotions"],
   [r"\b(active|live|running|current|ongoing)\b.*\bcampaigns?\b", r"\bcampaigns?\b.*\b(active|live|running|now|currently|ongoing|today)\b", r"\bwhat campaigns\b"],
   "SELECT m.campaign_id, m.campaign_name, m.channel, o.location AS city, ROUND(m.budget,0) AS budget, ROUND(m.actual_roi,2) AS roi, m.end_date FROM marketing m" + OJ.format(a="m") + " WHERE m.end_date >= date('now') {AND} ORDER BY m.end_date ASC LIMIT ?",
   ["marketing", "outlets"], lambda d, p: f"**{len(d)}** campaigns are live right now (showing up to {p['_n']}); the next to end is **{d.iloc[0]['campaign_name']}** on {d.iloc[0]['end_date']}.", empty="No campaigns are running right now.", more=["Which marketing channel works best?", "Which campaigns have the best ROI?"])
_e("marketing_spend", "Marketing spend by channel", "marketing", ["What is the marketing spend by channel?", "How much did we spend on campaigns?"],
   [r"\b(marketing|campaign|advertising)\b.*\b(spend|spending|budget|cost)\b", r"\b(spend|spending|budget)\b.*\b(marketing|campaigns?|channel)\b", r"\bcost per conversion\b", r"\bhow much\b.*\b(spend|spent)\b"],
   "SELECT m.channel, COUNT(*) AS campaigns, ROUND(SUM(m.budget),0) AS total_budget, SUM(m.conversions) AS conversions, ROUND(SUM(m.budget)*1.0/NULLIF(SUM(m.conversions),0),0) AS cost_per_conversion FROM marketing m" + OJ.format(a="m") + " {WHERE} GROUP BY m.channel ORDER BY total_budget {DIR} LIMIT ?",
   ["marketing", "outlets"], lambda d, p: f"Total campaign budget is **{_f(d.total_budget.sum())}**; **{d.iloc[0]['channel']}** takes the {'smallest' if p['_dir'] == 'ASC' else 'largest'} share (**{_f(d.iloc[0]['total_budget'])}**).", uses=["budget"], more=["Which marketing channel works best?", "Which campaigns have the best ROI?"])
# ---- feedback
_e("negative_feedback", "Recent negative feedback", "feedback", ["Show recent negative feedback", "Which customers complained recently?"],
   [r"\bnegative\b.*\b(feedback|reviews?|comments?)\b", r"\b(complaints?|complained)\b", r"\blow (rating|rated|ratings)\b", r"\bbad (feedback|reviews?)\b", r"\bunhappy customers?\b", r"\bnegative (feedback|reviews?)\b", r"\bnegative\b.*\bratings?\b", r"\bnegative customers?\b"],
   "SELECT f.feedback_id, f.outlet_id, o.location AS city, f.rating, f.date, f.comment FROM feedback f" + OJ.format(a="f") + " WHERE f.rating <= ? {AND} ORDER BY f.date DESC LIMIT ?",
   ["feedback", "outlets"], lambda d, p: f"**{len(d)}** recent reviews rated {p['_rmax']} or lower (showing up to {p['_n']}); latest is for **{d.iloc[0]['outlet_id']}** ({d.iloc[0]['city']}) on {d.iloc[0]['date']}.", empty="No reviews with a rating that low were found.", more=["Which outlets have the best customer ratings?", "Which outlets have customer satisfaction below 3.5?"])
_e("best_rated_outlets", "Best-rated outlets", "feedback", ["Which outlets have the best customer ratings?", "Which outlets are rated the lowest?"],
   [r"\b(best|top|highest|lowest|worst)\b.*\b(rated|ratings?)\b", r"\b(rated|ratings?)\b.*\b(best|top|highest|lowest|worst)\b", r"\bmost loved\b", r"\baverage rating\b.*\boutlets?\b", r"\boutlets?\b.*\baverage rating\b"],
   "SELECT o.outlet_id, o.outlet_name, o.location AS city, COUNT(*) AS reviews, ROUND(AVG(f.rating),2) AS avg_rating FROM feedback f" + OJ.format(a="f") + " {WHERE} GROUP BY o.outlet_id, o.outlet_name, o.location HAVING COUNT(*) >= 3 ORDER BY avg_rating {DIR} LIMIT ?",
   ["feedback", "outlets"], lambda d, p: f"{'Lowest' if p['_dir'] == 'ASC' else 'Best'} rated (outlets with 3+ reviews): **{d.iloc[0]['outlet_id']}** ({d.iloc[0]['city']}) at **{d.iloc[0]['avg_rating']}/5** from {int(d.iloc[0]['reviews'])} reviews.", more=["Show recent negative feedback", "Which outlets have customer satisfaction below 3.5?"])
# ---- audits
_e("failed_audits", "Audits needing action", "audit", ["Which audits failed?", "Show audits that need action"],
   [r"\b(failed|failing|failure|failures)\b.*\baudits?\b", r"\baudits?\b.*\b(failed|failing|fail|need action|needs action|action required|non[- ]?compliant)\b", r"\bnon[- ]?compliance\b", r"\baudits? (needing|requiring) action\b", r"\bwhich audits\b"],
   "SELECT a.audit_id, a.outlet_id, o.location AS city, a.category, a.audit_date, ROUND(a.score,1) AS score, a.violations, a.status FROM audits a" + OJ.format(a="a") + " WHERE a.status = 'Action Required' {AND} ORDER BY a.score ASC LIMIT ?",
   ["audits", "outlets"], lambda d, p: f"**{len(d)}** audits are marked *Action Required* (showing up to {p['_n']}); the lowest score is **{d.iloc[0]['score']}** at **{d.iloc[0]['outlet_id']}** ({d.iloc[0]['category']}).", empty="No audit is currently marked Action Required.", more=["Which audit category is weakest?", "Which outlets have the most audit violations?"])
_e("audit_by_category", "Audit results by category", "audit", ["Which audit category is weakest?", "Compliance score by category"],
   [r"\baudit\b.*\bcategor", r"\bcompliance\b.*\b(by|per|each)\b.*\bcategor", r"\bcategor\w*\b.*\b(audit|compliance)\b", r"\b(hygiene|food safety|fire|financial)\b.*\b(audit|compliance|score)\b", r"\b(weakest|worst)\b.*\b(audit|compliance)\b"],
   "SELECT a.category, COUNT(*) AS audits, ROUND(AVG(a.score),1) AS avg_score, SUM(a.violations) AS violations, SUM(CASE WHEN a.status = 'Action Required' THEN 1 ELSE 0 END) AS action_required FROM audits a" + OJ.format(a="a") + " {WHERE} GROUP BY a.category ORDER BY avg_score {DIR} LIMIT ?",
   ["audits", "outlets"], lambda d, p: f"**{d.iloc[0]['category']}** has the {'highest' if p['_dir'] == 'DESC' else 'lowest'} average audit score (**{d.iloc[0]['avg_score']}**), with {int(d.iloc[0]['violations'])} violations.", dflt="ASC", more=["Which audits failed?", "Which outlets have the most audit violations?"])
_e("violations_by_outlet", "Outlets with most audit violations", "audit", ["Which outlets have the most audit violations?", "Where are the most violations?"],
   [r"\bviolations?\b", r"\bmost (breaches|infractions)\b"],
   "SELECT o.outlet_id, o.location AS city, COUNT(*) AS audits, SUM(a.violations) AS total_violations, ROUND(AVG(a.score),1) AS avg_score FROM audits a" + OJ.format(a="a") + " {WHERE} GROUP BY o.outlet_id, o.location ORDER BY total_violations {DIR} LIMIT ?",
   ["audits", "outlets"], lambda d, p: f"Most audit violations: **{d.iloc[0]['outlet_id']}** ({d.iloc[0]['city']}) with **{int(d.iloc[0]['total_violations'])}** across {int(d.iloc[0]['audits'])} audits.", bad=True, more=["Which audits failed?", "Which audit category is weakest?"])
# ---- tickets and sales
_e("tickets_by_severity", "Open tickets by severity", "ops", ["How many open tickets are there by severity?", "Open issues by type"],
   [r"\b(open )?(tickets?|alerts?|issues?)\b.*\b(by|per|each)\b.*\b(severity|type)\b", r"\bseverity\b", r"\bhow many\b.*\b(open )?(tickets?|alerts?|issues?)\b", r"\b(open|pending) (tickets?|alerts?|issues?)\b.*\b(count|summary|breakdown)\b"],
   "SELECT t.severity, t.type, COUNT(*) AS tickets FROM agent_actions t JOIN outlets o ON o.outlet_id = t.target WHERE t.status IN ('open','proposed') {AND} GROUP BY t.severity, t.type ORDER BY tickets {DIR} LIMIT ?",
   ["agent_actions", "outlets"], lambda d, p: f"**{int(d.tickets.sum()):,}** open tickets across {len(d)} severity/type groups; the largest is **{d.iloc[0]['type']}** ({d.iloc[0]['severity']}, {int(d.iloc[0]['tickets'])}).", empty="There are no open tickets at the moment.", n=15, more=["Show the open critical tickets", "Which outlets are at highest risk?"])
_e("critical_tickets", "Open critical tickets", "ops", ["Show the open critical tickets", "What needs urgent action?"],
   [r"\bcritical\b.*\b(tickets?|alerts?|issues?|actions?)\b", r"\b(tickets?|alerts?|issues?)\b.*\bcritical\b", r"\burgent\b", r"\bimmediate (action|attention)\b"],
   "SELECT t.action_id, t.target AS outlet_id, o.location AS city, t.type, t.severity, t.detail, t.owner, t.due FROM agent_actions t JOIN outlets o ON o.outlet_id = t.target WHERE t.severity = 'Critical' AND t.status IN ('open','proposed') {AND} ORDER BY t.action_id DESC LIMIT ?",
   ["agent_actions", "outlets"], lambda d, p: f"**{len(d)}** critical tickets are open (showing up to {p['_n']}); newest: **{d.iloc[0]['type']}** on {d.iloc[0]['outlet_id']}.", empty="There are no open critical tickets.", more=["How many open tickets are there by severity?", "Which outlets are at highest risk?"])
_e("sales_by_day", "Daily sales", "outlet", ["Show daily sales for the last 7 days", "What were sales in the last 14 days?"],
   [r"\bdaily sales\b", r"\bsales\b.*\b(last|past|previous)\s*\d+\s*days?\b", r"\b(last|past|previous)\s*\d+\s*days?\b.*\bsales\b", r"\bsales\b.*\b(by day|per day|each day|day by day|day wise)\b", r"\bsales (today|yesterday|this week|last week)\b", r"\bsales\b.*\b(this|last|past) (week|month)\b"],
   "SELECT substr(s.ts,1,10) AS day, ROUND(SUM(s.amount),0) AS total_sales, SUM(s.tx_count) AS transactions, COUNT(DISTINCT s.outlet_id) AS outlets_reporting FROM sales s" + OJ.format(a="s") + " WHERE substr(s.ts,1,10) >= date('now', ?) {AND} GROUP BY substr(s.ts,1,10) ORDER BY day DESC LIMIT ?",
   ["sales", "outlets"], lambda d, p: f"Sales over the last {p['_days']} days: **{_f(d.total_sales.sum())}** across {int(d.transactions.sum()):,} transactions. Best day: **{d.loc[d.total_sales.idxmax(), 'day']}** ({_f(d.total_sales.max())}).", uses=["revenue"], empty="No sales were recorded in that period.", n=30,
   more=["Which outlets sold the most in the last 30 days?", "Revenue by city"])
_e("top_sellers_period", "Top-selling outlets in a period", "outlet", ["Which outlets sold the most in the last 30 days?", "Top selling outlets this week"],
   [r"\b(outlets?|stores?)\b.*\bsold\b.*\b(most|least)\b", r"\bsold the (most|least)\b", r"\b(top|best|worst|lowest)\b.*\b(selling|sales)\b.*\boutlets?\b", r"\boutlets?\b.*\bsales\b.*\b(last|past|this)\b.*\b(days?|week|month)\b", r"\bbest selling\b"],
   "SELECT o.outlet_id, o.location AS city, o.tier, ROUND(SUM(s.amount),0) AS total_sales, SUM(s.tx_count) AS transactions FROM sales s" + OJ.format(a="s") + " WHERE substr(s.ts,1,10) >= date('now', ?) {AND} GROUP BY o.outlet_id, o.location, o.tier ORDER BY total_sales {DIR} LIMIT ?",
   ["sales", "outlets"], lambda d, p: f"{'Lowest' if p['_dir'] == 'ASC' else 'Top'} seller over the last {p['_days']} days: **{d.iloc[0]['outlet_id']}** ({d.iloc[0]['city']}) with **{_f(d.iloc[0]['total_sales'])}**.", uses=["revenue"], chain=True, empty="No sales were recorded in that period.", more=["Show daily sales for the last 7 days", "Revenue by city"])

WL_BY_KEY = {e["key"]: e for e in WL}
_DAYS_DEFAULT = {"sales_by_day": 7, "top_sellers_period": 30}

def entry_access(e, role):
    """-> (allowed, reason). Checks the role's tables, restricted data tags and whether the query only makes sense chain-wide."""
    pol = policy_for(role); r = canon_role(role)
    if pol["tables"] != ALL:
        no = sorted(t for t in e["tables"] if t not in pol["tables"])
        if no: return False, f"{PL.get(no[0], no[0]).capitalize()} data is not available to the {r} role"
    no = sorted(e["uses"] & pol["deny"])
    if no: return False, f"{ {'salary': 'Salary', 'attrition': 'Attrition', 'revenue': 'Revenue and profit', 'operating_costs': 'Cost and profit', 'net_margin': 'Profit', 'budget': 'Marketing budget', 'headcount': 'Headcount', 'personal': 'Personal'}.get(no[0], no[0]) } data is not available to the {r} role"
    if e["chain"] and pol["scoped"]: return False, f"This is a chain-wide comparison, and the {r} role only sees its own outlet"
    return True, ""

def match_whitelist(qn):
    """Best catalogue entry for a normalised question -> (entry, score) or (None, 0). Score = number of its patterns that match."""
    ql = qn.lower(); best, bs = None, 0
    for e in WL:
        s = sum(1 for rx in e["pats"] if rx.search(ql))
        if s > bs: best, bs = e, s
    return best, bs

def build_entry(e, p, scope=None):
    """Compose the SQL text and bound parameters for a catalogue entry from the extracted parameters. -> (sql, binds, resolved_params).
    Bind order is always: entry pre-binds (days / rating cut-off), fixed-condition binds, filter binds, LIMIT."""
    rp = dict(p); high = rp.get("dir") == "high"; low = rp.get("dir") == "low"
    if e["bad"]:
        if rp.get("worst"): high, low = True, False
        elif rp.get("best"): high, low = False, True
    d = "DESC" if (high and not low) else "ASC" if (low and not high) else e["dflt"]
    cmp_ = rp.get("cmp") or (">" if e["key"] == "attrition_staff" else "<"); conds, cbinds = [], []
    if e["key"] == "csat_threshold" and not (high or low) and cmp_ == ">": d = "DESC"
    for cond, spec in e["where"]:
        conds.append(cond.replace("{CMP}", cmp_))
        if spec: v = rp.get(spec[0], spec[1]); cbinds.append(v); rp[f"_{spec[0]}"] = v
    pre = []
    if e["key"] in _DAYS_DEFAULT:
        days = int(rp.get("days") or _DAYS_DEFAULT[e["key"]]); rp["_days"] = days; pre = [f"-{days} days"]
    if e["key"] == "negative_feedback":
        t = rp.get("thr") if rp.get("cmp") == "<" and rp.get("thr") else None
        rmax = int(t - 1 if float(t) == int(t) else int(t)) if t else 2; pre = [max(1, rmax)]; rp["_rmax"] = pre[0]
    flt = []
    if "city" in e["filters"] and rp.get("city"): flt.append(("o.location = ?", rp["city"]))
    if "tier" in e["filters"] and rp.get("tier"): flt.append(("o.tier = ?", rp["tier"]))
    out = scope or (rp.get("outlet") if "outlet" in e["filters"] else None)
    if out: flt.append(("o.outlet_id = ?", out))
    sql = e["sql"]
    if "{WHERE}" in sql: sql = sql.replace("{WHERE}", (" WHERE " + " AND ".join(conds + [c for c, _ in flt])) if (conds or flt) else "")
    else: sql = sql.replace("{AND}", "".join(" AND " + c for c, _ in flt))
    n = int(rp.get("n") or e["n"]); rp["_n"] = n; rp["_dir"] = d; rp["_cmp"] = cmp_
    return sql.replace("{DIR}", d), pre + cbinds + [v for _, v in flt] + [n], rp

def run_entry(e, p, role, scope=None):
    """Execute a catalogue entry for this role and outlet scope -> dict(answer, df, sql, rows, key, params, status)."""
    ok, why = entry_access(e, role)
    if not ok: return {"status": "denied", "answer": why + ".", "df": None, "sql": "", "rows": 0, "key": e["key"], "params": p}
    pol = policy_for(role)
    if pol["scoped"] and not scope: return {"status": "denied", "answer": "Your account is not linked to an outlet yet, so I cannot show outlet data. Please ask an administrator to assign one.", "df": None, "sql": "", "rows": 0, "key": e["key"], "params": p}
    if scope and p.get("outlet") and p["outlet"] != scope: return {"status": "denied", "answer": f"{p['outlet']} is outside your outlet scope ({scope}).", "df": None, "sql": "", "rows": 0, "key": e["key"], "params": p}
    sql, binds, rp = build_entry(e, p, scope)
    try: df = run_sql(sql, binds, allowed=e["tables"])
    except SQLRejected as ex: return {"status": "error", "answer": "That query was blocked by the SQL safety check (" + str(ex) + ").", "df": None, "sql": render_sql(sql, binds), "rows": 0, "key": e["key"], "params": p}
    except Exception as ex:
        msg = str(ex)
        if "no such table" in msg: return {"status": "empty", "answer": f"The {PL.get(sorted(e['tables'])[0], 'requested')} data has not been created yet. It appears once the live agents have run, so please try again in a minute.", "df": None, "sql": render_sql(sql, binds), "rows": 0, "key": e["key"], "params": p}
        return {"status": "error", "answer": "I could not run that query just now. Please try again in a moment.", "df": None, "sql": render_sql(sql, binds), "rows": 0, "key": e["key"], "params": p}
    for c, tag in e["col_tags"].items():
        if tag in pol["deny"] and c in df.columns: df = df.drop(columns=[c])
    shown = render_sql(sql, binds)
    if not len(df):
        flt = ", ".join(f"{k} = {v}" for k, v in (("city", rp.get("city")), ("tier", rp.get("tier")), ("outlet", scope or rp.get("outlet"))) if v)
        msg = e["empty"] or "No rows matched."
        return {"status": "empty", "answer": f"{msg}" + (f" (Filters applied: {flt}.)" if flt else "") + ("" if e["empty"] else " Try removing a filter or widening the range."), "df": None, "sql": shown, "rows": 0, "key": e["key"], "params": p, "empty": True}
    lead = e["lead"](df, rp)
    note = f" Filtered to {', '.join(str(v) for v in (rp.get('city'), rp.get('tier'), scope or rp.get('outlet')) if v)}." if (rp.get("city") or rp.get("tier") or scope or rp.get("outlet")) else ""
    return {"status": "ok", "answer": lead + note, "df": df, "sql": shown, "rows": int(len(df)), "key": e["key"], "params": {k: v for k, v in rp.items() if not k.startswith("_")}, "title": e["title"], "limit": rp["_n"], "more": e["more"]}

# ------------------------------------------------------------------ 6. suggestions
def allowed_examples(role, scoped_ok=True):
    """Example questions the role can actually ask, grouped by catalogue order."""
    out = []
    for e in WL:
        ok, _ = entry_access(e, role)
        if ok: out.extend(e["examples"][:1])
    return out
def suggest_questions(qn, role, n=3, exclude=()):
    """Three valid, role-allowed questions, ranked by word overlap with what the user typed (so the hints are on topic)."""
    words = set(re.findall(r"[a-z]{4,}", qn.lower())) - _STOP; cand = []
    dom = None
    for e in WL:
        ok, _ = entry_access(e, role)
        if not ok or e["examples"][0] in exclude: continue
        text = " ".join(e["examples"] + [e["title"]]).lower(); toks = set(re.findall(r"[a-z]{4,}", text))
        sc = len(words & toks) + 0.5 * sum(1 for w in words if any(difflib.SequenceMatcher(None, w, t).ratio() > 0.8 for t in toks))
        cand.append((sc, -WL.index(e), e["examples"][0]))
    cand.sort(reverse=True); picks = [c[2] for c in cand[:n]]
    if len(picks) < n:
        picks += [x for x in allowed_examples(role) if x not in picks][: n - len(picks)]
    return picks[:n]

# ------------------------------------------------------------------ public answer(): the original engine plus role table access, row counts and the SQL that actually ran
def answer(kind, q, scope=None, mask=False, allowed=None):
    """-> dict(answer, sql, df, rows, ...) when the question maps to the data, else None. `sql` now shows the literal values that were bound."""
    _LOG.d = {}
    r = _answer_core(kind, q, scope, mask, allowed)
    if not r: return None
    sql = r.get("sql") or ""; params = _LOG.d.get(sql)
    if params is not None: r["sql"] = render_sql(sql, params)
    if r.get("rows") is None: r["rows"] = int(len(r["df"])) if r.get("df") is not None else 0
    return r


In [ ]:
%%writefile /content/app/scale_data.py
"""Scale both apps to production-like volume. Idempotent: only tops tables up to the target counts.
Coordinates are approximate (1-2 decimals): replace/extend through the Data Feeder with UN/LOCODE or World Port Index data."""
import random, datetime, math
from live_engine import conn, _DB_LOCK, now

PORTS = {
"China": "Dalian 38.9 121.7;Xiamen 24.5 118.1;Fuzhou 26.0 119.4;Lianyungang 34.7 119.4;Yantai 37.6 121.4;Rizhao 35.4 119.5;Nanjing 32.1 118.8;Zhanjiang 21.2 110.4;Haikou 20.0 110.3;Zhuhai 22.2 113.6;Taicang 31.5 121.2;Wuhan 30.6 114.3;Shantou 23.4 116.7;Dongguan 22.9 113.6",
"Japan": "Yokohama 35.4 139.7;Nagoya 35.1 136.9;Osaka 34.6 135.4;Kobe 34.7 135.2;Hakata 33.6 130.4;Chiba 35.6 140.1",
"South Korea": "Incheon 37.4 126.6;Gwangyang 34.9 127.7;Ulsan 35.5 129.4;Pyeongtaek 36.9 126.8",
"Taiwan": "Keelung 25.1 121.7;Taichung 24.3 120.5", "Vietnam": "Hai Phong 20.8 106.7;Da Nang 16.1 108.2;Vung Tau 10.4 107.1;Quy Nhon 13.8 109.3",
"Thailand": "Bangkok 13.7 100.6;Songkhla 7.2 100.6", "Malaysia": "Penang 5.4 100.3;Johor 1.4 103.9;Kuantan 3.97 103.4;Kuching 1.6 110.3",
"Indonesia": "Surabaya -7.2 112.7;Semarang -6.9 110.4;Belawan 3.8 98.7;Makassar -5.1 119.4;Balikpapan -1.3 116.8;Palembang -3.0 104.8;Bitung 1.4 125.2",
"Philippines": "Cebu 10.3 123.9;Davao 7.1 125.6;Subic Bay 14.8 120.3", "Myanmar": "Yangon 16.8 96.2", "Cambodia": "Sihanoukville 10.6 103.5",
"Bangladesh": "Chittagong 22.3 91.8;Mongla 22.5 89.6", "Sri Lanka": "Hambantota 6.1 81.1;Galle 6.0 80.2", "Pakistan": "Port Qasim 24.8 67.3;Gwadar 25.1 62.3", "Iran": "Chabahar 25.3 60.6",
"India": "Mormugao 15.4 73.8;Pipavav 20.9 71.5;Hazira 21.1 72.6;Ennore 13.2 80.3;Krishnapatnam 14.3 80.1;Kakinada 16.9 82.3;Gangavaram 17.6 83.2;Dahej 21.7 72.5;Mumbai Port 18.9 72.8;Navlakhi 22.9 70.5;Karaikal 10.9 79.9;Kamarajar 13.3 80.3;Vizhinjam 8.4 77.0;Okha 22.5 69.1;Porbandar 21.6 69.6",
"UAE": "Khalifa Port 24.8 54.6;Port Rashid 25.3 55.3;Fujairah 25.1 56.4;Khor Fakkan 25.3 56.4", "Oman": "Sohar 24.5 56.6;Duqm 19.7 57.7",
"Saudi Arabia": "Dammam 26.5 50.2;King Abdullah Port 22.5 39.1;Yanbu 24.1 38.1", "Qatar": "Hamad Port 25.0 51.6", "Kuwait": "Shuwaikh 29.4 47.9", "Bahrain": "Khalifa Bin Salman 26.2 50.6",
"Iraq": "Umm Qasr 30.0 47.9", "Jordan": "Aqaba 29.5 35.0", "Israel": "Haifa 32.8 35.0;Ashdod 31.8 34.6", "Turkey": "Mersin 36.8 34.6;Ambarli 41.0 28.7;Izmir 38.4 27.1",
"Yemen": "Aden 12.8 45.0", "Djibouti": "Djibouti 11.6 43.1", "Sudan": "Port Sudan 19.6 37.2",
"Germany": "Bremerhaven 53.6 8.5", "Belgium": "Zeebrugge 51.3 3.2", "Netherlands": "Amsterdam 52.4 4.9", "United Kingdom": "Southampton 50.9 -1.4;London Gateway 51.5 0.5;Liverpool 53.4 -3.0",
"Ireland": "Dublin 53.3 -6.2", "France": "Marseille Fos 43.4 4.9", "Spain": "Barcelona 41.4 2.2;Bilbao 43.3 -3.0", "Portugal": "Lisbon 38.7 -9.1;Sines 37.9 -8.9",
"Italy": "Naples 40.8 14.3;La Spezia 44.1 9.8;Livorno 43.6 10.3;Trieste 45.6 13.8;Gioia Tauro 38.4 15.9;Taranto 40.5 17.2", "Slovenia": "Koper 45.5 13.7", "Croatia": "Rijeka 45.3 14.4",
"Greece": "Thessaloniki 40.6 22.9", "Romania": "Constanta 44.2 28.7", "Bulgaria": "Varna 43.2 27.9", "Ukraine": "Odesa 46.5 30.7", "Sweden": "Gothenburg 57.7 11.9", "Denmark": "Aarhus 56.2 10.2",
"Norway": "Oslo 59.9 10.7", "Finland": "Helsinki 60.2 25.0", "Poland": "Gdynia 54.5 18.5", "Lithuania": "Klaipeda 55.7 21.1", "Latvia": "Riga 57.0 24.1", "Estonia": "Tallinn 59.4 24.8",
"Russia": "St Petersburg 59.9 30.2;Novorossiysk 44.7 37.8", "Malta": "Marsaxlokk 35.8 14.5", "Cyprus": "Limassol 34.7 33.0",
"Egypt": "Alexandria 31.2 29.9;Port Said 31.3 32.3;Damietta 31.5 31.8;Ain Sokhna 29.6 32.4", "Libya": "Tripoli 32.9 13.2", "Tunisia": "Rades 36.8 10.3", "Algeria": "Algiers 36.8 3.1",
"Morocco": "Casablanca 33.6 -7.6", "Senegal": "Dakar 14.7 -17.4", "Ivory Coast": "Abidjan 5.3 -4.0", "Ghana": "Tema 5.6 0.0", "Togo": "Lome 6.1 1.3", "Benin": "Cotonou 6.4 2.4",
"Nigeria": "Onne 4.7 7.2", "Cameroon": "Douala 4.0 9.7", "Angola": "Luanda -8.8 13.2", "Namibia": "Walvis Bay -22.9 14.5", "South Africa": "Cape Town -33.9 18.4;Ngqura -33.8 25.7",
"Mozambique": "Maputo -25.97 32.6;Beira -19.8 34.9", "Tanzania": "Dar es Salaam -6.8 39.3", "Kenya": "Mombasa -4.1 39.7", "Somaliland": "Berbera 10.4 45.0", "Mauritius": "Port Louis -20.2 57.5", "Madagascar": "Toamasina -18.1 49.4",
"USA": "Oakland 37.8 -122.3;Seattle 47.6 -122.3;Tacoma 47.3 -122.4;Houston 29.7 -95.3;New Orleans 29.9 -90.1;Miami 25.8 -80.2;Charleston 32.8 -79.9;Norfolk 36.9 -76.3;Baltimore 39.3 -76.6;Boston 42.4 -71.0",
"Canada": "Vancouver 49.3 -123.1;Prince Rupert 54.3 -130.3;Halifax 44.6 -63.6;Montreal 45.5 -73.5", "Mexico": "Manzanillo 19.1 -104.3;Veracruz 19.2 -96.1;Lazaro Cardenas 17.9 -102.2",
"Panama": "Balboa 8.95 -79.6;Colon 9.35 -79.9", "Colombia": "Cartagena 10.4 -75.5;Buenaventura 3.9 -77.1", "Peru": "Callao -12.05 -77.15", "Ecuador": "Guayaquil -2.3 -79.9",
"Chile": "San Antonio -33.6 -71.6;Valparaiso -33.0 -71.6", "Argentina": "Buenos Aires -34.6 -58.4", "Uruguay": "Montevideo -34.9 -56.2",
"Brazil": "Santos -23.98 -46.3;Paranagua -25.5 -48.5;Rio Grande -32.0 -52.1;Itajai -26.9 -48.7;Suape -8.4 -35.0;Pecem -3.5 -38.8", "Jamaica": "Kingston 17.97 -76.8", "Dominican Republic": "Caucedo 18.4 -69.6", "Bahamas": "Freeport 26.5 -78.8",
"Australia": "Sydney -33.9 151.2;Melbourne -37.8 144.9;Brisbane -27.4 153.2;Fremantle -32.05 115.7;Adelaide -34.8 138.5;Port Hedland -20.3 118.6", "New Zealand": "Auckland -36.8 174.8;Tauranga -37.65 176.2;Lyttelton -43.6 172.7",
"Fiji": "Suva -18.1 178.4", "New Caledonia": "Noumea -22.3 166.4", "Papua New Guinea": "Port Moresby -9.5 147.1"}
CARRIERS = ["Evergreen Marine","HMM","Yang Ming","ZIM Integrated","Pacific International Lines","Wan Hai Lines","OOCL","APL","Hamburg Sud","Matson","SM Line","Sinokor","KMTC","Namsung Shipping","SITC",
"X-Press Feeders","Gold Star Line","Arkas Line","Turkon Line","Grimaldi Lines","Messina Line","CCNI","Seaboard Marine","Crowley Maritime","Tropical Shipping","Swire Shipping","Hoegh Autoliners",
"Wallenius Wilhelmsen","NYK Line","Mitsui OSK Lines","K Line","Safmarine","Emirates Shipping Line","Shipping Corporation of India","Great Eastern Shipping","Samudera Shipping","Salam Pacific",
"Interasia Lines","Heung-A Line","Unifeeder","Containerships","Finnlines","DFDS","Stena Line","Atlantic Container Line","King Ocean","Bengal Tiger Line","Transworld Shipping","Sealand Maersk Asia","Pan Ocean","Cosco Hong Kong"]
def region_of(lat, lon):
    if -170 < lon < -30: return "Americas"
    if -30 <= lon < 50 and lat > 12: return "Europe/Mediterranean" if lat > 30 else "Africa"
    if -30 <= lon < 52 and lat <= 12: return "Africa"
    if 112 < lon and lat < -10: return "Oceania"
    if 25 < lon < 60 and 12 < lat < 33: return "Middle East"
    return "Asia"

def scale_freight(target_ports=220, target_carriers=110, target_shipments=400):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        have = {r[0].lower() for r in c.execute("SELECT port_name FROM ports")}; n = c.execute("SELECT COUNT(*) FROM ports").fetchone()[0]
        for ctry, spec in PORTS.items():
            for e in spec.split(";"):
                if n >= target_ports + 40: break
                nm, la, lo = e.rsplit(" ", 2); name = f"{nm} Port"
                if name.lower() in have: continue
                n += 1; have.add(name.lower())
                c.execute("INSERT OR REPLACE INTO ports(port_id,port_name,country,congestion_index,avg_dwell_days,lat,lon,region) VALUES(?,?,?,?,?,?,?,?)",
                          (f"PORT-{n:03d}", name, ctry, round(R.uniform(1.0, 4.6), 1), round(R.uniform(1.5, 7.0), 1), float(la), float(lo), region_of(float(la), float(lo))))
        k = c.execute("SELECT COUNT(*) FROM carriers").fetchone()[0]; names = {r[0] for r in c.execute("SELECT name FROM carriers")}
        pool = [x for x in CARRIERS if x not in names] + [f"Regional Feeder Line {i:02d}" for i in range(1, 80)]
        for nm in pool:
            if k >= target_carriers: break
            k += 1
            c.execute("INSERT OR REPLACE INTO carriers(carrier_id,name,rating,on_time_pct,avg_cost_index,risk_level) VALUES(?,?,?,?,?,?)",
                      (f"CAR-{k:03d}", nm, round(R.uniform(3.2, 4.9), 2), round(R.uniform(68, 97), 1), round(R.uniform(.8, 1.25), 2), R.choice(["Low", "Low", "Medium", "High"])))
        ports = c.execute("SELECT port_name FROM ports").fetchall(); cars = [r[0] for r in c.execute("SELECT name FROM carriers")]
        s = c.execute("SELECT COUNT(*) FROM shipments").fetchone()[0]
        while s < target_shipments:
            s += 1; a, b = R.sample(ports, 2)
            c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)",
                      (f"SHP-{s:04d}", a[0], b[0], R.choice(cars), R.choices(["In Transit", "Delivered", "Customs Hold", "Port Congestion Delay", "Anchorage Pending"], [55, 20, 8, 10, 7])[0],
                       (datetime.date.today() + datetime.timedelta(days=R.randint(2, 40))).isoformat(), round(R.uniform(800, 28000), 1), f"HS-{R.choice([30,84,85,87,10,62,27,39,72,22])}{R.randint(10,99)}", round(R.uniform(.03, .6), 2)))

CITIES = {"Mumbai": (19.08, 72.88), "Delhi": (28.61, 77.21), "Bengaluru": (12.97, 77.59), "Chennai": (13.08, 80.27), "Hyderabad": (17.39, 78.49), "Pune": (18.52, 73.86), "Ahmedabad": (23.03, 72.57),
"Jaipur": (26.91, 75.79), "Kolkata": (22.57, 88.36), "Surat": (21.17, 72.83), "Kochi": (9.93, 76.27), "Coimbatore": (11.02, 76.96), "Lucknow": (26.85, 80.95), "Nagpur": (21.15, 79.09), "Indore": (22.72, 75.86),
"Bhopal": (23.26, 77.41), "Patna": (25.59, 85.14), "Chandigarh": (30.73, 76.78), "Visakhapatnam": (17.69, 83.22), "Vadodara": (22.31, 73.18), "Thiruvananthapuram": (8.52, 76.94), "Mysuru": (12.30, 76.64),
"Guwahati": (26.14, 91.74), "Bhubaneswar": (20.30, 85.82), "Ranchi": (23.34, 85.31), "Raipur": (21.25, 81.63), "Dehradun": (30.32, 78.03), "Goa": (15.50, 73.83), "Madurai": (9.93, 78.12), "Nashik": (19.99, 73.79)}
SKUS = [("Buns", "Food"), ("Cooking Oil", "Food"), ("Sauce", "Food"), ("Frozen Fries", "Food"), ("Chicken", "Food"), ("Cheese", "Food"), ("Lettuce", "Food"), ("Packaging", "Supplies"), ("Beverage Syrup", "Beverage"), ("Potatoes", "Food"), ("Flour", "Food"), ("Spice Mix", "Food")]
def scale_franchise(outlets=120, staff=1250, per_outlet_sku=12):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        c.executescript("""CREATE TABLE IF NOT EXISTS outlet_geo(outlet_id TEXT PRIMARY KEY, lat REAL, lon REAL);
        CREATE TABLE IF NOT EXISTS customers_fr(customer_id TEXT PRIMARY KEY, outlet_id TEXT, total_spend_90d REAL, visit_freq REAL, avg_order_value REAL, days_since_last_visit INT);""")
        n = c.execute("SELECT COUNT(*) FROM outlets").fetchone()[0]; tiers = ["Metro Flagship", "Urban", "Suburban", "Highway"]; cities = list(CITIES)
        while n < outlets:
            n += 1; oid = f"OUT-{n:03d}"; city = R.choice(cities); rev = R.uniform(900000, 5200000)
            c.execute("INSERT OR REPLACE INTO outlets VALUES(?,?,?,?,?,?,?,?)", (oid, f"Franchise Outlet {n:03d}", city, R.choice(tiers), rev, rev * R.uniform(.55, .82), round(R.uniform(3.0, 4.9), 2), R.randint(12, 30)))
        for oid, city in c.execute("SELECT outlet_id, location FROM outlets").fetchall():
            if not c.execute("SELECT 1 FROM outlet_geo WHERE outlet_id=?", (oid,)).fetchone():
                la, lo = CITIES.get(city, CITIES["Mumbai"]); c.execute("INSERT INTO outlet_geo VALUES(?,?,?)", (oid, la + R.uniform(-.06, .06), lo + R.uniform(-.06, .06)))
        s = c.execute("SELECT COUNT(*) FROM staff").fetchone()[0]; oids = [r[0] for r in c.execute("SELECT outlet_id FROM outlets")]
        while s < staff:
            s += 1; sal = R.uniform(18000, 80000); sat = R.randint(1, 5); ot = R.uniform(0, 35)
            p = max(.05, min(.95, .9 - sal / 100000 - sat / 10 + ot / 100))
            c.execute("INSERT OR REPLACE INTO staff VALUES(?,?,?,?,?,?,?,?,?,?,?)", (f"STF-{s:04d}", R.choice(oids), f"Employee {s:04d}", R.choice(["Crew", "Crew", "Cashier", "Cook", "Shift Lead", "Store Manager"]), sal, round(ot, 1), sat, R.randint(19, 58), R.randint(0, 12), R.randint(1, 5), round(p, 2)))
        for oid in oids:
            have = {r[0] for r in c.execute("SELECT sku_name FROM inventory WHERE outlet_id=?", (oid,))}
            for sku, cat in SKUS[:per_outlet_sku]:
                if sku in have or sku in {"Oil"} and "Cooking Oil" == sku: continue
                wk = R.uniform(80, 700); lead = R.randint(2, 9)
                c.execute("INSERT OR REPLACE INTO inventory VALUES(?,?,?,?,?,?,?,?,?)", (f"INV-{oid[4:]}-{sku[:3].upper()}", oid, sku, cat, int(wk / 7 * (lead + R.uniform(2, 12))), int(wk / 7 * lead), round(wk, 1), lead, .1))
            if c.execute("SELECT COUNT(*) FROM marketing WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    c.execute("INSERT OR REPLACE INTO marketing VALUES(?,?,?,?,?,?,?,?,?,?)", (f"CMP-{oid[4:]}-{j}", oid, f"Campaign {oid[4:]}-{j}", R.choice(["Social", "Email", "Influencer", "Offline"]), R.uniform(20000, 120000), round(R.uniform(1.2, 5.2), 2), R.randint(8000, 60000), R.randint(200, 2500), "2024-01-01", "2024-12-31"))
            if c.execute("SELECT COUNT(*) FROM audits WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    sc = max(40, min(100, R.gauss(84, 10)))
                    c.execute("INSERT OR REPLACE INTO audits VALUES(?,?,?,?,?,?,?,?)", (f"AUD-{oid[4:]}-{j}", oid, (datetime.date.today() - datetime.timedelta(days=30 * (j + 1))).isoformat(), sc, int((100 - sc) / 6), R.choice(["Food Safety", "Hygiene", "SOP Compliance"]), "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", "Monthly audit"))
            if c.execute("SELECT COUNT(*) FROM feedback WHERE outlet_id=?", (oid,)).fetchone()[0] < 8:
                for j in range(8):
                    r = R.choices([1, 2, 3, 4, 5], [5, 8, 15, 35, 37])[0]
                    c.execute("INSERT OR REPLACE INTO feedback VALUES(?,?,?,?,?,?)", (f"FB-{oid[4:]}-{j}", oid, r, R.choice(["Great taste", "Slow service", "Clean ambience", "Staff was rude", "Cold food", "Good value", "Long wait time", "Dirty tables"]), datetime.date.today().isoformat(), (r - 3) / 2))
            if c.execute("SELECT COUNT(*) FROM customers_fr WHERE outlet_id=?", (oid,)).fetchone()[0] < 30:
                for j in range(30):
                    seg = R.choices(["hv", "freq", "churn"], [.2, .4, .4])[0]
                    f = {"hv": (6, 12), "freq": (3, 8), "churn": (.2, 1.5)}[seg]; aov = R.uniform(250, 900) if seg == "hv" else R.uniform(150, 450)
                    fr = R.uniform(*f); c.execute("INSERT OR REPLACE INTO customers_fr VALUES(?,?,?,?,?,?)", (f"C-{oid[4:]}-{j}", oid, round(fr * aov * 3, 0), round(fr, 1), round(aov, 0), R.randint(60, 120) if seg == "churn" else R.randint(1, 20)))


In [ ]:
%%writefile /content/app/seanav.py
"""Sea navigation: an ocean mask built from the world coastline (Natural Earth 110m land, bundled with the app), real canal/strait corridors
(Suez, Panama, Malacca, Gibraltar, Dover, Danish straits, Bosphorus, Hormuz, Sunda, Lombok...) and Dijkstra routing over a 1 degree water grid.
Routes are then straightened with a line-of-sight check on a finer 0.25 degree mask, so no leg crosses land. Everything is pure numpy/scipy."""
import os, json, math, heapq, threading
import numpy as np
from matplotlib.path import Path as MPath
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

HERE = os.path.dirname(os.path.abspath(__file__))
TOPO = [os.path.join(HERE, "static", "vendor", "world.topo.json")]
R_NM = 3440.065
ACCESS = {   # river / bay / canal-side ports: where a ship actually enters open water (lat, lon). Matched on the port name.
    "wuhan": (31.2, 122.6), "nanjing": (31.2, 122.6), "taicang": (31.3, 122.4), "shanghai": (31.1, 122.4), "guangzhou": (22.0, 114.0), "dongguan": (22.0, 114.0), "zhuhai": (21.9, 113.8),
    "tokyo": (34.9, 139.9), "yokohama": (34.9, 139.9), "chiba": (34.9, 139.9), "nagoya": (34.2, 137.0), "osaka": (34.2, 135.0), "kobe": (34.2, 135.0), "hamburg": (54.1, 8.2), "antwerp": (51.45, 3.4),
    "houston": (29.1, -94.6), "new orleans": (28.9, -89.3), "baltimore": (37.0, -75.9), "norfolk": (36.9, -75.9), "montreal": (49.3, -66.0), "palembang": (-2.0, 105.4), "yangon": (15.7, 96.3), "mongla": (21.6, 89.5), "kolkata": (21.6, 88.2),
    "ho chi minh": (10.3, 107.1), "kandla": (22.8, 69.6), "navlakhi": (22.9, 70.0), "dahej": (21.7, 72.0), "hazira": (21.0, 72.4), "ambarli": (40.9, 28.3), "izmir": (38.4, 26.6), "amsterdam": (52.5, 4.3), "london gateway": (51.5, 1.2),
    "oslo": (59.2, 10.6), "douala": (3.9, 9.4), "fuzhou": (26.0, 119.9), "lianyungang": (34.7, 119.6), "port klang": (2.9, 101.0), "belawan": (4.1, 98.6), "rotterdam": (51.95, 3.9), "buenos aires": (-35.0, -57.0), "montevideo": (-35.0, -56.2), "guayaquil": (-2.8, -80.4),
    "seattle": (47.9, -122.7), "tacoma": (47.8, -122.8), "vancouver": (49.2, -123.4), "oakland": (37.8, -122.6), "kuching": (2.0, 110.3), "bangkok": (13.3, 100.6), "haikou": (20.3, 110.3), "port said": (31.4, 32.3), "ain sokhna": (29.6, 32.6), "aqaba": (28.9, 34.9),
}
CORRIDORS = {   # name -> centre line (lat, lon); cells along it (and 0.3 degree either side) are forced to water
    "Suez Canal": [(31.9, 32.3), (31.3, 32.33), (30.6, 32.34), (30.0, 32.55), (29.4, 32.55), (28.9, 32.8), (28.3, 33.2)],
    "Panama Canal": [(9.9, -79.95), (9.35, -79.88), (9.05, -79.6), (8.6, -79.5), (8.0, -79.4)],
    "Strait of Malacca": [(1.1, 104.3), (1.25, 103.7), (1.9, 102.6), (2.7, 101.5), (3.6, 100.5), (4.6, 99.9), (5.6, 99.0), (6.3, 97.5)],
    "Singapore Strait": [(1.2, 104.0), (1.4, 104.6), (2.2, 105.5)],
    "Strait of Gibraltar": [(36.5, -7.0), (36.0, -5.8), (36.0, -5.3), (36.2, -4.5), (36.4, -3.5)],
    "Dover Strait": [(49.6, -6.0), (49.8, -3.0), (50.4, -0.5), (50.9, 1.1), (51.4, 2.1), (52.0, 3.0)],
    "Danish Straits": [(57.7, 7.5), (57.8, 10.3), (57.0, 11.6), (56.2, 12.5), (55.5, 12.8), (55.0, 13.0), (54.9, 14.0), (55.2, 16.0), (56.0, 18.0), (57.5, 19.7), (58.8, 21.0), (59.6, 23.5), (59.9, 26.0), (60.0, 29.0)],
    "Bosphorus": [(39.9, 25.9), (40.2, 26.5), (40.7, 27.7), (41.0, 28.9), (41.4, 29.15), (42.0, 29.4), (42.8, 29.8)],
    "Strait of Hormuz": [(24.5, 58.6), (25.7, 56.8), (26.5, 56.3), (26.4, 54.8), (26.6, 52.8), (27.4, 51.0), (28.8, 49.8), (29.6, 48.9)],
    "Bab-el-Mandeb": [(12.0, 43.9), (12.7, 43.3), (13.6, 42.6)],
    "Sunda Strait": [(-5.7, 105.9), (-6.1, 105.8), (-6.8, 105.3)],
    "Lombok Strait": [(-7.6, 115.7), (-8.4, 115.7), (-9.2, 115.6)],
    "Messina": [(38.4, 15.6), (38.0, 15.6), (37.5, 15.3)],
    "Seto Inland Sea": [(33.9, 135.0), (34.4, 134.9), (34.5, 135.1)],
    "Korea Strait": [(35.0, 129.2), (34.3, 129.8), (33.5, 130.3)],
    "Aegean": [(36.2, 22.8), (36.6, 24.0), (37.5, 24.2), (38.5, 24.6), (39.5, 24.6), (40.0, 25.8)],
    "Adriatic": [(39.8, 18.9), (41.0, 18.7), (42.5, 17.5), (43.8, 15.3), (45.0, 13.5), (45.6, 13.4)],
    "Red Sea": [(27.5, 34.2), (24.0, 36.2), (20.0, 38.4), (16.0, 41.2), (13.5, 42.8)],
    "Gulf of Aden": [(12.5, 43.5), (12.0, 47.0), (12.4, 51.5), (12.0, 54.5)],
    "Persian Gulf": [(24.5, 58.6), (25.5, 56.7), (26.4, 54.5), (27.0, 52.0), (28.0, 50.5), (29.3, 48.8)],
    "Gulf of Mexico": [(24.0, -84.0), (23.0, -87.0), (26.5, -90.0), (28.0, -91.0), (28.0, -95.0), (26.0, -97.0), (21.0, -97.0), (19.0, -95.0)],
    "Baltic South": [(54.2, 18.9), (54.6, 19.5)],
}
BLOCK_GROUPS = {"suez": ["Suez Canal"], "panama": ["Panama Canal"], "malacca": ["Strait of Malacca"], "hormuz": ["Strait of Hormuz"], "bosphorus": ["Bosphorus"]}
CHOKE_LABEL = {"Suez Canal": "Suez Canal", "Panama Canal": "Panama Canal", "Strait of Malacca": "Strait of Malacca", "Strait of Gibraltar": "Gibraltar", "Dover Strait": "English Channel", "Strait of Hormuz": "Hormuz", "Bab-el-Mandeb": "Bab-el-Mandeb", "Bosphorus": "Bosphorus", "Danish Straits": "Danish Straits", "Sunda Strait": "Sunda Strait", "Lombok Strait": "Lombok Strait"}

def _hav(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1])); h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return R_NM * 2 * math.asin(min(1.0, math.sqrt(h)))

def _land_polys():
    p = next((x for x in TOPO if os.path.exists(x)), None)
    if not p: raise FileNotFoundError("world.topo.json (bundled with the app) not found")
    t = json.load(open(p)); sx, sy = t["transform"]["scale"]; tx, ty = t["transform"]["translate"]; arcs = []
    for a in t["arcs"]:
        x = y = 0; pts = []
        for dx, dy in a: x += dx; y += dy; pts.append((x * sx + tx, y * sy + ty))
        arcs.append(pts)
    def ring(idx):
        out = []
        for i in idx:
            seg = arcs[i] if i >= 0 else arcs[~i][::-1]; out += seg if not out else seg[1:]
        return out
    polys = []
    def geoms(o):
        if o["type"] == "GeometryCollection":
            for g in o["geometries"]: yield from geoms(g)
        else: yield o
    for g in geoms(t["objects"]["land"]):
        pl = g["arcs"] if g["type"] == "MultiPolygon" else [g["arcs"]]
        for poly in pl: polys.append([_unwrap_ring(ring(r)) for r in poly])        # [outer, holes...]
    return polys
def _unwrap_ring(pts):
    """Rings that cross the antimeridian jump by ~360 degrees between two points; make the longitude continuous (it may then exceed +-180)."""
    out = [pts[0]]
    for x, y in pts[1:]:
        px = out[-1][0]
        while x - px > 180: x -= 360
        while x - px < -180: x += 360
        out.append((x, y))
    return out

def gc_arr(a, b, step_nm=60.0, include_ends=True):
    """Vectorised great-circle sampling: (lat array, lon array); longitudes continuous with a's."""
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1]))
    v1 = np.array([math.cos(la1) * math.cos(lo1), math.cos(la1) * math.sin(lo1), math.sin(la1)]); v2 = np.array([math.cos(la2) * math.cos(lo2), math.cos(la2) * math.sin(lo2), math.sin(la2)])
    om = math.acos(max(-1.0, min(1.0, float(v1 @ v2)))); n = max(1, int(om * R_NM / step_nm)); k = np.arange(0 if include_ends else 1, n + 1 if include_ends else n); t = k / n
    if om > 1e-9: w1 = np.sin((1 - t) * om) / math.sin(om); w2 = np.sin(t * om) / math.sin(om)
    else: w1, w2 = 1 - t, t
    v = w1[:, None] * v1[None, :] + w2[:, None] * v2[None, :]; la = np.degrees(np.arcsin(np.clip(v[:, 2], -1, 1))); lo = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    lo = a[1] + (((lo - a[1]) + 180) % 360 - 180)          # continuous with a's longitude
    return la, lo
def gc_points(a, b, step_nm=60.0, include_ends=True):
    """Points along the great circle from a to b (lat, lon), spaced about step_nm apart; longitudes continuous with a's."""
    la, lo = gc_arr(a, b, step_nm, include_ends); out = list(zip(la.tolist(), lo.tolist()))
    for q in range(1, len(out)):          # keep successive longitudes continuous (no jumps > 180)
        while out[q][1] - out[q - 1][1] > 180: out[q] = (out[q][0], out[q][1] - 360)
        while out[q][1] - out[q - 1][1] < -180: out[q] = (out[q][0], out[q][1] + 360)
    return out

class Sea:
    def __init__(self, step=1.0, fine=0.25, latmin=-78.0, latmax=82.0):
        self.step, self.fine, self.latmin, self.latmax = step, fine, latmin, latmax
        self.lats = np.arange(latmin + step / 2, latmax, step); self.lons = np.arange(-180 + step / 2, 180, step); self.nlat, self.nlon = len(self.lats), len(self.lons)
        self.flats = np.arange(latmin + fine / 2, latmax, fine); self.flons = np.arange(-180 + fine / 2, 180, fine)
        polys = _land_polys(); self.land = self._raster(polys, self.lats, self.lons); self.fland = self._raster(polys, self.flats, self.flons)
        self.corr_cells = {}                                                   # corridor name -> set of coarse node ids
        for name, line in CORRIDORS.items(): self._carve(name, line)
        pol = (self.lats < -52) | (self.lats > 76); drake = (self.lons > -82) & (self.lons < -48)
        self.land[np.ix_(pol & (self.lats < 0), ~drake)] = True; self.land[np.ix_(self.lats > 76, np.ones(self.nlon, bool))] = True
        fpol = (self.flats < -52); fdrake = (self.flons > -82) & (self.flons < -48); self.fland[np.ix_(fpol, ~fdrake)] = True; self.fland[self.flats > 76, :] = True
        self.ocean = ~self.land; self.nid = -np.ones(self.land.shape, int); self.nid[self.ocean] = np.arange(int(self.ocean.sum()))
        self.nodes = np.argwhere(self.ocean); self.n = len(self.nodes); self._build_graph()
        self._cache = {}; self._variants = {}; self._lock = threading.RLock()
    def _raster(self, polys, lats, lons):
        LON, LAT = np.meshgrid(lons, lats); pts = np.c_[LON.ravel(), LAT.ravel()]; inside = np.zeros(len(pts), bool)
        for rings in polys:
            outer = np.array(rings[0]); x0, y0 = outer.min(0); x1, y1 = outer.max(0)
            for off in (0, 360, -360):
                mo = (pts[:, 0] + off >= x0) & (pts[:, 0] + off <= x1) & (pts[:, 1] >= y0) & (pts[:, 1] <= y1)
                if not mo.any(): continue
                idx = np.where(mo)[0]; q = pts[idx] + np.array([off, 0]); ins = MPath(outer).contains_points(q)
                for h in rings[1:]: ins &= ~MPath(np.array(h)).contains_points(q)
                inside[idx[ins]] = True
        return inside.reshape(len(lats), len(lons))
    def _carve(self, name, line):
        cells = set()
        for (a, b) in zip(line[:-1], line[1:]):
            n = max(2, int(_hav(a, b) / 8))
            for t in np.linspace(0, 1, n):
                la, lo = a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t
                i = int((la - self.latmin) / self.step); j = int((lo + 180) / self.step) % self.nlon
                if 0 <= i < self.nlat: self.land[i, j] = False; cells.add((i, j))
                fi = int((la - self.latmin) / self.fine); fj = int((lo + 180) / self.fine) % len(self.flons)
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        a_, b_ = fi + di, (fj + dj) % len(self.flons)
                        if 0 <= a_ < len(self.flats): self.fland[a_, b_] = False
        self.corr_cells[name] = cells
    def _build_graph(self):
        rows, cols, w = [], [], []
        for k, (i, j) in enumerate(self.nodes):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0: continue
                    a, b = i + di, (j + dj) % self.nlon
                    if not (0 <= a < self.nlat) or not self.ocean[a, b]: continue
                    if di != 0 and dj != 0 and not (self.ocean[i, b] or self.ocean[a, j]): continue         # no diagonal squeeze between two land cells
                    rows.append(k); cols.append(self.nid[a, b]); w.append(_hav((self.lats[i], self.lons[j]), (self.lats[a], self.lons[b])))
        self.A = csr_matrix((w, (rows, cols)), shape=(self.n, self.n))
        nc, lab = connected_components(self.A, directed=False); self.main = lab == np.bincount(lab).argmax(); self.comp = lab
    # ------------------------------------------------------------ helpers
    def is_land(self, la, lo):
        i = int((la - self.latmin) / self.fine); j = int(((lo + 180) % 360) / self.fine)
        return bool(self.fland[i, j]) if 0 <= i < len(self.flats) else True
    def dist_to_water_nm(self, la, lo, maxr=2.0):
        """Distance in nm from a point to the nearest water cell of the fine mask (0 if the point is in water)."""
        i0 = int((la - self.latmin) / self.fine); j0 = int(((lo + 180) % 360) / self.fine); r = int(maxr / self.fine); best = None
        for di in range(-r, r + 1):
            for dj in range(-r, r + 1):
                i, j = i0 + di, (j0 + dj) % len(self.flons)
                if 0 <= i < len(self.flats) and not self.fland[i, j]:
                    d = _hav((la, lo), (self.flats[i], self.flons[j])); best = d if best is None or d < best else best
        return best
    def _clear(self, a, b, ends=0.6):
        la, lo = gc_arr(a, b, 7.0)
        keep = ~(((np.abs(la - a[0]) < ends) & (np.abs(lo - a[1]) < ends)) | ((np.abs(la - b[0]) < ends) & (np.abs(lo - b[1]) < ends)))
        la, lo = la[keep], lo[keep]
        if not len(la): return True
        i = ((la - self.latmin) / self.fine).astype(int); j = (((lo + 180) % 360) / self.fine).astype(int); ok = (i >= 0) & (i < len(self.flats))
        if not ok.all(): return False
        return not bool(self.fland[i, j].any())
    def _near_nodes(self, la, lo, k=12, main_only=True):
        d = (self.lats[self.nodes[:, 0]] - la) ** 2 + ((((self.lons[self.nodes[:, 1]] - lo + 180) % 360) - 180) * math.cos(math.radians(la))) ** 2
        if main_only: d = np.where(self.main, d, 1e9)
        return np.argsort(d)[:k]
    def _snap(self, la, lo):
        near = self._near_nodes(la, lo)
        for k in near:
            p = (float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]]))
            if self._clear((la, lo), p, 0.45): return int(k)
        return int(near[0])
    def _graph_for(self, blocked):
        key = tuple(sorted(blocked))
        if not key: return self.A
        with self._lock:
            if key in self._variants: return self._variants[key]
            ok = np.ones(self.n, bool)
            for g in key:
                for name in BLOCK_GROUPS.get(g, [g]):
                    for (i, j) in self.corr_cells.get(name, ()):
                        if self.nid[i, j] >= 0: ok[self.nid[i, j]] = False
            D = csr_matrix(ok.astype(float)).T if False else None
            A = self.A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros(); self._variants[key] = A; return A
    # ------------------------------------------------------------ public API
    def route(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """o, d = (lat, lon). Returns {"pts": [(lat, lon), ...] (longitudes unwrapped), "nm": float, "via": [names]} or None."""
        key = (round(o[0], 2), round(o[1], 2), round(d[0], 2), round(d[1], 2), oa, da, tuple(sorted(blocked)), tuple((round(s[0]), round(s[1]), round(s[2])) for s in storms))
        with self._lock:
            if key in self._cache: return self._cache[key]
        A = self._graph_for(blocked)
        if storms:
            ok = np.ones(self.n, bool)
            for (sla, slo, rn) in storms:
                dd = np.array([_hav((sla, slo), (self.lats[i], self.lons[j])) for i, j in self.nodes]) if False else None
                la_ = self.lats[self.nodes[:, 0]]; lo_ = self.lons[self.nodes[:, 1]]
                dist = R_NM * np.arccos(np.clip(np.sin(np.radians(la_)) * math.sin(math.radians(sla)) + np.cos(np.radians(la_)) * math.cos(math.radians(sla)) * np.cos(np.radians(lo_ - slo)), -1, 1)); ok &= dist > rn
            A = A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros()
        s, t = self._snap(*(oa or o)), self._snap(*(da or d))
        dist, pred = dijkstra(A, directed=True, indices=s, return_predecessors=True)
        if not np.isfinite(dist[t]): res = None
        else:
            path = [t]
            while path[-1] != s: path.append(pred[path[-1]])
            path.reverse(); pts = [o] + ([oa] if oa else []) + [(float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]])) for k in path] + ([da] if da else []) + [d]; pts = self._unwrap(self._straighten(pts)); pts = self._densify(pts)
            res = {"pts": [(round(a, 3), round(b, 3)) for a, b in pts], "nm": round(sum(_hav(pts[i], pts[i + 1]) for i in range(len(pts) - 1)), 1), "via": self._via(pts)}
        with self._lock: self._cache[key] = res
        return res
    def _straighten(self, pts):
        out = [pts[0]]; i = 0
        while i < len(pts) - 1:
            j = len(pts) - 1
            while j > i + 1 and not self._clear(pts[i], pts[j], 0.5): j -= 1
            out.append(pts[j]); i = j
        return out
    @staticmethod
    def _densify(pts, step=90.0):
        out = [pts[0]]
        for a, b in zip(pts[:-1], pts[1:]): out += gc_points(a, b, step)[1:]
        return out
    @staticmethod
    def _unwrap(pts):
        out = [pts[0]]
        for la, lo in pts[1:]:
            prev = out[-1][1]
            while lo - prev > 180: lo -= 360
            while lo - prev < -180: lo += 360
            out.append((la, lo))
        return out
    def _via(self, pts):
        names = []; dense = []
        for a, b in zip(pts[:-1], pts[1:]):
            for t in np.linspace(0, 1, max(2, int(_hav(a, b) / 15))): dense.append((a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t))
        order = []
        for la, lo in dense:
            i = int((la - self.latmin) / self.step); j = int(((lo + 180) % 360) / self.step) % self.nlon
            for name, cells in self.corr_cells.items():
                if name in CHOKE_LABEL and (i, j) in cells and CHOKE_LABEL[name] not in order: order.append(CHOKE_LABEL[name])
        if any(-47 < la < -30 and 10 < (lo % 360 if lo < 0 else lo) < 35 for la, lo in dense): order.append("Cape of Good Hope")
        if any(la < -50 and (-80 < lo < -55) for la, lo in dense): order.append("Cape Horn / Drake Passage")
        return order
    def options(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """Up to three sea routes: shortest, and the alternatives that avoid Suez / Panama / Malacca when the shortest uses them."""
        base = self.route(o, d, tuple(blocked), storms, oa, da); out = []
        if not base: return out
        def label(r):
            v = r["via"]; return ("via " + " + ".join(v[:3])) if v else "direct sea route"
        out.append({"name": label(base), "pts": base["pts"], "nm": base["nm"], "via": base["via"]})
        for g, nm_ in (("suez", "Suez Canal"), ("panama", "Panama Canal"), ("malacca", "Strait of Malacca")):
            if nm_ in base["via"] and g not in blocked:
                alt = self.route(o, d, tuple(blocked) + (g,), storms, oa, da)
                if alt and abs(alt["nm"] - base["nm"]) > 50 and all(abs(alt["nm"] - x["nm"]) > 50 for x in out): out.append({"name": label(alt) + f" (without {nm_})", "pts": alt["pts"], "nm": alt["nm"], "via": alt["via"]})
        return out

def access_for(name):
    n = (name or "").lower()
    for k, v in ACCESS.items():
        if k in n: return v
    return None
_SEA = None; _SL = threading.Lock(); _VER = "v5"
def _cache_path():
    d = os.environ.get("APP_DATA_DIR") or os.path.dirname(os.path.abspath(__file__)); return os.path.join(d, f"seanav_{_VER}.pkl")
def sea():
    """Singleton. The raster + graph (about 6 s to build) is pickled next to the database, so later starts load in well under a second."""
    global _SEA
    with _SL:
        if _SEA is None:
            import pickle
            p = _cache_path()
            try:
                with open(p, "rb") as f: _SEA = pickle.load(f)
                _SEA._cache = {}; _SEA._variants = {}; _SEA._lock = threading.RLock()
            except Exception:
                _SEA = Sea()
                try:
                    st = {k: v for k, v in _SEA.__dict__.items() if k not in ("_lock", "_cache", "_variants")}
                    class _P: pass
                    o = Sea.__new__(Sea); o.__dict__.update(st)
                    with open(p + ".tmp", "wb") as f: pickle.dump(o, f, protocol=4)
                    os.replace(p + ".tmp", p)
                except Exception: pass
    return _SEA


In [ ]:
%%writefile /content/app/seed_data.py
import random, sqlite3, datetime
import pandas as pd
from db import get_conn

BASE_PORTS = [
    ("JNPT Nhava Sheva (Mumbai)", "India", 3.4, 4, 18.95, 72.95, "Asia"),
    ("Mundra Port", "India", 2.8, 3, 22.84, 69.70, "Asia"),
    ("Chennai Port", "India", 3.1, 4, 13.10, 80.30, "Asia"),
    ("Tuticorin VOC Port", "India", 2.5, 3, 8.75, 78.18, "Asia"),
    ("Cochin Port", "India", 2.2, 3, 9.96, 76.26, "Asia"),
    ("Visakhapatnam Port", "India", 2.9, 3, 17.68, 83.28, "Asia"),
    ("Kolkata Haldia Port", "India", 3.5, 5, 22.03, 88.11, "Asia"),
    ("Kandla Deendayal Port", "India", 3.0, 4, 23.01, 70.22, "Asia"),
    ("New Mangalore Port", "India", 2.3, 3, 12.92, 74.81, "Asia"),
    ("Paradip Port", "India", 3.2, 4, 20.26, 86.67, "Asia"),
    ("Shanghai Port", "China", 4.2, 5, 31.23, 121.47, "Asia"),
    ("Singapore Port", "Singapore", 1.2, 2, 1.29, 103.85, "Asia"),
    ("Busan Port", "South Korea", 1.8, 3, 35.10, 129.04, "Asia"),
    ("Tokyo Port", "Japan", 2.2, 3, 35.62, 139.77, "Asia"),
    ("Colombo Port", "Sri Lanka", 2.7, 3, 6.94, 79.84, "Asia"),
    ("Dubai Jebel Ali Port", "UAE", 1.9, 2, 25.20, 55.27, "Middle East"),
    ("Rotterdam Port", "Netherlands", 1.5, 2, 51.92, 4.47, "Europe"),
    ("Antwerp Port", "Belgium", 2.6, 3, 51.22, 4.40, "Europe"),
    ("Hamburg Port", "Germany", 2.5, 3, 53.55, 9.99, "Europe"),
    ("Los Angeles Port", "USA", 3.6, 4, 33.74, -118.27, "Americas")
]

def safe_exec(conn, sql, params=()):
    try:
        conn.execute(sql, params)
    except Exception as e:
        pass

def seed_all():
    with get_conn() as conn:
        # 1. Ports
        safe_exec(conn, "DELETE FROM ports;")
        for i, p in enumerate(BASE_PORTS, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO ports (port_id, port_name, country, congestion_index, avg_dwell_days, lat, lon, region) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"PORT-{i:03d}", p[0], p[1], p[2], p[3], p[4], p[5], p[6])
            )

        # 2. Demo accounts — one per role, INCLUDING Admin, all seeded the same guaranteed way.
        # Admin uses your ADMIN_EMAIL_ID/ADMIN_PASSWORD secrets if set, else falls back to a fixed default
        # (admin@infosys.com / Admin@123) — same predictable pattern as the other 3 roles.
        try:
            import hashlib
            from auth import hash_password
            from config import ADMIN_EMAIL_ID, ADMIN_PASSWORD

            demo_pw = hash_password("Demo@1234")
            demo_ans = hashlib.sha256("infosys".encode("utf-8")).hexdigest()
            admin_pw = hash_password(ADMIN_PASSWORD)
            admin_ans = hashlib.sha256("infosys".encode("utf-8")).hexdigest()

            demo_users = [
                (1, "admin",    ADMIN_EMAIL_ID,        "Admin",            admin_pw),
                (2, "owner",    "owner@infosys.com",   "Franchise Owner",  demo_pw),
                (3, "manager",  "manager@infosys.com", "Store Manager",    demo_pw),
                (4, "staff",    "staff@infosys.com",   "Staff",             demo_pw),
                (5, "auditor",  "auditor@infosys.com", "Auditor",           demo_pw),
            ]

            # Upsert demo accounts. This is intentionally limited to the
            # authentication fields so stale passwords/lock states cannot
            # prevent non-admin users from logging in.
            for uid, uname, uemail, urole, pw_hash in demo_users:
                safe_exec(
                    conn,
                    "INSERT INTO users (id, username, email, password_hash, role, security_question, "
                    "security_answer, failed_attempts, lock_until, account_status) "
                    "VALUES (?, ?, ?, ?, ?, 'What is your favorite book/movie?', ?, 0, NULL, 'active') "
                    "ON CONFLICT(id) DO UPDATE SET "
                    "username=excluded.username, email=excluded.email, password_hash=excluded.password_hash, "
                    "role=excluded.role, security_question=excluded.security_question, "
                    "security_answer=excluded.security_answer, failed_attempts=0, lock_until=NULL, "
                    "account_status='active';",
                    (uid, uname, uemail, pw_hash, urole,
                     admin_ans if urole == "Admin" else demo_ans)
                )

            # Assign Store Manager to first outlet
            try:
                first_out = conn.execute("SELECT outlet_id FROM outlets ORDER BY outlet_id LIMIT 1").fetchone()
                if first_out:
                    conn.execute("UPDATE users SET outlet_id=? WHERE username='manager' OR email='manager@infosys.com'", (first_out[0],))
                    conn.execute("INSERT OR REPLACE INTO user_outlet_assignments(username, outlet_id) VALUES('manager', ?)", (first_out[0],))
            except Exception:
                pass

        except Exception:
            pass

        # 3. Shipments
        safe_exec(conn, "DELETE FROM shipments;")
        carriers_list = ["Maersk Line", "MSC Cargo", "CMA CGM", "COSCO Shipping", "Hapag-Lloyd", "ONE Ocean Express"]
        statuses = ["In Transit", "Customs Hold", "Delivered", "Port Congestion Delay", "Anchorage Pending"]
        cargos = ["Electronics", "Pharmaceuticals", "Automotive Parts", "Textiles", "Heavy Machinery", "Perishables"]

        for i in range(1, 101):
            p1 = BASE_PORTS[i % len(BASE_PORTS)][0]
            p2 = BASE_PORTS[(i+3) % len(BASE_PORTS)][0]
            shp_id = f"SHP-{i:04d}"
            weight = round(random.uniform(500.0, 45000.0), 1)
            dist = round(random.uniform(800.0, 18000.0), 1)
            sev = random.randint(1, 5)
            ch_prob = round(random.uniform(0.02, 0.45), 2)
            cong = round(random.uniform(1.0, 4.8), 1)
            d_risk = round((cong / 5.0) * 0.5 + (ch_prob) * 0.3 + (sev / 5.0) * 0.2, 2)
            co2 = round(weight * dist * 0.00012, 1)
            margin = round(random.uniform(8.5, 28.0), 1)
            dwell = random.randint(1, 8)
            cargo = random.choice(cargos)
            hs = f"HS-{random.randint(8400, 8900)}"

            safe_exec(conn,
                "INSERT OR REPLACE INTO shipments (shipment_id, origin_port, dest_port, carrier, status, weight_kg, distance_km, weather_severity, customs_hold_prob, congestion_index, predicted_delay_risk, co2_emissions_kg, freight_margin, port_dwell_days, cargo_type, hs_code) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (shp_id, p1, p2, random.choice(carriers_list), random.choice(statuses), weight, dist, sev, ch_prob, cong, d_risk, co2, margin, dwell, cargo, hs)
            )

        # 4. Weather Risks
        safe_exec(conn, "DELETE FROM weather_risks;")
        for i in range(1, 21):
            pname = BASE_PORTS[(i - 1) % len(BASE_PORTS)][0]
            sev = random.randint(1, 4)
            fore = "Category 3 Typhoon Warning" if sev >= 3 else "Clear Maritime Conditions"
            w_spd = round(random.uniform(12.0, 58.0), 1)
            wv_ht = round(random.uniform(0.8, 5.5), 1)
            temp = round(random.uniform(14.0, 38.0), 1)
            safe_exec(conn,
                "INSERT OR REPLACE INTO weather_risks (port_name, current_severity, forecast, wind_speed, wave_height, temperature) VALUES (?, ?, ?, ?, ?, ?);",
                (pname, sev, fore, w_spd, wv_ht, temp)
            )

        # 5. Alerts
        safe_exec(conn, "DELETE FROM alerts;")
        categories = ["Customs Hold", "Typhoon Storm", "Port Congestion", "Vessel Mechanical", "Bunker Fuel Surcharge"]
        severities = ["Critical", "High", "Medium", "Low"]
        for i in range(1, 51):
            shp_id = f"SHP-{i:04d}"
            sev = random.choice(severities)
            cat = random.choice(categories)
            msg = f"Alert #{i:03d}: Severe {cat} operational delay reported on {shp_id}."
            safe_exec(conn,
                "INSERT INTO alerts (shipment_id, severity, category, message, date, resolved) VALUES (?, ?, ?, ?, ?, ?);",
                (shp_id, sev, cat, msg, "2024-08-11", 0)
            )

        # 6. Carriers
        safe_exec(conn, "DELETE FROM carriers;")
        for idx, name in enumerate(carriers_list, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO carriers (carrier_id, name, rating, on_time_pct, avg_cost_index, risk_level) VALUES (?, ?, ?, ?, ?, ?);",
                (f"CAR-{idx:03d}", name, round(random.uniform(3.8, 4.9), 2), round(random.uniform(78, 96), 1), round(random.uniform(0.86, 1.18), 2), "Low")
            )

        # 7. Customers
        safe_exec(conn, "DELETE FROM customers;")
        for i in range(1, 25):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customers (customer_id, name, industry, priority_tier, credit_risk) VALUES (?, ?, ?, ?, ?);",
                (f"CUST-{i:03d}", f"Corporate Client {i:03d}", random.choice(["Food Service", "Retail", "QSR", "Hospitality"]), random.choice(["Platinum", "Gold", "Silver"]), round(random.uniform(0.02, 0.18), 2))
            )

        # 8. Freight Quotes
        safe_exec(conn, "DELETE FROM freight_quotes;")
        for i in range(1, 51):
            base = round(random.uniform(1200, 9000), 2)
            margin_pct = round(random.uniform(9, 24), 1)
            final = round(base * (1 + margin_pct / 100), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO freight_quotes (quote_id, shipment_id, customer_id, base_cost, insurance, customs_fee, fuel_surcharge, final_price, margin_pct, status, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"QTE-{i:04d}", f"SHP-{random.randint(1, 50):04d}", f"CUST-{random.randint(1, 20):03d}", base, round(base * 0.02, 2), round(random.uniform(100, 600), 2), round(base * 0.08, 2), final, margin_pct, random.choice(["Draft", "Accepted", "Submitted"]), datetime.date.today().isoformat())
            )

        # 9. Customs Tariffs
        safe_exec(conn, "DELETE FROM customs_tariffs;")
        for i, cargo in enumerate(cargos, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customs_tariffs (tariff_id, hs_code, cargo_type, origin_country, destination_country, duty_rate, clearance_risk, required_docs, advisory) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"TAR-{i:03d}", f"HS-{8400 + i * 31}", cargo, "India", "UAE", round(random.uniform(4, 16), 2), round(random.uniform(0.08, 0.42), 2), "Commercial invoice, packing list, bill of lading", "Validate HS code and pre-clear high-risk lanes.")
            )

        # 10. Outlets (FranchiseOps)
        outlet_cities = ["Chennai", "Bengaluru", "Hyderabad", "Mumbai", "Pune", "Delhi", "Kochi", "Coimbatore", "Ahmedabad", "Kolkata"]
        tiers = ["Metro Flagship", "Urban", "Express", "Mall"]
        safe_exec(conn, "DELETE FROM outlets;")
        for i in range(1, 51):
            revenue = round(random.uniform(850000, 6400000), 2)
            cost = round(revenue * random.uniform(0.58, 0.82), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO outlets (outlet_id, outlet_name, location, tier, revenue, operating_costs, customer_satisfaction, staff_headcount) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"OUT-{i:03d}", f"Franchise Outlet {i:03d}", random.choice(outlet_cities), random.choice(tiers), revenue, cost, round(random.uniform(3.2, 4.9), 2), random.randint(12, 55))
            )

        # 11. Staff
        roles = ["Store Manager", "Shift Lead", "Crew", "Chef", "Cashier", "Inventory Associate"]
        safe_exec(conn, "DELETE FROM staff;")
        for i in range(1, 151):
            satisfaction = random.randint(1, 5)
            overtime = round(random.uniform(0, 42), 1)
            attrition = min(0.95, max(0.03, 0.55 - satisfaction * 0.08 + overtime * 0.009 + random.uniform(-0.08, 0.08)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO staff (staff_id, outlet_id, name, role, salary, overtime_hrs, job_satisfaction, age, tenure_years, work_life_balance, predicted_attrition_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"STF-{i:04d}", f"OUT-{random.randint(1, 50):03d}", f"Employee {i:04d}", random.choice(roles), round(random.uniform(18000, 95000), 2), overtime, satisfaction, random.randint(19, 56), random.randint(0, 14), random.randint(1, 5), round(attrition, 2))
            )

        # 12. Inventory
        skus = ["Buns", "Cheese", "Sauce", "Chicken", "Paneer", "Coffee Beans", "Packaging", "Oil", "Frozen Fries", "Dessert Mix"]
        safe_exec(conn, "DELETE FROM inventory;")
        for i in range(1, 151):
            demand = round(random.uniform(20, 420), 1)
            threshold = random.randint(30, 180)
            stock = random.randint(5, 420)
            risk = min(0.95, max(0.02, (threshold - stock) / max(threshold, 1) + random.uniform(0.05, 0.28)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO inventory (record_id, outlet_id, sku_name, category, current_stock, reorder_threshold, weekly_demand, lead_time_days, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"INV-{i:04d}", f"OUT-{random.randint(1, 50):03d}", random.choice(skus), random.choice(["Food", "Beverage", "Packaging", "Consumable"]), stock, threshold, demand, random.randint(1, 9), round(risk, 2))
            )

        # 13. Marketing
        channels = ["Digital Ads", "Social Media", "Local Print", "Influencer Campaign", "Radio Spots"]
        safe_exec(conn, "DELETE FROM marketing;")
        for i in range(1, 51):
            budget = round(random.uniform(15000, 120000), 2)
            roi = round(random.uniform(1.8, 5.4), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO marketing (campaign_id, outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions, start_date, end_date) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"CMP-{i:03d}", f"OUT-{random.randint(1, 50):03d}", f"Campaign {i:03d}", random.choice(channels), budget, roi, random.randint(5000, 80000), random.randint(200, 4500), "2024-01-01", "2024-12-31")
            )

        # 14. Feedback
        safe_exec(conn, "DELETE FROM feedback;")
        comments = ["Great food!", "Slow service", "Clean ambience", "Polite staff", "Average experience"]
        for i in range(1, 101):
            rating = random.randint(1, 5)
            sentiment = round((rating - 3) / 2.0, 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO feedback (feedback_id, outlet_id, rating, comment, date, sentiment_score) VALUES (?, ?, ?, ?, ?, ?);",
                (f"FB-{i:04d}", f"OUT-{random.randint(1, 50):03d}", rating, random.choice(comments), "2024-08-01", sentiment)
            )

        # 15. Audits
        safe_exec(conn, "DELETE FROM audits;")
        categories_audit = ["Food Safety", "Hygiene & Sanitation", "Fire & Safety", "Financial Compliance"]
        for i in range(1, 51):
            score = round(random.uniform(65, 99), 1)
            status = "Pass" if score >= 85 else ("Conditional Pass" if score >= 75 else "Action Required")
            safe_exec(conn,
                "INSERT OR REPLACE INTO audits (audit_id, outlet_id, audit_date, score, violations, category, status, notes) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"AUD-{i:03d}", f"OUT-{random.randint(1, 50):03d}", "2024-08-01", score, int((100-score)/5), random.choice(categories_audit), status, "Audit completed cleanly.")
            )

        conn.commit()


In [ ]:
%%writefile /content/app/server.py
"""Web server: FastAPI JSON API + server-sent events + the single-page front end. Models live on the GPU in this process; the simulator and agent cycles run in worker.py."""
import sqlite3, os, sys, io, re, csv, json, time, asyncio, subprocess, threading, importlib, datetime
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config, rbac
from rbac import ROLE_CAPS, ROLE_MENU, ROLES, WRITE_CAPS
import intent_router as IR, analyst as AN, qa as QA, views_core as VC, demo_auth
from config import KIND, DB_PATH, JWT_SECRET_KEY, ADMIN_EMAIL_ID, ADMIN_PASSWORD, DEMO_MODE
import auth
from auth import (
    user_of as auth_user_of,
    hash_password,
    check_password,
    hash_security_answer,
    check_security_answer,
    create_access_token,
    decode_access_token,
    MAX_LOGIN_ATTEMPTS,
    generate_otp,
    make_otp_token,
    verify_otp_token,
    send_otp_email
)
import db
from fastapi import FastAPI, Request, Response, HTTPException, UploadFile, File, Form, Header
from fastapi.responses import JSONResponse, StreamingResponse, FileResponse, HTMLResponse, PlainTextResponse
from fastapi.staticfiles import StaticFiles
import pandas as pd
import numpy as np
import gpu, qa as QA, analyst as AN, live_engine as LE, entities as EN, blocks as B
import views_core as VC
from localuser import USER
dom = importlib.import_module("domain_franchise" if KIND == "Franchise" else "domain_freight")
V = importlib.import_module("views_franchise" if KIND == "Franchise" else "views_freight")
HERE = os.path.dirname(os.path.abspath(__file__)); STATIC = os.path.join(HERE, "static")
def ensure_showcase_users():
    """Guarantees the 5 showcase demo accounts exist in SQLite users table on startup."""
    try:
        from auth import hash_password
        users_to_seed = [
            ("admin", ADMIN_EMAIL_ID, ADMIN_PASSWORD or "Admin@123", "Admin"),
            ("owner", "owner@infosys.com", "Demo@1234", "Franchise Owner"),
            ("manager", "manager@infosys.com", "Demo@1234", "Store Manager"),
            ("staff", "staff@infosys.com", "Demo@1234", "Staff"),
            ("auditor", "auditor@infosys.com", "Demo@1234", "Auditor"),
        ]
        with sqlite3.connect(DB_PATH, timeout=30) as conn:
            for username, email, pwd, role in users_to_seed:
                pw_hash = hash_password(pwd)
                row = conn.execute("SELECT id FROM users WHERE email=? COLLATE NOCASE", (email,)).fetchone()
                if row:
                    conn.execute("UPDATE users SET password_hash=?, role=?, failed_attempts=0, lock_until=NULL, account_status='active' WHERE id=?", (pw_hash, role, row[0]))
                else:
                    conn.execute(
                        "INSERT INTO users (username, email, password_hash, role, security_question, security_answer, account_status, failed_attempts) "
                        "VALUES (?, ?, ?, ?, 'What is your favorite book/movie?', 'infosys', 'active', 0)",
                        (username, email.lower(), pw_hash, role)
                    )
            conn.commit()
    except Exception as e:
        print("ensure_showcase_users error:", e)

app = FastAPI(title=f"{V.NAME} Platform", docs_url=None, redoc_url=None, openapi_url=None) if os.environ.get("PUBLIC_DOCS") != "1" else FastAPI(title=f"{V.NAME} Platform"); _worker = None; _ready = {"db": False}
import hardening; hardening.install(app, KIND)

@app.middleware("http")
async def sec(request, call_next):
    r = await call_next(request); r.headers["X-Content-Type-Options"] = "nosniff"; r.headers["Referrer-Policy"] = "same-origin"; r.headers["Cache-Control"] = "no-store" if request.url.path.startswith("/api") else r.headers.get("Cache-Control", "public, max-age=300"); return r

def _boot():
    from db import init_db; from seed_data import seed_all
    init_db(); seed_all(); ensure_showcase_users(); LE.init_live_tables(); _ready["db"] = True
@app.on_event("startup")
def startup():
    global _worker
    _boot()
    if os.environ.get("NO_WORKER") != "1":
        env = {**os.environ, "APP_KIND": KIND}; _worker = subprocess.Popen([sys.executable, os.path.join(HERE, "worker.py")], env=env, cwd=HERE)
@app.on_event("shutdown")
def shutdown():
    if _worker:
        try: _worker.terminate()
        except Exception: pass

# ------------------------------------------------------------------ auth + context
def user_of(request: Request):
    return auth_user_of(request)
def ctx(u, params=None):
    if not u:
        u = {"role": "Staff", "email": ""}
    role = rbac.normalize_role(u.get("role"))
    user = {**u, "role": role}
    return {"dom": dom, "kind": KIND, "role": role, "email": user.get("email", ""),
            "user": user, "scope": rbac.scope_for(user), "params": params or {}}

def need(c, cap):
    return rbac.need(c, cap)

def menu_for(role):
    return rbac.menu_for(role)
def _audit_rbac_denial(user, what):
    try:
        LE.audit(user.get("email", "unknown"), "access_denied", what,
                 "role=" + rbac.normalize_role(user.get("role")))
    except Exception:
        pass

rbac.set_denial_hook(_audit_rbac_denial)

def ok(msg="Done", **kw): return {"ok": True, "msg": msg, **kw}
def bad(msg): return JSONResponse({"ok": False, "msg": msg}, status_code=400)

@app.get("/api/public")
def public():
    s = LE.get_state("boot", {}) if _ready["db"] else {}
    demo_accounts = []
    if DEMO_MODE:
        demo_accounts = [
            {"role": "Admin", "email": ADMIN_EMAIL_ID, "password": ADMIN_PASSWORD or "Admin@123", "icon": "👑"},
            {"role": "Franchise Owner", "email": "owner@infosys.com", "password": "Demo@1234", "icon": "🏢"},
            {"role": "Store Manager", "email": "manager@infosys.com", "password": "Demo@1234", "icon": "🏪"},
            {"role": "Staff", "email": "staff@infosys.com", "password": "Demo@1234", "icon": "👤"},
            {"role": "Auditor", "email": "auditor@infosys.com", "password": "Demo@1234", "icon": "📋"},
        ]
    questions = [
        "What was the name of your first pet?",
        "What city were you born in?",
        "What was your childhood nickname?",
        "What is your mother's maiden name?",
        "What is your favorite book/movie?"
    ]
    return {
        "kind": KIND,
        "name": V.NAME,
        "stage": (s or {}).get("stage", "starting"),
        "gpu": {"cuda": False, "name": None},
        "questions": questions,
        "email_enabled": bool(getattr(config, "EMAIL_ID", "")),
        "demo": demo_accounts,
        "demo_mode": DEMO_MODE
    }
@app.get("/api/me")
def me(request: Request):
    try:
        u = user_of(request)
        role = rbac.normalize_role(u.get("role"))
        assigned_scope = rbac.scope_for(u)
        menu_items = [{
            "id": m_id,
            "label": m_lbl,
            "icon": m_ico
        } for m_id, m_lbl, m_ico in (V.MENU + [('profile', 'Profile', 'user'), ('admin', 'Admin', 'shield')]) if m_id in rbac.ROLE_MENU[role]]
        return {
            "ok": True,
            "authenticated": True,
            "user": {
                "id": u.get("id"),
                "username": u.get("username", u.get("email", "").split("@")[0]),
                "email": u.get("email"),
                "role": role,
                "outlet_id": u.get("outlet_id"),
                "last_login": u.get("last_login"),
                "profile_picture": u.get("profile_picture")
            },
            "kind": KIND,
            "name": V.NAME,
            "menu": menu_items,
            "caps": list(rbac.ROLE_CAPS[role]),
            "scope": assigned_scope if assigned_scope != rbac.NO_OUTLET else None,
            "outlet_assigned": assigned_scope != rbac.NO_OUTLET,
            "langs": ["English"],
            "questions": []
        }
    except HTTPException as e:
        return JSONResponse({"ok": False, "authenticated": False, "user": None, "detail": e.detail}, status_code=e.status_code)
    except Exception as e:
        return JSONResponse({"ok": False, "authenticated": False, "user": None, "detail": str(e)}, status_code=401)

@app.post("/api/signup")
@app.post("/api/auth/signup")
async def signup_route(request: Request, response: Response):
    """Task 1 & 2: User signup with hashed password and security questions."""
    data = await request.json()
    username = str(data.get("username", "")).strip()
    email = str(data.get("email", "")).strip().lower()
    password = str(data.get("password", "")).strip()
    role = str(data.get("role", "Franchise Owner")).strip() or "Customer"
    sq = str(data.get("security_question", "")).strip()
    sa = str(data.get("security_answer", "")).strip()

    if not username or not email or not password:
        return bad("Username, email, and password are required")
    if len(password) < 6:
        return bad("Password must be at least 6 characters")

    if db.get_user_by_email(email):
        return bad("Email is already registered")
    if db.get_user_by_username(username):
        return bad("Username is already taken")

    pwd_hash = hash_password(password)
    sa_hash = hash_security_answer(sa) if sa else None

    uid = db.create_user(
        username=username,
        email=email,
        password_hash=pwd_hash,
        role=role,
        security_question=sq,
        security_answer=sa_hash
    )
    if not uid:
        return bad("Failed to create user")

    user = db.get_user_by_email(email)
    token = create_access_token(user)
    response.set_cookie(key="token", value=token, httponly=True, samesite="lax", max_age=86400)
    return {
        "ok": True,
        "msg": "Account created successfully",
        "token": token,
        "user": {
            "id": user["id"],
            "username": user["username"],
            "email": user["email"],
            "role": user["role"]
        }
    }

@app.post("/api/login")
@app.post("/api/auth/login")
async def login_route(request: Request, response: Response):
    """Task 1, 2, 6, 7: User login with 5-attempt lockout, last-login tracking, and JWT issuance."""
    data = await request.json()
    identifier = str(data.get("email") or data.get("username") or data.get("identifier") or "").strip()
    password = str(data.get("password") or "").strip()

    if not identifier or not password:
        return bad("Email/username and password are required")

    user = db.get_user_by_identifier(identifier)
    if not user:
        return JSONResponse({"ok": False, "msg": "Invalid credentials"}, status_code=401)

    # Task 6: Five-attempt account lockout check
    if db.is_user_locked(user):
        return JSONResponse({
            "ok": False,
            "msg": "Account is locked due to too many failed attempts (5-attempt lockout). Please try again in 15 minutes or reset your password.",
            "locked": True
        }, status_code=403)

    if not check_password(password, user["password_hash"]):
        failure_info = db.record_login_failure(identifier)
        attempts = failure_info.get("failed_attempts", 0)
        if failure_info.get("locked"):
            return JSONResponse({
                "ok": False,
                "msg": "Account has been locked after 5 failed login attempts. Please try again in 15 minutes or reset your password.",
                "locked": True,
                "remaining_attempts": 0
            }, status_code=403)
        rem = max(0, MAX_LOGIN_ATTEMPTS - attempts)
        return JSONResponse({
            "ok": False,
            "msg": f"Invalid password. {rem} attempt{'s' if rem != 1 else ''} remaining before lockout.",
            "remaining_attempts": rem
        }, status_code=401)

    # Task 7: Successful login: reset attempts and track last_login
    db.record_login_success(identifier)
    updated_user = db.get_user_by_identifier(identifier)

    token = create_access_token(updated_user)
    response.set_cookie(key="token", value=token, httponly=True, samesite="lax", max_age=86400)
    return {
        "ok": True,
        "msg": "Login successful",
        "token": token,
        "user": {
            "id": updated_user["id"],
            "username": updated_user["username"],
            "email": updated_user["email"],
            "role": updated_user["role"],
            "last_login": updated_user.get("last_login")
        }
    }

@app.post("/api/forgot-password")
@app.post("/api/auth/forgot-password")
async def forgot_password_route(request: Request):
    """Task 2: Forgot password flow fetching security questions and validating answer."""
    data = await request.json()
    identifier = str(data.get("email") or data.get("username") or data.get("identifier") or "").strip()
    if not identifier:
        return bad("Email or username is required")

    user = db.get_user_by_identifier(identifier)
    if not user:
        return bad("User account not found")
    if db.is_user_locked(user) or user.get("account_status") == "locked":
        return bad("Account is locked for 15 minutes after five failed attempts. Please wait before recovery.")

    sa = data.get("security_answer")
    if sa is not None:
        sa = str(sa).strip()
        stored_sa = user.get("security_answer")
        if not stored_sa:
            return bad("No security answer on file for this account")
        if not check_security_answer(sa, stored_sa):
            return bad("Incorrect security answer")
        import datetime as _dt
        reset_token = create_access_token({"sub": user["email"], "email": user["email"], "role": "reset_flow"}, _dt.timedelta(minutes=15))
        return {
            "ok": True,
            "msg": "Security answer verified",
            "verified": True,
            "reset_token": reset_token
        }

    sq = user.get("security_question") or "What is your favorite book/movie?"
    return {
        "ok": True,
        "msg": "Security question retrieved",
        "security_question": sq,
        "email": user["email"]
    }


# -----------------------------------------------------------------------------
# Milestone 1: OTP Signup & Password Recovery Endpoints (Verify Portal)
# -----------------------------------------------------------------------------
@app.post("/api/signup/request-otp")
@app.post("/api/auth/signup/request-otp")
async def signup_request_otp_route(request: Request):
    """Step 1 of Signup: Validates credentials and sends 6-digit OTP."""
    data = await request.json()
    username = str(data.get("username") or "").strip()
    email = str(data.get("email") or "").strip().lower()
    password = str(data.get("password") or "").strip()

    if not username or not email or not password:
        return bad("Username, email, and password are required")
    if len(username) < 3:
        return bad("Username must be at least 3 characters")
    if "@" not in email or "." not in email:
        return bad("Please enter a valid email address")
    if len(password) < 8:
        return bad("Password must be at least 8 characters")

    # Check existence
    if db.get_user_by_email(email):
        return bad("An account with this email address already exists")
    if db.get_user_by_username(username):
        return bad("This username is already taken")

    # Rate limit check
    can_req, r_msg = db.can_request_otp(email)
    if not can_req:
        return bad(r_msg)

    otp = generate_otp()
    otp_token = make_otp_token(email, otp, purpose="signup_otp", minutes=10)
    db.record_otp_request(email)
    ok, send_msg, demo_otp = send_otp_email(email, otp, is_signup=True)

    return {
        "ok": True,
        "msg": send_msg,
        "otp_token": otp_token,
        "email": email,
        "demo_otp": demo_otp
    }

@app.post("/api/signup/verify")
@app.post("/api/auth/signup/verify")
async def signup_verify_route(request: Request):
    """Step 2 of Signup: Verifies 6-digit passcode and creates account."""
    data = await request.json()
    otp_token = str(data.get("otp_token") or "").strip()
    otp_code = str(data.get("otp_code") or data.get("otp") or "").strip()
    email = str(data.get("email") or "").strip().lower()
    username = str(data.get("username") or "").strip()
    password = str(data.get("password") or "").strip()
    role = str(data.get("role") or "Franchise Owner").strip()
    sec_q = str(data.get("security_question") or "").strip()
    sec_a = str(data.get("security_answer") or "").strip()

    if not otp_token or not otp_code or not email:
        return bad("Verification token and 6-digit code are required")

    ok, v_msg = verify_otp_token(otp_token, otp_code, email, purpose="signup_otp")
    if not ok:
        return bad(v_msg)

    # Verification passed! Create user in database
    pw_hash = hash_password(password)
    sa_hash = hash_security_answer(sec_a) if sec_a else ""
    created = db.create_user(
        username=username,
        email=email,
        password_hash=pw_hash,
        role=role,
        security_question=sec_q,
        security_answer=sa_hash
    )
    if not created:
        return bad("Failed to create account. Username or email may already be registered.")

    return {
        "ok": True,
        "msg": "Account created successfully! You can now sign in."
    }

@app.post("/api/forgot-password/send-otp")
@app.post("/api/auth/forgot-password/send-otp")
async def forgot_password_send_otp_route(request: Request):
    """Method 1 of Forgot Password: Sends 6-digit OTP to user's registered email."""
    data = await request.json()
    identifier = str(data.get("identifier") or data.get("email") or data.get("username") or "").strip()
    if not identifier:
        return bad("Email or username is required")

    user = db.get_user_by_identifier(identifier)
    if not user:
        return bad("No registered account found matching that username or email")
    if db.is_user_locked(user) or user.get("account_status") == "locked":
        return bad("Account is locked for 15 minutes after five failed attempts. Please wait before recovery.")

    user_email = user["email"]
    can_req, r_msg = db.can_request_otp(user_email)
    if not can_req:
        return bad(r_msg)

    otp = generate_otp()
    reset_otp_token = make_otp_token(user_email, otp, purpose="password_reset_otp", minutes=10)
    db.record_otp_request(user_email)
    ok, send_msg, demo_otp = send_otp_email(user_email, otp, is_signup=False)

    # Mask email for display: d***i@gmail.com
    parts = user_email.split("@")
    if len(parts) == 2 and len(parts[0]) > 2:
        masked = parts[0][0] + "***" + parts[0][-1] + "@" + parts[1]
    else:
        masked = user_email

    return {
        "ok": True,
        "msg": send_msg,
        "email": user_email,
        "email_masked": masked,
        "reset_otp_token": reset_otp_token,
        "demo_otp": demo_otp
    }

@app.post("/api/forgot-password/verify-otp")
@app.post("/api/auth/forgot-password/verify-otp")
async def forgot_password_verify_otp_route(request: Request):
    """Verifies OTP and generates a password reset authorization token."""
    data = await request.json()
    token = str(data.get("reset_otp_token") or data.get("token") or "").strip()
    otp_code = str(data.get("otp_code") or data.get("otp") or "").strip()
    email = str(data.get("email") or "").strip().lower()

    if not token or not otp_code or not email:
        return bad("Verification token, 6-digit code, and email are required")

    ok, v_msg = verify_otp_token(token, otp_code, email, purpose="password_reset_otp")
    if not ok:
        return bad(v_msg)

    # Issue password reset token (15 minute validity)
    import datetime as _dt
    reset_token = create_access_token({"sub": email, "email": email, "role": "reset_flow"}, _dt.timedelta(minutes=15))
    return {
        "ok": True,
        "msg": "Code verified successfully.",
        "reset_token": reset_token
    }

@app.post("/api/reset-password")
@app.post("/api/auth/reset-password")
async def reset_password_route(request: Request):
    """Task 2: Resets user password upon verified token or answer, and resets lockout."""
    data = await request.json()
    identifier = str(data.get("email") or data.get("username") or data.get("identifier") or "").strip()
    new_pw = str(data.get("new_password") or data.get("password") or "").strip()
    sa = str(data.get("security_answer") or "").strip()
    reset_token = str(data.get("reset_token") or "").strip()

    if not identifier and reset_token:
        p = decode_access_token(reset_token)
        if p and p.get("email"):
            identifier = p["email"]

    if not identifier or not new_pw:
        return bad("Identifier and new password are required")
    if len(new_pw) < 6:
        return bad("New password must be at least 6 characters")

    user = db.get_user_by_identifier(identifier)
    if not user:
        return bad("User account not found")
    if db.is_user_locked(user) or user.get("account_status") == "locked":
        return bad("Account is locked for 15 minutes after five failed attempts. Please wait before recovery.")

    verified = False
    if reset_token:
        payload = decode_access_token(reset_token)
        if payload and (payload.get("email") == user["email"] or payload.get("sub") == user["email"]):
            verified = True
    if not verified and sa:
        stored_sa = user.get("security_answer")
        if stored_sa and check_security_answer(sa, stored_sa):
            verified = True

    if not verified:
        return bad("Verification failed. Please provide a valid security answer or reset token.")

    new_hash = hash_password(new_pw)
    db.update_user_password(user["email"], new_hash)
    db.unlock_user(user["email"])

    return {
        "ok": True,
        "msg": "Password reset successfully. You can now log in with your new password."
    }

@app.post("/api/logout")
@app.post("/api/auth/logout")
def logout_route(response: Response):
    """Task 7: Logout endpoint clearing authentication cookies."""
    response.delete_cookie("token")
    response.delete_cookie("access_token")
    response.delete_cookie("session")
    return {"ok": True, "msg": "Logged out successfully"}

# ------------------------------------------------------------------ live pulse
def pulse():
    rt = LE.get_state("runtime", {}) or {}; boot = LE.get_state("boot", {}) or {}; d = VC.decision()
    oa = B.Q("SELECT status, severity, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed') GROUP BY status, severity")
    n = lambda st, sv=None: int(oa[(oa.status == st) & ((oa.severity == sv) if sv else True)].n.sum()) if len(oa) else 0
    return {"stage": boot.get("stage", "starting"), "ticks": rt.get("ticks", 0), "cycles": rt.get("cycles", 0), "last_cycle": rt.get("last_cycle"), "health": d["score"] if d else None, "risk": d["risk"] if d else None, "dec": d["id"] if d else 0,
            "open": n("open"), "critical": n("open", "Critical"), "proposed": 0, "gpu": {"cuda": True, "name": "", "used": 0, "total": 0, "llm": "off", "llm_name": "", "llm_done": 0, "llm_total": 0}, "worker_alive": bool(rt.get("last_cycle"))}
@app.get("/api/pulse")
def pulse_http(request: Request):
    c = ctx(user_of(request)); result = pulse()
    if c["scope"] is not None: result["health"] = None; result["risk"] = None
    return result
@app.get("/api/stream")
async def stream(request: Request):
    user_of(request)
    async def gen():
        last = None; t0 = time.time()
        while True:
            if await request.is_disconnected(): break
            try: p = await asyncio.to_thread(pulse)
            except Exception as e: p = {"error": str(e)}
            k = json.dumps(p, sort_keys=True, default=str)
            if k != last or time.time() - t0 > 10: last = k; t0 = time.time(); yield f"data: {k}\n\n"
            await asyncio.sleep(1.5)
    return StreamingResponse(gen(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

# ------------------------------------------------------------------ pages
_TABS = {}
def _tabs_of(c, f):
    if f not in _TABS: _TABS[f] = f(c, None).get("tabs")
    return _TABS[f]
def _page(c, pid, tab):
    if pid == "copilot": return B.page("AI Copilot", "Ask about your data in plain English. Answers are exact database queries.", [{"t": "chat", "examples": EXAMPLES, "langs": ["English"]}], [], live=0)
    f = V.PAGES.get(pid)
    if not f: raise HTTPException(404, "unknown page")
    o = f(c, tab); _TABS[f] = o.get("tabs"); return o
@app.get("/api/page/{pid}")
def page(pid: str, request: Request, tab: str = "", p: str = ""):
    u = user_of(request)
    c = ctx(u, json.loads(p) if p else {})
    # My access is a shared, authenticated information page rather than a
    # role-specific menu entry.
    if pid != "access" and pid not in rbac.ROLE_MENU[c["role"]]:
        rbac.need(c, "__page_" + pid)
    t = time.time()
    out = _page(c, pid, tab or None)
    out["id"] = pid
    out["ms"] = round((time.time() - t) * 1000)
    return JSONResponse(B.clean(out))

@app.get("/api/detail/{eid:path}")
def detail(eid: str, request: Request):
    u = user_of(request)
    c = ctx(u)
    rbac.enforce_outlet(c, eid)
    return JSONResponse(B.clean(V.detail(c, eid)))

# ------------------------------------------------------------------ tickets, cycles, policy
@app.post("/api/run_cycle")
def run_cycle(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); t = time.time(); LE.run_cycle(dom, "manual", ""); LE.audit(u["email"], "manual_cycle", "all"); return ok(f"Agents finished in {time.time() - t:.1f}s")
@app.post("/api/run_agent")
async def run_agent(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); a = (await request.json()).get("agent")
    if a not in dom.AGENTS: raise HTTPException(404, "unknown agent")
    await asyncio.to_thread(_run_one, a); LE.audit(u["email"], "manual_agent", a); return ok(f"{a} re-ran")
def _run_one(agent):
    cid = time.strftime("%m%d-%H%M%S") + "-page"; t = time.time(); s = {"cycle_id": cid, "trigger": "page", "request": "", "queue": [], "results": {}}
    if agent == "digest":
        for a in ("outlet", "inventory", "staff", "marketing", "audit"): s["results"][a] = dom.AGENTS[a](s)
    res = dom.AGENTS[agent](s); LE._log_run(cid, "page", res, int((time.time() - t) * 1000)); s["results"][agent] = res; LE.act(dom, {"actions": res.actions}, cid, {agent: res})

# ------------------------------------------------------------------ admin users

# ------------------------------------------------------------------ digital twin

# ------------------------------------------------------------------ data feeds

# ------------------------------------------------------------------ Franchise actions
@app.post("/api/franchise/review")
async def fr_review(request: Request):
    import uuid; u = user_of(request); c = ctx(u); need(c, "review"); d = await request.json(); rbac.enforce_outlet(c, d.get("outlet")); tx = str(d.get("comment", "")).strip()
    if not tx: return bad("Write a comment first")
    src = "eng_Latn"; en = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+|\+?\d[\d -]{8,}\d", "[removed]", tx)
    try: from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer; sc = SentimentIntensityAnalyzer().polarity_scores(en)["compound"]
    except Exception: sc = (int(d["rating"]) - 3) / 2
    rt = int(d["rating"])
    with LE._DB_LOCK, LE.conn() as cn: cn.execute("INSERT INTO feedback(feedback_id,outlet_id,rating,comment,date,sentiment_score) VALUES(?,?,?,?,?,?)", ("FB-U" + uuid.uuid4().hex[:10], d["outlet"], rt, en, datetime.date.today().isoformat(), .6 * (rt - 3) / 2 + .4 * sc))
    LE.emit("review", d["outlet"], {"rating": rt}); return ok(f"Stored for {d['outlet']} (sentiment {sc:+.2f}{', translated from ' + src if src != 'eng_Latn' else ''}). The Sentiment agent includes it in its next cycle.", refresh=True)
@app.post("/api/franchise/audit")
async def fr_audit(request: Request):
    import uuid; u = user_of(request); c = ctx(u); need(c, "audit_write"); d = await request.json(); rbac.enforce_outlet(c, d.get("outlet")); sc = int(d["score"])
    with LE._DB_LOCK, LE.conn() as cn: cn.execute("INSERT INTO audits(audit_id,outlet_id,audit_date,score,violations,category,status,notes) VALUES(?,?,?,?,?,?,?,?)", ("AUD-U" + uuid.uuid4().hex[:10], d["outlet"], datetime.date.today().isoformat(), sc, int((100 - sc) / 6), d["category"], "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", d.get("notes", "")))
    LE.emit("audit", d["outlet"], {"score": sc}); LE.audit(u["email"], "audit_logged", d["outlet"], str(sc)); await asyncio.to_thread(_run_one, "audit"); return ok("Saved and evaluated by the Audit agent.", refresh=True)

# ------------------------------------------------------------------ Freight actions
@app.post("/api/freight/quote")
async def fq_quote(request: Request):
    u = user_of(request); c = ctx(u); need(c, "quote"); d = await request.json()
    if d["o"] == d["d"]: return bad("Origin and destination must differ.")
    rid = dom.submit_request(d["o"], d["d"], d["cargo"], float(d["weight"]), d["hs"], d["customer"], "English", d.get("docs") or []); await asyncio.to_thread(dom.run_quote, rid); LE.audit(u["email"], "quote_created", str(rid), f"{d['o']}->{d['d']}")
    return ok("Quote generated", blocks=B.clean(V.quote_result(rid)))
@app.post("/api/freight/storm")
async def fq_storm(request: Request):
    PRESET_STORMS = {"Red Sea": (18.0, 40.0), "Arabian Sea": (16.0, 66.0), "Bay of Bengal": (14.0, 88.0), "South China Sea": (14.0, 114.0), "Mediterranean": (35.0, 18.0)}; u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json() if request.headers.get("content-length", "0") not in ("0", "2") else {}
    la, lo = PRESET_STORMS.get(d.get("area") or "Red Sea"); dom.inject_storm(la, lo, wind=float(d.get("wind", 24))); LE.audit(u["email"], "inject_storm", d.get("area", "Red Sea")); return ok("Storm injected. The weather agent will propose re-routes on its next cycle.", refresh=True)
@app.post("/api/freight/reroute")
async def fq_reroute(request: Request):
    u = user_of(request); c = ctx(u); need(c, "approve"); j = await request.json(); sid = j["shipment"]; opt = j.get("opt")
    idx = int(str(opt).split(".")[0]) - 1 if opt else None; act = {"type": "reroute", "target": sid}
    if idx is not None: act["opt_label"] = str(opt).split(" · ", 1)[-1]; act["opt_idx"] = idx
    with LE._DB_LOCK, LE.conn() as cn: note = dom.apply(act, cn)
    LE.audit(u["email"], "manual_reroute", sid, note); return ok(note or "no change", refresh=True)
@app.post("/api/freight/attach")
async def fq_attach(request: Request):
    u = user_of(request); c = ctx(u); need(c, "attach_doc")
    d = await request.json(); dom.attach_doc(d["ref"], d["doc"], u["email"]); LE.audit(u["email"], "doc_attached", d["ref"], d["doc"]); return ok("Attached. Quotes re-run immediately; shipments clear on the next customs cycle.", refresh=True)
@app.post("/api/freight/readdoc")
async def fq_readdoc(request: Request, file: UploadFile = File(...)):
    u = user_of(request); data = await file.read(); txt = ""
    try:
        if file.filename.lower().endswith(".pdf"):
            import pdfplumber
            with pdfplumber.open(io.BytesIO(data)) as pdf: txt = "\n".join((pg.extract_text() or "") for pg in pdf.pages)
        else: txt = data.decode("utf-8", errors="ignore")
    except Exception as e: return bad(f"Could not read the file: {e}")
    hs = re.findall(r"HS[- ]?\d{2,8}", txt); q = re.findall(r"QTE-L\d+", txt); wt = re.findall(r"(\d+(?:\.\d+)?)\s*(?:t|tons|kg)\b", txt)
    return ok("Read", blocks=[B.table([{"field": "HS code", "value": ", ".join(dict.fromkeys(hs)) or "-"}, {"field": "Quote reference", "value": ", ".join(dict.fromkeys(q)) or "-"}, {"field": "Weights", "value": ", ".join(wt[:3]) or "-"}], ["field", "value"], title="Fields found", h=170, key="ocr"), B.md("```\n" + txt[:2500] + "\n```")])

# ------------------------------------------------------------------ language, knowledge base

# ------------------------------------------------------------------ Copilot (agents first, then the GPU LLM; NLLB for every language) + executive brief
def _sse(ev, **kw): return f"data: {json.dumps({'ev': ev, **kw}, default=str)}\n\n"
SYSTEM = "You are the operations copilot of an agentic platform. Answer ONLY from the EVIDENCE. Quote numbers exactly as given, never invent any. Be concise: a one-line answer, then up to 5 short bullets with the entity ids. If the evidence does not answer, say so. When you use a numbered source, cite it as [1], [2]."
_BRIEF = {}
def _chat_db():
    c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("CREATE TABLE IF NOT EXISTS copilot_chats(id TEXT PRIMARY KEY, email TEXT, title TEXT, ts TEXT, data TEXT)"); c.execute("CREATE INDEX IF NOT EXISTS ix_chat_email ON copilot_chats(email, ts)"); return c
@app.get("/api/chats")
async def chats_list(request: Request):
    u = user_of(request)
    c = ctx(u)
    if "copilot" not in rbac.ROLE_MENU[c["role"]]: rbac.need(c, "__copilot")
    with _chat_db() as c: rows = c.execute("SELECT id, title, ts FROM copilot_chats WHERE email=? ORDER BY ts DESC LIMIT 60", (u["email"],)).fetchall()
    return {"ok": True, "chats": [{"id": r[0], "title": r[1], "ts": r[2]} for r in rows]}
@app.get("/api/chats/{cid}")
async def chats_get(cid: str, request: Request):
    u = user_of(request)
    c = ctx(u)
    if "copilot" not in rbac.ROLE_MENU[c["role"]]: rbac.need(c, "__copilot")
    with _chat_db() as c: r = c.execute("SELECT title, data FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"])).fetchone()
    if not r: raise HTTPException(404, "chat not found")
    return {"ok": True, "title": r[0], "msgs": json.loads(r[1])}
@app.post("/api/chats/save")
async def chats_save(request: Request):
    u = user_of(request)
    c = ctx(u)
    if "copilot" not in rbac.ROLE_MENU[c["role"]]: rbac.need(c, "__copilot")
    d = await request.json(); cid = re.sub(r"[^a-zA-Z0-9_-]", "", str(d.get("id") or ""))[:40] or ("c" + str(int(time.time() * 1000))); msgs = d.get("msgs") or []
    if not isinstance(msgs, list) or not msgs: return bad("nothing to save")
    msgs = [{"role": m.get("role"), "text": str(m.get("text", ""))[:20000], "en": (str(m["en"])[:20000] if m.get("en") else None), "lang": m.get("lang"), "sql": str(m.get("sql") or "")[:1500]} for m in msgs[-80:]]
    title = next((m["text"] for m in msgs if m["role"] == "user"), "Chat")[:70]; now = datetime.datetime.now().isoformat(timespec="seconds")
    with _chat_db() as c:
        o = c.execute("SELECT email FROM copilot_chats WHERE id=?", (cid,)).fetchone()
        if o and o[0] != u["email"]: raise HTTPException(403, "not your chat")
        c.execute("INSERT OR REPLACE INTO copilot_chats VALUES(?,?,?,?,?)", (cid, u["email"], title, now, json.dumps(msgs)))
    return {"ok": True, "id": cid}
@app.delete("/api/chats/{cid}")
async def chats_del(cid: str, request: Request):
    u = user_of(request)
    c = ctx(u)
    if "copilot" not in rbac.ROLE_MENU[c["role"]]: rbac.need(c, "__copilot")
    with _chat_db() as c: c.execute("DELETE FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"]))
    return {"ok": True}
@app.post("/api/weather/refresh")
async def wx_refresh(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run")
    if KIND != "Freight": raise HTTPException(404)
    import weather as WX; n, msg = await asyncio.to_thread(WX.refresh); LE.audit(u["email"], "weather_refresh", "ports", msg)
    return ok(("Real weather updated: " if n else "Could not reach Open-Meteo, simulated values stay: ") + msg, refresh=True)

# ------------------------------------------------------------------ reports, exports, profile, invitations


@app.get("/api/health")
def health(): return {"ok": True, "gpu": gpu.status()}
@app.get("/")
def index(): return FileResponse(os.path.join(STATIC, "index.html"), headers={"Cache-Control": "no-store"})
app.mount("/static", StaticFiles(directory=STATIC), name="static")


# ------------------------------------------------------------------ AI Copilot (SQL only: greetings + exact database answers)
EXAMPLES = ["Which outlets are at highest risk?", "Why is OUT-007 flagged?", "How is inventory holding up?", "Top 5 outlets by revenue", "Which stores have the lowest customer satisfaction?"]
GREET = re.compile(r"^\W*(hi|hello|hey|hii|good (morning|afternoon|evening)|namaste|yo)\b[\w\s!.,]*$", re.I)
THANKS = re.compile(r"^\W*(thanks|thank you|thx|ok thanks|great|cool)\b[\w\s!.,]*$", re.I)
@app.post("/api/copilot")
async def copilot(request: Request):
    u = user_of(request); c = ctx(u)
    if "copilot" not in rbac.ROLE_MENU[c["role"]]: rbac.need(c, "__copilot")
    d = await request.json(); q = str(d.get("q", "")).strip()[:800]
    state = d.get("state") if isinstance(d.get("state"), dict) else None
    role = c.get("role") or u.get("role") or "Staff"; scope = c.get("scope") or u.get("outlet_id") or None
    def gen():
        t0 = time.time()
        try:
            r = IR.respond(KIND, q, role, scope, state, VC.decision(), dom)
        except Exception as e:
            print("copilot failed:", repr(e), file=sys.stderr)
            r = {"text": "Sorry, something went wrong while answering that. Please try again or rephrase your question.", "sql": "", "rows": None, "status": "error", "suggestions": QA.suggest_questions("", role), "followups": [], "state": {}, "validated": False, "intent": {}, "corrections": [], "ms": round((time.time() - t0) * 1000)}
        yield _sse("intent", intent=r.get("intent"), role=r.get("role"), status=r.get("status"))
        if r.get("sql"): yield _sse("exact", sql=r["sql"], rows=r.get("rows"))
        yield _sse("final", text=r["text"], sql=r.get("sql", ""), rows=r.get("rows"), status=r.get("status"), suggestions=r.get("suggestions") or [], followups=r.get("followups") or [], state=r.get("state") or {}, validated=r.get("validated", False),
                   corrections=r.get("corrections") or [], intent=r.get("intent"), role=r.get("role"), ms=r.get("ms", round((time.time() - t0) * 1000)))
        yield _sse("done")
    return StreamingResponse(gen(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})


# ------------------------------------------------------------------------------
# RBAC Management & Demo Routes
# ------------------------------------------------------------------------------
@app.get("/api/admin/users")
def get_users(request: Request):
    user = user_of(request)
    rbac.need(ctx(user), "users")
    import db
    return {"ok": True, "users": db.list_users()}

@app.post("/api/admin/users/role")
async def change_role(request: Request):
    user = user_of(request)
    rbac.need(ctx(user), "users")
    params = await request.json()
    target_username = params.get("username")
    target_role = params.get("role")

    if target_role not in rbac.ROLES:
        raise HTTPException(status_code=400, detail="Unsupported platform role name")

    import db
    with db.get_conn() as con:
        current = con.execute("SELECT role FROM users WHERE username=?", (target_username,)).fetchone()
        admins = con.execute("SELECT COUNT(*) FROM users WHERE role='Admin'").fetchone()[0]
        if current and current[0] == "Admin" and target_role != "Admin" and admins <= 1:
            return bad("The last Admin cannot be demoted")
        con.execute("UPDATE users SET role = ? WHERE username = ?", (target_role, target_username))
        con.commit()
    LE.audit(user["email"], "admin_role_changed", target_username, target_role)
    return {"ok": True, "msg": f"User '{target_username}' updated to role '{target_role}' successfully!"}

@app.post("/api/admin/assign-outlet")
async def assign_outlet(request: Request):
    user = user_of(request)
    rbac.need(ctx(user), "assign_outlet")
    params = await request.json()
    target_username = params.get("username")
    target_outlet_id = params.get("outlet_id")

    import db
    db.assign_user_outlet(target_username, target_outlet_id)
    LE.audit(user["email"], "admin_outlet_assigned", target_username, target_outlet_id)
    return {"ok": True, "msg": f"Store Manager '{target_username}' successfully assigned to outlet {target_outlet_id}!"}

@app.get("/api/admin/outlets")
def admin_outlets(request: Request):
    rbac.need(ctx(user_of(request)), "assign_outlet")
    import blocks as B
    df = B.Q("SELECT outlet_id FROM outlets ORDER BY outlet_id")
    return {"ok": True, "outlets": df.outlet_id.tolist() if len(df) else []}

@app.get("/api/demo/roles")
def demo_roles():
    import demo_auth
    return demo_auth.get_roles_data()

@app.post("/api/demo/login")
async def demo_login_route(request: Request, response: Response):
    d = await request.json()
    role = str(d.get("role", ""))
    if role not in rbac.ROLES:
        return JSONResponse({"ok": False, "msg": "Choose one of the five roles"}, status_code=400)
    import db
    users_dict = {r: (u, e) for u, e, r in db.DEMO_ROLE_USERS}
    if role not in users_dict:
        return JSONResponse({"ok": False, "msg": "Role account not found"}, status_code=404)
    email = users_dict[role][1]
    user_db = db.get_user_by_identifier(email)
    if not user_db:
        db.seed_role_users()
        user_db = db.get_user_by_identifier(email)
    import auth
    token = auth.create_access_token(user_db)
    res = JSONResponse({"ok": True, "msg": "Signed in as " + role, "role": role, "token": token, "user": user_db})
    res.set_cookie(key="token", value=token, httponly=True, samesite="lax", max_age=86400)
    return res

@app.post("/api/demo/logout")
def demo_logout_route(response: Response):
    res = JSONResponse({"ok": True, "msg": "Signed out"})
    res.delete_cookie(key="token")
    return res


# ------------------------------------------------------------------ Milestone 2 Profile and Admin pages
_base_page = _page
def _page(c, pid, tab):
    if pid == "access":
        role = c["role"]
        labels = {m_id: m_label for m_id, m_label, _ in (V.MENU + [('profile', 'Profile', 'user'), ('admin', 'Admin', 'shield')])}
        pages = [{"page": labels.get(page_id, page_id.replace("_", " ").title())} for page_id in sorted(rbac.ROLE_MENU.get(role, set()))]
        permissions = [{"permission": cap.replace("_", " ").title()} for cap in sorted(rbac.ROLE_CAPS.get(role, set()))]
        account = [{"field": "Role", "value": role}, {"field": "Outlet access", "value": c.get("scope") or "All permitted outlets"}, {"field": "Mode", "value": "Read-only" if not (rbac.ROLE_CAPS.get(role, set()) & rbac.WRITE_CAPS) else "Operational access"}]
        return B.page("My access", "Your role, permitted pages and permissions", [
            B.table(pd.DataFrame(account), ["field", "value"], title="Account access", h=180, key="access_account"),
            B.table(pd.DataFrame(pages), ["page"], title="Available pages", h=280, key="access_pages"),
            B.table(pd.DataFrame(permissions), ["permission"], title="Permissions", h=280, key="access_permissions")])
    if pid == "profile":
        u = c["user"]
        rows = [{"field": "Username", "value": u.get("username", "")}, {"field": "Email", "value": u.get("email", "")},
                {"field": "Role", "value": c["role"]}, {"field": "Joined", "value": u.get("created_at", "-")}, {"field": "Last login", "value": u.get("last_login", "-")}]
        photo = u.get("profile_picture")
        photo_section = B.card("Current profile photo", {"t": "profile_photo", "src": photo, "alt": "Current profile photo"}) if photo else B.callout("No profile photo yet. Choose an image below to add one.", "info", "Profile photo")
        return B.page("Profile", "Your account and security settings", [
            B.table(pd.DataFrame(rows), ["field", "value"], title="Account details", h=260, key="profile_details"),
            photo_section,
            B.form("profile_photo", [{"name":"photo","label":"Choose profile photo (PNG, JPG or WEBP; max 2 MB)","type":"file","accept":"image/png,image/jpeg,image/webp"}], "Change profile photo" if photo else "Save profile photo", "/api/profile/photo", title="Profile photo", sub="Your photo is shown in the account card in the sidebar."),
            B.button("profile_photo_delete", "Delete profile photo", "/api/profile/photo/delete", tone="ghost", confirm="Delete your current profile photo?") if photo else B.md(""),
            B.form("profile_name", [{"name":"username","label":"Username","type":"text","value":u.get("username", "")}], "Save username", "/api/profile/username", title="Update username"),
            B.form("profile_password", [{"name":"current_password","label":"Current password","type":"password"}, {"name":"new_password","label":"New password","type":"password"}, {"name":"confirm_password","label":"Confirm new password","type":"password"}], "Change password", "/api/profile/password", title="Change password", sub="Use 8+ characters with upper-case, lower-case, number and special character.")])
    if pid == "admin":
        rbac.need(c, "manage_users")
        users = B.Q("SELECT username,email,role,account_status,created_at,last_login FROM users ORDER BY created_at DESC")
        summary = B.Q("SELECT role, COUNT(*) users FROM users GROUP BY role ORDER BY role")
        locked = int(B.Q("SELECT COUNT(*) n FROM users WHERE account_status='locked'").n.iloc[0])
        role_kpis = [("Total users", len(users)), ("Locked users", locked, None, "Warning")] + [(str(r.role), int(r.users)) for r in summary.itertuples()]
        return B.page("Admin dashboard", "Users, access control and recent security activity", [
            B.kpis(*role_kpis),
            B.table(users, ["username","email","role",B.pill_col("account_status"),"created_at","last_login"], title="Users", h=340, key="admin_users"),
            B.form("admin_add", [{"name":"username","label":"Username","type":"text"},{"name":"email","label":"Email","type":"text"},{"name":"password","label":"Temporary password","type":"password"},{"name":"role","label":"Role","type":"select","options":list(rbac.ROLES),"value":"Staff"}], "Add user", "/api/admin/users/add", title="Add user"),
            B.form("admin_status", [{"name":"username","label":"Username","type":"text"},{"name":"action","label":"Action","type":"select","options":["lock","unlock","delete"],"value":"lock"},{"name":"confirm_delete","label":"For delete only, type DELETE to confirm","type":"text"}], "Apply", "/api/admin/users/status", title="Lock, unlock or delete user", sub="You cannot lock/delete yourself or remove the last Admin. Delete requires explicit confirmation."),
            _activity_log_table()])
    return _base_page(c, pid, tab)

def _activity_log_table():
    """Present audit values for people, not internal key=value notation."""
    activity = B.Q("SELECT ts,actor,event,target,detail FROM audit_log ORDER BY audit_id DESC LIMIT 50")
    activity = activity.rename(columns={"actor": "user"})
    for column in ("target", "detail"):
        if column in activity:
            activity[column] = activity[column].fillna("").astype(str).str.replace(
                r"^(?:role|outlet)\s*[:=]\s*", "", regex=True)
    return B.table(activity, ["ts", "user", "event", "target", "detail"], title="Activity log (latest 50)", h=320, key="admin_audit")

def _valid_password(value):
    return len(value or "") >= 8 and all(re.search(p, value or "") for p in (r"[A-Z]", r"[a-z]", r"\d", r"[^\w\s]"))

@app.post("/api/profile/username")
async def profile_username(request: Request):
    u=user_of(request); d=await request.json(); name=str(d.get("username") or "").strip()
    if len(name) < 2: return bad("Username must contain at least 2 characters")
    with db.get_conn() as cn: cn.execute("UPDATE users SET username=? WHERE email=?", (name, u["email"])); cn.commit()
    LE.audit(u["email"], "profile_username_changed", u["email"]); return ok("Username updated", refresh=True)

@app.post("/api/profile/photo")
async def profile_photo(request: Request, photo: UploadFile = File(...)):
    """Store a small, validated image in the existing users.profile_picture column.
    A data URI keeps this portable in Colab and avoids exposing an upload folder."""
    u = user_of(request)
    mime = (photo.content_type or "").lower()
    allowed = {"image/png", "image/jpeg", "image/webp"}
    if mime not in allowed:
        return bad("Choose a PNG, JPG or WEBP image")
    raw = await photo.read()
    if not raw or len(raw) > 2 * 1024 * 1024:
        return bad("Profile photo must be smaller than 2 MB")
    valid_magic = ((mime == "image/png" and raw.startswith(b"\x89PNG\r\n\x1a\n")) or
                   (mime == "image/jpeg" and raw.startswith(b"\xff\xd8\xff")) or
                   (mime == "image/webp" and raw.startswith(b"RIFF") and raw[8:12] == b"WEBP"))
    if not valid_magic:
        return bad("The selected file is not a valid image")
    import base64
    picture = "data:" + mime + ";base64," + base64.b64encode(raw).decode("ascii")
    with db.get_conn() as cn:
        cn.execute("UPDATE users SET profile_picture=? WHERE email=?", (picture, u["email"]))
        cn.commit()
    LE.audit(u["email"], "profile_photo_changed", u["email"])
    return ok("Profile photo updated", refresh=True)

@app.post("/api/profile/photo/delete")
async def delete_profile_photo(request: Request):
    u = user_of(request)
    with db.get_conn() as cn:
        cn.execute("UPDATE users SET profile_picture=NULL WHERE email=?", (u["email"],))
        cn.commit()
    LE.audit(u["email"], "profile_photo_deleted", u["email"])
    return ok("Profile photo deleted", refresh=True)

@app.post("/api/profile/password")
async def profile_password(request: Request):
    u=user_of(request); d=await request.json(); current=str(d.get("current_password") or ""); new=str(d.get("new_password") or "")
    if not check_password(current, u.get("password_hash", "")): return bad("Current password is incorrect")
    if new != str(d.get("confirm_password") or ""): return bad("New passwords do not match")
    if not _valid_password(new): return bad("Password must have 8+ characters, upper-case, lower-case, number and special character")
    db.update_user_password(u["email"], hash_password(new)); LE.audit(u["email"], "profile_password_changed", u["email"]); return ok("Password changed", refresh=True)

@app.post("/api/admin/users/add")
async def admin_add_user(request: Request):
    actor=user_of(request); rbac.need(ctx(actor), "users"); d=await request.json(); email=str(d.get("email") or "").strip().lower(); username=str(d.get("username") or "").strip(); password=str(d.get("password") or ""); role=str(d.get("role") or "")
    if not re.match(r"^[^@\s]{2,}@[^^@\s]{2,}\.[^@\s]{2,}$", email) or len(username)<2 or role not in rbac.ROLES or not _valid_password(password): return bad("Provide a valid email, username, role and strong temporary password")
    try:
        with db.get_conn() as cn: cn.execute("INSERT INTO users(username,email,password_hash,role,account_status) VALUES(?,?,?,?, 'active')", (username,email,hash_password(password),role)); cn.commit()
    except Exception: return bad("Username or email already exists")
    LE.audit(actor["email"], "admin_user_added", email, role); return ok("User added", refresh=True)

@app.post("/api/admin/users/status")
async def admin_user_status(request: Request):
    actor=user_of(request); rbac.need(ctx(actor), "users"); d=await request.json(); username=str(d.get("username") or "").strip(); action=str(d.get("action") or "")
    if action == "delete" and str(d.get("confirm_delete") or "") != "DELETE": return bad("Type DELETE to confirm permanent account deletion")
    with db.get_conn() as cn:
        row=cn.execute("SELECT username,email,role FROM users WHERE username=?", (username,)).fetchone()
        if not row: return bad("User not found")
        if row[1].lower()==str(actor.get("email", "")).lower(): return bad("You cannot lock or delete your own account")
        admins=cn.execute("SELECT COUNT(*) FROM users WHERE role='Admin'").fetchone()[0]
        if row[2]=='Admin' and action in ('delete', 'lock') and admins<=1: return bad("The last Admin cannot be removed or locked")
        if action=='lock': cn.execute("UPDATE users SET account_status='locked' WHERE username=?", (username,))
        elif action=='unlock': cn.execute("UPDATE users SET account_status='active', failed_attempts=0, lock_until=NULL WHERE username=?", (username,))
        elif action=='delete': cn.execute("DELETE FROM users WHERE username=?", (username,))
        else: return bad("Unsupported action")
        cn.commit()
    LE.audit(actor["email"], "admin_user_"+action, username); return ok("User " + action + "ed", refresh=True)

@app.exception_handler(HTTPException)
async def friendly_http_errors(request: Request, exc: HTTPException):
    """Show people a useful 403 page for direct browser URLs, while API calls
    still receive JSON that the single-page application can handle."""
    if exc.status_code == 403:
        detail = str(exc.detail or "Access denied")
        if "__page_" in detail:
            page = detail.split("__page_", 1)[1].replace("_", " ").title()
            message = f"Your role does not have permission to open the {page} page."
        else:
            message = "Your role does not have permission to perform this action."
        if "text/html" in request.headers.get("accept", ""):
            return HTMLResponse(f"""<!doctype html><html><head><title>Access denied</title><style>body{{margin:0;min-height:100vh;display:grid;place-items:center;background:#f6f7fb;color:#1c2024;font:16px system-ui}}main{{max-width:510px;margin:24px;background:#fff;border:1px solid #e6e8f0;border-radius:18px;padding:38px;box-shadow:0 12px 35px #1c202420;text-align:center}}h1{{margin:0 0 10px;color:#c62a2f}}p{{color:#586174;line-height:1.6}}a{{display:inline-block;margin-top:14px;padding:10px 16px;border-radius:10px;background:#5b5bd6;color:#fff;text-decoration:none;font-weight:700}}</style></head><body><main><div style='font-size:42px'>🛡️</div><h1>Access denied</h1><p>{message}</p><p>Please return to a page available for your role.</p><a href='/'>Return to FranchiseOps</a></main></body></html>""", status_code=403)
        return JSONResponse({"ok": False, "msg": message, "detail": message}, status_code=403)
    return JSONResponse({"ok": False, "msg": str(exc.detail or "Request failed")}, status_code=exc.status_code)


In [ ]:
%%writefile /content/app/views_core.py
"""Shared page builders: helpers, Command Center (tickets, agents, anomalies), Knowledge Graph."""
import json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, TONE
from rbac import has_capability

def can(c, cap): return has_capability(c, cap)
def latest(agent):
    r = LE.conn().execute("SELECT ts,score,risk,latency_ms,summary,output,cycle_id FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,)).fetchone()
    if not r: return None
    try: out = json.loads(r[5])
    except Exception: out = {}
    return {"ts": r[0], "score": r[1], "risk": r[2], "ms": r[3], "summary": r[4], "out": out, "cycle": r[6]}
def decision():
    d = Q("SELECT decision_id, ts, score, risk, narrative, evidence FROM decisions ORDER BY decision_id DESC LIMIT 1")
    if not len(d): return None
    try: ev = json.loads(d.evidence[0])
    except Exception: ev = {}
    return {"id": int(d.decision_id[0]), "ts": d.ts[0], "score": float(d.score[0]), "risk": d.risk[0], "narrative": d.narrative[0], "ev": ev}
def scoped(df, c, col="target"):
    """Restrict Store Manager ticket/entity rows to the assigned outlet identifier."""
    scope = c.get("scope")
    if scope and len(df) and col in df.columns:
        values = df[col].astype(str)
        return df[(values == str(scope)) | values.str.startswith(str(scope) + "-")]
    return df

# ------------------------------------------------------------------ tickets
def tk_df(c, agent=None, types=None, target=None, statuses=("open", "proposed"), limit=300):
    dom = c["dom"]; sql = "SELECT action_id id, severity, status, type, target, detail, owner, assignee, due, note, ts FROM agent_actions WHERE status IN (%s)" % ",".join("?" * len(statuses)); p = list(statuses)
    ts = types or ([t for t, a in dom.ACTION_AGENT.items() if a == agent] if agent else None)
    if ts: sql += f" AND type IN ({','.join('?' * len(ts))})"; p += list(ts)
    if target: sql += " AND target=?"; p.append(target)
    df = Q(sql + " ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT " + str(limit), p)
    return scoped(df, c)
def ticket_block(c, agent=None, types=None, title="Open tickets", target=None, key=None, limit=60):
    """Read-only list of what the agent has raised (type, target, severity, detail). There is no approval workflow in this version."""
    df = tk_df(c, agent, types, target, limit=limit)
    if not len(df): return B.callout("Nothing raised right now.", "info", title)
    return B.table(df, ["id", "ts", B.pill_col("severity"), "type", "target", "detail"], title=title, h=300, key=key or f"tk_{agent or target or 'all'}", open={"col": "target"} if False else None)
def hero(c, agent, title, blurb): return B.hero(agent, title, blurb, latest(agent), can_run=can(c, "run"))
def cross(c, targets, own, title="Flagged by other agents for these entities"):
    t = [str(x) for x in list(dict.fromkeys(targets))][:60]
    if not t: return None
    df = Q("SELECT type, target, severity, detail FROM agent_actions WHERE status IN ('open','proposed') AND target IN (%s)" % ",".join("?" * len(t)), t)
    df["agent"] = df.type.map(lambda x: c["dom"].ACTION_AGENT.get(x, "engine")); df = df[df.agent != own].head(30)
    if not len(df): return None
    return B.table(df, ["target", "agent", "type", B.pill_col("severity"), "detail"], title=title, h=240, open={"col": "target"}, key="cross_" + own)

# ------------------------------------------------------------------ Command Center

# ------------------------------------------------------------------ Approvals inbox (human in the loop)
SOURCE = {"manual_request": "Approval request", "quote_approval": "Quote approval", "ml_review": "ML second opinion", "twin_mitigation": "Digital Twin"}





# ------------------------------------------------------------------ Knowledge graph
def entity_explorer(c):
    kind, dom = c["kind"], c["dom"]; e = (c.get("params") or {}).get("e")
    ents = Q("SELECT DISTINCT target FROM agent_actions WHERE status IN ('open','proposed') LIMIT 300").target.tolist()
    base = Q("SELECT outlet_id e FROM outlets ORDER BY outlet_id").e.tolist() if kind == "Franchise" else Q("SELECT port_name e FROM ports ORDER BY congestion_index DESC LIMIT 60").e.tolist()
    opts = list(dict.fromkeys(([c["scope"]] if c.get("scope") else ents + base))); e = e if e and (not c.get("scope") or e == c["scope"]) else opts[0]      # a scoped user cannot explore another outlet through the URL parameter
    d = EN.describe(kind, dom, e); nodes = [{"id": e, "kind": "focus", "size": 34, "color": "#ffffff", "tip": "selected", "label": True}]; links = []
    for n, k, tip in d["rel"]: nodes.append({"id": n, "kind": k, "size": 16, "tip": tip, "label": True, "open": n if EN.kind_of(n) != "port" or kind == "Freight" else None}); links.append((e, n))
    for r in d["tickets"][d["tickets"].status.isin(["open", "proposed"])].itertuples():
        n = f"#{r.id} {r.type}"; ag = dom.ACTION_AGENT.get(r.type, "engine"); nodes += [{"id": n, "kind": "ticket", "size": 14, "color": TONE.get(r.severity), "tip": r.detail[:120], "label": True}, {"id": ag, "kind": "agent", "size": 26, "color": TEAL, "tip": "agent", "label": True}]; links += [(e, n), (n, ag)]
    seen = set(); nodes = [n for n in nodes if not (n["id"] in seen or seen.add(n["id"]))]
    sel = {"t": "form", "id": "kg_pick", "fields": [{"name": "e", "label": "Entity", "type": "select", "options": opts, "value": e}], "submit": "Explore", "url": "page:kg:explore", "title": None, "tone": "primary", "inline": True}
    info = B.table([{"field": k, "value": v} for k, v in d["info"].items()], ["field", "value"], title="What the platform knows", h=230, key="kg_info") if d["info"] else B.callout("No master record for this entity.", "info")
    tk = d["tickets"]
    return [sel, B.row(B.card("Facts", info, B.table(tk, ["id", B.pill_col("severity"), B.pill_col("status"), "type", "owner", "detail"], title="Why is it flagged", h=260, key="kg_tk")), B.card("Relations", B.chart(B.graph(nodes, links), h=470, open={"graph": True}, key="kg_ent")), ratio=[2, 3])]


In [ ]:
%%writefile /content/app/views_franchise.py
"""FranchiseOps AI pages (server-driven blocks)."""
import json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import domain_franchise as dom
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, GREEN, TONE
from views_core import can, latest, decision, ticket_block, tk_df, cross, hero, scoped

NAME = "FranchiseOps AI"
MENU = [('copilot', 'AI Copilot', 'bot'), ('workforce', 'Agent 1 · Workforce', 'users'), ('outlets', 'Agent 2 · Outlets', 'store'), ('inventory', 'Agent 3 · Inventory', 'boxes'), ('marketing', 'Agent 4 · Marketing', 'megaphone'), ('sentiment', 'Agent 5 · Sentiment', 'messages'), ('audit', 'Agent 6 · Audit', 'clipboard')]
def color_h(h): return RED if h < 60 else AMBER if h < 80 else TEAL
def _sc(c, col="outlet_id"): return (f" WHERE {col}=?", (c["scope"],)) if c.get("scope") else ("", ())
def model_card(name, keys):
    m = Q("SELECT metrics, trained_at FROM ml_models WHERE model_name=? ORDER BY model_id DESC LIMIT 1", (name,))
    if not len(m): return None
    try: d = json.loads(m.metrics[0])
    except Exception: d = {}
    return B.kpis(*[(k, d.get(k, "–")) for k in keys], ("trained", (m.trained_at[0] or "")[5:16]))

# ------------------------------------------------------------------ Command Center: Operations
def ops(c):
    d = decision(); health = (d or {}).get("ev", {}).get("health", {}) if d else {}
    g = Q("SELECT g.outlet_id, g.lat, g.lon, o.outlet_name, o.location FROM outlet_geo g JOIN outlets o USING(outlet_id)"); g = g[g.outlet_id == c["scope"]] if c.get("scope") else g
    tk = Q("SELECT target, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed') GROUP BY target").set_index("target").n; oa = tk_df(c, limit=500)
    pts = [{"id": r.outlet_id, "lat": r.lat, "lon": r.lon, "color": color_h(health.get(r.outlet_id, 70)), "r": 6 + min(10, int(tk.get(r.outlet_id, 0)) * 1.2), "tip": f"<b>{r.outlet_id}</b> {r.outlet_name}<br>{r.location}<br>health <b>{health.get(r.outlet_id, '–')}</b> · {int(tk.get(r.outlet_id, 0))} open tickets", "open": r.outlet_id} for r in g.itertuples() if r.outlet_id in health]
    mp = B.map_([{"type": "points", "id": "outlets", "pts": pts}], h=500, view={"bounds": [[7, 69], [34, 90]]}, key="ops_map", legend=[("Outlet health ≥ 80", TEAL), ("Health 60–79 (watch)", AMBER), ("Health < 60 (critical)", RED), ("Bigger dot = more open tickets", "#8b8d98")])
    hs = pd.Series(health).sort_values() if health else pd.Series(dtype=float); hs = hs[hs.index == c["scope"]] if c.get("scope") else hs
    need = pd.DataFrame({"outlet": hs.index[:10], "health": hs.values[:10]}); need["tickets"] = need.outlet.map(tk).fillna(0).astype(int)
    h = Q("SELECT substr(ts,12,8) t, score FROM decisions ORDER BY decision_id DESC LIMIT 80").iloc[::-1]; sales = Q("SELECT substr(ts,12,8) time, outlet_id, ROUND(amount) amount, source FROM sales ORDER BY sale_id DESC LIMIT 7")
    by_ag = oa.assign(agent=oa.type.map(lambda t: dom.ACTION_AGENT.get(t, "engine"))).groupby("agent").size().sort_values() if len(oa) else pd.Series(dtype=int); ex = dom.kpi_extra()
    bl = [B.kpis(("Franchise health", d["score"] if d else "…", (d or {}).get("risk", "") + " risk", "Critical" if d and d["risk"] == "High" else "Warning" if d and d["risk"] == "Medium" else "Low"), ("Open tickets", int((oa.status == "open").sum()) if len(oa) else 0), ("Critical", int(((oa.severity == "Critical") & (oa.status == "open")).sum()) if len(oa) else 0, None, "Critical"),
                  ("Awaiting approval", int((oa.status == "proposed").sum()) if len(oa) else 0, "agent proposals", "Warning"), (ex[0][0], ex[0][1]), (ex[1][0], ex[1][1]))]
    if d: bl.append({"t": "brief", "text": d["narrative"], "id": d["id"], "can_brief": True})
    bl.append(B.row(B.card("Outlet map · click an outlet", mp), B.col(B.card("Needs attention first", B.table(need, ["outlet", B.num_col("health", "health", "bar", 100), B.num_col("tickets", "tickets", "int")], h=250, open={"col": "outlet"}, key="need")),
                                                                B.card("Health trend", B.chart(B.line(h.t, h.score, ymin=0, ymax=100, colors=[TEAL]), h=150, key="trend"))), ratio=[3, 2]))
    bl.append(B.row(B.card("Live sales stream", B.table(sales, ["time", "outlet_id", B.num_col("amount", "amount", "money"), B.pill_col("source")], h=240, open={"col": "outlet_id"}, key="sales")), B.card("Open tickets by agent", B.chart(B.bar(by_ag.index, by_ag.values, horizontal=True, colors=[BLUE]), h=240, key="byag"))))
    return bl

# ------------------------------------------------------------------ detail drawer
def detail(c, e):
    k = EN.kind_of(e)
    if k != "outlet": return {"title": e, "blocks": [B.callout("No detail view for this entity.", "info")]}
    d = decision(); health = (d or {}).get("ev", {}).get("health", {}); o = Q("SELECT outlet_name, location, revenue, customer_satisfaction csat, staff_headcount hc FROM outlets WHERE outlet_id=?", (e,))
    if not len(o): return {"title": e, "blocks": [B.callout("Unknown outlet.", "warn")]}
    r = o.iloc[0]; bd = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(e)
        if v is not None: bd[a] = round(v, 1)
    s = Q("SELECT substr(ts,12,8) t, amount FROM sales WHERE outlet_id=? ORDER BY sale_id DESC LIMIT 50", (e,)).iloc[::-1]
    lo = Q("SELECT sku_name, current_stock stock, weekly_demand wk, lead_time_days lead FROM inventory WHERE outlet_id=? ORDER BY current_stock*1.0/MAX(weekly_demand,1) LIMIT 5", (e,))
    h = health.get(e); bl = [B.kpis(("Health", h if h is not None else "–", None, "Critical" if h is not None and h < 60 else "Warning" if h is not None and h < 80 else "Low"), ("Satisfaction", f"{r.csat:.2f}"), ("Headcount", int(r.hc)), ("Revenue", f"{r.revenue:,.0f}"))]
    if bd: bl.append(B.card("Why: score from each agent", B.chart(B.bar(list(bd), list(bd.values()), horizontal=True, max_=100, itemcolors=[color_h(v) for v in bd.values()]), h=200, key="d_bd"), sub="weights: outlet 30% · inventory 20% · staff 20% · marketing 15% · audit 15%"))
    if len(s): bl.append(B.card("Live sales stream", B.chart(B.line(s.t, s.amount, colors=[TEAL]), h=170, key="d_s")))
    bl.append(B.table(lo, ["sku_name", "stock", "wk", "lead"], title="Lowest-cover SKUs", h=190, key="d_lo"))
    try:
        import alts; bl += alts.outlet_alternatives(e)
    except Exception as ex: bl.append(B.callout(f"Transfer suggestions unavailable: {ex}", "info"))
    bl.append(ticket_block(c, target=e, title=f"Tickets for {e}", key="d_tk"))
    if can(c, "run"): bl.append(B.button("rerun", "Re-run all agents now", "/api/run_cycle", "ghost"))
    return {"title": f"{e} · {r.outlet_name}", "sub": r.location, "blocks": bl}

# ------------------------------------------------------------------ agent pages
def workforce(c, tab=None):
    w, p = _sc(c); s = Q("SELECT staff_id, outlet_id, name, role, salary, overtime_hrs, job_satisfaction sat, predicted_attrition_prob p FROM staff" + w, p)
    bl = [hero(c, "staff", "Workforce agent", "Attrition risk (LogisticRegression), attendance and overtime for every employee. Feeds 20% of the outlet health score."),
          B.kpis(("Employees", f"{len(s):,}"), ("Avg attrition risk", f"{s.p.mean():.0%}"), ("High risk (>60%)", int((s.p > .6).sum()), None, "Warning"), ("Avg overtime h", round(s.overtime_hrs.mean(), 1)))]
    g = s.groupby("outlet_id").p.mean().sort_values(ascending=False).head(12); at = Q("SELECT status, COUNT(*) n FROM attendance" + w + " GROUP BY status", p)
    bl.append(B.row(B.card("Attrition probability distribution", B.chart(B.hist(s.p, 24, TEAL, pct=True), h=270, key="wf_h")), B.card("Outlets with highest average attrition risk (click)", B.chart(B.bar(g.index, (g.values * 100).round(1), horizontal=True, colors=[AMBER], pct=True), h=270, open={"bar": "y"}, key="wf_b"))))
    t = dom.attrition_table(c.get("scope"), 15); imp = dom.attrition_importance()
    bl.append(B.row(B.card("Live attendance (all recorded shifts)", B.chart(B.pie(at.set_index("status").n.to_dict(), colors={"Present": TEAL, "Late": AMBER, "Absent": RED}), h=270, key="wf_a")),
                    B.card("Highest-risk employees and their main driver", B.table(t[["staff_id", "outlet_id", "name", "role", "prob", "driver"]] if len(t) else pd.DataFrame(), ["staff_id", "outlet_id", "name", "role", B.num_col("prob", "attrition", "pct"), "driver"], h=270, open={"col": "outlet_id"}, key="wf_t"))))
    if imp: ip = pd.Series(imp).sort_values(); bl.append(B.card("What pushes attrition up (positive) or down (negative)", B.chart(B.bar(ip.index, ip.values.round(3), horizontal=True, colors=[VIOLET]), h=240, key="wf_i")))
    mc = model_card("logreg_attrition", ["accuracy", "auc", "source"]); bl += [mc] if mc else []
    x = cross(c, s.groupby("outlet_id").p.mean().sort_values(ascending=False).head(15).index, "staff"); bl += [x] if x else []
    bl.append(ticket_block(c, "staff", title="Open tickets from this agent", key="tk_staff")); return B.page("Workforce", "Agent 1 · attrition, attendance, overtime", bl, live=12)

def outlets(c, tab=None):
    r = latest("outlet"); out = (r or {}).get("out", {}); tiers = pd.Series(out.get("tiers", {})); d = decision(); health = (d or {}).get("ev", {}).get("health", {})
    o = Q("SELECT o.outlet_id, o.outlet_name, o.location, o.revenue, o.customer_satisfaction csat, o.staff_headcount hc, g.lat, g.lon FROM outlets o LEFT JOIN outlet_geo g USING(outlet_id)" + (" WHERE o.outlet_id=?" if c.get("scope") else ""), (c["scope"],) if c.get("scope") else ()); o["tier"] = o.outlet_id.map(tiers); o["health"] = o.outlet_id.map(health); o = o[o.outlet_id == c["scope"]] if c.get("scope") else o
    order = ["Excellent", "Good", "Needs Attention", "Critical"]; col = {"Excellent": TEAL, "Good": BLUE, "Needs Attention": AMBER, "Critical": RED}; tc = o.tier.value_counts().reindex(order).fillna(0)
    bl = [hero(c, "outlet", "Outlet performance agent", "KMeans (k=4) over revenue, orders and customers ranks every outlet. Drives 30% of the health score."),
          B.kpis(("Outlets", len(o)), ("Avg health", f"{o.health.mean():.1f}" if o.health.notna().any() else "…"), ("Critical tier", int((o.tier == "Critical").sum()), None, "Critical"), ("Silhouette", out.get("silhouette", "n/a")))]
    sc = {t: [{"x": a.revenue, "y": a.csat, "label": a.outlet_id, "size": 6 + a.hc / 4} for a in o[o.tier == t].itertuples()] for t in order if (o.tier == t).any()}
    bl.append(B.row(B.card("Outlets per KMeans tier", B.chart(B.bar(order, tc.values.astype(int).tolist(), itemcolors=[col[t] for t in order]), h=290, key="ou_t")), B.card("Revenue vs satisfaction (size = headcount · click a dot)", B.chart(B.scatter(sc, "revenue", "satisfaction", colors=col, sized=True), h=290, open={"scatter": True}, key="ou_s"))))
    bl.append(B.table(o.sort_values("health")[["outlet_id", "outlet_name", "location", "tier", "health", "csat", "hc"]], ["outlet_id", "outlet_name", "location", B.pill_col("tier"), B.num_col("health", "health", "bar", 100), B.num_col("csat", "csat", "num"), B.num_col("hc", "staff", "int")], title="All outlets · click a row for the drill-down", h=360, open={"col": "outlet_id"}, key="ou_tb", search=True))
    x = cross(c, o.sort_values("health").head(15).outlet_id, "outlet"); bl += [x] if x else []
    allo = o.sort_values("outlet_id").outlet_id.tolist(); picks = [p for p in ((c.get("params") or {}).get("cmp") or []) if p in allo][:5] or o.sort_values("health", ascending=False).outlet_id.tolist()[:2] + o.sort_values("health").outlet_id.tolist()[:1]
    names = ["outlet", "inventory", "staff", "marketing", "audit", "feedback"]; ser = {}; rows = []
    for p in picks:
        vals = [round(float(EN.latest_out(a).get("per_outlet", {}).get(p, 0) or 0), 1) for a in names]; ser[p] = vals; rr = o[o.outlet_id == p].iloc[0]; rows.append({"outlet": p, "location": rr.location, "health": health.get(p), "revenue": rr.revenue, "csat": rr.csat, "hc": rr.hc, **dict(zip(names, vals))})
    bl.append({"t": "form", "id": "cmp", "fields": [{"name": "cmp", "label": "Benchmark up to 5 outlets", "type": "multi", "options": allo, "value": picks}], "submit": "Compare", "url": "page:outlets", "title": None, "tone": "ghost", "inline": True})
    bl.append(B.row(B.card("Outlet benchmarking: score from each agent", B.chart(B.radar(names, ser), h=330, key="cmp_r")), B.card("Side by side", B.table(rows, ["outlet", "location", B.num_col("health", "health", "bar", 100)] + [B.num_col(a, a, "num") for a in names] + [B.num_col("revenue", "revenue", "money")], h=330, open={"col": "outlet"}, key="cmp_t")), ratio=[2, 3]))
    bl.append(ticket_block(c, "outlet", ["outlet_review", "health_alert"], title="Open tickets from this agent", key="tk_out")); return B.page("Outlets", "Agent 2 · KMeans tiers and the live health score", bl, live=12)

def inventory(c, tab=None):
    w, p = _sc(c); d = Q("SELECT outlet_id, sku_name, category, current_stock, reorder_threshold mn, weekly_demand wd, lead_time_days lead, stockout_risk_prob risk FROM inventory" + w, p)
    mult = LE.get_state("demand_mult", 1.0) or 1.0; d["cover_days"] = (d.current_stock / (d.wd.clip(lower=1) / 7 * mult)).round(1); d["short"] = d.cover_days < d.lead + 2
    po = Q("SELECT po_id, outlet_id, sku_name, qty, ts, eta_ts, status FROM purchase_orders" + w + " ORDER BY po_id DESC LIMIT 300", p)
    bl = [hero(c, "inventory", "Inventory agent", "Cover days = stock ÷ (forecast demand × live demand multiplier). Below lead time + 2 days raises a reorder and creates a purchase order."),
          B.kpis(("SKUs tracked", f"{len(d):,}"), ("Flagged short", int(d.short.sum()), None, "Warning"), ("POs in transit", int((po.status == "ordered").sum())), ("POs delivered", int((po.status == "delivered").sum())), ("Demand multiplier", f"x{mult}", "weather + holiday feed"))]
    cat = d.groupby("category").short.mean().mul(100).round(1)
    bl.append(B.row(B.card("Stock cover (days)", B.chart(B.hist(d.cover_days, 28, BLUE), h=270, key="iv_h")), B.card("% of SKUs short, by category", B.chart(B.bar(cat.index, cat.values, colors=[AMBER], pct=True), h=270, key="iv_c"))))
    bl.append(B.row(B.card("Most urgent SKUs", B.table(d[d.short].sort_values("cover_days").head(15)[["outlet_id", "sku_name", "current_stock", "cover_days", "lead", "risk"]], ["outlet_id", "sku_name", "current_stock", B.num_col("cover_days", "cover days", "num"), "lead", B.num_col("risk", "stock-out risk", "pct")], h=290, open={"col": "outlet_id"}, key="iv_u")),
                    B.card("Purchase orders created by this agent (they deliver into stock, closing the loop)", B.table(po.head(20), ["po_id", "outlet_id", "sku_name", "qty", B.pill_col("status"), "eta_ts"], h=290, open={"col": "outlet_id"}, key="iv_p"))))
    outs = sorted(d.outlet_id.unique()); sel = (c.get("params") or {}).get("o") or (outs[0] if outs else None); meth = (c.get("params") or {}).get("m") or "Prophet (if installed)"
    bl.append({"t": "form", "id": "inv_fc", "fields": [{"name": "o", "label": "Demand forecast for outlet", "type": "select", "options": outs, "value": sel}, {"name": "m", "label": "Method", "type": "select", "options": ["Prophet (if installed)", "Trend x weekday"], "value": meth}], "submit": "Forecast", "url": "page:inventory", "title": None, "tone": "ghost", "inline": True})
    if sel:
        h, f, how = dom.forecast_series(sel, 14, "trend" if meth.startswith("Trend") else None)
        if len(f): x = list(h.d.astype(str)) + list(f.d.astype(str)); bl.append(B.card(f"{sel}: daily sales and 14-day forecast ({how})", B.chart(B.line(x, {"actual": list(h.a) + [None] * len(f), "forecast": [None] * (len(h) - 1) + [h.a.iloc[-1]] + list(f.a)}, colors=[TEAL, AMBER], dashed=("forecast",), band=([None] * len(h) + list(f.lo), [None] * len(h) + list(f.hi))), h=290, key="iv_f")))
    x = cross(c, d[d.short].outlet_id.unique(), "inventory"); bl += [x] if x else []
    bl.append(ticket_block(c, "inventory", title="Open tickets from this agent", key="tk_inv")); return B.page("Inventory", "Agent 3 · cover days, reorders, forecast", bl, live=10)

def marketing(c, tab=None):
    w, p = _sc(c); m = Q("SELECT outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions, start_date, end_date FROM marketing" + w, p); today = datetime.date.today().isoformat()
    m["status"] = np.where(m.end_date >= today, "live", "ended / paused"); m["roi_pct"] = ((m.actual_roi - 1) * 100).round(0); r = (latest("marketing") or {}).get("out", {})
    bl = [hero(c, "marketing", "Marketing agent", "Campaign ROI = (revenue − budget) ÷ budget, plus KMeans (k=3) customer segments. Campaigns are paused when Inventory reports a shortage."),
          B.kpis(("Campaigns", len(m)), ("Live now", int((m.status == "live").sum())), ("Avg ROI", f"{m.roi_pct.mean():.0f}%"), ("Best channel", r.get("best_channel", "…")))]
    ch = m.groupby("channel").actual_roi.mean().sub(1).mul(100).round(0).sort_values(); seg = r.get("segments", {})
    bl.append(B.row(B.card("Average ROI % by channel", B.chart(B.bar(ch.index, ch.values, horizontal=True, colors=[TEAL], pct=True), h=270, key="mk_c")), B.card("Customer segments (KMeans k=3)", B.chart(B.pie(seg), h=270, key="mk_s")) if seg else B.callout("Segments appear after the first agent run.", "info")))
    bl.append(B.table(m[m.status == "live"].sort_values("roi_pct")[["outlet_id", "campaign_name", "channel", "budget", "roi_pct", "end_date"]].head(30), ["outlet_id", "campaign_name", "channel", B.num_col("budget", "budget", "money"), B.num_col("roi_pct", "ROI %", "num"), "end_date"], title="Live campaigns (lowest ROI first)", h=300, open={"col": "outlet_id"}, key="mk_l"))
    paused = Q("SELECT target, detail, ts FROM agent_actions WHERE type='pause_promo' ORDER BY action_id DESC LIMIT 8")
    if len(paused): bl.append(B.table(paused, ["target", "detail", "ts"], title="Promotions paused by the cross-agent rule (stock shortage + live campaign)", h=220, open={"col": "target"}, key="mk_p"))
    x = cross(c, m[m.status == "live"].outlet_id.unique(), "marketing"); bl += [x] if x else []
    bl.append(ticket_block(c, "marketing", title="Open tickets from this agent", key="tk_mk")); return B.page("Marketing", "Agent 4 · ROI, segments, campaign control", bl, live=12)

def sentiment(c, tab=None):
    w, p = _sc(c); f = Q("SELECT outlet_id, rating, comment, date, sentiment_score s FROM feedback" + w + " ORDER BY rowid DESC LIMIT 4000", p); r = (latest("feedback") or {}).get("out", {})
    bl = [hero(c, "feedback", "Sentiment agent", "VADER + star rating on every review, with issue categories. A review submitted here is analysed at once and picked up by the next agent cycle."),
          B.kpis(("Reviews analysed", len(f)), ("Avg rating", f"{f.rating.mean():.2f}" if len(f) else "n/a"), ("Outlets scoring < 45", sum(v < 45 for v in r.get("per_outlet", {}).values()), None, "Warning"))]
    iss = pd.Series(r.get("issues", {})).sort_values(); rd = f.rating.value_counts().sort_index()
    bl.append(B.row(B.card("Issue categories in reviews", B.chart(B.bar(iss.index, iss.values, horizontal=True, colors=[AMBER]), h=270, key="se_i")) if len(iss) else B.callout("Issue categories appear after the first run.", "info"), B.card("Rating distribution", B.chart(B.bar(rd.index.astype(int).astype(str), rd.values.astype(int), colors=[TEAL]), h=270, key="se_r"))))
    per = pd.Series(r.get("per_outlet", {})); per = per[per.index == c["scope"]] if c.get("scope") else per.sort_values().head(10)
    if len(per): bl.append(B.table(per.rename("score").reset_index().rename(columns={"index": "outlet"}), ["outlet", B.num_col("score", "score", "bar", 100)], title="Lowest sentiment scores (0-100)", h=250, open={"col": "outlet"}, key="se_l"))
    bl.append(B.table(f.head(14)[["outlet_id", "rating", "comment", "date"]], ["outlet_id", "rating", "comment", "date"], title="Latest reviews", h=300, open={"col": "outlet_id"}, key="se_t"))
    outs = [c["scope"]] if c.get("scope") else Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()
    if can(c, "review"): bl.append(B.form("review", [{"name": "outlet", "label": "Outlet", "type": "select", "options": outs, "value": outs[0]}, {"name": "rating", "label": "Rating", "type": "range", "min": 1, "max": 5, "value": 4}, {"name": "comment", "label": "Comment (any language, translated by NLLB)", "type": "textarea"}], "Submit review", "/api/franchise/review", title="Add a review", sub="Stored and analysed immediately; the agent includes it in its next cycle."))
    if can(c, "view"): bl.append(ticket_block(c, "feedback", title="Open tickets from this agent", key="tk_fb"))
    return B.page("Sentiment", "Agent 5 · reviews, VADER, issue categories", bl, live=15)

def audit(c, tab=None):
    w, p = _sc(c); a = Q("SELECT outlet_id, audit_date, score, violations, category, status, notes FROM audits" + w + " ORDER BY audit_date DESC, rowid DESC", p); last = a.drop_duplicates("outlet_id")
    bl = [hero(c, "audit", "Audit & compliance agent", "Latest inspection per outlet. Pass ≥ 80, Warning 60–79, Fail < 60. A fail opens a 72-hour corrective-action ticket and re-plans the Staff and Sentiment agents."),
          B.kpis(("Outlets audited", len(last)), ("Pass", int((last.score >= 80).sum()), None, "Low"), ("Warning", int(((last.score >= 60) & (last.score < 80)).sum()), None, "Warning"), ("Fail", int((last.score < 60).sum()), None, "Critical"))]
    cg = a.groupby("category").score.mean().round(1)
    bl.append(B.row(B.card("Latest audit scores", B.chart(B.hist(last.score, 20, TEAL), h=270, key="au_h")), B.card("Average score by category", B.chart(B.bar(cg.index, cg.values, colors=[BLUE]), h=270, key="au_c"))))
    bl.append(B.table(last.sort_values("score").head(15), ["outlet_id", "audit_date", B.num_col("score", "score", "bar", 100), "violations", "category", B.pill_col("status"), "notes"], title="Lowest-scoring outlets", h=320, open={"col": "outlet_id"}, key="au_l"))
    if can(c, "audit_write"):
        outs = [c["scope"]] if c.get("scope") else Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()
        bl.append(B.form("audit_log", [{"name": "outlet", "label": "Outlet", "type": "select", "options": outs, "value": outs[0]}, {"name": "score", "label": "Score", "type": "range", "min": 0, "max": 100, "value": 82}, {"name": "category", "label": "Category", "type": "select", "options": ["Food Safety", "Hygiene", "Documentation", "Equipment", "Staff Training"], "value": "Food Safety"}, {"name": "notes", "label": "Notes", "type": "text"}], "Save inspection", "/api/franchise/audit", title="Log an inspection", sub="Runs the Audit agent straight away; a fail opens a corrective-action ticket."))
    x = cross(c, last[last.score < 80].outlet_id, "audit"); bl += [x] if x else []
    bl.append(ticket_block(c, "audit", title="Open tickets from this agent", key="tk_au")); return B.page("Audit", "Agent 6 · inspections and compliance", bl, live=15)

PAGES = {"workforce": workforce, "outlets": outlets, "inventory": inventory, "marketing": marketing, "sentiment": sentiment, "audit": audit}


In [ ]:
%%writefile /content/app/weather.py
"""Real weather for every port: Open-Meteo forecast + marine APIs (free, no key), fetched in batches and stored in wx_live.
Falls back to the simulated values already in weather_risks when the network is closed; the page says which one it is showing."""
import time, threading, datetime, json
import live_engine as LE

try: import requests
except Exception: requests = None

FORECAST = "https://api.open-meteo.com/v1/forecast"
MARINE = "https://marine-api.open-meteo.com/v1/marine"
WMO = {0: "Clear", 1: "Mostly clear", 2: "Partly cloudy", 3: "Overcast", 45: "Fog", 48: "Rime fog", 51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle", 56: "Freezing drizzle", 57: "Freezing drizzle", 61: "Light rain", 63: "Rain", 65: "Heavy rain", 66: "Freezing rain", 67: "Freezing rain",
       71: "Light snow", 73: "Snow", 75: "Heavy snow", 77: "Snow grains", 80: "Rain showers", 81: "Rain showers", 82: "Violent showers", 85: "Snow showers", 86: "Snow showers", 95: "Thunderstorm", 96: "Thunderstorm, hail", 99: "Thunderstorm, heavy hail"}
_get = None   # tests can replace this with a function(url, params) -> json

def ensure():
    c = LE.conn(); c.execute("CREATE TABLE IF NOT EXISTS wx_live(port_name TEXT PRIMARY KEY, ts TEXT, wind_ms REAL, gust_ms REAL, wind_dir REAL, temp_c REAL, precip_mm REAL, wcode INT, wave_m REAL, wave_dir REAL, src TEXT)"); c.commit()

def _fetch(url, params):
    if _get: return _get(url, params)
    r = requests.get(url, params=params, timeout=10); r.raise_for_status(); return r.json()

def _listify(j): return j if isinstance(j, list) else [j]

def refresh(batch=50):
    """Fetch current weather for every port. Returns (rows_updated, message)."""
    ensure(); c = LE.conn(); ports = c.execute("SELECT port_name, lat, lon FROM ports").fetchall()
    if not ports: return 0, "no ports"
    n = 0; err = ""; ts = datetime.datetime.utcnow().isoformat(timespec="seconds")
    for i in range(0, len(ports), batch):
        chunk = ports[i:i + batch]; la = ",".join(f"{p[1]:.3f}" for p in chunk); lo = ",".join(f"{p[2]:.3f}" for p in chunk)
        try:
            f = _listify(_fetch(FORECAST, {"latitude": la, "longitude": lo, "current": "temperature_2m,precipitation,weather_code,wind_speed_10m,wind_direction_10m,wind_gusts_10m", "wind_speed_unit": "ms", "timezone": "UTC"}))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"; break
        try: m = _listify(_fetch(MARINE, {"latitude": la, "longitude": lo, "current": "wave_height,wave_direction", "timezone": "UTC"}))
        except Exception: m = [{}] * len(chunk)
        for p, a, b in zip(chunk, f, m + [{}] * (len(chunk) - len(m))):
            cur = (a or {}).get("current") or {}
            if cur.get("wind_speed_10m") is None: continue
            mc = (b or {}).get("current") or {}; wave = mc.get("wave_height")
            c.execute("INSERT OR REPLACE INTO wx_live VALUES(?,?,?,?,?,?,?,?,?,?,?)", (p[0], ts, cur.get("wind_speed_10m"), cur.get("wind_gusts_10m"), cur.get("wind_direction_10m"), cur.get("temperature_2m"), cur.get("precipitation") or 0.0, cur.get("weather_code"), wave, mc.get("wave_direction"), "Open-Meteo"))
            wind = float(cur["wind_speed_10m"]); lvl = "High" if wind > 15 else "Medium" if wind >= 8 else "Low"
            c.execute("INSERT OR REPLACE INTO weather_risks(port_name,current_severity,forecast,wind_speed,wave_height,temperature) VALUES(?,?,?,?,?,?)", (p[0], lvl, f"Open-Meteo live {ts[11:16]} UTC · {WMO.get(cur.get('weather_code'), '')}", wind * 3.6, float(wave) if wave is not None else 1.2, cur.get("temperature_2m")))
            n += 1
    c.commit(); LE.set_state("wx_status", {"ts": ts if n else (LE.get_state("wx_status", {}) or {}).get("ts"), "ok": n, "total": len(ports), "err": err, "tried": ts})
    return n, (err or f"{n} ports updated")

def status():
    s = LE.get_state("wx_status", {}) or {}; ok = int(s.get("ok") or 0); age = None
    if s.get("ts"):
        try: age = (datetime.datetime.utcnow() - datetime.datetime.fromisoformat(s["ts"])).total_seconds() / 60
        except Exception: pass
    live = bool(ok) and age is not None and age < 120
    return {"live": live, "ts": s.get("ts"), "age_min": age, "ok": ok, "total": s.get("total"), "err": s.get("err", ""), "tried": s.get("tried")}

def fresh_ports(max_age_min=90):
    """Ports whose real weather is recent: the simulator must not overwrite them."""
    try:
        cut = (datetime.datetime.utcnow() - datetime.timedelta(minutes=max_age_min)).isoformat(timespec="seconds")
        return {r[0] for r in LE.conn().execute("SELECT port_name FROM wx_live WHERE ts>=?", (cut,)).fetchall()}
    except Exception: return set()

_T = {"t": None}
def start(every_min=20):
    if _T["t"]: return
    def loop():
        time.sleep(5)
        while True:
            try: refresh()
            except Exception: pass
            time.sleep(every_min * 60 if status()["live"] else 120)
    _T["t"] = threading.Thread(target=loop, daemon=True); _T["t"].start()

def table():
    import pandas as pd
    ensure()
    return pd.read_sql("SELECT w.*, p.lat, p.lon FROM wx_live w JOIN ports p USING(port_name)", LE.conn())


In [ ]:
%%writefile /content/app/weather_context.py
import requests

CITY_COORDS = {
    "Mumbai": (19.0760, 72.8777), "Delhi": (28.61, 77.21), "Bangalore": (12.97, 77.59),
    "Chennai": (13.0827, 80.2707), "Hyderabad": (17.39, 78.49), "Pune": (18.52, 73.86),
    "Ahmedabad": (23.03, 72.57), "Jaipur": (26.91, 75.79), "Kolkata": (22.5726, 88.3639), "Surat": (21.1702, 72.8311)
}

def get_weather_report(city):
    try:
        lat, lon = CITY_COORDS.get(city, (19.08, 72.88))
        r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true", timeout=5)
        if r.status_code == 200:
            cw = r.json().get("current_weather", {})
            return {"temp": cw.get("temperature", 28), "wind": cw.get("windspeed", 10), "city": city}
    except: pass
    return {"temp": 28, "wind": 10, "city": city}

def get_city_weather(city):
    return get_weather_report(city)

def get_route_weather_multiplier(origin, dest):
    try:
        w1 = get_weather_report(origin)
        w2 = get_weather_report(dest)
        avg_wind = (w1["wind"] + w2["wind"]) / 2
        return 1.0 + (avg_wind / 100)
    except:
        return 1.0


In [ ]:
%%writefile /content/app/worker.py
"""Background worker process: the simulator, the agent cycles, escalation and the feed threads run here so the web server (and the GPU models) never share a GIL with them."""
import os, sys, time, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config, live_engine as LE
dom = importlib.import_module("domain_franchise" if config.KIND == "Franchise" else "domain_freight")
if __name__ == "__main__":
    rt = LE.Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    if config.KIND == "Freight":
        import weather as WX; WX.start()
    while True: time.sleep(3600)


In [ ]:
%%writefile /content/app/static/app.js
"use strict";
const $=(s,r=document)=>r.querySelector(s);
const esc=s=>String(s==null?"":s).replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
function h(tag,attrs,...kids){const e=document.createElement(tag);for(const k in(attrs||{})){const v=attrs[k];if(v==null||v===false)continue;if(k==="class")e.className=v;else if(k==="html"||k==="innerHTML")e.innerHTML=v;else if(k.startsWith("on"))e.addEventListener(k.slice(2),v);else e.setAttribute(k,v===true?"":v)}for(const c of kids.flat())if(c!=null&&c!==false)e.append(c.nodeType?c:document.createTextNode(c));return e}
const TONE={Critical:"#cd2b31",High:"#cd2b31",Fail:"#cd2b31",rejected:"#cd2b31","On Hold":"#cd2b31",overdue:"#cd2b31",Warning:"#ab6400",Medium:"#ab6400","Pending Approval":"#ab6400",proposed:"#793aaf",Info:"#3451b2",open:"#3451b2",Low:"#0b7a6c",Pass:"#0b7a6c",Approved:"#0b7a6c",approved:"#0b7a6c",resolved:"#0b7a6c","auto-closed":"#6b7080",live:"#0b7a6c",Healthy:"#0b7a6c",Watch:"#ab6400",Rejected:"#6b7080"};
const tone=v=>TONE[v]||(/crit|high|fail|bad|reject|hold/i.test(v)?"#cd2b31":/warn|med|pend/i.test(v)?"#ab6400":/ok|pass|low|good|appr|done|clear|live/i.test(v)?"#0b7a6c":"#6b7080");
const ICON={bot:"M12 8V4H8M4 12a2 2 0 0 1 2-2h12a2 2 0 0 1 2 2v6a2 2 0 0 1-2 2H6a2 2 0 0 1-2-2zM2 14h2M20 14h2M9 15v1M15 15v1",users:"M16 21v-2a4 4 0 0 0-4-4H6a4 4 0 0 0-4 4v2M9 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8M22 21v-2a4 4 0 0 0-3-3.9M16 3.1a4 4 0 0 1 0 7.8",store:"M3 9l1-5h16l1 5M3 9v11h18V9M3 9a3 3 0 0 0 6 0 3 3 0 0 0 6 0 3 3 0 0 0 6 0M9 20v-6h6v6",boxes:"M21 8l-9-5-9 5 9 5zM3 8v8l9 5 9-5V8M12 13v8",megaphone:"M3 11v2a1 1 0 0 0 1 1h3l8 5V5L7 10H4a1 1 0 0 0-1 1zM19 8a5 5 0 0 1 0 8",messages:"M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z",clipboard:"M9 4h6v3H9zM8 5H6a2 2 0 0 0-2 2v13a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V7a2 2 0 0 0-2-2h-2M9 13l2 2 4-4",bell:"M18 8a6 6 0 0 0-12 0c0 7-3 9-3 9h18s-3-2-3-9M13.7 21a2 2 0 0 1-3.4 0",network:"M12 2v6M12 16v6M5 6l5 4M19 6l-5 4M5 18l5-4M19 18l-5-4M12 8a4 4 0 1 0 0 8 4 4 0 0 0 0-8",flask:"M10 2v7L4 20a1 1 0 0 0 1 1h14a1 1 0 0 0 1-1l-6-11V2M8 2h8M7 15h10",upload:"M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4M17 8l-5-5-5 5M12 3v12",shield:"M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10",radar:"M12 12l7-7M12 2a10 10 0 1 0 10 10M12 6a6 6 0 1 0 6 6M12 10a2 2 0 1 0 2 2",route:"M6 19a3 3 0 1 0 0-6 3 3 0 0 0 0 6M18 11a3 3 0 1 0 0-6 3 3 0 0 0 0 6M9 16h6a3 3 0 0 0 0-6H9a3 3 0 0 1 0-6h3",dollar:"M12 2v20M17 5H9.5a3.5 3.5 0 0 0 0 7h5a3.5 3.5 0 0 1 0 7H6",truck:"M1 3h15v13H1zM16 8h4l3 3v5h-7M5.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5M18.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5",cloud:"M17.5 19a4.5 4.5 0 1 0-1.4-8.8A6 6 0 1 0 5 15a4 4 0 0 0 1 4z",trend:"M3 17l6-6 4 4 8-8M15 7h6v6",chart:"M3 3v18h18M7 15l4-4 3 3 5-6",check:"M22 11.1V12a10 10 0 1 1-5.9-9.1M22 4L12 14l-3-3",user:"M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2M12 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8",file:"M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8zM14 2v6h6M9 13h6M9 17h6"};
const icon=n=>{const s=document.createElementNS("http://www.w3.org/2000/svg","svg");s.setAttribute("viewBox","0 0 24 24");const p=document.createElementNS("http://www.w3.org/2000/svg","path");p.setAttribute("d",ICON[n]||ICON.file);s.append(p);return s};
const S={me:null,page:null,tab:null,params:{},lang:localStorage.getItem("lang")||"English",pulse:null,tree:null,timer:null,seq:0,boot:null};

const applyTheme=theme=>{document.documentElement.dataset.theme=theme;localStorage.setItem("franchiseops_theme",theme)};
const toggleTheme=()=>applyTheme(document.documentElement.dataset.theme==="dark"?"light":"dark");
applyTheme(localStorage.getItem("franchiseops_theme")||"light");
const themeButton=()=>h("button",{class:"btn sm ghost",title:"Toggle dark mode",onclick:e=>{toggleTheme();e.currentTarget.textContent=document.documentElement.dataset.theme==="dark"?"Light mode":"Dark mode"}},document.documentElement.dataset.theme==="dark"?"Light mode":"Dark mode");


const indiaTime=ts=>{if(!ts)return "";const raw=String(ts);const dated=/([zZ]|[+-]\d{2}:?\d{2})$/.test(raw)?raw:raw+"Z";const value=new Date(dated);return Number.isNaN(value.getTime())?raw:new Intl.DateTimeFormat("en-IN",{timeZone:"Asia/Kolkata",hour:"2-digit",minute:"2-digit",second:"2-digit",hour12:false}).format(value)+" IST"};

const LANGS_CODE={};
async function api(path,opt={}){const o={credentials:"same-origin",...opt};if(opt.json!==undefined){o.method=o.method||"POST";o.headers={"Content-Type":"application/json"};o.body=JSON.stringify(opt.json)}
 const r=await fetch(path,o);let d=null;try{d=await r.json()}catch(e){}
 if(r.status===401&&S.me&&!path.includes("login")){S.me=null;render();return{ok:false,msg:"Session ended"}}
 if(!r.ok&&d&&!d.msg)d.msg=d.detail||("Error "+r.status);if(!d)d={ok:false,msg:"Error "+r.status};return d}
function toast(msg,bad){const t=h("div",{class:bad?"bad":""},msg);$("#toast").append(t);setTimeout(()=>t.remove(),bad?7000:4500)}
const fmt={num:v=>v==null?"–":(typeof v==="number"?(Math.abs(v)>=1000?v.toLocaleString(undefined,{maximumFractionDigits:1}):+v.toFixed(2)):v),int:v=>v==null?"–":Math.round(v).toLocaleString(),money:v=>v==null?"–":"$"+Math.round(v).toLocaleString()};
function md(t){let s=esc(t||"");s=s.replace(/```([\s\S]*?)```/g,(_,c)=>"<pre>"+c.trim()+"</pre>").replace(/`([^`]+)`/g,"<code>$1</code>").replace(/\*\*([^*]+)\*\*/g,"<b>$1</b>");const L=s.split("\n");let o="",inl=false,tb=null;const row=l=>l.trim().replace(/^\||\|$/g,"").split("|").map(x=>x.trim());const endT=()=>{if(tb){o+='<div class="tbl" style="margin:8px 0"><table><thead><tr>'+row(tb[0]).map(x=>"<th>"+x+"</th>").join("")+"</tr></thead><tbody>"+tb.slice(2).map(r=>"<tr>"+row(r).map(x=>"<td>"+x+"</td>").join("")+"</tr>").join("")+"</tbody></table></div>";tb=null}};for(let i=0;i<L.length;i++){const l=L[i];if(/^\s*\|.*\|\s*$/.test(l)){if(inl){o+="</ul>";inl=false}(tb=tb||[]).push(l);continue}endT();const m=l.match(/^\s*[-*•]\s+(.*)/);if(m){if(!inl){o+="<ul>";inl=true}o+="<li>"+m[1]+"</li>"}else{if(inl){o+="</ul>";inl=false}const hd=l.match(/^(#{1,3})\s+(.*)/);o+=hd?`<h${hd[1].length}>${hd[2]}</h${hd[1].length}>`:(l.trim()?l+"<br>":"")}}endT();if(inl)o+="</ul>";return o}

// ---------------------------------------------------------------- block renderers (each returns {el, update(b)})
function mount(b,key){const f=R[b.t];if(!f){return{el:h("div",{class:"card"},"Unsupported block: "+b.t),update(){}}}const m=f(b,key);m.t=b.t;m.k=b.key||b.id||"";return m}
function mountList(parent,blocks,key){const hs=blocks.map((b,i)=>mount(b,key+"/"+i));hs.forEach(x=>parent.append(x.el));return hs}
function updList(parent,hs,blocks,key){if(hs.length!==blocks.length||hs.some((x,i)=>x.t!==blocks[i].t||x.k!==(blocks[i].key||blocks[i].id||""))){parent.replaceChildren();return mountList(parent,blocks,key)}hs.forEach((x,i)=>x.update(blocks[i]));return hs}
const R={};
R.kpis=b=>{const el=h("div",{class:"kpis"});const up=b=>{el.replaceChildren(...b.items.map(i=>h("div",{class:"kpi "+(i.tone||"")},h("div",{class:"l"},i.label),h("div",{class:"v"},typeof i.value==="number"?fmt.num(i.value):i.value),i.sub?h("div",{class:"s"},i.sub):null)))};up(b);return{el,update:up}};
R.callout=b=>{const el=h("div");const up=b=>{el.className="callout "+(b.tone||"info");el.replaceChildren(h("div",null,b.title?h("b",null,b.title+" "):null,h("span",{html:md(b.text)})))};up(b);return{el,update:up}};
R.md=b=>{const el=h("div",{class:"md"});const up=b=>el.innerHTML=md(b.text);up(b);return{el,update:up}};
R.row=(b,k)=>{const el=h("div",{class:"row"});const sty=r=>el.style.gridTemplateColumns=(r||b.cols.map(()=>1)).map(x=>x+"fr").join(" ");sty(b.ratio);let hs=[];const wrap=[];const up=nb=>{sty(nb.ratio);if(hs.length!==nb.cols.length||hs.some((x,i)=>x.t!==nb.cols[i].t)){el.replaceChildren();hs=nb.cols.map((c,i)=>{const m=mount(c,k+"/"+i);el.append(m.el);return m})}else hs.forEach((x,i)=>x.update(nb.cols[i]))};up(b);return{el,update:up}};
R.col=(b,k)=>{const el=h("div",{class:"col"});let hs=mountList(el,b.blocks,k);return{el,update:nb=>{hs=updList(el,hs,nb.blocks,k)}}};
R.card=(b,k)=>{const el=h("div",{class:"card"});const hd=h("h3"),sub=h("div",{class:"sub"}),body=h("div");el.append(hd,sub,body);let hs=[];const up=(nb,first)=>{hd.textContent=nb.title||"";hd.style.display=nb.title?"":"none";sub.textContent=nb.sub||"";sub.style.display=nb.sub?"":"none";hs=first?mountList(body,nb.blocks,k):updList(body,hs,nb.blocks,k)};up(b,true);return{el,update:nb=>up(nb)}};
R.hero=b=>{const el=h("div",{class:"hero"});const up=b=>{const c=b.risk?tone(b.risk):"#8b8d98";el.replaceChildren(h("div",{class:"sc",style:"color:"+c},b.score==null?"–":Math.round(b.score)),h("div",{style:"flex:1"},h("h2",null,b.title," ",b.risk?h("span",{class:"pill",style:`color:${c};border-color:${c}`},b.risk):null),h("p",null,b.blurb),h("p",{style:"color:#454a58"},b.summary||"No run yet."),h("p",null,b.ts?`last run ${indiaTime(b.ts)} · ${b.ms} ms`:"")),b.can_run?h("button",{class:"btn primary",onclick:async e=>{e.target.disabled=true;const r=await api("/api/run_agent",{json:{agent:b.agent}});toast(r.msg,!r.ok);e.target.disabled=false;reloadPage()}},"Run now"):null)};up(b);return{el,update:up}};
R.brief=b=>{const el=h("div",{class:"card"});const out=h("div",{class:"md"});const up=b=>{out.dataset.def=b.text;if(!out.dataset.ai)out.innerHTML=md(b.text)};el.append(h("h3",null,"Executive briefing"),out,b.can_brief?h("button",{class:"btn sm",style:"margin-top:8px",onclick:async e=>{e.target.disabled=true;out.dataset.ai="1";out.textContent="";let acc="";await sse("/api/brief",{lang:S.lang},ev=>{if(ev.ev==="tok"){acc+=ev.text;out.innerHTML=md(acc)}if(ev.ev==="final")out.innerHTML=md(ev.text)});e.target.disabled=false}},"Write with the GPU model"):null);up(b);return{el,update:up}};
R.button=b=>{const el=h("button",{class:"btn "+(b.tone||"primary"),onclick:async()=>{if(b.payload&&b.payload.download){window.open(b.url,"_blank");return}if(b.confirm&&!confirm(b.confirm))return;el.disabled=true;const r=await api(b.url,{json:b.payload||{}});toast(r.msg||"Done",!r.ok);el.disabled=false;if(r.refresh)reloadPage()}},b.label);return{el,update(){}}};
R.table=b=>{const el=h("div"),title=h("h3",{style:"margin:0 0 6px;font-size:13.5px"}),sub=h("div",{class:"sub"}),srch=h("input",{class:"srch",placeholder:"Search…",oninput:()=>{st.q=srch.value.toLowerCase();draw()}}),box=h("div",{class:"tbl"}),st={sort:null,dir:1,q:""};let cur=b;el.append(title,sub,srch,box);
 const draw=()=>{const b=cur;title.textContent=b.title||"";title.style.display=b.title?"":"none";sub.textContent=b.sub||"";sub.style.display=b.sub?"":"none";srch.style.display=b.search?"":"none";if(b.h)box.style.maxHeight=b.h+"px";
  let rows=b.rows;if(st.q)rows=rows.filter(r=>Object.values(r).some(v=>String(v).toLowerCase().includes(st.q)));if(st.sort)rows=[...rows].sort((x,y)=>{const a=x[st.sort],c=y[st.sort];return(a==null)-(c==null)||(a>c?1:a<c?-1:0)*st.dir});
  const mx={};b.cols.forEach(c=>{if(c.type==="pct"){mx[c.k]=Math.max(...b.rows.map(r=>Math.abs(r[c.k]||0)))<=1.0001?100:1}});
  const th=h("tr",null,...b.cols.map(c=>h("th",{onclick:()=>{st.dir=st.sort===c.k?-st.dir:1;st.sort=c.k;draw()}},c.label+(st.sort===c.k?(st.dir>0?" ▲":" ▼"):""))));
  const tb=rows.slice(0,400).map(r=>{const tr=h("tr",{class:b.open?"cl":""},...b.cols.map(c=>{const v=r[c.k];const t=c.type||"text";let n;
    if(t==="pill"&&v!=null&&v!=="")n=h("span",{class:"pill",style:`color:${tone(v)};border-color:${tone(v)}55;background:${tone(v)}18`},v);
    else if(t==="bar"){const m=c.max||100,p=Math.max(0,Math.min(100,(v||0)/m*100)),col=c.k.match(/risk|prob|congest|ci|index/i)?(p>66?"#e5484d":p>40?"#f5a524":"#12a594"):(p<60?"#e5484d":p<80?"#f5a524":"#12a594");n=h("div",{class:"bar"},h("span",null,v==null?"–":fmt.num(v)),h("i",null,h("b",{style:`width:${p}%;background:${col}`})))}
    else if(t==="pct")n=document.createTextNode(v==null?"–":(+(v*mx[c.k]).toFixed(1))+"%");else if(t==="money")n=document.createTextNode(fmt.money(v));else if(t==="int")n=document.createTextNode(fmt.int(v));else if(t==="num")n=document.createTextNode(fmt.num(v));else n=document.createTextNode(v==null?"":v);
    return h("td",{title:typeof v==="string"&&v.length>40?v:null},n)}));if(b.open)tr.onclick=()=>UI.openEntity(r[b.open.col]);return tr});
  const sc=box.scrollTop;box.replaceChildren(h("table",null,h("thead",null,th),h("tbody",null,...tb)));box.scrollTop=sc;if(!rows.length)box.append(h("div",{class:"sub",style:"padding:12px"},"No rows."))};
 draw();return{el,update:nb=>{cur=nb;draw()}}};
R.tickets=b=>{const el=h("div"),st={f:"all",open:new Set()};let cur=b;const body=h("div"),fl=h("div",{class:"filters"});el.append(h("h3",{style:"margin:0 0 6px;font-size:13.5px"},b.title||"Tickets"),h("div",{class:"sub"},b.sub||""),fl,body);
 const act=async(id,a,json)=>{const r=await api(`/api/ticket/${id}/${a}`,{json:json||{}});toast(r.msg,!r.ok);if(document.activeElement)document.activeElement.blur();st.sig=null;reloadPage(true)};
 const draw=()=>{const items=cur.items;st.sig=JSON.stringify(items)+st.f+[...st.open];const cnt=k=>k==="all"?items.length:k==="proposed"?items.filter(i=>i.status==="proposed").length:k==="overdue"?items.filter(i=>i.overdue).length:items.filter(i=>i.sev===k).length;
  fl.replaceChildren(...["all","proposed","Critical","Warning","Info","overdue"].map(k=>h("span",{class:"chip"+(st.f===k?" on":""),onclick:()=>{st.f=k;draw()}},(k==="proposed"?"awaiting approval":k)+" · "+cnt(k))));
  const list=items.filter(i=>st.f==="all"||(st.f==="proposed"?i.status==="proposed":st.f==="overdue"?i.overdue:i.sev===st.f));
  body.replaceChildren(...list.slice(0,80).map(i=>{const c=tone(i.sev);const t=h("div",{class:"tk "+i.sev+" "+i.status},
   h("div",{class:"h"},h("b",null,"#"+i.id+" "+i.type.replace(/_/g," ")),h("span",{class:"pill",style:`color:${c};border-color:${c}66`},i.sev),i.status==="proposed"?h("span",{class:"pill",style:"color:#8e4ec6;border-color:#8e4ec666"},"awaiting approval"):null,i.overdue?h("span",{class:"pill",style:"color:#e5484d;border-color:#e5484d66"},"overdue"):null,h("a",{href:"#",onclick:e=>{e.preventDefault();UI.openEntity(i.target)}},i.target)),
   h("div",{class:"d"},i.detail),h("div",{class:"m"},h("span",null,"agent: "+i.agent),h("span",null,"owner: "+(i.owner||"–")),h("span",null,"assignee: "+(i.assignee||"unassigned")),h("span",null,"due "+(i.due||"–")),h("span",null,i.ts)),
   i.chain&&i.chain.length?h("div",{class:"m"},h("span",null,"approval chain: "),...i.chain.map(s=>h("span",{class:"pill",style:`color:${s.status==="approved"?"#1a9e6e":s.status==="rejected"?"#e5484d":"#8e4ec6"};border-color:currentColor`},"L"+s.step+" "+s.role+(s.status==="approved"?" ✓ "+s.by:s.status==="rejected"?" ✗":" …")))):null,
   i.status==="proposed"&&i.turn&&!i.can_approve?h("div",{class:"m"},"Waiting for a "+i.turn+" to decide"):null,
   i.note?h("div",{class:"m"},"note: "+i.note):null,
   h("div",{class:"a"},i.status==="proposed"&&i.can_approve?[h("button",{class:"btn ok sm",onclick:()=>act(i.id,"approve")},"Approve & execute"),h("button",{class:"btn danger sm",onclick:()=>act(i.id,"reject")},"Reject")]:null,
    i.quote&&i.can_approve&&i.status!=="proposed"?[h("button",{class:"btn ok sm",onclick:()=>act(i.id,"approve")},"Approve quote"),h("button",{class:"btn danger sm",onclick:()=>act(i.id,"reject")},"Reject")]:null,
    i.can_resolve?[h("button",{class:"btn sm",onclick:()=>act(i.id,"resolve")},"Resolve"),(()=>{const a=h("input",{class:"in",style:"width:170px",placeholder:"assign to email",value:i.assignee||""});return h("span",{style:"display:flex;gap:4px"},a,h("button",{class:"btn sm",onclick:()=>a.value&&act(i.id,"assign",{assignee:a.value})},"Assign"))})()]:null,
    h("button",{class:"btn sm",onclick:()=>{st.open.has(i.id)?st.open.delete(i.id):st.open.add(i.id);draw()}},"Comments ("+i.comments.length+")")),
   st.open.has(i.id)?h("div",{class:"cm"},...i.comments.map(c=>h("div",null,h("b",null,c.by),` ${c.ts}: ${c.body}`)),(()=>{const t=h("input",{class:"in",placeholder:"Write a comment and press Enter",onkeydown:e=>{if(e.key==="Enter"&&t.value.trim())act(i.id,"comment",{body:t.value})}});return t})()):null);return t}));if(!list.length)body.append(h("div",{class:"sub"},"Nothing here."))};
 draw();return{el,update:nb=>{if(el.contains(document.activeElement)&&document.activeElement.tagName==="INPUT"){cur=nb;return}if(JSON.stringify(nb.items)+st.f+[...st.open]===st.sig)return;cur=nb;draw()}}};
// ---- charts
const FN={size3:v=>Math.max(8,Math.min(34,Math.sqrt(Math.abs(v[3]||10))*1.6)),scatterTip:p=>`<b>${esc(p.value[2]||"")}</b><br>${esc(p.seriesName||"")}<br>${p.value[0]}, ${p.value[1]}`,nodeTip:p=>p.dataType==="node"?`<b>${esc(p.name)}</b><br>${esc(p.data.tip||"")}`:"",nodeLabel:p=>p.data.showLabel?p.name:""};
function revive(o){if(Array.isArray(o))return o.map(revive);if(o&&typeof o==="object"){if(o.__fn)return FN[o.__fn];const r={};for(const k in o)r[k]=revive(o[k]);return r}return o}
echarts.registerTheme("lt",{color:["#5b5bd6","#12a594","#f5a524","#8e4ec6","#e5484d","#30a46c","#d6409f","#3e63dd","#8b8d98"],backgroundColor:"transparent",textStyle:{fontFamily:"Inter, system-ui, sans-serif",color:"#454a58"},legend:{textStyle:{color:"#454a58"}},categoryAxis:{axisLine:{lineStyle:{color:"#d9dce8"}},axisTick:{show:false},axisLabel:{color:"#6b7080"},splitLine:{lineStyle:{color:"#eceef6"}}},valueAxis:{axisLine:{show:false},axisTick:{show:false},axisLabel:{color:"#6b7080"},splitLine:{lineStyle:{color:"#eceef6"}}},tooltip:{backgroundColor:"#fff",borderColor:"#e6e8f0",textStyle:{color:"#1c2024"},extraCssText:"box-shadow:0 8px 24px rgba(20,24,40,.12);border-radius:10px"}});
const CH=[];function lazy(box,fn){let d=0;const t=()=>{if(d)return;if(box.isConnected&&box.clientWidth>0){d=1;fn()}else setTimeout(t,60)};t()}
R.chart=b=>{const el=h("div"),box=h("div",{style:`height:${b.h||290}px`});el.append(h("h3",{style:"margin:0 0 4px;font-size:13.5px;display:"+(b.title?"block":"none")},b.title||""),box);let ch=null,cur=b;
 const init=()=>{ch=echarts.init(box,"lt",{renderer:"canvas"});CH.push(ch);ch.setOption(revive(cur.opt),true);
  ch.on("click",p=>{const o=cur.open;if(!o)return;let id=null;if(o.bar)id=p.name;else if(o.scatter)id=p.value&&p.value[2];else if(o.graph)id=p.data&&p.data.open;else if(p.data&&p.data.open)id=p.data.open;if(id)UI.openEntity(id)});
  if(cur.open)box.style.cursor="pointer"};
 lazy(box,init);
 return{el,update:nb=>{cur=nb;if(ch){const r=JSON.stringify(nb.opt);if(box._h!==r){box._h=r;ch.setOption(revive(nb.opt),{replaceMerge:["series","xAxis","yAxis"]})}}},};};
window.addEventListener("resize",()=>CH.forEach(c=>{try{c.resize()}catch(e){}}));
// ---- map
function shiftGeo(g,sh){const mv=c=>typeof c[0]==="number"?[c[0]+sh,c[1]]:c.map(mv);return{type:"FeatureCollection",features:g.features.map(f=>({type:"Feature",properties:f.properties,geometry:{type:f.geometry.type,coordinates:mv(f.geometry.coordinates)}}))}}
let WORLD=null;async function world(){if(WORLD)return WORLD;try{const t=await(await fetch("/static/vendor/world.topo.json")).json();WORLD=topojson.feature(t,t.objects.countries)}catch(e){WORLD={type:"FeatureCollection",features:[]}}return WORLD}
R.map=b=>{const el=h("div"),box=h("div",{class:"map",style:`height:${b.h||470}px`}),lg=h("div",{class:"lg"});el.append(box,lg);let M=null,grp=null,cur=b;
 const draw=()=>{grp.clearLayers();const SH=[-360,0,360];cur.layers.forEach(L=>{if(L.type==="lines")L.lines.forEach(l=>{if(l.pts.length<2)return;SH.forEach(sh=>{const p=L2.polyline(l.pts.map(q=>[q[0],q[1]+sh]),{color:l.color||"#5b5bd6",weight:l.w||2,dashArray:l.dash||null,opacity:.9,lineJoin:"round",renderer:M._r});if(l.tip)p.bindTooltip(l.tip,{sticky:true});grp.addLayer(p)})});
   else if(L.type==="circles")L.items.forEach(c=>SH.forEach(sh=>{const k=L2.circle([c.lat,c.lon+sh],{radius:c.r_m,color:c.color||"#e5484d",weight:1,fillOpacity:.18,renderer:M._r});if(c.tip)k.bindTooltip(c.tip);grp.addLayer(k)}));
   else if(L.type==="points")[...L.pts].sort((a,c)=>(a.z||0)-(c.z||0)).forEach(p=>SH.forEach(sh=>{const m=L2.circleMarker([p.lat,p.lon+sh],{radius:p.r||5,color:"#ffffff",weight:1.2,fillColor:p.color||"#5b5bd6",fillOpacity:.95,renderer:M._r});if(p.tip)m.bindTooltip(p.tip,{sticky:true});const o=p.open||(cur.open?p.id:null)||p.id;if(o)m.on("click",()=>UI.openEntity(o));grp.addLayer(m)}))});
  lg.replaceChildren(...(cur.legend||[]).map(([t,c])=>h("span",null,h("i",{style:"background:"+c}),t)))};
 const L2=L;const go=async()=>{M=L2.map(box,{zoomControl:true,worldCopyJump:true,attributionControl:false,minZoom:1,preferCanvas:true});M._r=L2.canvas({padding:.5});const w=await world();[-360,0,360].forEach(sh=>L2.geoJSON(sh?shiftGeo(w,sh):w,{renderer:M._r,style:{color:"#b9b2a0",weight:.8,fillColor:"#f3efe6",fillOpacity:1},interactive:false}).addTo(M));grp=L2.layerGroup().addTo(M);
  const v=cur.view||{};if(v.bounds)M.fitBounds(v.bounds);else M.setView(v.center||[20,0],v.zoom||2);draw();setTimeout(()=>M.invalidateSize(),50)};
 lazy(box,go);
 return{el,update:nb=>{cur=nb;if(M&&grp)draw()}}};
// ---- forms
R.form=b=>{const el=h("div",{class:b.title||b.inline?"":""});const res=h("div",{class:"res"});const f=h("form",{class:"f"+(b.inline?" inline":"")});let ins={};
 b.fields.forEach(fl=>{const w=h("div",{class:"fld"+(fl.type==="textarea"||fl.type==="checks"||fl.type==="multi"?" wide":"")});if(fl.type!=="hidden")w.append(h("label",null,fl.label||fl.name));let i;
  if(fl.type==="select")i=h("select",{name:fl.name},...fl.options.map(o=>h("option",{value:o,selected:String(o)===String(fl.value)},o)));
  else if(fl.type==="multi"){i=h("select",{name:fl.name,multiple:true,size:5},...fl.options.map(o=>h("option",{value:o,selected:(fl.value||[]).includes(o)},o)))}
  else if(fl.type==="checks"){i=h("div",{class:"chk"},...fl.options.map(o=>h("label",null,h("input",{type:"checkbox",value:o,checked:(fl.value||[]).includes(o)}),o)));i.get=()=>[...i.querySelectorAll("input:checked")].map(x=>x.value)}
  else if(fl.type==="textarea")i=h("textarea",{name:fl.name,placeholder:fl.placeholder||""},fl.value||"");
  else if(fl.type==="range"){const v=h("span",{style:"color:#12a594"},fl.value);i=h("input",{type:"range",name:fl.name,min:fl.min,max:fl.max,step:fl.step||1,value:fl.value,oninput:e=>v.textContent=e.target.value});w.firstChild.append(" ",v)}
  else if(fl.type==="file")i=h("input",{type:"file",name:fl.name,accept:fl.accept||null});
  else if(fl.type==="hidden"){i=h("input",{type:"hidden",name:fl.name,value:fl.value});w.style.display="none"}
  else i=h("input",{type:fl.type==="number"?"number":fl.type==="date"?"date":fl.type==="password"?"password":"text",name:fl.name,value:fl.value==null?"":fl.value,min:fl.min,max:fl.max,step:fl.step||(fl.type==="number"?"any":null),placeholder:fl.placeholder||""});
  i.fl=fl;ins[fl.name]=i;w.append(i);f.append(w)});
 const btn=h("button",{class:"btn "+(b.tone||"primary"),type:"submit"},b.submit||"Submit");f.append(h("div",{class:"fld"},btn));
 f.addEventListener("submit",async e=>{e.preventDefault();const v={};let fd=null;for(const n in ins){const i=ins[n],fl=i.fl;if(fl.type==="file"){if(i.files[0]){fd=fd||new FormData();fd.append(n,i.files[0])}}else if(fl.type==="checks")v[n]=i.get();else if(fl.type==="multi")v[n]=[...i.selectedOptions].map(o=>o.value);else if(fl.type==="number"||fl.type==="range")v[n]=i.value===""?null:+i.value;else v[n]=i.value}
  if(b.url.startsWith("get:")){const q=new URLSearchParams();for(const n in v)if(v[n]!==""&&v[n]!=null)q.set(n,v[n]);window.open(b.url.slice(4)+"?"+q,"_blank");return}
  if(b.url.startsWith("page:")){const[,id,tab]=b.url.split(":");UI.go(id,tab||S.tab,v);return}
  btn.disabled=true;const t0=btn.textContent;btn.textContent="Working…";let r;
  if(fd){for(const n in v)fd.append(n,Array.isArray(v[n])?v[n].join(","):v[n]);r=await api(b.url,{method:"POST",body:fd})}else r=await api(b.url,{json:v});
  btn.disabled=false;btn.textContent=t0;toast(r.msg||(r.ok?"Done":"Failed"),!r.ok);res.replaceChildren();if(r.blocks){const hs=mountList(res,r.blocks,"res");}if(r.refresh)reloadPage(true)});
 if(b.title||b.sub){const c=h("div",{class:"card"},b.title?h("h3",null,b.title):null,b.sub?h("div",{class:"sub"},b.sub):null,f,res);el.append(c)}else el.append(f,res);return{el,update(){}}};

R.profile_photo=b=>{const el=h("div",{class:"profile-photo-preview"});const up=nb=>{el.replaceChildren(h("img",{src:nb.src,alt:nb.alt||"Profile photo"}))};up(b);return{el,update:up}};

// ---- chat
async function sse(url,json,on){const r=await fetch(url,{method:"POST",credentials:"same-origin",headers:{"Content-Type":"application/json"},body:JSON.stringify(json)});if(!r.ok){on({ev:"final",text:"Error "+r.status});return}const rd=r.body.getReader(),dec=new TextDecoder();let buf="";for(;;){const{value,done}=await rd.read();if(done)break;buf+=dec.decode(value,{stream:true});let i;while((i=buf.indexOf("\n\n"))>=0){const m=buf.slice(0,i);buf=buf.slice(i+2);if(m.startsWith("data: ")){try{on(JSON.parse(m.slice(6)))}catch(e){}}}}}
R.chat=b=>{let cid=null,hist=[],lastState=null; /* COPILOT ENHANCEMENT: lastState = follow-up context returned by the server */const el=h("div",{class:"card chat"}),msgs=h("div",{class:"msgs"}),inp=h("input",{placeholder:"Ask about your outlets, staff, stock, campaigns, feedback or audits. Spelling mistakes are fine.",onkeydown:e=>{if(e.key==="Enter")send(inp.value)}});
 const lang=h("select",{class:"in",style:"width:auto",onchange:()=>{S.lang=lang.value;localStorage.setItem("lang",S.lang)}},...b.langs.map(l=>h("option",{value:l,selected:l===S.lang},l)));
 const hsel=h("select",{class:"in",style:"width:auto;max-width:260px",onchange:()=>{if(hsel.value)openChat(hsel.value)}},h("option",{value:""},"Previous chats…"));
 const listChats=async()=>{const r=await api("/api/chats");if(!r.chats)return;hsel.replaceChildren(h("option",{value:""},r.chats.length?`Previous chats (${r.chats.length})`:"No saved chats yet"),...r.chats.map(c=>h("option",{value:c.id,selected:c.id===cid},c.title+" · "+(c.ts||"").replace("T"," ").slice(5,16))));return r.chats};
 const isSql=x=>/^\s*(SELECT|WITH)\b/i.test(x||"");
 const chipRow=(label,arr)=>arr&&arr.length?h("div",{style:"margin-top:6px"},h("div",{class:"st"},label),h("div",{class:"filters"},...arr.map(e=>h("span",{class:"chip",onclick:()=>send(e)},e)))):null;
 const meta=(a,m,live)=>{const bar=h("div",{class:"st",style:"display:flex;gap:8px;flex-wrap:wrap;align-items:center;margin-top:6px"});
  if(typeof m.rows==="number")bar.append(h("span",{class:"pill",style:"color:#0b7a6c;border-color:#0b7a6c55"},m.rows+(m.rows===1?" row":" rows")+" returned"));
  if(m.status&&!["ok","greeting"].includes(m.status))bar.append(h("span",{class:"pill",style:"color:"+(m.status==="empty"?"#ab6400":m.status==="unsupported"?"#6b7080":"#cd2b31")+";border-color:currentColor"},m.status));
  if(isSql(m.sql))bar.append(h("span",{class:"pill",style:"color:#3451b2;border-color:#3451b255"},"read-only SQL · validated"));
  if(m.corrections&&m.corrections.length)bar.append(h("span",null,"Spelling corrected: "+m.corrections.map(c=>c[0]+" → "+c[1]).join(", ")));
  if(bar.children.length)a.append(bar);
  if(m.sql)a.append(h("details",{class:"cites",open:true},h("summary",null,isSql(m.sql)?"SQL used for this answer":"How this answer was computed"),h("pre",{style:"white-space:pre-wrap;font-size:11px;margin:4px 0"},m.sql),isSql(m.sql)?h("a",{class:"lnk",style:"cursor:pointer;font-size:12px",onclick:ev=>{try{navigator.clipboard.writeText(m.sql);ev.target.textContent="Copied"}catch(e){}}},"Copy SQL"):null));
  const c1=chipRow(live?"You could try:":"Try asking:",live?m.suggestions:m.sug),c2=live?chipRow("Follow up:",m.followups):null;if(c1)a.append(c1);if(c2)a.append(c2)};
 const showMsg=m=>{if(m.role==="user")msgs.append(h("div",{class:"m u"},m.text));else{const o=h("div",{class:"md"});o.innerHTML=md(m.text);const a=h("div",{class:"m a"},o);if(m.en){const tg=h("a",{class:"lnk",style:"cursor:pointer;font-size:12px",onclick:()=>{const on=tg._e=!tg._e;o.innerHTML=md(on?m.en:m.text);tg.textContent=on?"Show "+(m.lang||"translation"):"Show English original"}},"Show English original");a.append(h("div",{class:"st"},"Translated with NLLB-200 ",tg))}meta(a,m,false);msgs.append(a)}};
 const openChat=async id=>{const r=await api("/api/chats/"+encodeURIComponent(id));if(!r.msgs)return;cid=id;hist=r.msgs;lastState=([...hist].reverse().find(x=>x.state&&Object.keys(x.state).length)||{}).state||null;msgs.replaceChildren();hist.forEach(showMsg);msgs.lastChild&&msgs.lastChild.scrollIntoView({block:"nearest"});listChats()};
 const newChat=()=>{cid=null;hist=[];lastState=null;msgs.replaceChildren();hsel.value="";inp.focus()};
 const delChat=async()=>{if(!cid||!confirm("Delete this chat?"))return;await api("/api/chats/"+encodeURIComponent(cid),{method:"DELETE"});newChat();listChats()};
 const saveChat=async()=>{const r=await api("/api/chats/save",{json:{id:cid,msgs:hist}});if(r.id){cid=r.id;listChats()}};
 listChats().then(cs=>{if(cs&&cs.length&&!hist.length)openChat(cs[0].id)});
 const send=async q=>{q=q.trim();if(!q)return;inp.value="";hist.push({role:"user",text:q});msgs.append(h("div",{class:"m u"},q));const a=h("div",{class:"m a"}),stg=h("div"),tr=h("div"),out=h("div",{class:"md"});a.append(stg,tr,out);msgs.append(a);let acc="";const sc=()=>{msgs.lastChild.scrollIntoView({block:"nearest"})};
  await sse("/api/copilot",{q,lang:S.lang,state:lastState},ev=>{if(ev.ev==="stage"){stg.append(h("div",{class:"st"},ev.text))}else if(ev.ev==="cites"){const d=h("details",{class:"cites"},h("summary",null,`Sources (${ev.rows.length})`),...ev.rows.map(r=>h("div",{class:"st",title:r.text},h("b",null,`[${r.n}] `),`${r.title} · ${r.source} · ${r.why||r.score}`)));stg.append(d)}else if(ev.ev==="trace"){tr.replaceChildren(h("div",{class:"tbl",style:"margin:6px 0"},h("table",null,h("tbody",null,...ev.rows.map(r=>h("tr",null,h("td",null,r.agent),h("td",null,h("span",{class:"pill",style:`color:${tone(r.risk)};border-color:${tone(r.risk)}55`},r.risk)),h("td",{style:"white-space:normal"},r.finding)))))))}else if(ev.ev==="tok"){acc+=ev.text;out.innerHTML=md(acc)}else if(ev.ev==="final"){out.innerHTML=md(ev.text);lastState=ev.state||null;hist.push({role:"assistant",text:ev.text,en:ev.en,lang:ev.lang,sql:ev.sql,rows:ev.rows,status:ev.status,sug:[...(ev.suggestions||[]),...(ev.followups||[])].slice(0,5),state:ev.state});saveChat();if(ev.en){const tg=h("a",{class:"lnk",style:"cursor:pointer;font-size:12px;margin-left:8px",onclick:()=>{const on=tg._e=!tg._e;out.innerHTML=md(on?ev.en:ev.text);tg.textContent=on?"Show "+(ev.lang||"translation"):"Show English original"}},"Show English original");stg.append(h("div",{class:"st"},"Translated with NLLB-200 ",tg))}meta(a,ev,true);stg.append(h("div",{class:"st"},`done in ${(ev.ms/1000).toFixed(1)}s`+(ev.tok_s?` · ${ev.tok_s} tok/s`:"")))}sc()})};
 el.append(h("div",{style:"display:flex;gap:8px;align-items:center;flex-wrap:wrap"},h("button",{class:"btn sm",onclick:newChat},"+ New chat"),hsel,h("button",{class:"btn sm ghost",onclick:delChat,title:"Delete this chat"},"Delete"),h("b",{style:"margin-left:8px"},"Answer language"),lang,h("span",{class:"sub",style:"margin:0"},"NLLB-200 translates in and out")),msgs,h("div",{class:"filters"},...b.examples.map(e=>h("span",{class:"chip",onclick:()=>send(e)},e))),h("div",{class:"cin"},inp,(()=>{const SR=window.SpeechRecognition||window.webkitSpeechRecognition;if(!SR)return null;const VL={Hindi:"hi-IN",Tamil:"ta-IN",Telugu:"te-IN",Kannada:"kn-IN",Malayalam:"ml-IN",Bengali:"bn-IN",Gujarati:"gu-IN",Marathi:"mr-IN",Punjabi:"pa-IN",Urdu:"ur-IN",Arabic:"ar-SA",French:"fr-FR",German:"de-DE",Spanish:"es-ES",Portuguese:"pt-PT",Chinese:"zh-CN",Japanese:"ja-JP",Korean:"ko-KR",Russian:"ru-RU",Indonesian:"id-ID",Turkish:"tr-TR",Italian:"it-IT",Dutch:"nl-NL",Thai:"th-TH",Vietnamese:"vi-VN"};let rec=null;const mic=h("button",{class:"btn",title:"Speak your question (browser speech recognition)",onclick:()=>{if(rec){rec.stop();return}rec=new SR();const k=Object.keys(VL).find(x=>S.lang.startsWith(x));rec.lang=k?VL[k]:"en-IN";rec.interimResults=true;rec.onresult=e=>{inp.value=[...e.results].map(r=>r[0].transcript).join(" ")};rec.onend=()=>{rec=null;mic.textContent="🎤";if(inp.value.trim())send(inp.value)};mic.textContent="⏺";try{rec.start()}catch(e){rec=null;mic.textContent="🎤"}}},"🎤");return mic})(),h("button",{class:"btn primary",onclick:()=>send(inp.value)},"Ask")));return{el,update(){}}};

// ---------------------------------------------------------------- shell
const UI={};
UI.openEntity=async id=>{if(!id)return;$("#dt").textContent=id;$("#db").replaceChildren(h("div",{class:"boot"},h("div",{class:"spin"}),"Loading…"));$("#drawer").classList.add("on");const d=await api("/api/detail/"+encodeURIComponent(id));if(d.title===undefined){$("#db").replaceChildren(h("div",{class:"callout warn"},d.msg||"Not available"));return}$("#dt").textContent=d.title||id;$("#db").replaceChildren();mountList($("#db"),d.blocks||[],"d");translateUI($("#db"))};
UI.closeDrawer=()=>$("#drawer").classList.remove("on");
UI.go=(id,tab,params)=>{S.page=id;S.tab=tab||null;S.params=params||{};history.replaceState(null,"","#"+id+(S.tab?":"+S.tab:""));loadPage(false);renderNav()};
async function loadPage(silent){const seq=++S.seq;clearTimeout(S.timer);const q=new URLSearchParams();if(S.tab)q.set("tab",S.tab);if(Object.keys(S.params).length)q.set("p",JSON.stringify(S.params));
 if(!silent){S.tree=null;$("#blocks").replaceChildren(h("div",{class:"boot"},h("div",{class:"spin"}),"Loading…"))}
 const t=performance.now();const d=await api(`/api/page/${S.page}?`+q);if(seq!==S.seq)return;
 if(d.blocks===undefined){$("#blocks").replaceChildren(h("div",{class:"callout error"},d.msg||"Could not load"));return}
 S.ms=Math.round(performance.now()-t);$("#ttl").textContent=d.title;$("#sub").textContent=d.sub||"";document.title=d.title+" · "+S.me.name;
 const tabs=$("#tabs");tabs.replaceChildren(...(d.tabs||[]).map(x=>h("b",{class:(S.tab||d.tabs[0].id)===x.id?"on":"",onclick:()=>{S.params={};UI.go(S.page,x.id)}},x.label)));tabs.style.display=d.tabs?"":"none";
 $("#acts").replaceChildren(...(d.actions||[]).map(a=>h("button",{class:"btn "+(a.tone||"primary"),onclick:async e=>{e.target.disabled=true;const r=await api(a.url,{json:{}});toast(r.msg,!r.ok);e.target.disabled=false;reloadPage(true)}},a.label)));
 const box=$("#blocks");const focus=document.activeElement&&box.contains(document.activeElement)&&["INPUT","TEXTAREA","SELECT"].includes(document.activeElement.tagName);
 if(!S.tree||silent===false||!S.tree.ok){box.replaceChildren();S.tree={hs:mountList(box,d.blocks,"p"),ok:1,id:S.page,tab:S.tab};translateUI(box)}else S.tree.hs=updList(box,S.tree.hs,d.blocks,"p");
 S.live=d.live||0;sched()}
function sched(){clearTimeout(S.timer);if(S.live&&S.lang==="English"||S.live&&true)S.timer=setTimeout(()=>{if(!document.hidden)loadPage(true);else sched()},S.live*1000)}
function reloadPage(soft){loadPage(!!soft)}
async function translateUI(root){if(S.lang==="English")return;const nodes=[];const w=document.createTreeWalker(root,NodeFilter.SHOW_TEXT);let n;while((n=w.nextNode())){const t=n.nodeValue.trim();if(t.length>2&&/[A-Za-z]{3}/.test(t)&&!n.parentElement.closest("script,style,pre,code,.m.u,.mono,.noTr,input,select,option")&&!n._tr&&!(n.parentElement.closest("td")&&t.split(/\s+/).length<4)&&!/^[A-Z]{2,6}-?[A-Z]?\d+$/.test(t))nodes.push(n)}
 const list=nodes.slice(0,300);if(!list.length)return;const r=await api("/api/translate_batch",{json:{texts:list.map(n=>n.nodeValue.trim()),lang:S.lang}});if(r.out)list.forEach((n,i)=>{if(r.out[i]){n.nodeValue=r.out[i];n._tr=1}})}
const GROUPS=[["Assistant",["copilot","approvals"]],["Agents",["workforce","outlets","inventory","marketing","sentiment","audit","route","quotes","carriers","weather","margin","customs"]],["Insights",["twin","reports","notifications"]],["Platform",["feeds","admin","profile"]]];
function renderNav(){const nav=$("#nav");if(!nav)return;const items=[];const used=new Set();const mk=m=>h("div",{class:"nav"+(m.id===S.page?" on":"")+(m.id==="command"?" cmd":""),"data-id":m.id,onclick:()=>UI.go(m.id)},icon(m.icon),h("span",null,m.label.replace(/^Agent \d+ · /,"")),h("em",{class:"bd",style:"display:none"}));
 GROUPS.forEach(([g,ids])=>{const ms=S.me.menu.filter(m=>ids.includes(m.id));if(!ms.length)return;items.push(h("div",{class:"ngrp"},g));ms.forEach(m=>{used.add(m.id);items.push(mk(m))})});
 S.me.menu.filter(m=>!used.has(m.id)&&m.id!=="command").forEach(m=>items.push(mk(m)));const c=S.me.menu.find(m=>m.id==="command");if(c)items.push(mk(c));
 nav.replaceChildren(...items);updBadges()}
// ---- command palette (Ctrl/Cmd+K): pages, entities, ask the copilot
UI.palette=()=>{if($(".kpal"))return;let sel=0,rows=[];const inp=h("input",{placeholder:"Jump to a page, open an entity (e.g. OUT-007, QTE-0012, SHP-…), or ask the Copilot…"}),lst=h("div",{class:"lst"}),ov=h("div",{class:"kpal",onclick:e=>{if(e.target===ov)close()}},h("div",null,inp,lst));
 const close=()=>{ov.remove();document.removeEventListener("keydown",kd,true)};
 const build=()=>{const q=inp.value.trim(),ql=q.toLowerCase();rows=[];S.me.menu.filter(m=>!ql||m.label.toLowerCase().includes(ql)||m.id.includes(ql)).forEach(m=>rows.push({g:"Pages",t:m.label.replace(/^Agent \d+ · /,""),s:"go to",f:()=>UI.go(m.id)}));
  if(/^[A-Za-z]{2,5}-?[A-Za-z]?\d+$/.test(q))rows.unshift({g:"Open",t:q.toUpperCase(),s:"entity",f:()=>UI.openEntity(q.toUpperCase())});
  if(q.length>2)rows.push({g:"Ask",t:"Ask the Copilot: "+q,s:"agents + RAG",f:()=>{UI.go("copilot");setTimeout(()=>{const i=document.querySelector(".cin input");if(i){i.value=q;i.dispatchEvent(new KeyboardEvent("keydown",{key:"Enter"}))}},700)}});
  if(sel>=rows.length)sel=rows.length-1;
  let last="";lst.replaceChildren(...rows.flatMap((r,i)=>{const e=[];if(r.g!==last){last=r.g;e.push(h("div",{class:"gl"},r.g))}e.push(h("div",{class:"it"+(i===sel?" on":""),onclick:()=>{close();r.f()}},r.t,h("small",null,r.s)));return e}))};
 const kd=e=>{if(e.key==="Escape"){close()}else if(e.key==="ArrowDown"){sel=Math.min(rows.length-1,sel+1);build();e.preventDefault()}else if(e.key==="ArrowUp"){sel=Math.max(0,sel-1);build();e.preventDefault()}else if(e.key==="Enter"&&document.activeElement===inp){const r=rows[sel];if(r){close();r.f()}}};
 document.addEventListener("keydown",kd,true);inp.addEventListener("input",()=>{sel=0;build()});document.body.append(ov);build();inp.focus()};
document.addEventListener("keydown",e=>{if((e.ctrlKey||e.metaKey)&&e.key.toLowerCase()==="k"&&S.me){e.preventDefault();UI.palette()}});
function updBadges(){const p=S.pulse;if(!p)return;const cc=$('.nav[data-id="command"] .bd');if(cc){const n=p.critical;cc.style.display=n?"":"none";cc.textContent=n;cc.className="bd"}const ap=$('.nav[data-id="approvals"] .bd');if(ap){const n=p.proposed;ap.style.display=n?"":"none";ap.textContent=n;ap.className="bd a"}
 const g=p.gpu||{};$("#gpu").innerHTML=`<span class="dot ${g.cuda?"g":"r"}"></span>${g.cuda?esc(g.name||"GPU")+" · "+g.used+"/"+g.total+" GB":"NO GPU"} · LLM ${g.llm==="downloading"?"downloading "+(g.llm_done||0)+"/"+(g.llm_total||"?")+" GB":g.llm} · NLLB ${g.nllb}${g.tok_s?" · "+g.tok_s+" tok/s":""}`;
 $("#hl").innerHTML=`<span class="dot g live"></span>live · health ${p.health==null?"–":Math.round(p.health)} · open ${p.open} · crit ${p.critical} · cycle ${p.cycles}`;$("#warn").style.display=g.cuda?"none":"";if(!p.worker_alive)$("#hl").innerHTML+=' · <b style="color:#e5484d">worker down</b>'}
function openStream(){if(S.es)S.es.close();const es=new EventSource("/api/stream");S.es=es;es.onmessage=e=>{const p=JSON.parse(e.data);if(p.error)return;const old=S.pulse;S.pulse=p;updBadges();if(old&&p.dec!==old.dec&&S.live)loadPage(true)};es.onerror=()=>{}}

// ---------------------------------------------------------------- Milestone 1 Auth Screens (Verify Portal)
function renderAuth(app, pub) {
  pub = pub || {};
  let view = "login";

  // State across views
  let signupData = null;
  let signupOtpToken = "";
  let signupEmail = "";

  let resetEmail = "";
  let resetToken = "";
  let resetOtpToken = "";
  let recoveryQuestion = "";

  const container = h("div", {class: "portal-auth-bg"});

  function svgIcon(name) {
    if (name === "envelope") {
      return '<svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="3" y="5" width="18" height="14" rx="2"/><path d="m3 7 9 6 9-6"/></svg>';
    } else if (name === "envelope-header") {
      return '<svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="3" y="5" width="18" height="14" rx="2"/><path d="m3 7 9 6 9-6"/><path d="M9 3h6"/></svg>';
    } else if (name === "check") {
      return '<svg width="44" height="44" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><path d="M5 13l4 4L19 7"/></svg>';
    } else if (name === "pw-box" || name === "lock") {
      return '<svg width="26" height="26" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="3" y="11" width="18" height="11" rx="2" ry="2"/><path d="M7 11V7a5 5 0 0 1 10 0v4"/></svg>';
    }
    return '';
  }

  function draw() {
    container.replaceChildren();

    // =========================================================================
    // 1. LOGIN SCREEN (Matches Screenshot 2)
    // =========================================================================
    if (view === "login") {
      const card = h("main", {class: "card vp-login-card"},
        h("div", {class: "icon"}, "🔐"),
        h("h2", null, "Sign in to your account"),
        h("p", null, "Securely access your Verify Portal account"),
        (()=>{
          const form = h("form", {class: "form", id: "form"});
          const notice = h("div", {id: "notice"});
          form.append(notice);

          const emailWrap = h("div", {class: "wrap"},
            h("span", {class: "symbol"}, "👤"),
            h("input", {id: "email", class: "field", type: "text", placeholder: "Enter your username or email", required: true})
          );
          const passWrap = h("div", {class: "wrap"},
            h("span", {class: "symbol"}, "🔒"),
            h("input", {id: "password", class: "field", type: "password", placeholder: "Enter your password", required: true}),
            h("button", {id: "eye", class: "eye", type: "button", onclick: () => {
              const inp = passWrap.querySelector("input");
              inp.type = inp.type === "password" ? "text" : "password";
              eyeBtn.textContent = inp.type === "password" ? "◉" : "◌";
            }}, "◉")
          );
          const eyeBtn = passWrap.querySelector("#eye");

          const submitBtn = h("button", {class: "primary", type: "submit"}, "Sign In to Portal →");

          // 5-Role Instant Login at Bottom
          const demoRoles = (pub && pub.demo && pub.demo.length) ? pub.demo : [
            {role: "Admin", email: (pub && pub.demo && pub.demo[0] && pub.demo[0].email) || "admin@infosys.com", password: (pub && pub.demo && pub.demo[0] && pub.demo[0].password) || "Admin@123", icon: "👑"},
            {role: "Franchise Owner", email: "owner@infosys.com", password: "Demo@1234", icon: "🏢"},
            {role: "Store Manager", email: "manager@infosys.com", password: "Demo@1234", icon: "🏪"},
            {role: "Staff", email: "staff@infosys.com", password: "Demo@1234", icon: "👤"},
            {role: "Auditor", email: "auditor@infosys.com", password: "Demo@1234", icon: "📋"}
          ];

          const demoPanel = h("div", {class: "demo-showcase-panel"},
            h("div", {class: "demo-showcase-title"}, "Instant Login"),
            h("div", {class: "demo-roles-grid"},
              demoRoles.map(item => {
                return h("button", {
                  type: "button",
                  class: "demo-role-btn",
                  title: `Login as ${item.role}`,
                  onclick: () => {
                    const emInp = emailWrap.querySelector("input");
                    const pwInp = passWrap.querySelector("input");
                    emInp.value = item.email;
                    pwInp.value = item.password;
                    toast(`Logging in as ${item.role}...`);
                    if (form.requestSubmit) {
                      form.requestSubmit();
                    } else {
                      form.dispatchEvent(new Event("submit", {cancelable: true}));
                    }
                  }
                },
                  h("span", {class: "role-icon"}, item.icon || "👤"),
                  h("span", {class: "role-name"}, item.role)
                );
              })
            )
          );

          form.append(
            h("label", null, "Username or Email Address"), emailWrap,
            h("label", null, "Password"), passWrap,
            submitBtn,
            demoPanel
          );

          form.onsubmit = async (e) => {
            e.preventDefault();
            notice.replaceChildren();
            submitBtn.disabled = true;
            submitBtn.textContent = "Signing in...";

            const em = emailWrap.querySelector("input").value.trim();
            const pw = passWrap.querySelector("input").value;

            const res = await api("/api/login", {json: {email: em, password: pw}});
            submitBtn.disabled = false;
            submitBtn.textContent = "Sign In to Portal →";

            if (res.ok) {
              toast("Signed in successfully!");
              render();
            } else {
              notice.innerHTML = `<div class="notice error">${esc(res.msg || "Authentication failed")}</div>`;
              if (res.locked) {
                submitBtn.disabled = true;
                submitBtn.textContent = "Account Locked";
              }
            }
          };

          return form;
        })(),
        h("div", {class: "links"},
          h("button", {id: "signup", type: "button", onclick: () => { view = "signup"; draw(); }}, "✨ Create account"),
          h("button", {id: "forgot", type: "button", onclick: () => { view = "reset_email"; draw(); }}, "🔑 Forgot password?")
        )
      );
      container.append(card);
    }

    // =========================================================================
    // 2. CREATE ACCOUNT SCREEN (Matches Screenshot 3)
    // =========================================================================
    else if (view === "signup") {
      const card = h("main", {class: "card vp-signup-card"},
        h("div", {class: "icon"}, "✦"),
        h("h2", null, "Create your account"),
        h("p", {class: "subtitle"}, "Register for secure Verify Portal access"),
        (()=>{
          const form = h("form", {class: "form", id: "form"});
          const notice = h("div", {id: "notice"});

          const uWrap = h("div", {class: "field"},
            h("label", null, "Full Name"),
            h("div", {class: "wrap"},
              h("span", {class: "mark"}, "◉"),
              h("input", {id: "username", type: "text", required: true, placeholder: "John Doe", value: signupData ? signupData.username : ""})
            )
          );
          const eWrap = h("div", {class: "field"},
            h("label", null, "Email Address"),
            h("div", {class: "wrap"},
              h("span", {class: "mark"}, "✉"),
              h("input", {id: "email", type: "email", required: true, placeholder: "you@company.com", value: signupData ? signupData.email : ""})
            )
          );
          const rWrap = h("div", {class: "field"},
            h("label", null, "Account Role"),
            h("div", {class: "wrap"},
              h("span", {class: "mark"}, "👤"),
              h("select", {id: "role", required: true, style: "width:100%; height:46px; border:1.5px solid #d6dfec; border-radius:8px; padding-left: 48px !important; padding-right: 36px; font-size:14px; background:#fff; color:#17233b; appearance:auto; -webkit-appearance:auto;"},
                h("option", {value: "Franchise Owner", selected: (signupData && signupData.role === "Franchise Owner") || !signupData}, "Franchise Owner (Owns several outlets)"),
                h("option", {value: "Store Manager", selected: signupData && signupData.role === "Store Manager"}, "Store Manager (Runs one outlet)"),
                h("option", {value: "Staff", selected: signupData && signupData.role === "Staff"}, "Staff (Works at an outlet)"),
                h("option", {value: "Auditor", selected: signupData && signupData.role === "Auditor"}, "Auditor (Compliance reviewer)"),
                h("option", {value: "Admin", selected: signupData && signupData.role === "Admin"}, "Admin (Platform operator)")
              )
            )
          );
          const pWrap = h("div", {class: "field"},
            h("label", null, "Password"),
            h("div", {class: "wrap"},
              h("span", {class: "mark"}, "🔒"),
              h("input", {id: "password", type: "password", required: true, placeholder: "At least 8 characters"}),
              h("button", {class: "eye", type: "button", onclick: (e) => {
                const inp = pWrap.querySelector("input");
                inp.type = inp.type === "password" ? "text" : "password";
                e.target.textContent = inp.type === "password" ? "◉" : "◌";
              }}, "◉")
            )
          );
          const cWrap = h("div", {class: "field"},
            h("label", null, "Confirm Password"),
            h("div", {class: "wrap"},
              h("span", {class: "mark"}, "🔒"),
              h("input", {id: "confirm", type: "password", required: true, placeholder: "Re-enter password"}),
              h("button", {class: "eye", type: "button", onclick: (e) => {
                const inp = cWrap.querySelector("input");
                inp.type = inp.type === "password" ? "text" : "password";
                e.target.textContent = inp.type === "password" ? "◉" : "◌";
              }}, "◉")
            )
          );

          // Password strength meter
          const meter = h("div", {id: "meter", class: "meter"}, h("i"), h("i"), h("i"), h("i"), h("i"));
          const strength = h("div", {id: "strength", class: "rule"}, "Password strength: —");
          const rules = h("div", {id: "rules"},
            h("div", {class: "rule", "data-rule": "length"}, "○ 8 or more characters"),
            h("div", {class: "rule", "data-rule": "upper"}, "○ At least 1 uppercase letter"),
            h("div", {class: "rule", "data-rule": "lower"}, "○ At least 1 lowercase letter"),
            h("div", {class: "rule", "data-rule": "digit"}, "○ At least 1 number"),
            h("div", {class: "rule", "data-rule": "special"}, "○ At least 1 special character")
          );
          const match = h("div", {id: "match", class: "rule"});

          // Security question
          const secBox = h("div", {class: "security"},
            h("h3", null, "Security Question"),
            h("p", null, "Required for secure account recovery if you forget your password."),
            h("div", {class: "field"},
              h("label", null, "Choose a security question"),
              h("div", {class: "wrap"},
                h("select", {id: "question", required: true},
                  h("option", {value: "", disabled: true, selected: true}, "Select a security question"),
                  h("option", null, "What was the name of your first pet?"),
                  h("option", null, "What is your favourite city?"),
                  h("option", null, "What was your first school?"),
                  h("option", null, "In what city or town was your first job?"),
                  h("option", null, "What was the name of the street you grew up on?"),
                  h("option", null, "What was the name of your childhood best friend?"),
                  h("option", null, "What is your favorite book or movie?")
                )
              )
            ),
            h("div", {class: "field"},
              h("label", null, "Security answer"),
              h("div", {class: "wrap"},
                h("span", {class: "mark"}, "🔐"),
                h("input", {id: "answer", type: "text", required: true, placeholder: "Enter your secret answer"})
              )
            )
          );

          const submitBtn = h("button", {id: "submit", class: "submit", type: "submit", disabled: true}, "Create Account →");

          form.append(notice, uWrap, eWrap, rWrap, pWrap, cWrap, meter, strength, rules, match, secBox, submitBtn);

          function isSecurityAnswerValid(ans) {
            const trimmed = (ans || '').trim();
            if (trimmed.length < 3) return false;
            const stripped = trimmed.replace(/\s+/g, '');
            if (/^(.)\1+$/.test(stripped)) return false;
            const words = trimmed.split(/\s+/).filter(Boolean);
            if (words.length >= 3 && words.every(w => w.length === 1)) return false;
            return true;
          }

          function validate() {
            const p = pWrap.querySelector("input").value;
            const c = cWrap.querySelector("input").value;
            const un = uWrap.querySelector("input").value.trim();
            const em = eWrap.querySelector("input").value.trim();
            const q = secBox.querySelector("#question").value.trim();
            const a = secBox.querySelector("#answer").value;

            const checks = {
              length: p.length >= 8,
              upper: /[A-Z]/.test(p),
              lower: /[a-z]/.test(p),
              digit: /\d/.test(p),
              special: /[!@#$%^&*(),.?":{}|<>]/.test(p)
            };
            const score = Object.values(checks).filter(Boolean).length;
            rules.querySelectorAll('[data-rule]').forEach(x => {
              let ok = checks[x.dataset.rule];
              x.className = 'rule ' + (ok ? 'ok' : '');
              x.textContent = (ok ? '● ' : '○ ') + x.textContent.slice(2);
            });
            meter.className = 'meter ' + (score <= 2 ? 'weak' : score <= 4 ? 'medium' : 'strong');
            strength.textContent = 'Password strength: ' + (score === 0 ? '—' : score <= 2 ? 'Weak' : score <= 4 ? 'Moderate' : 'Strong');
            match.textContent = c ? (p === c ? '● Passwords match' : '○ Passwords do not match') : '';
            match.className = 'rule ' + (c && p === c ? 'ok' : 'bad');

            submitBtn.disabled = !(Object.values(checks).every(Boolean) && p === c && un && em && q && isSecurityAnswerValid(a));
          }

          form.addEventListener("input", validate);
          form.addEventListener("change", validate);

          form.onsubmit = async (e) => {
            e.preventDefault();
            notice.replaceChildren();

            const un = uWrap.querySelector("input").value.trim();
            const em = eWrap.querySelector("input").value.trim().toLowerCase();
            const roleEl = rWrap.querySelector("select");
            const ro = roleEl ? roleEl.value : "Franchise Owner";
            const pw = pWrap.querySelector("input").value;
            const q = secBox.querySelector("#question").value;
            const a = secBox.querySelector("#answer").value.trim();

            submitBtn.disabled = true;
            submitBtn.textContent = "Sending OTP...";

            const res = await api("/api/signup/request-otp", {
              json: {username: un, email: em, password: pw, role: ro}
            });
            submitBtn.disabled = false;
            submitBtn.textContent = "Create Account →";

            if (res.ok) {
              signupData = {username: un, email: em, password: pw, role: ro, question: q, answer: a};
              signupOtpToken = res.otp_token;
              signupEmail = em;
              view = "signup_otp";
              draw();
            } else {
              notice.innerHTML = `<div class="notice error">${esc(res.msg || "Could not send verification code")}</div>`;
            }
          };

          return form;
        })(),
        h("button", {id: "signin", class: "link", type: "button", onclick: () => { view = "login"; draw(); }}, "Already have an account? Sign in")
      );
      container.append(card);
    }

    // =========================================================================
    // 3. VERIFY SCREEN (Matches Screenshot 4)
    // =========================================================================
    else if (view === "signup_otp" || view === "otp_verification") {
      const isSignupFlow = view === "signup_otp";
      const targetEmail = isSignupFlow ? signupEmail : resetEmail;
      let countdownTimer = null;

      const card = h("main", {class: "card vp-otp-card"},
        h("div", {class: "stepper"},
          h("div", {class: "step step-done"}, h("div", {class: "circle"}, "1"), h("div", {class: "step-title"}, "Account")),
          h("div", {class: "line line-done"}),
          h("div", {class: "step step-active"}, h("div", {class: "circle"}, "2"), h("div", {class: "step-title"}, "Verify")),
          h("div", {class: "line line-inactive"}),
          h("div", {class: "step step-pending"}, h("div", {class: "circle"}, "3"), h("div", {class: "step-title"}, "Complete"))
        ),
        h("div", {class: "icon", html: svgIcon("envelope-header")}),
        h("h2", null, "Verify"),
        h("p", {class: "subtext"}, "We sent a verification code to ", h("strong", {id: "email", style: "color: #3b82f6;"}, targetEmail), h("br"), "Enter the 6-digit code to continue"),
        (()=>{
          const form = h("form", {id: "form"});
          const notice = h("div", {id: "notice", class: "notice", style: "display:none;"});
          const enterLabel = h("div", {class: "enter-label"}, "Enter Code");

          const otpContainer = h("div", {class: "otp-container"});
          const inputs = [];
          for (let i = 0; i < 6; i++) {
            const inp = h("input", {class: "otp", inputmode: "numeric", maxlength: "1"});
            if (i === 0) setTimeout(() => inp.focus(), 50);
            inputs.push(inp);
            otpContainer.append(inp);
          }

          inputs.forEach((input, index) => {
            input.addEventListener('input', () => {
              inputs.forEach(x => x.classList.remove('input-error'));
              input.value = input.value.replace(/\D/g, '').slice(0, 1);
              if (input.value && inputs[index + 1]) inputs[index + 1].focus();
            });
            input.addEventListener('keydown', e => {
              if (e.key === 'Backspace' && !input.value && inputs[index - 1]) inputs[index - 1].focus();
            });
            input.addEventListener('paste', e => {
              e.preventDefault();
              inputs.forEach(x => x.classList.remove('input-error'));
              const chars = (e.clipboardData.getData('text').match(/\d/g) || []).slice(0, 6);
              chars.forEach((char, i) => { if (inputs[i]) inputs[i].value = char; });
              if (inputs[Math.min(chars.length, 5)]) inputs[Math.min(chars.length, 5)].focus();
            });
          });

          const spamNote = h("div", {class: "spam-note"}, "Check your spam folder if you don't see the email");

          const resendBtn = h("button", {class: "link", type: "button", id: "resend", disabled: true}, "↻  Resend Code (30s)");
          const resendRow = h("div", {class: "resend"}, "Didn't receive the code? ", resendBtn);

          const submitBtn = h("button", {class: "verify", type: "submit"}, "• Verify");

          form.append(enterLabel, otpContainer, spamNote, resendRow, submitBtn);

          function startResendCountdown(seconds = 30) {
            if (countdownTimer) clearInterval(countdownTimer);
            let remaining = seconds;
            resendBtn.disabled = true;
            resendBtn.innerHTML = `↻&nbsp; Resend Code (<span id="timer">${remaining}</span>s)`;

            countdownTimer = setInterval(() => {
              remaining -= 1;
              const timerEl = card.querySelector('#timer');
              if (remaining <= 0) {
                clearInterval(countdownTimer);
                countdownTimer = null;
                resendBtn.disabled = false;
                resendBtn.textContent = '↻  Resend Code';
              } else if (timerEl) {
                timerEl.textContent = remaining;
              }
            }, 1000);
          }

          startResendCountdown(30);

          resendBtn.onclick = async () => {
            inputs.forEach(x => { x.value = ''; x.disabled = false; x.classList.remove('input-error'); });
            inputs[0].focus();
            toast("Resending verification code...");
            const url = isSignupFlow ? "/api/signup/request-otp" : "/api/forgot-password/send-otp";
            const payload = isSignupFlow ? {username: signupData.username, email: signupData.email, password: signupData.password} : {identifier: resetEmail};
            const res = await api(url, {json: payload});
            if (res.ok) {
              if (isSignupFlow) signupOtpToken = res.otp_token;
              else resetOtpToken = res.reset_otp_token;
              toast("Verification code sent!");
              startResendCountdown(30);
            } else {
              toast(res.msg || "Failed to resend");
            }
          };

          form.onsubmit = async (e) => {
            e.preventDefault();
            const code = inputs.map(x => x.value).join('');
            if (code.length !== 6) {
              notice.style.display = "block";
              notice.className = "notice error";
              notice.textContent = "Please enter all 6 digits of the verification code.";
              inputs.forEach(x => x.classList.add('input-error'));
              return;
            }

            submitBtn.disabled = true;
            submitBtn.textContent = "Verifying...";

            if (isSignupFlow) {
              const res = await api("/api/signup/verify", {
                json: {
                  otp_token: signupOtpToken,
                  otp_code: code,
                  username: signupData.username,
                  email: signupData.email,
                  password: signupData.password,
                  role: signupData.role || "Franchise Owner",
                  security_question: signupData.question,
                  security_answer: signupData.answer
                }
              });
              submitBtn.disabled = false;
              submitBtn.textContent = "• Verify";

              if (res.ok) {
                if (countdownTimer) clearInterval(countdownTimer);
                view = "signup_complete";
                draw();
              } else {
                notice.style.display = "block";
                notice.className = "notice error";
                notice.textContent = res.msg || "Invalid code";
                inputs.forEach(x => x.classList.add('input-error'));
              }
            } else {
              const res = await api("/api/forgot-password/verify-otp", {
                json: {reset_otp_token: resetOtpToken, otp_code: code, email: resetEmail}
              });
              submitBtn.disabled = false;
              submitBtn.textContent = "• Verify";

              if (res.ok && res.reset_token) {
                if (countdownTimer) clearInterval(countdownTimer);
                resetToken = res.reset_token;
                view = "reset_password";
                draw();
              } else {
                notice.style.display = "block";
                notice.className = "notice error";
                notice.textContent = res.msg || "Invalid code";
                inputs.forEach(x => x.classList.add('input-error'));
              }
            }
          };

          return h("div", null, notice, form);
        })(),
        h("div", {class: "trouble-divider"}, "having trouble?"),
        h("div", {class: "trouble-buttons"},
          h("button", {class: "trouble-btn", type: "button", onclick: () => {
            if (isSignupFlow) view = "signup";
            else view = "reset_email";
            draw();
          }}, "◇ Change Email Address"),
          h("button", {class: "trouble-btn", type: "button", onclick: () => {
            toast("Support: Please contact your platform administrator.");
          }}, "◇ Contact Support")
        ),
        h("button", {class: "link", type: "button", onclick: () => { view = "login"; draw(); }}, "Already verified? Sign in to your account")
      );
      container.append(card);
    }

    // =========================================================================
    // 4. SIGNUP COMPLETE SCREEN
    // =========================================================================
    else if (view === "signup_complete") {
      let redirectTimer = null;
      let count = 5;

      const card = h("main", {class: "card vp-complete-card"},
        h("div", {class: "stepper"},
          h("div", {class: "step step-done"}, h("div", {class: "circle"}, "✓"), h("div", {class: "step-title"}, "Account")),
          h("div", {class: "line line-done"}),
          h("div", {class: "step step-done"}, h("div", {class: "circle"}, "✓"), h("div", {class: "step-title"}, "Verify Email")),
          h("div", {class: "line line-done"}),
          h("div", {class: "step step-done"}, h("div", {class: "circle"}, "✓"), h("div", {class: "step-title"}, "Complete"))
        ),
        h("div", {class: "success-icon", html: svgIcon("check")}),
        h("h2", null, "Account Created Successfully! 🎉"),
        h("p", {class: "subtext"}, "Your account for ", h("strong", null, signupEmail), " has been verified and created."),
        h("p", {class: "subtext"}, "You can now sign in with your credentials."),
        h("div", {class: "info-box"},
          h("strong", null, "✓ What's ready:"), h("br"),
          "• Email address verified", h("br"),
          "• Password securely hashed", h("br"),
          "• Account activated and ready to use"
        ),
        h("button", {
          class: "signin-btn",
          id: "signin",
          onclick: () => {
            if (redirectTimer) clearInterval(redirectTimer);
            view = "login";
            draw();
          }
        }, "Sign In to Your Account →"),
        h("div", {class: "countdown"}, "Redirecting to sign in in ", h("span", {id: "counter"}, "5"), " seconds...")
      );

      redirectTimer = setInterval(() => {
        count -= 1;
        const cEl = card.querySelector('#counter');
        if (cEl) cEl.textContent = count;
        if (count <= 0) {
          clearInterval(redirectTimer);
          view = "login";
          draw();
        }
      }, 1000);

      container.append(card);
    }

    // =========================================================================
    // 5. RESET EMAIL SCREEN
    // =========================================================================
    else if (view === "reset_email") {
      const card = h("main", {class: "card vp-reset-card"},
        h("div", {class: "steps"},
          h("div", {class: "step active"}, h("div", {class: "circle"}, "1"), "Account"),
          h("div", {class: "line"}),
          h("div", {class: "step"}, h("div", {class: "circle"}, "2"), "Verify"),
          h("div", {class: "line"}),
          h("div", {class: "step"}, h("div", {class: "circle"}, "3"), "Complete")
        ),
        h("div", {class: "icon"}, "✉"),
        h("h1", null, "Reset Password"),
        h("p", {html: "Enter your registered email address.<br>We'll send a 6-digit verification code to continue."}),
        (()=>{
          const form = h("form", {class: "form", id: "form"});
          const notice = h("div", {id: "notice"});

          const emInput = h("input", {id: "email", type: "text", placeholder: "you@company.com or username", required: true});
          const submitBtn = h("button", {class: "submit", id: "submitBtn", type: "submit"}, "Send Verification Code   →");
          const orDiv = h("div", {class: "or"}, "or");
          const secBtn = h("button", {
            class: "alt",
            id: "security",
            type: "button",
            onclick: async () => {
              const val = emInput.value.trim();
              if (!val) {
                notice.innerHTML = `<div class="notice error">Please enter your username or email address.</div>`;
                emInput.classList.add("input-error");
                return;
              }
              secBtn.disabled = true;
              secBtn.textContent = "Checking...";
              const res = await api("/api/forgot-password", {json: {email: val}});
              secBtn.disabled = false;
              secBtn.textContent = "🔐 Use security question instead";

              if (res.ok && res.security_question) {
                resetEmail = res.email || val;
                recoveryQuestion = res.security_question;
                view = "security_recovery";
                draw();
              } else {
                notice.innerHTML = `<div class="notice error">${esc(res.msg || "No account found matching that username or email.")}</div>`;
                emInput.classList.add("input-error");
              }
            }
          }, "🔐 Use security question instead");

          form.append(notice, h("label", null, "Email Address or Username"), emInput, submitBtn, orDiv, secBtn);

          form.onsubmit = async (e) => {
            e.preventDefault();
            notice.replaceChildren();
            const val = emInput.value.trim();
            if (!val) return;

            submitBtn.disabled = true;
            submitBtn.textContent = "Sending OTP...";

            const res = await api("/api/forgot-password/send-otp", {json: {identifier: val}});
            submitBtn.disabled = false;
            submitBtn.textContent = "Send Verification Code   →";

            if (res.ok) {
              resetEmail = res.email;
              resetOtpToken = res.reset_otp_token;
              view = "otp_verification";
              draw();
            } else {
              notice.innerHTML = `<div class="notice error">${esc(res.msg || "Account not found")}</div>`;
              emInput.classList.add("input-error");
            }
          };

          return form;
        })(),
        h("button", {class: "signin", id: "signin", type: "button", onclick: () => { view = "login"; draw(); }}, "Remembered your password? Sign in to your account")
      );
      container.append(card);
    }

    // =========================================================================
    // 6. SECURITY RECOVERY SCREEN
    // =========================================================================
    else if (view === "security_recovery") {
      const card = h("main", {class: "card vp-sec-card"},
        h("div", {class: "steps"},
          h("div", {class: "step done"}, h("div", {class: "circle"}, "✓"), "Account"),
          h("div", {class: "line line-done"}),
          h("div", {class: "step active"}, h("div", {class: "circle"}, "2"), "Verify"),
          h("div", {class: "line"}),
          h("div", {class: "step"}, h("div", {class: "circle"}, "3"), "Complete")
        ),
        h("div", {class: "icon"}, "🔐"),
        h("h1", null, "Answer security question"),
        h("p", {class: "description"}, "Verify your identity to reset the password for ", h("strong", null, resetEmail), "."),
        (()=>{
          const form = h("form", {class: "form", id: "form"});
          const notice = h("div", {id: "notice"});

          const qBox = h("div", {class: "question", id: "question"}, recoveryQuestion || "Security Question");
          const ansInput = h("input", {id: "answer", type: "text", autocomplete: "off", required: true, placeholder: "Enter your answer"});
          const submitBtn = h("button", {class: "submit", type: "submit"}, "Continue to new password   →");

          form.append(notice, h("label", null, "Your security question"), qBox, h("label", null, "Security answer"), ansInput, submitBtn);

          form.onsubmit = async (e) => {
            e.preventDefault();
            notice.replaceChildren();
            const sa = ansInput.value.trim();
            if (sa.length < 2) {
              notice.innerHTML = `<div class="notice error">Please enter your answer.</div>`;
              ansInput.classList.add("input-error");
              return;
            }

            submitBtn.disabled = true;
            submitBtn.textContent = "Verifying...";

            const res = await api("/api/forgot-password", {
              json: {email: resetEmail, security_answer: sa}
            });
            submitBtn.disabled = false;
            submitBtn.textContent = "Continue to new password   →";

            if (res.ok && res.reset_token) {
              resetToken = res.reset_token;
              view = "reset_password";
              draw();
            } else {
              notice.innerHTML = `<div class="notice error">${esc(res.msg || "Incorrect security answer")}</div>`;
              ansInput.classList.add("input-error");
            }
          };

          return form;
        })(),
        h("div", {class: "bottom"},
          h("button", {id: "back", type: "button", onclick: () => { view = "reset_email"; draw(); }}, "← Use email verification"),
          h("button", {id: "signin", type: "button", onclick: () => { view = "login"; draw(); }}, "Sign in instead")
        )
      );
      container.append(card);
    }

    // =========================================================================
    // 7. CREATE NEW PASSWORD SCREEN (Matches Screenshot 5)
    // =========================================================================
    else if (view === "reset_password") {
      const card = h("main", {class: "card vp-pw-card"},
        h("div", {class: "stepper"},
          h("div", {class: "step step-done"}, h("div", {class: "circle"}, "✓"), h("div", {class: "step-title"}, "Account")),
          h("div", {class: "line line-done"}),
          h("div", {class: "step step-done"}, h("div", {class: "circle"}, "✓"), h("div", {class: "step-title"}, "Verify")),
          h("div", {class: "line line-done"}),
          h("div", {class: "step step-active"}, h("div", {class: "circle"}, "3"), h("div", {class: "step-title"}, "Complete"))
        ),
        h("div", {class: "icon", html: svgIcon("pw-box")}),
        h("h2", null, "Create New Password"),
        h("p", {class: "sub"}, "Choose a strong password to secure your account."),
        (()=>{
          const form = h("form", {class: "form", id: "form"});
          const notice = h("div", {id: "notice", class: "notice", style: "display:none;"});

          const pWrap = h("div", {class: "field"},
            h("label", null, "New Password"),
            h("div", {class: "wrap"},
              h("span", {class: "lock"}, "🔒"),
              h("input", {id: "password", class: "password", type: "password", autocomplete: "new-password", required: true}),
              h("button", {class: "eye", type: "button", onclick: (e) => {
                const inp = pWrap.querySelector("input");
                inp.type = inp.type === "password" ? "text" : "password";
                e.target.textContent = inp.type === "password" ? "◉" : "◌";
              }}, "◉")
            )
          );

          const cWrap = h("div", {class: "field"},
            h("label", null, "Confirm New Password"),
            h("div", {class: "wrap"},
              h("span", {class: "lock"}, "🔒"),
              h("input", {id: "confirm", class: "password", type: "password", autocomplete: "new-password", required: true}),
              h("button", {class: "eye", type: "button", onclick: (e) => {
                const inp = cWrap.querySelector("input");
                inp.type = inp.type === "password" ? "text" : "password";
                e.target.textContent = inp.type === "password" ? "◉" : "◌";
              }}, "◉")
            )
          );

          const rules = h("div", {class: "rules", style: "margin: 16px 0 10px;"},
            h("div", {class: "rule", "data-rule": "len"}, h("span", null, "✔"), " 8 or more characters"),
            h("div", {class: "rule", "data-rule": "upper"}, h("span", null, "✔"), " At least 1 uppercase letter"),
            h("div", {class: "rule", "data-rule": "lower"}, h("span", null, "✔"), " At least 1 lowercase letter"),
            h("div", {class: "rule", "data-rule": "num"}, h("span", null, "✔"), " At least 1 number"),
            h("div", {class: "rule", "data-rule": "special"}, h("span", null, "✔"), " At least 1 special character")
          );

          const submitBtn = h("button", {class: "submit", type: "submit", disabled: true}, "Change Password");

          form.append(notice, pWrap, cWrap, rules, submitBtn);

          function validate() {
            const p = pWrap.querySelector("input").value;
            const c = cWrap.querySelector("input").value;

            const checks = {
              len: p.length >= 8,
              upper: /[A-Z]/.test(p),
              lower: /[a-z]/.test(p),
              num: /\d/.test(p),
              special: /[!@#$%^&*(),.?":{}|<>]/.test(p)
            };

            rules.querySelectorAll('[data-rule]').forEach(x => {
              const ok = checks[x.dataset.rule];
              x.className = 'rule ' + (ok ? 'ok' : '');
            });

            submitBtn.disabled = !(Object.values(checks).every(Boolean) && p === c);
          }

          form.addEventListener("input", validate);

          form.onsubmit = async (e) => {
            e.preventDefault();
            const np = pWrap.querySelector("input").value;

            submitBtn.disabled = true;
            submitBtn.textContent = "Updating...";

            const res = await api("/api/reset-password", {
              json: {email: resetEmail, new_password: np, reset_token: resetToken}
            });
            submitBtn.disabled = false;
            submitBtn.textContent = "Change Password";

            if (res.ok) {
              toast("Password reset successfully! Please sign in.");
              view = "login";
              draw();
            } else {
              notice.style.display = "block";
              notice.className = "notice error";
              notice.textContent = res.msg || "Password reset failed";
            }
          };

          return form;
        })(),
        h("button", {class: "link", type: "button", onclick: () => { view = "login"; draw(); }}, "Want to start over? Sign in to your account")
      );
      container.append(card);
    }
  }

  draw();
  app.replaceChildren(container);
}

async function render(){const app=$("#app");if(S.es){S.es.close();S.es=null}
 const pub=await api("/api/public")||{};S.pub=pub;
 const m=await api("/api/me");if(m&&m.ok&&m.user){S.me=m}else{S.me=null}
 if(!S.me){renderAuth(app,pub);return}
 app.replaceChildren(h("div",{class:"side"},
   h("div",{class:"brand"},
     h("i",null,icon("radar")),
     h("div",null,S.me.name,h("small",null,S.me.user.role+" · "+S.me.user.email)),
     h("button",{class:"btn sm ghost",style:"margin-top:6px;font-size:11px;padding:2px 8px;width:fit-content;",onclick:async()=>{await api("/api/logout",{json:{}});S.me=null;toast("Signed out");render()}},"Sign out")
   ),
   h("div",{class:"ksearch",onclick:()=>UI.palette()},"Search or ask…",h("kbd",null,"Ctrl K")),
   h("div",{id:"nav"}),
   h("div",{class:"sp"}),userCard()
 ),
 h("div",{class:"main"},
   h("div",{id:"warn",class:"warnbar",style:"display:none"},"No CUDA GPU detected: models run on CPU and will be slow. In Colab choose Runtime → Change runtime type → T4 GPU, then restart."),
   h("div",{class:"top"},
     h("div",null,h("h1",{id:"ttl"}),h("div",{class:"sub",id:"sub"})),
     h("div",{class:"sp"}),
     h("div",{id:"acts",style:"display:flex;gap:8px"}),themeButton(),
     h("span",{class:"chip",id:"hl"},"connecting…"),
     h("span",{class:"chip",id:"gpu"}),
     h("button",{class:"btn sm ghost",style:"font-size:12px",onclick:async()=>{await api("/api/logout",{json:{}});S.me=null;toast("Signed out");render()}},"Sign Out")
   ),
   h("div",{class:"content"},roleBanner(),h("div",{class:"tabs",id:"tabs"}),h("div",{id:"pg"},h("div",{class:"blocks",id:"blocks"}))))
 );

 renderNav();const hs=(location.hash||"").slice(1).split(":");const ok=S.me.menu.find(m=>m.id===hs[0])||(hs[0]==="access"?{id:"access"}:null);S.page=ok?hs[0]:S.me.menu[0].id;S.tab=ok&&hs[1]||null;S.params={};
 loadPage(false);openStream();api("/api/pulse").then(p=>{if(!p.error){S.pulse=p;updBadges()}})}


// ---------------------------------------------------------------- RBAC Components
const ROLE_UI={"Admin":{c:"#8e4ec6",bg:"#f5eefb"},"Franchise Owner":{c:"#3e63dd",bg:"#edf2fe"},"Store Manager":{c:"#0b7a6c",bg:"#e6f7f4"},"Staff":{c:"#ab6400",bg:"#fff6e0"},"Auditor":{c:"#555b6b",bg:"#eef0f4"}};
const PAGE_LABEL={copilot:"AI Copilot",workforce:"Workforce",outlets:"Outlets",inventory:"Inventory",marketing:"Marketing",sentiment:"Sentiment",audit:"Audit"};
const CAP_LABEL={view:"View data",run:"Run agents / cycles",review:"Submit customer reviews",audit_write:"Log audits",resolve:"Resolve tickets",quote:"Create quotes",approve:"Approve actions",admin:"Admin tools",manage_users:"Manage users & roles",assign_outlet:"Assign outlets",export:"Export data",weather_refresh:"Refresh weather",attach_doc:"Attach documents",inject_storm:"Inject storm scenarios"};
const rpill=r=>{const u=ROLE_UI[r]||ROLE_UI.Staff;return h("span",{class:"pill",style:`color:${u.c};background:${u.bg};border-color:${u.c}55`},r)};
async function signIn(role){const r=await api("/api/demo/login",{json:{role}});if(!r.ok){toast(r.msg,true);return}S.me=null;S.demo=null;location.hash="";render()}
async function signOut(){await api("/api/demo/logout",{json:{}});S.me=null;S.demo=null;location.hash="";render()}
function loginScreen(app){const d=S.demo;
 app.replaceChildren(h("div",{class:"rolelogin"},
  h("div",{class:"lhead"},h("i",null,icon("radar")),h("div",null,h("h1",null,"FranchiseOps AI"),h("p",null,"Choose a role to sign in. Each role sees different pages and can do different things, enforced by the server."))),
  h("div",{class:"rgrid"},...d.roles.map(r=>{const u=ROLE_UI[r.role];const wr=r.caps.filter(c=>d.write_caps.includes(c)).length;
   return h("button",{class:"rcard",style:`--rc:${u.c};--rbg:${u.bg}`,onclick:()=>signIn(r.role)},
    h("div",{class:"rt"},h("b",null,r.role),r.read_only?h("span",{class:"pill",style:"color:#555b6b;border-color:#555b6b55"},"read-only"):null,r.outlet_scoped?h("span",{class:"pill",style:"color:#0b7a6c;border-color:#0b7a6c55"},"1 outlet"):null),
    h("p",null,r.blurb),
    h("div",{class:"rm"},...r.menu.map(m=>h("span",{class:"chip"},PAGE_LABEL[m]||m))),
    h("div",{class:"rf"},h("span",null,r.menu.length+" page"+(r.menu.length>1?"s":"")+" · "+wr+" write permission"+(wr===1?"":"s")),h("span",{class:"go"},"Sign in →")))})),
  h("p",{class:"lnote"},"Demo sign-in: no password. Role and outlet are read from the database on every request, so the browser cannot choose its own permissions.")))}
function userCard(){const u=S.me.user,r=u.role;const sel=S.demo?h("select",{class:"rsw",title:"Switch role",onchange:e=>signIn(e.target.value)},...S.demo.roles.map(x=>h("option",{value:x.role,selected:x.role===r},"Switch: "+x.role))):null;
 const initials=String(u.username||u.email||"U").trim().slice(0,1).toUpperCase();const avatar=u.profile_picture?h("img",{class:"uavatar",src:u.profile_picture,alt:"Profile photo"}):h("span",{class:"uavatar uph",title:"Profile photo"},initials);
 return h("div",{class:"ucard"},h("div",{class:"ur"},avatar,rpill(r),S.me.scope?h("span",{class:"chip"},S.me.scope):null),h("div",{class:"ue"},u.email),
  h("div",{class:"ua"},h("a",{class:"lnk",href:"#access",role:"button",onclick:e=>{e.preventDefault();UI.go("access")}},"My access"),S.demo?h("a",{class:"lnk",onclick:signOut},"Sign out"):null),sel)}
function roleBanner(){const r=S.me.user.role,caps=S.me.caps||[];const ro=!caps.some(c=>["run","review","audit_write","resolve","quote","approve","admin","manage_users"].includes(c));
 let m=null,t="info";
 if(r==="Store Manager"){if(S.me.outlet_assigned){m=h("span",null,h("b",null,"Store Manager view: "),"you only see and act on outlet ",h("b",null,S.me.scope),". Other outlets are blocked by the server.")}else{t="warn";m=h("span",null,h("b",null,"No outlet assigned yet. "),"You will see no data until an Admin assigns you an outlet (Admin → My access → Users).")}}
 else if(ro){t="info";m=h("span",null,h("b",null,r+" is read-only. "),"You can view and export, but running agents, approving, resolving and logging are disabled.")}
 else if(r==="Staff"){m=h("span",null,h("b",null,"Staff view: "),"Sentiment page only. You can submit customer reviews.")}
 return m?h("div",{class:"callout "+t,style:"margin-bottom:14px"},m):h("span")}
async function renderAccess(box){const me=S.me,d=(S.demo&&Array.isArray(S.demo.roles))?S.demo:await api("/api/demo/roles");if(!d||!Array.isArray(d.roles)){box.replaceChildren(h("div",{class:"callout warn"},(d&&d.msg)||"Access details are temporarily unavailable. Please refresh."));return}const cur=me.user.role;
 const ticks=(has)=>has?h("span",{class:"rtk y"},"✓"):h("span",{class:"rtk"},"–");
 const cols=d.roles;
 const mat=(title,rows,get)=>h("div",{class:"card"},h("h3",null,title),h("div",{class:"tbl",style:"margin-top:8px"},h("table",{class:"mx"},h("thead",null,h("tr",null,h("th",null,""),...cols.map(c=>h("th",{class:c.role===cur?"me":""},c.role)))),h("tbody",null,...rows.map(([id,lab])=>h("tr",null,h("td",null,lab),...cols.map(c=>h("td",{class:c.role===cur?"me":""},ticks(get(c,id))))))))));
 const who=h("div",{class:"card"},h("h3",null,"Signed in as"),h("div",{style:"display:flex;gap:10px;align-items:center;margin-top:6px"},rpill(cur),h("b",null,me.user.email),me.scope?h("span",{class:"chip"},"outlet "+me.scope):null),
  h("p",{class:"sub",style:"margin:8px 0 0"},d.roles.find(r=>r.role===cur).blurb));
 box.replaceChildren(who,mat("Pages by role",Object.entries(PAGE_LABEL),(c,id)=>c.menu.includes(id)),mat("Permissions by role",d.all_caps.map(c=>[c,(CAP_LABEL[c]||c)+(d.write_caps.includes(c)?"":" (read)")]),(c,id)=>c.caps.includes(id)));
 if(me.caps.includes("manage_users")){const users=h("div",{class:"card"},h("h3",null,"Users & outlet assignment"),h("div",{class:"sub"},"Admin only. A Store Manager has exactly one outlet; other roles are never outlet-scoped."),h("div",{id:"ulist",style:"margin-top:8px"},"Loading…"));box.append(users);loadUsers(d)}}
async function loadUsers(d){const [u,o]=await Promise.all([api("/api/admin/users"),api("/api/admin/outlets")]);const el=$("#ulist");if(!el)return;if(!u.users){el.textContent=u.msg||"Could not load users";return}
 const outs=o.outlets||[];
 el.replaceChildren(h("div",{class:"tbl"},h("table",null,h("thead",null,h("tr",null,...["User","Email","Role","Outlet",""].map(x=>h("th",null,x)))),h("tbody",null,...u.users.map(x=>{
  const rs=h("select",null,...d.roles.map(r=>h("option",{value:r.role,selected:r.role===x.role},r.role)));
  const os=h("select",null,h("option",{value:""},"— none —"),...outs.map(i=>h("option",{value:i,selected:i===x.outlet_id},i)));
  const sync=()=>{os.disabled=rs.value!=="Store Manager";if(os.disabled)os.value=""};rs.onchange=sync;sync();
  const save=h("button",{class:"btn sm primary",onclick:async e=>{e.target.disabled=true;let r={ok:true};
    if(rs.value!==x.role)r=await api("/api/admin/users/role",{json:{username:x.username,role:rs.value}});
    if(r.ok&&rs.value==="Store Manager"&&os.value&&os.value!==x.outlet_id)r=await api("/api/admin/assign-outlet",{json:{username:x.username,outlet_id:os.value}});
    toast(r.msg||"Saved",!r.ok);e.target.disabled=false;loadUsers(d)}},"Save");
  return h("tr",null,h("td",null,x.username),h("td",null,x.email),h("td",null,rs),h("td",null,os),h("td",null,save))})))))}



window.UI=UI;render();


In [ ]:
%%writefile /content/app/static/index.html
<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Operations Platform</title>
<link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Crect width='32' height='32' rx='8' fill='%235b5bd6'/%3E%3Cpath d='M8 20l5-6 4 4 7-9' stroke='%23fff' stroke-width='3' fill='none' stroke-linecap='round'/%3E%3C/svg%3E">
<link rel="stylesheet" href="/static/vendor/leaflet.css"><link rel="stylesheet" href="/static/style.css"></head>
<body><div id="app"></div><div id="toast"></div><div id="drawer"><div class="dh"><b id="dt"></b><button class="x" onclick="UI.closeDrawer()">✕</button></div><div id="db"></div></div>
<script src="/static/vendor/echarts.min.js"></script><script src="/static/vendor/leaflet.js"></script><script src="/static/vendor/topojson-client.min.js"></script><script src="/static/app.js"></script></body></html>


In [ ]:
%%writefile /content/app/static/style.css
@font-face{font-family:"Inter";font-style:normal;font-weight:100 900;font-display:swap;src:url(/static/vendor/fonts/inter-latin-wght-normal.woff2) format("woff2");unicode-range:U+0000-00FF,U+0131,U+0152-0153,U+02BB-02BC,U+02C6,U+02DA,U+02DC,U+2000-206F,U+20AC,U+2122,U+2191,U+2193,U+2212,U+2215,U+FEFF,U+FFFD}
@font-face{font-family:"Inter";font-style:normal;font-weight:100 900;font-display:swap;src:url(/static/vendor/fonts/inter-latin-ext-wght-normal.woff2) format("woff2");unicode-range:U+0100-02BA,U+02BD-02C5,U+02C7-02CC,U+02CE-02D7,U+02DD-02FF,U+0304,U+0308,U+0329,U+1D00-1DBF,U+1E00-1E9F,U+1EF2-1EFF,U+2020,U+20A0-20AB,U+20AD-20C0,U+2113,U+2C60-2C7F,U+A720-A7FF}
:root{--bg:#f6f7fb;--card:#fff;--line:#e6e8f0;--line2:#d9dce8;--tx:#1c2024;--tx2:#454a58;--mut:#6b7080;--pri:#5b5bd6;--pri2:#4a4ac4;--pri-bg:#eef0ff;--teal:#12a594;--teal-bg:#e6f7f4;--blue:#3e63dd;--blue-bg:#edf2fe;--amber:#f5a524;--amber-bg:#fff6e0;--red:#e5484d;--red-bg:#feebec;--vio:#8e4ec6;--vio-bg:#f5eefb;--green:#30a46c;--r:14px;--sh:0 1px 2px rgba(20,24,40,.04),0 4px 14px rgba(20,24,40,.05)}
*{box-sizing:border-box}html,body{margin:0;height:100%;background:var(--bg);color:var(--tx);font:14px/1.5 "Inter",ui-sans-serif,system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;font-feature-settings:"cv11","ss01";-webkit-font-smoothing:antialiased}
button,input,select,textarea{font:inherit;color:inherit}a{color:var(--pri);text-decoration:none}a:hover{text-decoration:underline}
#app{display:flex;height:100vh}
.side{width:248px;flex:none;background:#fff;border-right:1px solid var(--line);display:flex;flex-direction:column;padding:14px 12px;overflow:auto}
.brand{display:flex;gap:11px;align-items:center;padding:4px 6px 14px;font-weight:700;font-size:15px;letter-spacing:-.01em}.brand i{width:34px;height:34px;border-radius:11px;background:linear-gradient(135deg,var(--pri),#8b8bf0);display:grid;place-items:center;color:#fff;box-shadow:0 4px 12px rgba(91,91,214,.35)}.brand i svg{width:18px;height:18px;stroke:#fff;fill:none;stroke-width:2}
.brand small{display:block;color:var(--mut);font-weight:500;font-size:11px;max-width:150px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.ksearch{display:flex;align-items:center;gap:8px;margin:0 2px 10px;padding:8px 10px;border:1px solid var(--line);border-radius:10px;color:var(--mut);cursor:pointer;font-size:12.5px;background:var(--bg)}.ksearch:hover{border-color:var(--line2)}.ksearch kbd{margin-left:auto;font:600 10.5px "Inter";border:1px solid var(--line2);border-radius:5px;padding:1px 5px;background:#fff;color:var(--mut)}
.ngrp{font-size:10.5px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;color:#9a9eb0;padding:12px 10px 4px}
.nav{display:flex;align-items:center;gap:10px;padding:8px 10px;border-radius:10px;color:var(--tx2);cursor:pointer;border:1px solid transparent;margin-bottom:1px;user-select:none;font-weight:500;transition:background .12s}
.nav:hover{background:#f1f2f8}.nav.on{background:var(--pri-bg);color:var(--pri2);font-weight:600}
.nav svg{width:17px;height:17px;flex:none;stroke:currentColor;fill:none;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round;opacity:.85}.nav .bd{margin-left:auto;background:var(--red);color:#fff;border-radius:99px;font-size:10.5px;padding:1px 7px;font-weight:700}.nav .bd.a{background:var(--amber);color:#3d2600}
.nav.cmd{margin-top:10px;background:linear-gradient(135deg,#5b5bd6,#7a6be0);color:#fff;font-weight:600;box-shadow:0 6px 16px rgba(91,91,214,.28)}.nav.cmd:hover{filter:brightness(1.05)}.nav.cmd.on{background:linear-gradient(135deg,#4a4ac4,#6a5bd8);color:#fff}
.main{flex:1;min-width:0;display:flex;flex-direction:column}
.top{display:flex;align-items:center;gap:12px;padding:12px 24px;border-bottom:1px solid var(--line);background:rgba(255,255,255,.85);backdrop-filter:blur(10px);flex-wrap:wrap}
.top h1{font-size:19px;margin:0;letter-spacing:-.015em}.top .sub{color:var(--mut);font-size:12.5px;margin:0}.sp{flex:1}
.chip{display:inline-flex;gap:6px;align-items:center;padding:4px 11px;border-radius:99px;border:1px solid var(--line);background:#fff;font-size:12px;white-space:nowrap;color:var(--tx2)}
.dot{width:8px;height:8px;border-radius:50%;background:#b4b8c8}.dot.g{background:var(--green);box-shadow:0 0 0 3px rgba(48,164,108,.18)}.dot.r{background:var(--red);box-shadow:0 0 0 3px rgba(229,72,77,.16)}.dot.a{background:var(--amber)}
.live{animation:pl 1.6s infinite}@keyframes pl{50%{opacity:.4}}
.content{flex:1;overflow:auto;padding:20px 24px 70px}
.tabs{display:flex;gap:2px;border-bottom:1px solid var(--line);margin-bottom:16px;flex-wrap:wrap}.tabs b{padding:9px 14px;cursor:pointer;color:var(--mut);border-bottom:2px solid transparent;font-weight:600;margin-bottom:-1px}.tabs b:hover{color:var(--tx)}.tabs b.on{color:var(--pri2);border-color:var(--pri)}
.blocks>*{margin-bottom:16px}
.card{background:var(--card);border:1px solid var(--line);border-radius:var(--r);padding:16px;box-shadow:var(--sh);min-width:0}.card>h3{margin:0 0 2px;font-size:14px;letter-spacing:-.005em}.card>.sub,.sub{color:var(--mut);font-size:12.5px;margin-bottom:8px}
.row{display:grid;gap:16px}.row>*{min-width:0}.col>*{margin-bottom:16px}.col>*:last-child{margin-bottom:0}
@media(max-width:1000px){.row{grid-template-columns:1fr!important}.side{width:64px}.nav span,.brand div,.side .foot,.ngrp,.ksearch{display:none}}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:14px}.kpi{background:var(--card);border:1px solid var(--line);border-radius:var(--r);padding:14px 16px;position:relative;overflow:hidden;box-shadow:var(--sh)}
.kpi:before{content:"";position:absolute;left:0;top:14px;bottom:14px;width:3px;border-radius:0 3px 3px 0;background:var(--teal)}.kpi.Critical:before,.kpi.High:before{background:var(--red)}.kpi.Warning:before,.kpi.Medium:before{background:var(--amber)}.kpi.Info:before{background:var(--blue)}
.kpi .l{color:var(--mut);font-size:11.5px;font-weight:600;text-transform:uppercase;letter-spacing:.05em}.kpi .v{font-size:26px;font-weight:700;margin-top:3px;letter-spacing:-.02em}.kpi .s{color:var(--mut);font-size:12px}
.pill{display:inline-block;padding:1px 9px;border-radius:99px;font-size:11.5px;font-weight:600;border:1px solid}
.tbl{overflow:auto;border:1px solid var(--line);border-radius:12px;background:#fff}.tbl table{width:100%;border-collapse:collapse;font-size:12.8px}.tbl th{position:sticky;top:0;background:#fafbfe;text-align:left;padding:9px 12px;color:var(--mut);font-weight:600;cursor:pointer;white-space:nowrap;text-transform:capitalize;z-index:1;border-bottom:1px solid var(--line)}
.tbl td{padding:8px 12px;border-top:1px solid #eff0f6;max-width:380px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.tbl tr.cl{cursor:pointer}.tbl tr.cl:hover td{background:var(--pri-bg)}
.bar{display:flex;align-items:center;gap:8px;min-width:110px}.bar i{display:block;height:6px;border-radius:4px;flex:1;background:#eceef6;overflow:hidden}.bar i b{display:block;height:100%;border-radius:4px}
.srch{margin:0 0 8px;padding:7px 11px;border-radius:9px;border:1px solid var(--line2);background:#fff;width:240px}
.callout{padding:11px 14px;border-radius:12px;border:1px solid;display:flex;gap:10px;color:var(--tx2)}.callout.info{border-color:#cfd9fb;background:var(--blue-bg)}.callout.warn,.callout.warning{border-color:#f6d98f;background:var(--amber-bg)}.callout.error,.callout.critical,.callout.bad{border-color:#f5bcbe;background:var(--red-bg)}.callout.good,.callout.ok,.callout.success{border-color:#b6e3da;background:var(--teal-bg)}
.md{line-height:1.65}.md code,.md pre{background:#f1f2f8;border-radius:6px;padding:1px 5px;font-family:ui-monospace,"JetBrains Mono",monospace;font-size:12px}.md pre{padding:10px;overflow:auto;white-space:pre-wrap}.md h1,.md h2,.md h3{margin:.7em 0 .3em;letter-spacing:-.01em}.md ul{margin:.3em 0;padding-left:20px}.md table{border-collapse:collapse;margin:.5em 0}.md td,.md th{border:1px solid var(--line);padding:5px 10px}
.btn{border:1px solid var(--line2);background:#fff;padding:7px 14px;border-radius:10px;cursor:pointer;font-weight:600;color:var(--tx2);transition:all .12s}.btn:hover{background:#f4f5fb;border-color:#c9cde0}
.btn.primary{background:var(--pri);color:#fff;border-color:var(--pri);box-shadow:0 2px 8px rgba(91,91,214,.3)}.btn.primary:hover{background:var(--pri2)}.btn.danger{background:var(--red-bg);border-color:#f5bcbe;color:#c62a2f}.btn.ok{background:var(--teal-bg);border-color:#b6e3da;color:#0b7a6c}.btn.ghost{background:transparent}.btn.sm{padding:3px 10px;font-size:12px}.btn:disabled{opacity:.5;cursor:wait}
form.f{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;align-items:end}form.f.inline{grid-template-columns:repeat(auto-fit,minmax(160px,max-content))}
.fld label{display:block;color:var(--mut);font-size:12px;font-weight:500;margin-bottom:4px}.fld input,.fld select,.fld textarea,.in{width:100%;padding:8px 11px;border-radius:10px;border:1px solid var(--line2);background:#fff;transition:border .12s,box-shadow .12s}.fld input:focus,.fld select:focus,.fld textarea:focus,.in:focus,.srch:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(91,91,214,.15)}.fld textarea{min-height:80px;grid-column:1/-1}.fld.wide{grid-column:1/-1}.fld input[type=range]{padding:0;accent-color:var(--pri)}
.chk{display:flex;flex-wrap:wrap;gap:6px}.chk label{display:flex;gap:6px;align-items:center;margin:0;color:var(--tx2);padding:3px 10px;border:1px solid var(--line2);border-radius:99px;font-size:12px;background:#fff}
.res{margin-top:12px}
.tk{border:1px solid var(--line);border-left:4px solid var(--blue);border-radius:12px;padding:12px 14px;margin-bottom:10px;background:#fff;box-shadow:var(--sh)}.tk.Critical{border-left-color:var(--red)}.tk.Warning{border-left-color:var(--amber)}.tk.proposed{background:var(--vio-bg)}
.tk .h{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.tk .d{margin:6px 0;color:var(--tx2)}.tk .m{color:var(--mut);font-size:11.8px;display:flex;gap:10px;flex-wrap:wrap}.tk .a{display:flex;gap:6px;margin-top:8px;flex-wrap:wrap;align-items:center}.tk .cm{margin-top:8px;border-top:1px dashed var(--line2);padding-top:6px;font-size:12.5px}
.filters{display:flex;gap:6px;margin-bottom:12px;flex-wrap:wrap;align-items:center}.filters .chip{cursor:pointer}.filters .chip:hover{border-color:var(--pri)}.filters .chip.on{border-color:var(--pri);background:var(--pri-bg);color:var(--pri2);font-weight:600}
.hero{display:flex;gap:18px;align-items:center;background:linear-gradient(120deg,#eef0ff,#e8f6f4);border:1px solid #d9def7;border-radius:16px;padding:16px 20px}.hero .sc{font-size:36px;font-weight:800;min-width:84px;text-align:center;letter-spacing:-.03em}.hero h2{margin:0;font-size:16px}.hero p{margin:2px 0;color:var(--mut);font-size:12.5px}
.map{border-radius:12px;overflow:hidden;border:1px solid var(--line);background:#cfe5f7}.leaflet-container{background:#cfe5f7;font:inherit}.lg{display:flex;gap:12px;font-size:12px;color:var(--mut);margin-top:8px;flex-wrap:wrap}.lg i{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}
.leaflet-tooltip{background:#fff;color:var(--tx);border:1px solid var(--line);border-radius:8px;box-shadow:var(--sh)}
#toast{position:fixed;right:18px;bottom:18px;display:flex;flex-direction:column;gap:8px;z-index:99}#toast div{background:#1c2024;color:#fff;padding:10px 16px;border-radius:12px;max-width:380px;box-shadow:0 10px 30px rgba(20,24,40,.25);animation:up .25s}#toast div.bad{background:#c62a2f}@keyframes up{from{transform:translateY(10px);opacity:0}}
#drawer{z-index:2000!important;position:fixed;right:0;top:0;bottom:0;width:min(720px,96vw);background:#fff;border-left:1px solid var(--line);transform:translateX(105%);transition:.25s;z-index:50;display:flex;flex-direction:column;box-shadow:-20px 0 60px rgba(20,24,40,.12)}#drawer.on{transform:none}
#drawer .dh{display:flex;justify-content:space-between;padding:14px 18px;border-bottom:1px solid var(--line);font-size:16px}#db{padding:16px;overflow:auto;flex:1;background:var(--bg)}.x{background:none;border:0;color:var(--mut);cursor:pointer;font-size:16px}
.login{display:grid;grid-template-columns:1.1fr 1fr;width:100%;min-height:100vh;background:#fff}.lp{background:radial-gradient(900px 500px at 20% 0%,#7c7cf0 0%,transparent 60%),linear-gradient(145deg,#4a4ac4,#5b5bd6 55%,#12a594);color:#fff;padding:56px;display:flex;flex-direction:column;justify-content:center;gap:18px}.lp h2{font-size:34px;line-height:1.15;margin:0;letter-spacing:-.025em}.lp p{margin:0;color:rgba(255,255,255,.85);max-width:460px}.lp ul{list-style:none;margin:8px 0 0;padding:0;display:grid;gap:10px}.lp li{display:flex;gap:10px;align-items:center;color:rgba(255,255,255,.92)}.lp li:before{content:"";width:8px;height:8px;border-radius:50%;background:#7ff3df;flex:none}
.auth{margin:auto;width:min(440px,92vw);padding:26px;border:0;box-shadow:none}.auth h1{margin:0 0 4px;letter-spacing:-.02em}.auth .fld{margin-bottom:12px}.demo{display:flex;gap:6px;flex-wrap:wrap;margin-top:14px}.demo .chip{cursor:pointer}.demo .chip:hover{border-color:var(--pri);color:var(--pri2)}
@media(max-width:860px){.login{grid-template-columns:1fr}.lp{display:none}}
.boot{text-align:center;padding:60px;color:var(--mut)}.spin{width:28px;height:28px;border:3px solid var(--line);border-top-color:var(--pri);border-radius:50%;animation:sp .8s linear infinite;margin:0 auto 12px}@keyframes sp{to{transform:rotate(360deg)}}
.warnbar{background:var(--amber-bg);border-bottom:1px solid #f6d98f;color:#7a4b00;padding:7px 24px;font-size:12.5px}
.chat{display:flex;flex-direction:column;gap:12px}.msgs{display:flex;flex-direction:column;gap:12px;min-height:280px}.m{max-width:88%;padding:11px 15px;border-radius:16px;border:1px solid var(--line)}.m.u{align-self:flex-end;background:var(--pri);color:#fff;border-color:var(--pri);border-bottom-right-radius:5px}.m.a{background:#fff;box-shadow:var(--sh);border-bottom-left-radius:5px}.st{color:var(--mut);font-size:12px}.st:before{content:"› "}.cites{margin:4px 0;font-size:12px;color:var(--mut)}.cites summary{cursor:pointer;color:var(--pri2);font-weight:600}
.cin{display:flex;gap:8px}.cin input{flex:1;padding:11px 15px;border-radius:12px;border:1px solid var(--line2);background:#fff}.cin input:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(91,91,214,.15)}
.kpal{position:fixed;inset:0;background:rgba(20,24,40,.35);backdrop-filter:blur(2px);z-index:90;display:flex;justify-content:center;padding-top:12vh}.kpal>div{width:min(620px,94vw);background:#fff;border-radius:16px;box-shadow:0 30px 80px rgba(20,24,40,.3);overflow:hidden;align-self:flex-start}.kpal input{width:100%;border:0;border-bottom:1px solid var(--line);padding:16px 18px;font-size:15px;outline:0}.kpal .it{display:flex;gap:10px;padding:10px 18px;cursor:pointer;align-items:center;color:var(--tx2)}.kpal .it.on{background:var(--pri-bg);color:var(--pri2)}.kpal .it small{margin-left:auto;color:var(--mut)}.kpal .lst{max-height:50vh;overflow:auto;padding:6px 0}.kpal .gl{font-size:10.5px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;color:#9a9eb0;padding:8px 18px 2px}
::-webkit-scrollbar{width:9px;height:9px}::-webkit-scrollbar-thumb{background:#d5d8e6;border-radius:9px}
@media(max-width:760px){#app{flex-direction:column-reverse}.side{width:100%;flex-direction:row;padding:4px;overflow-x:auto;border-right:0;border-top:1px solid var(--line);position:sticky;bottom:0;z-index:20;gap:2px}.side .brand,.side .foot,.side .sp,.ngrp,.ksearch{display:none}#nav{display:flex;gap:2px}.nav{flex-direction:column;gap:2px;padding:6px 10px;font-size:10px;min-width:62px;text-align:center;margin:0}.nav.cmd{margin-top:0}.nav span{display:block!important;white-space:nowrap;max-width:70px;overflow:hidden;text-overflow:ellipsis}.nav .bd{position:absolute;margin:0}.nav{position:relative}.content{padding:12px 10px 20px}.top{padding:8px 10px}.top h1{font-size:15px}.kpis{grid-template-columns:repeat(2,1fr)}.hero{flex-direction:column;align-items:flex-start}#drawer{width:100vw}.tbl td{max-width:200px}.srch{width:100%}.chip{font-size:11px}}
.top>div:first-child{min-width:0;flex:1 1 280px}.top .sub{white-space:nowrap;overflow:hidden;text-overflow:ellipsis;max-width:100%}

.leaflet-container{isolation:isolate;overflow:hidden!important}.mapw,.map,.leafmap{isolation:isolate;position:relative;z-index:0}#drawer.on{z-index:2000}.kpal{z-index:3000!important}#toast{z-index:3500!important}

/* ==========================================================================
   Exact Verify Portal Milestone 1 UI Styles (Pixel-Perfect from Screenshots)
   ========================================================================== */

.portal-auth-bg {
  position: fixed;
  top: 0;
  left: 0;
  right: 0;
  bottom: 0;
  width: 100vw;
  height: 100vh;
  margin: 0;
  padding: 40px 15px 90px !important;
  box-sizing: border-box;
  display: block !important;
  overflow-y: auto !important;
  overflow-x: hidden !important;
  -webkit-overflow-scrolling: touch;
  background-color: #5542e8 !important;
  background-image: 
    linear-gradient(to right, rgba(255, 255, 255, 0.08) 1px, transparent 1px),
    linear-gradient(to bottom, rgba(255, 255, 255, 0.08) 1px, transparent 1px),
    radial-gradient(circle at 50% 50%, #6854f5 0%, #4834dc 100%) !important;
  background-size: 80px 80px, 80px 80px, 100% 100% !important;
  background-attachment: fixed !important;
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
  color: #17233b;
  position: fixed;
  top: 0;
  left: 0;
  z-index: 99999;
  overflow-y: auto;
}

/* Universal Card Base */
.portal-auth-bg .card {
  background: #ffffff;
  border-radius: 16px;
  box-shadow: 0 20px 50px rgba(0, 0, 0, 0.2);
  margin: auto;
  box-sizing: border-box;
  text-align: center;
}

.portal-auth-bg .icon {
  width: 58px;
  height: 58px;
  margin: 0 auto 20px;
  display: grid;
  place-items: center;
  border-radius: 50%;
  background: linear-gradient(135deg, #6260e6, #8343ad);
  color: white;
  font-size: 24px;
  box-shadow: 0 8px 20px rgba(98, 96, 230, 0.35);
}

.portal-auth-bg .icon svg {
  display: block;
}

.portal-auth-bg h1,
.portal-auth-bg h2 {
  margin: 0 0 10px;
  font-size: 26px;
  font-weight: 700;
  color: #17233b;
}

.portal-auth-bg p,
.portal-auth-bg .subtitle,
.portal-auth-bg .subtext,
.portal-auth-bg .sub {
  margin: 0 0 24px;
  color: #64748b;
  font-size: 14px;
  line-height: 1.5;
}

/* Stepper (Account -> Verify -> Complete) */
.portal-auth-bg .stepper,
.portal-auth-bg .steps {
  display: flex;
  align-items: center;
  justify-content: center;
  max-width: 380px;
  margin: 0 auto 32px;
  gap: 4px;
}

.portal-auth-bg .step {
  min-width: 65px;
  display: flex;
  flex-direction: column;
  align-items: center;
  font-size: 11px;
  font-weight: 600;
  color: #94a3b8;
  white-space: nowrap;
}

.portal-auth-bg .circle {
  width: 32px;
  height: 32px;
  border-radius: 50%;
  display: grid;
  place-items: center;
  font-size: 13px;
  font-weight: 700;
  background: #f1f5f9;
  color: #94a3b8;
  margin-bottom: 6px;
}

.portal-auth-bg .step-done .circle,
.portal-auth-bg .step.done .circle {
  background: #00b982 !important;
  color: #fff !important;
}

.portal-auth-bg .step-active .circle,
.portal-auth-bg .step.active .circle,
.portal-auth-bg .step.current .circle {
  background: #6260e6 !important;
  color: #fff !important;
  box-shadow: 0 0 0 4px #e8eaff !important;
}

.portal-auth-bg .step-done,
.portal-auth-bg .step.done {
  color: #00b982 !important;
}

.portal-auth-bg .step-active,
.portal-auth-bg .step.active,
.portal-auth-bg .step.current {
  color: #6260e6 !important;
}

.portal-auth-bg .line {
  height: 2px;
  flex: 1;
  min-width: 30px;
  margin: 0 8px 18px;
  border-radius: 2px;
  background: #e2e8f0;
}

.portal-auth-bg .line-done,
.portal-auth-bg .line-active {
  background: #00b982 !important;
}

/* ==========================================================================
   1. LOGIN SCREEN (.vp-login-card) - Matches Screenshot 2
   ========================================================================== */
.portal-auth-bg .card.vp-login-card {
  width: min(620px, 94vw);
  max-width: 620px;
  padding: 44px 48px 38px;
}

.portal-auth-bg .vp-login-card .form {
  width: 535px;
  max-width: 100%;
  margin: 32px auto 28px;
  text-align: left;
}

.portal-auth-bg .vp-login-card label {
  display: block;
  font-weight: 700;
  font-size: 13px;
  margin: 17px 0 8px;
  color: #17233b;
}

.portal-auth-bg .vp-login-card .wrap {
  position: relative;
}

.portal-auth-bg .vp-login-card .field {
  width: 100%;
  height: 56px;
  border: 1.5px solid #d6dfec;
  border-radius: 8px;
  padding: 0 48px 0 45px;
  font-size: 15px;
  color: #17233b;
  background: #fff;
  box-sizing: border-box;
}

.portal-auth-bg .vp-login-card .field:focus {
  border-color: #6260e6;
  outline: none;
  box-shadow: 0 0 0 3px rgba(98, 96, 230, 0.15);
}

.portal-auth-bg .vp-login-card .symbol {
  position: absolute;
  left: 15px;
  top: 18px;
  color: #64748b;
  font-size: 16px;
  pointer-events: none;
}

.portal-auth-bg .vp-login-card .eye {
  position: absolute;
  right: 12px;
  top: 14px;
  border: 0;
  background: none;
  color: #64748b;
  cursor: pointer;
  font-size: 20px;
}

.portal-auth-bg .vp-login-card .primary {
  width: 100%;
  height: 56px;
  margin-top: 26px;
  border: 0;
  border-radius: 8px;
  background: linear-gradient(90deg, #6260e6, #8343ad);
  color: #fff;
  font-weight: 700;
  font-size: 15px;
  box-shadow: 0 5px 14px rgba(98, 96, 230, 0.35);
  cursor: pointer;
}

.portal-auth-bg .vp-login-card .links {
  display: flex;
  justify-content: center;
  gap: 16px;
}

.portal-auth-bg .vp-login-card .links button {
  min-width: 200px;
  border: 1px solid #e2e8f0;
  background: #fff;
  border-radius: 8px;
  padding: 14px 20px;
  color: #475569;
  font-weight: 700;
  cursor: pointer;
}

.portal-auth-bg .vp-login-card .links button:hover {
  border-color: #cbd5e1;
  background: #f8fafc;
}

/* ==========================================================================
   2. CREATE ACCOUNT SCREEN (.vp-signup-card) - Matches Screenshot 3
   ========================================================================== */
.portal-auth-bg .card.vp-signup-card {
  width: min(680px, 94vw);
  max-width: 680px;
  padding: 40px 48px 36px;
}

.portal-auth-bg .vp-signup-card .form {
  width: 590px;
  max-width: 100%;
  margin: 0 auto;
  text-align: left;
}

.portal-auth-bg .vp-signup-card .field {
  margin: 14px 0 !important;
  width: 100% !important;
  height: auto !important;
  border: none !important;
  padding: 0 !important;
  background: transparent !important;
}

.portal-auth-bg .vp-signup-card .field label {
  display: block;
  font-size: 13px;
  font-weight: 700;
  margin-bottom: 6px;
  color: #17233b;
}

.portal-auth-bg .vp-signup-card .wrap {
  position: relative;
  width: 100%;
}

.portal-auth-bg .vp-signup-card .wrap input,
.portal-auth-bg .vp-signup-card .wrap select {
  width: 100% !important;
  height: 48px !important;
  border: 1.5px solid #cbd5e1 !important;
  border-radius: 8px !important;
  padding: 0 44px !important;
  font-size: 14px !important;
  color: #17233b !important;
  background: #ffffff !important;
  box-sizing: border-box !important;
  outline: none;
}

.portal-auth-bg .vp-signup-card .wrap select#role {
  padding-left: 48px !important;
  padding-right: 36px !important;
  cursor: pointer;
  appearance: auto;
  -webkit-appearance: auto;
}

.portal-auth-bg .vp-signup-card .wrap select#question {
  padding-left: 16px !important;
  padding-right: 36px !important;
  cursor: pointer;
}

.portal-auth-bg .vp-signup-card .wrap input:focus,
.portal-auth-bg .vp-signup-card .wrap select:focus {
  border-color: #6260e6 !important;
  box-shadow: 0 0 0 3px rgba(98, 96, 230, 0.15) !important;
}

.portal-auth-bg .vp-signup-card .mark {
  position: absolute;
  left: 14px;
  top: 14px;
  color: #64748b;
  font-size: 16px;
  pointer-events: none;
}

.portal-auth-bg .vp-signup-card .eye {
  position: absolute;
  right: 12px;
  top: 12px;
  border: 0;
  background: transparent;
  color: #475569;
  font-size: 19px;
  cursor: pointer;
}

/* Password Strength Meter */
.portal-auth-bg .meter {
  display: flex;
  gap: 4px;
  margin: 12px 0 6px;
}

.portal-auth-bg .meter i {
  height: 4px;
  flex: 1;
  background: #e2e8f0;
  border-radius: 3px;
  display: block;
}

.portal-auth-bg .meter.weak i:first-child { background: #ef4444; }
.portal-auth-bg .meter.medium i:nth-child(-n+3) { background: #f59e0b; }
.portal-auth-bg .meter.strong i { background: #22c55e; }

.portal-auth-bg .rule {
  font-size: 12.5px;
  margin: 4px 0;
  color: #64748b;
  display: flex;
  align-items: center;
  gap: 6px;
}

.portal-auth-bg .rule.ok {
  color: #16a34a;
  font-weight: 600;
}

.portal-auth-bg .rule.bad {
  color: #dc2626;
}

/* Security Box */
.portal-auth-bg .security {
  border: 1px solid #e2e8f0;
  border-radius: 10px;
  padding: 14px 18px;
  margin: 18px 0 14px;
  background: #f8fafc;
}

.portal-auth-bg .security h3 {
  font-size: 13.5px;
  font-weight: 700;
  margin: 0 0 4px;
  color: #17233b;
}

.portal-auth-bg .security p {
  font-size: 12px;
  color: #64748b;
  margin: 0 0 10px;
}

.portal-auth-bg .vp-signup-card .submit {
  width: 100%;
  height: 50px;
  border: 0;
  border-radius: 8px;
  background: linear-gradient(90deg, #6260e6, #8343ad);
  color: white;
  font-weight: 700;
  font-size: 15px;
  margin-top: 14px;
  cursor: pointer;
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 8px;
  box-shadow: 0 5px 14px rgba(98, 96, 230, 0.35);
  transition: all .2s ease;
}

.portal-auth-bg .vp-signup-card .submit:disabled {
  opacity: 0.5;
  cursor: not-allowed;
  box-shadow: none;
}

.portal-auth-bg .link {
  display: block;
  margin: 20px auto 0;
  border: 0;
  background: none;
  color: #6260e6;
  font-weight: 700;
  font-size: 13.5px;
  cursor: pointer;
}

.portal-auth-bg .link:hover {
  text-decoration: underline;
}

/* ==========================================================================
   3. VERIFY SCREEN (.vp-otp-card) - Matches Screenshot 4
   ========================================================================== */
.portal-auth-bg .card.vp-otp-card {
  width: min(620px, 94vw);
  max-width: 620px;
  padding: 42px 44px 38px;
}

.portal-auth-bg .vp-otp-card .otp-container {
  display: flex;
  justify-content: center;
  gap: 12px;
  margin: 18px auto 22px;
}

.portal-auth-bg .vp-otp-card .otp {
  width: 48px !important;
  height: 56px !important;
  font-size: 22px !important;
  font-weight: 700 !important;
  text-align: center !important;
  border: 1.5px solid #34d399 !important;
  background: #f0fdf4 !important;
  border-radius: 8px !important;
  color: #17233b !important;
  outline: none !important;
  box-sizing: border-box !important;
  transition: all .2s ease !important;
}

.portal-auth-bg .vp-otp-card .otp:focus {
  border-color: #6260e6 !important;
  background: #ffffff !important;
  box-shadow: 0 0 0 3px rgba(98, 96, 230, 0.2);
}

.portal-auth-bg .vp-otp-card .spam-note {
  color: #94a3b8;
  font-size: 12.5px;
  margin: 14px 0 6px;
}

.portal-auth-bg .vp-otp-card .resend {
  margin: 8px 0 22px;
  font-size: 13px;
  color: #64748b;
}

.portal-auth-bg .vp-otp-card .verify {
  width: 100% !important;
  max-width: 340px !important;
  height: 48px !important;
  border: 0 !important;
  border-radius: 8px !important;
  background: linear-gradient(90deg, #6260e6, #8343ad) !important;
  color: #fff !important;
  font-weight: 700 !important;
  font-size: 15px !important;
  margin: 18px auto 0 !important;
  cursor: pointer !important;
  display: block !important;
  box-shadow: 0 5px 14px rgba(98, 96, 230, 0.35) !important;
}

.portal-auth-bg .trouble-divider {
  margin: 28px auto 14px;
  color: #94a3b8;
  font-size: 12px;
  text-align: center;
}

.portal-auth-bg .trouble-buttons {
  display: flex;
  justify-content: center;
  gap: 14px;
  margin-bottom: 20px;
}

.portal-auth-bg .trouble-btn {
  padding: 10px 18px;
  border: 1px solid #e2e8f0;
  background: #fff;
  border-radius: 8px;
  color: #475569;
  font-size: 12.5px;
  font-weight: 600;
  cursor: pointer;
  display: flex;
  align-items: center;
  gap: 6px;
  transition: all .15s;
}

.portal-auth-bg .trouble-btn:hover {
  background: #f8fafc;
  border-color: #cbd5e1;
}

/* ==========================================================================
   4. CREATE NEW PASSWORD SCREEN (.vp-pw-card) - Matches Screenshot 5
   ========================================================================== */
.portal-auth-bg .card.vp-pw-card {
  width: min(620px, 94vw);
  max-width: 620px;
  padding: 42px 48px 38px;
}

.portal-auth-bg .vp-pw-card .form {
  width: 535px;
  max-width: 100%;
  margin: 24px auto 0;
  text-align: left;
}

.portal-auth-bg .vp-pw-card .field {
  margin: 16px 0 !important;
  width: 100% !important;
  height: auto !important;
  border: none !important;
  padding: 0 !important;
  background: transparent !important;
}

.portal-auth-bg .vp-pw-card .field label {
  display: block;
  font-size: 13px;
  font-weight: 700;
  margin-bottom: 6px;
  color: #17233b;
}

.portal-auth-bg .vp-pw-card .wrap {
  position: relative;
  width: 100%;
}

.portal-auth-bg .vp-pw-card .wrap input {
  width: 100% !important;
  height: 50px !important;
  border: 1.5px solid #cbd5e1 !important;
  border-radius: 8px !important;
  padding: 0 45px !important;
  font-size: 14px !important;
  color: #17233b !important;
  background: #fff !important;
  box-sizing: border-box !important;
}

.portal-auth-bg .vp-pw-card .lock {
  position: absolute;
  left: 14px;
  top: 15px;
  color: #64748b;
  font-size: 16px;
}

.portal-auth-bg .vp-pw-card .eye {
  position: absolute;
  right: 12px;
  top: 14px;
  border: 0;
  background: none;
  color: #64748b;
  cursor: pointer;
  font-size: 18px;
}

.portal-auth-bg .vp-pw-card .submit {
  width: 100%;
  height: 52px;
  border: 0;
  border-radius: 8px;
  background: linear-gradient(90deg, #6260e6, #8343ad);
  color: #fff;
  font-weight: 700;
  font-size: 15px;
  margin-top: 20px;
  cursor: pointer;
  box-shadow: 0 5px 14px rgba(98, 96, 230, 0.35);
}

.portal-auth-bg .vp-pw-card .submit:disabled {
  opacity: 0.5;
  cursor: not-allowed;
  box-shadow: none;
}

/* Common Notices & Error states */
.portal-auth-bg .notice {
  padding: 12px 16px;
  border-radius: 8px;
  margin-bottom: 16px;
  font-size: 13.5px;
  text-align: center;
  line-height: 1.4;
  font-weight: 600;
}

.portal-auth-bg .notice.error {
  background: #fff1f2;
  color: #be123c;
  border: 1px solid #fda4af;
}

.portal-auth-bg .input-error {
  border-color: #ef4444 !important;
  background-color: #fef2f2 !important;
  box-shadow: 0 0 0 3px rgba(239, 68, 68, 0.25) !important;
  animation: shake 0.35s ease-in-out;
}

@keyframes shake {
  0%, 100% { transform: translateX(0); }
  20%, 60% { transform: translateX(-6px); }
  40%, 80% { transform: translateX(6px); }
}

/* Success & Reset Email Cards */
.portal-auth-bg .card.vp-complete-card {
  width: min(620px, 94vw);
  max-width: 620px;
  padding: 42px 48px 38px;
}

.portal-auth-bg .success-icon {
  width: 90px;
  height: 90px;
  border-radius: 50%;
  margin: 10px auto 28px;
  display: grid;
  place-items: center;
  background: linear-gradient(135deg, #00b982, #059669);
  box-shadow: 0 10px 30px rgba(0, 185, 130, 0.35);
}

.portal-auth-bg .info-box {
  background: #f0fdf4;
  border: 1px solid #bbf7d0;
  border-radius: 8px;
  padding: 16px;
  margin: 24px auto;
  max-width: 420px;
  font-size: 13px;
  color: #15803d;
  line-height: 1.6;
  text-align: left;
}

.portal-auth-bg .card.vp-reset-card,
.portal-auth-bg .card.vp-sec-card {
  width: min(620px, 94vw);
  max-width: 620px;
  padding: 42px 48px 38px;
}

.portal-auth-bg .vp-reset-card .form,
.portal-auth-bg .vp-sec-card .form {
  width: 535px;
  max-width: 100%;
  margin: 28px auto 0;
  text-align: left;
}

.portal-auth-bg .vp-reset-card label,
.portal-auth-bg .vp-sec-card label {
  display: block;
  font-weight: 700;
  font-size: 13px;
  margin: 16px 0 8px;
}

.portal-auth-bg .vp-reset-card input,
.portal-auth-bg .vp-sec-card input {
  width: 100%;
  height: 52px;
  border: 1.5px solid #cbd5e1;
  border-radius: 8px;
  padding: 0 16px;
  font-size: 14px;
  box-sizing: border-box;
}

.portal-auth-bg .vp-sec-card .question {
  padding: 14px 16px;
  border: 1px solid #dbe4ef;
  border-radius: 8px;
  background: #f8fafc;
  color: #334155;
  font-weight: 700;
  margin: 8px 0 16px;
}

.portal-auth-bg .vp-reset-card .or {
  text-align: center;
  margin: 14px 0;
  color: #94a3b8;
  font-size: 13px;
}

.portal-auth-bg .vp-reset-card .alt {
  width: 100%;
  height: 50px;
  border: 1px solid #d6dfec;
  border-radius: 8px;
  background: #f8fafc;
  color: #475569;
  font-weight: 700;
  font-size: 14px;
  cursor: pointer;
}

.portal-auth-bg .vp-reset-card .signin,
.portal-auth-bg .vp-sec-card .signin {
  display: block;
  margin: 20px auto 0;
  border: 0;
  background: none;
  color: #6260e6;
  font-weight: 700;
  font-size: 13px;
  cursor: pointer;
}


/* Universal Submit & Primary Button styles */
.portal-auth-bg .submit,
.portal-auth-bg #submitBtn,
.portal-auth-bg .primary,
.portal-auth-bg .signin-btn,
.portal-auth-bg button[type="submit"] {
  width: 100% !important;
  height: 52px !important;
  border: 0 !important;
  border-radius: 8px !important;
  background: linear-gradient(90deg, #6260e6, #8343ad) !important;
  color: #ffffff !important;
  font-weight: 700 !important;
  font-size: 15px !important;
  margin-top: 18px !important;
  cursor: pointer !important;
  box-shadow: 0 5px 14px rgba(98, 96, 230, 0.35) !important;
  display: flex !important;
  align-items: center !important;
  justify-content: center !important;
  box-sizing: border-box !important;
  transition: opacity .2s, transform .1s !important;
}

.portal-auth-bg .submit:hover,
.portal-auth-bg #submitBtn:hover,
.portal-auth-bg .primary:hover,
.portal-auth-bg .verify:hover,
.portal-auth-bg .signin-btn:hover,
.portal-auth-bg button[type="submit"]:hover {
  opacity: 0.95 !important;
}

.portal-auth-bg .submit:disabled,
.portal-auth-bg #submitBtn:disabled,
.portal-auth-bg button[type="submit"]:disabled {
  opacity: 0.5 !important;
  cursor: not-allowed !important;
  box-shadow: none !important;
}


/* Explicit Compact Sizes for 100% Zoom */
.portal-auth-bg .verify,
.portal-auth-bg .vp-otp-card .verify,
.portal-auth-bg .vp-otp-card button[type="submit"] {
  width: 100% !important;
  max-width: 340px !important;
  height: 48px !important;
  border-radius: 8px !important;
  margin: 18px auto 0 !important;
  display: flex !important;
  align-items: center !important;
  justify-content: center !important;
}

.portal-auth-bg .vp-otp-card .otp-container {
  max-width: 340px !important;
  margin: 16px auto 20px !important;
  gap: 8px !important;
}

.portal-auth-bg .vp-signup-card select#role {
  cursor: pointer;
}

/* Explicit Role Select Padding & Layout Increase */
.portal-auth-bg .card.vp-signup-card {
  width: min(680px, 94vw) !important;
  max-width: 680px !important;
  padding: 40px 48px 36px !important;
}

.portal-auth-bg .vp-signup-card .wrap select#role {
  padding-left: 48px !important;
  padding-right: 36px !important;
  cursor: pointer !important;
}

.portal-auth-bg .vp-signup-card .mark {
  z-index: 5 !important;
}

.portal-auth-bg .vp-otp-card .verify,
.portal-auth-bg .vp-otp-card button[type="submit"] {
  max-width: 400px !important;
}



.portal-auth-bg .demo-showcase-sub {
  font-size: 12px;
  color: #64748b;
  margin-bottom: 12px;
}

.portal-auth-bg .demo-roles-grid {
  display: flex;
  flex-wrap: wrap;
  justify-content: center;
  gap: 8px;
}

.portal-auth-bg .demo-role-btn {
  display: inline-flex;
  align-items: center;
  gap: 6px;
  padding: 8px 13px;
  background: #ffffff;
  border: 1.5px solid #cbd5e1;
  border-radius: 8px;
  font-size: 12.5px;
  font-weight: 600;
  color: #1e293b;
  cursor: pointer;
  transition: all 0.18s ease-in-out;
  box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);
}

.portal-auth-bg .demo-role-btn:hover {
  background: #eef2ff;
  border-color: #6366f1;
  color: #4338ca;
  transform: translateY(-2px);
  box-shadow: 0 4px 10px rgba(99, 102, 241, 0.2);
}

.portal-auth-bg .demo-role-btn:active {
  transform: translateY(0);
}

.portal-auth-bg .demo-role-btn .role-icon {
  font-size: 14px;
}

.portal-auth-bg .demo-role-btn .role-name {
  font-size: 12px;
  font-weight: 700;
}

/* ==========================================================================
   Instant Login Buttons at Bottom
   ========================================================================== */
.portal-auth-bg .demo-showcase-panel {
  background: #f8faff;
  border: 1.5px dashed #c7d2fe;
  border-radius: 12px;
  padding: 14px 16px 16px;
  margin: 22px 0 10px;
  text-align: center;
  box-sizing: border-box;
}

.portal-auth-bg .demo-showcase-title {
  font-size: 13px;
  font-weight: 700;
  color: #4f46e5;
  margin-bottom: 12px;
  text-transform: uppercase;
  letter-spacing: 0.6px;
}

.portal-auth-bg .demo-roles-grid {
  display: flex;
  flex-wrap: wrap;
  justify-content: center;
  gap: 8px;
}

.portal-auth-bg .demo-role-btn {
  display: inline-flex;
  align-items: center;
  gap: 6px;
  padding: 8px 12px;
  background: #ffffff;
  border: 1.5px solid #cbd5e1;
  border-radius: 8px;
  font-size: 12px;
  font-weight: 600;
  color: #1e293b;
  cursor: pointer;
  transition: all 0.15s ease-in-out;
  box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);
}

.portal-auth-bg .demo-role-btn:hover {
  background: #eef2ff;
  border-color: #6366f1;
  color: #4338ca;
  transform: translateY(-2px);
  box-shadow: 0 4px 10px rgba(99, 102, 241, 0.2);
}

.portal-auth-bg .demo-role-btn:active {
  transform: translateY(0);
}

.portal-auth-bg .demo-role-btn .role-icon {
  font-size: 14px;
}


/* ==========================================================================
   RBAC & Access Control Styles
   ========================================================================== */
.rolelogin{margin:0 auto;max-width:1060px;padding:48px 24px;width:100%;overflow:auto}
.lhead{display:flex;gap:14px;align-items:center;margin-bottom:26px}.lhead i{width:46px;height:46px;border-radius:14px;background:linear-gradient(135deg,var(--pri),#8b8bf0);display:grid;place-items:center;box-shadow:0 4px 12px rgba(91,91,214,.35)}.lhead i svg{width:24px;height:24px;stroke:#fff;fill:none;stroke-width:2}
.lhead h1{margin:0;font-size:24px;letter-spacing:-.02em}.lhead p{margin:2px 0 0;color:var(--mut)}
.rgrid{display:grid;grid-template-columns:repeat(auto-fill,minmax(310px,1fr));gap:14px}
.rcard{text-align:left;background:#fff;border:1px solid var(--line);border-top:4px solid var(--rc);border-radius:var(--r);padding:16px 18px;cursor:pointer;box-shadow:var(--sh);display:flex;flex-direction:column;gap:8px;transition:transform .12s,box-shadow .12s}
.rcard:hover,.rcard:focus-visible{transform:translateY(-2px);box-shadow:0 8px 24px rgba(20,24,40,.12);outline:none}
.rcard .rt{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.rcard .rt b{font-size:16px;color:var(--rc)}.rcard p{margin:0;color:var(--tx2);font-size:13px;flex:1}
.rcard .rm{display:flex;gap:4px;flex-wrap:wrap}.rcard .rm .chip{padding:1px 8px;font-size:11px;background:var(--rbg);border-color:transparent;color:var(--rc)}
.rcard .rf{display:flex;justify-content:space-between;font-size:12px;color:var(--mut);border-top:1px solid var(--line);padding-top:8px}.rcard .go{color:var(--rc);font-weight:700}
.lnote{color:var(--mut);font-size:12px;margin-top:18px}
.ucard{border:1px solid var(--line);border-radius:12px;padding:10px;background:#fafbfe;margin-top:10px;display:flex;flex-direction:column;gap:6px}
.ucard .ur{display:flex;gap:6px;align-items:center;flex-wrap:wrap}.ucard .ue{font-size:12px;color:var(--mut);overflow:hidden;text-overflow:ellipsis}.ucard .ua{display:flex;gap:12px;font-size:12.5px}.ucard .lnk{cursor:pointer}
.rsw{border:1px solid var(--line2);border-radius:8px;padding:4px 6px;background:#fff;font-size:12px}
table.mx th,table.mx td{text-align:center}table.mx th:first-child,table.mx td:first-child{text-align:left}table.mx .me{background:var(--pri-bg)}
.rtk{color:#b6bac8}.rtk.y{color:#0b7a6c;font-weight:800}
#blocks>.card{margin-bottom:14px}


/* Accessible dark-mode token override. Existing components consume these tokens. */
:root[data-theme="dark"]{--bg:#14161d;--card:#1e212b;--line:#353946;--line2:#464b5b;--tx:#f2f4f8;--tx2:#d5d8e1;--mut:#aeb4c2;--pri:#9b9bff;--pri2:#b2b2ff;--pri-bg:#292a4c;--teal:#42c6b2;--teal-bg:#193d3a;--blue:#86a8ff;--blue-bg:#1d3157;--amber:#ffc35c;--amber-bg:#4a3918;--red:#ff858a;--red-bg:#51282d;--vio:#c28cff;--vio-bg:#39254d}
:root[data-theme="dark"] .side,:root[data-theme="dark"] .tbl,:root[data-theme="dark"] .tbl th,:root[data-theme="dark"] .fld input,:root[data-theme="dark"] .fld select,:root[data-theme="dark"] .fld textarea,:root[data-theme="dark"] .in,:root[data-theme="dark"] .srch,:root[data-theme="dark"] .chk label{background:var(--card);color:var(--tx)}
:root[data-theme="dark"] .tbl td,:root[data-theme="dark"] .tbl th{border-color:var(--line)}
:root[data-theme="dark"] .tbl tr.cl:hover td{background:var(--pri-bg)}


/* Accessible dark-mode token override. Existing components consume these tokens. */
:root[data-theme="dark"]{--bg:#14161d;--card:#1e212b;--line:#353946;--line2:#464b5b;--tx:#f2f4f8;--tx2:#d5d8e1;--mut:#aeb4c2;--pri:#9b9bff;--pri2:#b2b2ff;--pri-bg:#292a4c;--teal:#42c6b2;--teal-bg:#193d3a;--blue:#86a8ff;--blue-bg:#1d3157;--amber:#ffc35c;--amber-bg:#4a3918;--red:#ff858a;--red-bg:#51282d;--vio:#c28cff;--vio-bg:#39254d}
:root[data-theme="dark"] .side,:root[data-theme="dark"] .tbl,:root[data-theme="dark"] .tbl th,:root[data-theme="dark"] .fld input,:root[data-theme="dark"] .fld select,:root[data-theme="dark"] .fld textarea,:root[data-theme="dark"] .in,:root[data-theme="dark"] .srch,:root[data-theme="dark"] .chk label{background:var(--card);color:var(--tx)}
:root[data-theme="dark"] .tbl td,:root[data-theme="dark"] .tbl th{border-color:var(--line)}
:root[data-theme="dark"] .tbl tr.cl:hover td{background:var(--pri-bg)}
:root[data-theme="dark"] .top{background:rgba(30,33,43,.94);color:var(--tx)}
:root[data-theme="dark"] .chip,:root[data-theme="dark"] .btn,:root[data-theme="dark"] .cin input,:root[data-theme="dark"] .m.a,:root[data-theme="dark"] .tk,:root[data-theme="dark"] #drawer,:root[data-theme="dark"] .kpal>div,:root[data-theme="dark"] .rcard,:root[data-theme="dark"] .ucard,:root[data-theme="dark"] .rsw{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .btn:hover,:root[data-theme="dark"] .nav:hover{background:#2a2e3b;color:var(--tx)}
:root[data-theme="dark"] .top h1,:root[data-theme="dark"] .chat,:root[data-theme="dark"] .card,:root[data-theme="dark"] .rcard p{color:var(--tx)}
:root[data-theme="dark"] .md code,:root[data-theme="dark"] .md pre{background:#292e3b;color:var(--tx)}
:root[data-theme="dark"] .hero{background:linear-gradient(120deg,#282a50,#193d3a);border-color:#45486e}
:root[data-theme="dark"] .ksearch kbd,:root[data-theme="dark"] .leaflet-tooltip{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .nav:hover{background:#2a2e3b}


/* Accessible dark-mode token override. Existing components consume these tokens. */
:root[data-theme="dark"]{--bg:#14161d;--card:#1e212b;--line:#353946;--line2:#464b5b;--tx:#f2f4f8;--tx2:#d5d8e1;--mut:#aeb4c2;--pri:#9b9bff;--pri2:#b2b2ff;--pri-bg:#292a4c;--teal:#42c6b2;--teal-bg:#193d3a;--blue:#86a8ff;--blue-bg:#1d3157;--amber:#ffc35c;--amber-bg:#4a3918;--red:#ff858a;--red-bg:#51282d;--vio:#c28cff;--vio-bg:#39254d}
:root[data-theme="dark"] .side,:root[data-theme="dark"] .tbl,:root[data-theme="dark"] .tbl th,:root[data-theme="dark"] .fld input,:root[data-theme="dark"] .fld select,:root[data-theme="dark"] .fld textarea,:root[data-theme="dark"] .in,:root[data-theme="dark"] .srch,:root[data-theme="dark"] .chk label{background:var(--card);color:var(--tx)}
:root[data-theme="dark"] .tbl td,:root[data-theme="dark"] .tbl th{border-color:var(--line)}
:root[data-theme="dark"] .tbl tr.cl:hover td{background:var(--pri-bg)}
:root[data-theme="dark"] .top{background:rgba(30,33,43,.94);color:var(--tx)}
:root[data-theme="dark"] .chip,:root[data-theme="dark"] .btn,:root[data-theme="dark"] .cin input,:root[data-theme="dark"] .m.a,:root[data-theme="dark"] .tk,:root[data-theme="dark"] #drawer,:root[data-theme="dark"] .kpal>div,:root[data-theme="dark"] .rcard,:root[data-theme="dark"] .ucard,:root[data-theme="dark"] .rsw{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .btn:hover,:root[data-theme="dark"] .nav:hover{background:#2a2e3b;color:var(--tx)}
:root[data-theme="dark"] .top h1,:root[data-theme="dark"] .chat,:root[data-theme="dark"] .card,:root[data-theme="dark"] .rcard p{color:var(--tx)}
:root[data-theme="dark"] .md code,:root[data-theme="dark"] .md pre{background:#292e3b;color:var(--tx)}
:root[data-theme="dark"] .hero{background:linear-gradient(120deg,#282a50,#193d3a);border-color:#45486e}
:root[data-theme="dark"] .ksearch kbd,:root[data-theme="dark"] .leaflet-tooltip{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .nav:hover{background:#2a2e3b}

/* Security-recovery secondary actions retain the same visual system as the
   rest of the portal instead of falling back to browser-default buttons. */
.portal-auth-bg .vp-sec-card .bottom{display:flex;justify-content:center;gap:10px;flex-wrap:wrap;margin-top:16px}
.portal-auth-bg .vp-sec-card .bottom button{border:1px solid #cbd5e1;background:#fff;color:#243b62;border-radius:8px;padding:9px 13px;font-weight:650;cursor:pointer;transition:background .15s,border-color .15s,transform .15s}
.portal-auth-bg .vp-sec-card .bottom button:hover{background:#eef2ff;border-color:#7c7cf0;transform:translateY(-1px)}


/* Accessible dark-mode token override. Existing components consume these tokens. */
:root[data-theme="dark"]{--bg:#14161d;--card:#1e212b;--line:#353946;--line2:#464b5b;--tx:#f2f4f8;--tx2:#d5d8e1;--mut:#aeb4c2;--pri:#9b9bff;--pri2:#b2b2ff;--pri-bg:#292a4c;--teal:#42c6b2;--teal-bg:#193d3a;--blue:#86a8ff;--blue-bg:#1d3157;--amber:#ffc35c;--amber-bg:#4a3918;--red:#ff858a;--red-bg:#51282d;--vio:#c28cff;--vio-bg:#39254d}
:root[data-theme="dark"] .side,:root[data-theme="dark"] .tbl,:root[data-theme="dark"] .tbl th,:root[data-theme="dark"] .fld input,:root[data-theme="dark"] .fld select,:root[data-theme="dark"] .fld textarea,:root[data-theme="dark"] .in,:root[data-theme="dark"] .srch,:root[data-theme="dark"] .chk label{background:var(--card);color:var(--tx)}
:root[data-theme="dark"] .tbl td,:root[data-theme="dark"] .tbl th{border-color:var(--line)}
:root[data-theme="dark"] .tbl tr.cl:hover td{background:var(--pri-bg)}
:root[data-theme="dark"] .top{background:rgba(30,33,43,.94);color:var(--tx)}
:root[data-theme="dark"] .chip,:root[data-theme="dark"] .btn,:root[data-theme="dark"] .cin input,:root[data-theme="dark"] .m.a,:root[data-theme="dark"] .tk,:root[data-theme="dark"] #drawer,:root[data-theme="dark"] .kpal>div,:root[data-theme="dark"] .rcard,:root[data-theme="dark"] .ucard,:root[data-theme="dark"] .rsw{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .btn:hover,:root[data-theme="dark"] .nav:hover{background:#2a2e3b;color:var(--tx)}
:root[data-theme="dark"] .top h1,:root[data-theme="dark"] .chat,:root[data-theme="dark"] .card,:root[data-theme="dark"] .rcard p{color:var(--tx)}
:root[data-theme="dark"] .md code,:root[data-theme="dark"] .md pre{background:#292e3b;color:var(--tx)}
:root[data-theme="dark"] .hero{background:linear-gradient(120deg,#282a50,#193d3a);border-color:#45486e}
:root[data-theme="dark"] .ksearch kbd,:root[data-theme="dark"] .leaflet-tooltip{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .nav:hover{background:#2a2e3b}

/* Security-recovery secondary actions retain the same visual system as the
   rest of the portal instead of falling back to browser-default buttons. */
.portal-auth-bg .vp-sec-card .bottom{display:flex;justify-content:center;gap:10px;flex-wrap:wrap;margin-top:16px}
.portal-auth-bg .vp-sec-card .bottom button{border:1px solid #cbd5e1;background:#fff;color:#243b62;border-radius:8px;padding:9px 13px;font-weight:650;cursor:pointer;transition:background .15s,border-color .15s,transform .15s}
.portal-auth-bg .vp-sec-card .bottom button:hover{background:#eef2ff;border-color:#7c7cf0;transform:translateY(-1px)}


/* Accessible dark-mode token override. Existing components consume these tokens. */
:root[data-theme="dark"]{--bg:#14161d;--card:#1e212b;--line:#353946;--line2:#464b5b;--tx:#f2f4f8;--tx2:#d5d8e1;--mut:#aeb4c2;--pri:#9b9bff;--pri2:#b2b2ff;--pri-bg:#292a4c;--teal:#42c6b2;--teal-bg:#193d3a;--blue:#86a8ff;--blue-bg:#1d3157;--amber:#ffc35c;--amber-bg:#4a3918;--red:#ff858a;--red-bg:#51282d;--vio:#c28cff;--vio-bg:#39254d}
:root[data-theme="dark"] .side,:root[data-theme="dark"] .tbl,:root[data-theme="dark"] .tbl th,:root[data-theme="dark"] .fld input,:root[data-theme="dark"] .fld select,:root[data-theme="dark"] .fld textarea,:root[data-theme="dark"] .in,:root[data-theme="dark"] .srch,:root[data-theme="dark"] .chk label{background:var(--card);color:var(--tx)}
:root[data-theme="dark"] .tbl td,:root[data-theme="dark"] .tbl th{border-color:var(--line)}
:root[data-theme="dark"] .tbl tr.cl:hover td{background:var(--pri-bg)}
:root[data-theme="dark"] .top{background:rgba(30,33,43,.94);color:var(--tx)}
:root[data-theme="dark"] .chip,:root[data-theme="dark"] .btn,:root[data-theme="dark"] .cin input,:root[data-theme="dark"] .m.a,:root[data-theme="dark"] .tk,:root[data-theme="dark"] #drawer,:root[data-theme="dark"] .kpal>div,:root[data-theme="dark"] .rcard,:root[data-theme="dark"] .ucard,:root[data-theme="dark"] .rsw{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .btn:hover,:root[data-theme="dark"] .nav:hover{background:#2a2e3b;color:var(--tx)}
:root[data-theme="dark"] .top h1,:root[data-theme="dark"] .chat,:root[data-theme="dark"] .card,:root[data-theme="dark"] .rcard p{color:var(--tx)}
:root[data-theme="dark"] .md code,:root[data-theme="dark"] .md pre{background:#292e3b;color:var(--tx)}
:root[data-theme="dark"] .hero{background:linear-gradient(120deg,#282a50,#193d3a);border-color:#45486e}
:root[data-theme="dark"] .ksearch kbd,:root[data-theme="dark"] .leaflet-tooltip{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .nav:hover{background:#2a2e3b}

/* Small avatar inside the existing account card; it does not change the card's
   controls or navigation layout. */
.uavatar{width:28px;height:28px;border-radius:50%;object-fit:cover;border:1px solid var(--line);background:var(--pri-bg);display:inline-grid;place-items:center;flex:0 0 28px}
.uph{color:var(--pri);font-size:12px;font-weight:800}

/* Security-recovery secondary actions retain the same visual system as the
   rest of the portal instead of falling back to browser-default buttons. */
.portal-auth-bg .vp-sec-card .bottom{display:flex;justify-content:center;gap:10px;flex-wrap:wrap;margin-top:16px}
.portal-auth-bg .vp-sec-card .bottom button{border:1px solid #cbd5e1;background:#fff;color:#243b62;border-radius:8px;padding:9px 13px;font-weight:650;cursor:pointer;transition:background .15s,border-color .15s,transform .15s}
.portal-auth-bg .vp-sec-card .bottom button:hover{background:#eef2ff;border-color:#7c7cf0;transform:translateY(-1px)}


/* Accessible dark-mode token override. Existing components consume these tokens. */
:root[data-theme="dark"]{--bg:#14161d;--card:#1e212b;--line:#353946;--line2:#464b5b;--tx:#f2f4f8;--tx2:#d5d8e1;--mut:#aeb4c2;--pri:#9b9bff;--pri2:#b2b2ff;--pri-bg:#292a4c;--teal:#42c6b2;--teal-bg:#193d3a;--blue:#86a8ff;--blue-bg:#1d3157;--amber:#ffc35c;--amber-bg:#4a3918;--red:#ff858a;--red-bg:#51282d;--vio:#c28cff;--vio-bg:#39254d}
:root[data-theme="dark"] .side,:root[data-theme="dark"] .tbl,:root[data-theme="dark"] .tbl th,:root[data-theme="dark"] .fld input,:root[data-theme="dark"] .fld select,:root[data-theme="dark"] .fld textarea,:root[data-theme="dark"] .in,:root[data-theme="dark"] .srch,:root[data-theme="dark"] .chk label{background:var(--card);color:var(--tx)}
:root[data-theme="dark"] .tbl td,:root[data-theme="dark"] .tbl th{border-color:var(--line)}
:root[data-theme="dark"] .tbl tr.cl:hover td{background:var(--pri-bg)}
:root[data-theme="dark"] .top{background:rgba(30,33,43,.94);color:var(--tx)}
:root[data-theme="dark"] .chip,:root[data-theme="dark"] .btn,:root[data-theme="dark"] .cin input,:root[data-theme="dark"] .m.a,:root[data-theme="dark"] .tk,:root[data-theme="dark"] #drawer,:root[data-theme="dark"] .kpal>div,:root[data-theme="dark"] .rcard,:root[data-theme="dark"] .ucard,:root[data-theme="dark"] .rsw{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .btn:hover,:root[data-theme="dark"] .nav:hover{background:#2a2e3b;color:var(--tx)}
:root[data-theme="dark"] .top h1,:root[data-theme="dark"] .chat,:root[data-theme="dark"] .card,:root[data-theme="dark"] .rcard p{color:var(--tx)}
:root[data-theme="dark"] .md code,:root[data-theme="dark"] .md pre{background:#292e3b;color:var(--tx)}
:root[data-theme="dark"] .hero{background:linear-gradient(120deg,#282a50,#193d3a);border-color:#45486e}
:root[data-theme="dark"] .ksearch kbd,:root[data-theme="dark"] .leaflet-tooltip{background:var(--card);color:var(--tx);border-color:var(--line2)}
:root[data-theme="dark"] .nav:hover{background:#2a2e3b}

/* Small avatar inside the existing account card; it does not change the card's
   controls or navigation layout. */
.uavatar{width:28px;height:28px;border-radius:50%;object-fit:cover;border:1px solid var(--line);background:var(--pri-bg);display:inline-grid;place-items:center;flex:0 0 28px}
.uph{color:var(--pri);font-size:12px;font-weight:800}
.profile-photo-preview{display:flex;align-items:center;justify-content:center;padding:8px 0}
.profile-photo-preview img{width:132px;height:132px;border-radius:50%;object-fit:cover;border:3px solid var(--pri-bg);box-shadow:0 4px 16px rgba(40,45,70,.16);background:var(--card)}

/* Security-recovery secondary actions retain the same visual system as the
   rest of the portal instead of falling back to browser-default buttons. */
.portal-auth-bg .vp-sec-card .bottom{display:flex;justify-content:center;gap:10px;flex-wrap:wrap;margin-top:16px}
.portal-auth-bg .vp-sec-card .bottom button{border:1px solid #cbd5e1;background:#fff;color:#243b62;border-radius:8px;padding:9px 13px;font-weight:650;cursor:pointer;transition:background .15s,border-color .15s,transform .15s}
.portal-auth-bg .vp-sec-card .bottom button:hover{background:#eef2ff;border-color:#7c7cf0;transform:translateY(-1px)}


In [ ]:
%%writefile /content/app/auth.py
"""Authentication, JWT tokens, password hashing, and user identity resolution.
Fulfills Milestone 1 integration into FranchiseOps:
1. Signup & login logic
2. FastAPI auth routes support
3. bcrypt + PBKDF2 password hashing & verification
4. JWT token generation (HS256) and decoding
5. Multi-user user_of(request) resolver replacing single-user admin dict
6. 5-attempt account lockout mechanism (15-minute cooldown)
7. Logout and last_login tracking
8. Invalidation of active sessions/tokens for locked accounts
"""
import os, time, hmac, hashlib, secrets, datetime as dt
from typing import Optional, Dict, Any
import jwt
from fastapi import Request, HTTPException
import config
from config import JWT_SECRET_KEY, ADMIN_EMAIL_ID, ADMIN_PASSWORD
import db

MAX_LOGIN_ATTEMPTS = 5
LOCKOUT_SECONDS = 900  # 15 minutes lockout
JWT_ALGORITHM = "HS256"
JWT_EXPIRATION_HOURS = 12

def hash_password(password: str) -> str:
    """Hashes a plaintext password using bcrypt, falling back to PBKDF2-HMAC-SHA256."""
    if not password:
        return ""
    try:
        import bcrypt
        return bcrypt.hashpw(password.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")
    except Exception:
        salt = secrets.token_hex(16)
        k = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt.encode("utf-8"), 100000)
        return f"pbkdf2_sha256${salt}${k.hex()}"

def check_password(password: str, hashed: str) -> bool:
    """Verifies a plaintext password against a stored bcrypt or PBKDF2 hash."""
    if not password or not hashed:
        return False
    if hashed.startswith("$2a$") or hashed.startswith("$2b$") or hashed.startswith("$2y$"):
        try:
            import bcrypt
            return bcrypt.checkpw(password.encode("utf-8"), hashed.encode("utf-8"))
        except Exception:
            return False
    if hashed.startswith("pbkdf2_sha256$"):
        try:
            parts = hashed.split("$")
            if len(parts) == 3:
                salt, key = parts[1], parts[2]
                k = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt.encode("utf-8"), 100000)
                return hmac.compare_digest(k.hex(), key)
        except Exception:
            return False
    # Backward compatibility with sha256 or plaintext
    try:
        if hmac.compare_digest(hashlib.sha256(password.encode("utf-8")).hexdigest(), hashed):
            return True
    except Exception:
        pass
    return password == hashed

def hash_security_answer(answer: str) -> str:
    """Normalizes and hashes security answer using SHA-256."""
    if not answer:
        return ""
    norm = str(answer).strip().lower().encode("utf-8")
    return hashlib.sha256(norm).hexdigest()

def check_security_answer(answer: str, hashed_answer: str) -> bool:
    """Compares normalized security answer with stored hash."""
    if not answer or not hashed_answer:
        return False
    norm_hash = hash_security_answer(answer)
    return hmac.compare_digest(norm_hash, str(hashed_answer)) or norm_hash == str(hashed_answer) or str(answer).strip().lower() == str(hashed_answer).strip().lower()

def create_access_token(user_dict: dict, expires_delta: Optional[dt.timedelta] = None) -> str:
    """Issues a signed JWT access token containing subject, email, username, role."""
    now = dt.datetime.now(dt.timezone.utc)
    exp = now + (expires_delta or dt.timedelta(hours=JWT_EXPIRATION_HOURS))
    payload = {
        "sub": str(user_dict.get("email") or user_dict.get("username")),
        "email": user_dict.get("email"),
        "username": user_dict.get("username"),
        "role": user_dict.get("role", "Customer"),
        "iat": int(now.timestamp()),
        "exp": int(exp.timestamp()),
        "type": "login"
    }
    return jwt.encode(payload, JWT_SECRET_KEY, algorithm=JWT_ALGORITHM)

def decode_access_token(token: str) -> Optional[Dict[str, Any]]:
    """Decodes and validates a JWT token."""
    if not token or not JWT_SECRET_KEY:
        return None
    try:
        payload = jwt.decode(token, JWT_SECRET_KEY, algorithms=[JWT_ALGORITHM])
        return payload
    except Exception:
        return None

def extract_token_from_request(request: Request) -> Optional[str]:
    """Extracts auth token from Authorization header, cookies, or query parameters."""
    auth_header = request.headers.get("Authorization") or request.headers.get("authorization")
    if auth_header and auth_header.startswith("Bearer "):
        return auth_header.split(" ", 1)[1].strip()

    # Check cookies
    for c_key in ("token", "access_token", "jwt", "session"):
        c_val = request.cookies.get(c_key)
        if c_val:
            return c_val.strip()

    # Check query params (e.g. for EventSource /api/stream)
    q_token = request.query_params.get("token")
    if q_token:
        return q_token.strip()

    return None

def user_of(request: Request) -> Dict[str, Any]:
    """Resolves the current authenticated user from request credentials.
    Replaces static localuser.
    TASK 8: Checks if user account is locked. If locked, actively rejects with 403,
    ensuring locked users' existing tokens stop working immediately!
    """
    token = extract_token_from_request(request)
    if not token:
        raise HTTPException(status_code=401, detail="Authentication required. Please log in.")

    payload = decode_access_token(token)
    if not payload:
        raise HTTPException(status_code=401, detail="Invalid or expired session token. Please log in again.")

    identifier = payload.get("email") or payload.get("sub") or payload.get("username")
    if not identifier:
        raise HTTPException(status_code=401, detail="Malformed token payload.")

    user = db.get_user_by_identifier(identifier)
    if not user:
        raise HTTPException(status_code=401, detail="User account no longer exists.")

    # Task 8: Invalidate tokens for locked users
    if db.is_user_locked(user):
        raise HTTPException(status_code=403, detail="Account is temporarily locked due to excessive failed attempts. Existing tokens are invalidated.")

    if user.get("account_status") in ("disabled", "suspended", "locked"):
        raise HTTPException(status_code=403, detail="Account is disabled or suspended.")

    return {
        "id": user.get("id"),
        "email": user.get("email"),
        "username": user.get("username"),
        "role": user.get("role", "Customer"),
        "account_status": user.get("account_status", "active"),
        "last_login": user.get("last_login"),
        "created_at": user.get("created_at"),
        "outlet_id": user.get("outlet_id"),
        "password_hash": user.get("password_hash"),
        "profile_picture": user.get("profile_picture")
    }

USER = {"email": ADMIN_EMAIL_ID, "username": "Admin", "role": "Admin"}


# -----------------------------------------------------------------------------
# Milestone 1: OTP Generation, Hashing & SMTP / Simulation Delivery
# -----------------------------------------------------------------------------
def generate_otp() -> str:
    """Generates a cryptographically secure 6-digit numeric passcode."""
    return str(secrets.randbelow(900000) + 100000)

def make_otp_token(email: str, otp: str, purpose: str = "password_reset_otp", minutes: int = 10) -> str:
    """Issues a signed JWT binding the user's email to the hashed OTP passcode."""
    now = dt.datetime.now(dt.timezone.utc)
    payload = {
        "sub": email.lower().strip(),
        "otp_hash": hash_password(otp),
        "purpose": purpose,
        "type": purpose,
        "iat": now,
        "exp": now + dt.timedelta(minutes=minutes)
    }
    return jwt.encode(payload, JWT_SECRET_KEY, algorithm=JWT_ALGORITHM)

def verify_otp_token(token: str, input_otp: str, email: str, purpose: str = "password_reset_otp") -> tuple:
    """Verify an OTP; existing accounts share the login lockout counter."""
    email = str(email or "").lower().strip()
    user = db.get_user_by_identifier(email)
    if user and db.is_user_locked(user):
        return False, "Account is locked for 15 minutes after five failed attempts. Please wait before trying again."
    if not token:
        return False, "This verification session has expired. Please request a new code."
    clean_otp = str(input_otp or "").strip()
    if not clean_otp or len(clean_otp) != 6 or not clean_otp.isdigit():
        return False, "Please enter the complete 6-digit verification code."
    try:
        payload = jwt.decode(token, JWT_SECRET_KEY, algorithms=[JWT_ALGORITHM])
        if payload.get("sub") != email or (payload.get("purpose") != purpose and payload.get("type") != purpose):
            return False, "This verification code does not match the active session."
        if not check_password(clean_otp, payload.get("otp_hash")):
            if user:
                result = db.record_login_failure(email, MAX_LOGIN_ATTEMPTS, LOCKOUT_SECONDS)
                if result.get("locked"):
                    return False, "Account locked for 15 minutes after five failed attempts."
                return False, f"Invalid verification code. {MAX_LOGIN_ATTEMPTS - result.get('failed_attempts', 0)} attempt(s) remaining."
            # A new signup has no account yet, so retain the OTP-only counter.
            db.record_failed_otp_attempt(email); count = db.get_otp_attempts(email)
            return False, f"Invalid verification code. {max(0, MAX_LOGIN_ATTEMPTS - count)} attempt(s) remaining."
        if not user:
            db.clear_otp_attempts(email)
        return True, "Verification successful."
    except jwt.ExpiredSignatureError:
        return False, "This verification code has expired. Please request a new code."
    except Exception:
        return False, "Invalid verification token. Please request a new code."

def get_secret(key, default=""):
    """Reads credentials from Google Colab Secrets (key icon), environment variables, or fallback."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val is not None:
            return str(val)
    except Exception:
        pass
    val = os.getenv(key)
    if val:
        return val
    return default

def send_otp_email(to_email: str, otp: str, is_signup: bool = False) -> tuple:
    """Sends OTP email via Gmail SMTP using the exact Verify Portal HTML template.
    Reads credentials from Colab Secrets: EMAIL_ID / EMAIL_ADDRESS and EMAIL_PASSWORD.
    """
    import smtplib
    from email.utils import formatdate, make_msgid
    from email.mime.text import MIMEText
    from email.mime.multipart import MIMEMultipart

    sender_email = get_secret("EMAIL_ADDRESS") or get_secret("EMAIL_ID", "")
    email_password = get_secret("EMAIL_PASSWORD", "")
    otp_expiry_minutes = 5

    if not sender_email or not email_password:
        print(f"[OTP WARNING] Email settings missing. Add EMAIL_ID and EMAIL_PASSWORD to Colab Secrets (key icon). Code for {to_email}: {otp}")
        return True, "Verification code generated (Check Colab output).", otp

    if is_signup:
        badge_text = "Account Verification"
        heading = "Verify Your Account"
        subject = "Your Verify Portal verification code"
        intro = (
            "Thank you for registering! Please use the 6-digit verification code below "
            "to complete your account setup and activate your portal access."
        )
        security_note = "If you didn't attempt to create an account, you can safely ignore this email."
        icon_symbol = "✦"
    else:
        badge_text = "Password Reset"
        heading = "Reset Your Password"
        subject = "Your Verify Portal password reset code"
        intro = (
            "We received a request to reset the password for your account. "
            "Please use the 6-digit verification code below to authorize this change."
        )
        security_note = "If you didn't request a password reset, you can safely ignore this email. Your password will remain unchanged."
        icon_symbol = "🔒"

    msg = MIMEMultipart("alternative")
    msg["From"] = f"Verify Portal <{sender_email}>"
    msg["To"] = to_email
    msg["Subject"] = subject
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = sender_email

    text_body = (
        f"Hi,\n\n"
        f"Your verification code is:\n\n"
        f"{otp}\n\n"
        f"This code expires in {otp_expiry_minutes} minutes.\n\n"
        f"Don't reply to this email or share this code with anyone. Verify Portal will never ask you for it.\n\n"
        f"{security_note}\n\n"
        f"--\nVerify Portal\n"
    )

    html_body = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{subject}</title>
</head>
<body style="margin: 0; padding: 0; background-color: #f1f3f9; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; -webkit-font-smoothing: antialiased;">
  <table role="presentation" width="100%" border="0" cellspacing="0" cellpadding="0" style="background-color: #f1f3f9; padding: 36px 12px;">
    <tr>
      <td align="center">
        <!-- Main Card Container -->
        <table role="presentation" width="100%" border="0" cellspacing="0" cellpadding="0" style="max-width: 540px; background-color: #ffffff; border-radius: 16px; border: 1px solid #e2e8f0; box-shadow: 0 12px 36px rgba(98, 96, 230, 0.08); overflow: hidden;">

          <!-- Top Accent Banner -->
          <tr>
            <td style="height: 6px; background: linear-gradient(90deg, #6260e6, #8343ad);"></td>
          </tr>

          <!-- Main Content -->
          <tr>
            <td style="padding: 42px 38px 36px; text-align: center;">

              <!-- Glowing Icon Circle -->
              <div style="display: inline-block; width: 64px; height: 64px; line-height: 64px; border-radius: 50%; background: linear-gradient(135deg, #6260e6 0%, #8343ad 100%); box-shadow: 0 8px 24px rgba(98, 96, 230, 0.35); margin-bottom: 20px; text-align: center;">
                <span style="font-size: 28px; color: #ffffff; vertical-align: middle;">{icon_symbol}</span>
              </div>

              <!-- Brand Name -->
              <div style="font-size: 13px; font-weight: 800; color: #6260e6; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 10px;">
                VERIFY PORTAL
              </div>

              <!-- Sleek Badge -->
              <div style="margin-bottom: 16px;">
                <span style="display: inline-block; background-color: #ede9fe; color: #5b21b6; border: 1px solid #ddd6fe; border-radius: 20px; padding: 4px 15px; font-size: 11px; font-weight: 700; letter-spacing: 0.8px; text-transform: uppercase;">
                  {badge_text}
                </span>
              </div>

              <!-- Main Heading -->
              <h1 style="margin: 0 0 12px; font-size: 25px; font-weight: 800; color: #1e1b4b; letter-spacing: -0.5px; line-height: 1.3;">
                {heading}
              </h1>

              <!-- Intro Subtitle -->
              <p style="margin: 0 auto 20px; font-size: 14.5px; color: #64748b; line-height: 1.6; max-width: 440px;">
                {intro}
              </p>

              <!-- Verification Code Container (Formatted for Gmail NLP Extraction) -->
              <div style="background: linear-gradient(180deg, #f8fafc 0%, #f1f5f9 100%); border: 2px dashed #6260e6; border-radius: 14px; padding: 24px 20px; margin: 24px 0; text-align: center;">
                <p style="margin: 0 0 12px; font-size: 14px; font-weight: 700; color: #1e1b4b; text-transform: uppercase; letter-spacing: 1px;">
                  Your verification code is:
                </p>

                <!-- Isolated Digits Box -->
                <div style="display: inline-block; background-color: #ffffff; border: 1.5px solid #e0e7ff; border-radius: 10px; padding: 14px 32px; box-shadow: 0 4px 14px rgba(98, 96, 230, 0.08); margin-bottom: 12px; user-select: all; -webkit-user-select: all; -moz-user-select: all;">
                  <span id="otpCode" style="font-family: 'SF Mono', Consolas, 'Liberation Mono', Menlo, Courier, monospace; font-size: 38px; font-weight: 800; letter-spacing: 8px; color: #1e1b4b; user-select: all; -webkit-user-select: all; -moz-user-select: all;">{otp}</span>
                </div>

                <div style="font-size: 13px; color: #64748b; margin-top: 4px;">
                  This code expires in <strong>{otp_expiry_minutes} minutes</strong>.
                </div>
              </div>

              <!-- Expiry Alert -->
              <table role="presentation" width="100%" border="0" cellspacing="0" cellpadding="0" style="background-color: #fdf4ff; border: 1px solid #f0abfc; border-radius: 10px; margin-bottom: 24px;">
                <tr>
                  <td style="padding: 12px 16px; text-align: left;">
                    <span style="font-size: 16px; vertical-align: middle; margin-right: 8px;">⏱</span>
                    <span style="font-size: 13px; font-weight: 600; color: #86198f; vertical-align: middle;">
                      This code is valid for <strong>{otp_expiry_minutes} minutes</strong>. Please do not share it.
                    </span>
                  </td>
                </tr>
              </table>

              <!-- Divider -->
              <div style="height: 1px; background-color: #e2e8f0; margin: 24px 0;"></div>

              <!-- Security Information -->
              <div style="text-align: left; font-size: 12.5px; line-height: 1.6; color: #64748b;">
                <p style="margin: 0 0 8px;">
                  🔒 <strong>Security Notice:</strong> Verify staff will never ask for your verification code or password.
                </p>
                <p style="margin: 0; color: #94a3b8;">
                  {security_note}
                </p>
              </div>

            </td>
          </tr>

          <!-- Footer -->
          <tr>
            <td style="background-color: #f8fafc; border-top: 1px solid #edf2f7; padding: 20px 36px; text-align: center; font-size: 11.5px; color: #94a3b8; line-height: 1.5;">
              <strong style="color: #64748b;">Verify Authentication Services</strong><br>
              Cryptographic One-Time Passcode Verification • Automatic Notification<br>
              This is an automated security message. Please do not reply directly.
            </td>
          </tr>
        </table>
      </td>
    </tr>
  </table>
</body>
</html>
"""

    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(sender_email, email_password)
            server.sendmail(sender_email, [to_email], msg.as_string())

        db.clear_otp_attempts(to_email)
        print(f"[OTP EMAIL] Successfully delivered OTP email to {to_email} via Gmail SMTP.")
        return True, "Email sent successfully.", None

    except Exception as e:
        print(f"[OTP EMAIL EXCEPTION] Failed sending via Gmail SMTP: {e}. Simulated OTP for {to_email}: {otp}")
        return True, f"Email delivery note ({e}). Code generated.", otp


In [ ]:
import base64, io, tarfile
B64 = (
'...'
)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64)), mode='r:gz').extractall('/content/app/static')
print('vendor files ready')


In [ ]:
%%writefile /content/app/test_copilot.py
"""AI Copilot Enhancement - automated tests. Run:  python test_copilot.py
Runs every new question (clean wording, alternative wording, misspelled), follow-ups, role restrictions, SQL validation, unsupported and empty-result
handling, and one end-to-end HTTP check. Writes COPILOT_TEST_REPORT.md with the SQL, row count and answer for each question."""
import os, sys, json, time, datetime
os.environ.setdefault("APP_KIND", "Franchise"); os.environ.setdefault("NO_WORKER", "1"); os.environ.setdefault("RATE_LIMIT_OFF", "1")
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config, live_engine as LE, domain_franchise as dom
from db import init_db; from seed_data import seed_all
init_db(); seed_all(); LE.init_live_tables()
try:                                                    # make sure the live tables and one agent cycle exist (the worker normally does this)
    dom.bootstrap()
    import sqlite3
    if sqlite3.connect(config.DB_PATH).execute("SELECT COUNT(*) FROM agent_runs").fetchone()[0] == 0: LE.run_cycle(dom, "manual", "")
except Exception as e: print("bootstrap note:", e)
import qa as QA, analyst as AN, intent_router as IR, views_core as VC
DEC = VC.decision()
COUNTS = {t: int(QA.Q(f'SELECT COUNT(*) n FROM {t}').n[0]) for t in ('outlets', 'staff', 'inventory')}   # taken before the HTTP test restarts the app (which re-seeds the base tables)
PASS, FAIL = [], []
REPORT = []
def check(name, ok, detail=""):
    (PASS if ok else FAIL).append(name)
    if not ok: print("  FAIL:", name, detail)
def ask(q, role="Admin", scope=None, state=None): return IR.respond("Franchise", q, role, scope, state, DEC, dom)
def preview(r, n=3):
    t = r["text"]; i = t.find("\n\n|")
    lead = (t[:i] if i > 0 else t).replace("\n", " ")
    return lead

# ------------------------------------------------------------------ A. the new question catalogue: 3 phrasings each
CASES = [  # key, [clean, alternative wording, misspelled]
 ("revenue_by_tier", ["Revenue by tier", "Which tier earns the most?", "revnue by teir"]),
 ("revenue_by_city", ["Which city earns the most revenue?", "Which city has the lowest revenue?", "revenu per citty"]),
 ("outlets_per_tier", ["How many outlets are in each tier?", "Number of outlets per tier", "how mny outlets in each teir"]),
 ("outlets_per_city", ["How many outlets are in each city?", "City wise outlet count", "outlets per citty"]),
 ("net_margin", ["Which outlets have the highest net margin?", "Which stores are most profitable?", "profit margn by outlet"]),
 ("loss_making", ["Which outlets are loss making?", "Which shops are making a loss?", "unprofitable outlts"]),
 ("csat_threshold", ["Which outlets have customer satisfaction below 3.5?", "Outlets with satisfaction under 3.2", "outlets wth custmer satisfacton above 4"]),
 ("attrition_staff", ["Which staff are at high risk of leaving?", "Who might resign?", "staf at high attrtion risk"]),
 ("overtime_outlets", ["Which outlets have the most overtime?", "Where are staff overworked?", "overtme by outlet"]),
 ("salary_by_role", ["What is the average salary by role?", "Compare pay across roles", "avrage salry by role"]),
 ("staff_per_role", ["How many staff are there per role?", "Team composition by role", "how many employees per role"]),
 ("staff_satisfaction", ["Which outlets have the lowest job satisfaction among staff?", "Where are employees least satisfied?", "job satisfction by outlet"]),
 ("attendance_summary", ["Show the attendance summary", "How many staff were absent?", "attendnce summary"]),
 ("low_stock_by_outlet", ["Which outlets have the most items below reorder level?", "Show low stock items by outlet", "outlets with items belw reorder level"]),
 ("stockout_by_category", ["Which item category has the highest stockout risk?", "Stock risk by category", "stockout risk by categry"]),
 ("longest_lead_time", ["Which items have the longest supplier lead time?", "Slowest items to arrive", "suplier lead tim"]),
 ("open_purchase_orders", ["Show pending purchase orders", "Which orders are on the way?", "pendng purchase ordrs"]),
 ("campaign_roi", ["Which campaigns have the best ROI?", "Show the worst performing campaigns", "campaing roi"]),
 ("channel_performance", ["Which marketing channel works best?", "Which channel works best?", "compare roi by chanel"]),
 ("active_campaigns", ["Which campaigns are running now?", "Show active promotions", "campaigns runing now"]),
 ("marketing_spend", ["What is the marketing spend by channel?", "How much did we spend on campaigns?", "marketng budget by channel"]),
 ("negative_feedback", ["Show recent negative feedback", "Which customers complained recently?", "negativ feedbak"]),
 ("best_rated_outlets", ["Which outlets have the best customer ratings?", "Which outlets are rated the lowest?", "best rated outlts"]),
 ("failed_audits", ["Which audits failed?", "Show audits that need action", "failed audts"]),
 ("audit_by_category", ["Which audit category is weakest?", "Compliance score by category", "audit categry weakest"]),
 ("violations_by_outlet", ["Which outlets have the most audit violations?", "Where are the most violations?", "outlets with most violaions"]),
 ("tickets_by_severity", ["How many open tickets are there by severity?", "Open issues by type", "tickets by severty"]),
 ("critical_tickets", ["Show the open critical tickets", "What needs urgent action?", "critcal tickets"]),
 ("sales_by_day", ["Show daily sales for the last 7 days", "What were sales in the last 14 days?", "daly sales for last 7 days"]),
 ("top_sellers_period", ["Which outlets sold the most in the last 30 days?", "Top selling outlets this week", "outlets that sold the most in last 30 days"]),
]
print(f"A. catalogue: {len(CASES)} queries x 3 phrasings"); REPORT.append("## A. New whitelisted questions (clean, alternative wording, misspelled)\n")
ANALYST_Q = [("Compare OUT-007 and OUT-012", "analyst"), ("Give me an overview of the business", "analyst"), ("How is Mumbai doing?", "analyst"), ("Tell me about OUT-007", "analyst")]
n_q = 0
for key, qs in CASES:
    REPORT.append(f"### `{key}` - {QA.WL_BY_KEY[key]['title']}\n")
    for q in qs:
        r = ask(q); n_q += 1
        okk = r["intent"].get("key") == key and r["status"] in ("ok", "empty") and (r["status"] == "empty" or (r["rows"] or 0) > 0) and r["sql"].strip().upper().startswith("SELECT") and r["validated"]
        check(f"A:{key}:{q}", okk, f"got key={r['intent'].get('key')} status={r['status']} rows={r['rows']}")
        tag = "PASS" if okk else "FAIL"; fixes = ", ".join(f"{a}→{b}" for a, b in r["corrections"])
        REPORT.append(f"**Q:** {q}  \n**Result:** {tag} · status `{r['status']}` · **{r['rows']} rows**" + (f" · spelling fixed: {fixes}" if fixes else "") + f"  \n**Answer:** {preview(r)}  \n```sql\n{r['sql']}\n```\n")
REPORT.append("### Multi-query analyst questions\n")
for q, key in ANALYST_Q:
    r = ask(q); n_q += 1; okk = r["status"] == "ok" and (r["rows"] or 0) > 0 and "SELECT" in r["sql"]; check(f"A:analyst:{q}", okk)
    REPORT.append(f"**Q:** {q}  \n**Result:** {'PASS' if okk else 'FAIL'} · **{r['rows']} rows**  \n**Answer:** {preview(r)}  \n```sql\n{r['sql'][:1800]}\n```\n")

# ------------------------------------------------------------------ B. parameters in the question
print("B. parameters"); REPORT.append("## B. Filters and parameters picked up from the question\n")
B = [("Revenue by city for Mall outlets", lambda r: "o.tier = 'Mall'" in r["sql"]), ("Which outlets have the highest net margin in Mumbai?", lambda r: "o.location = 'Mumbai'" in r["sql"]),
     ("Top 3 outlets by net margin", lambda r: r["rows"] == 3), ("Show the 15 worst performing campaigns", lambda r: r["rows"] == 15 and "ASC" in r["sql"]),
     ("Which staff have attrition risk above 80%?", lambda r: "> 0.8" in r["sql"]), ("Outlets with customer satisfaction below 3.3 in Mumbai", lambda r: "< 3.3" in r["sql"] and "Mumbai" in r["sql"]),
     ("Show daily sales for the last 3 days", lambda r: r["rows"] <= 3), ("Which outlets are loss making in Bangalore?", lambda r: "Bengaluru" in r["sql"]),
     ("Show pending purchase orders for OUT-007", lambda r: "OUT-007" in r["sql"] or r["status"] == "empty")]
for q, f in B:
    r = ask(q); okk = bool(f(r)); check("B:" + q, okk, r["sql"]); n_q += 1
    REPORT.append(f"**Q:** {q}  \n**Result:** {'PASS' if okk else 'FAIL'} · **{r['rows']} rows**  \n```sql\n{r['sql']}\n```\n")

# ------------------------------------------------------------------ C. questions that worked before still work
print("C. regression"); REPORT.append("## C. Existing questions still work\n")
C = ["Which outlets are at highest risk?", "Why is OUT-007 flagged?", "How is inventory holding up?", "Top 5 outlets by revenue", "Which stores have the lowest customer satisfaction?", "Which stock needs immediate refill?", "Which state has the most outlets?",
     "How many staff are there?", "Average salary of staff", "Hi", "Thanks"]
for q in C:
    r = ask(q); okk = r["status"] in ("ok", "greeting"); check("C:" + q, okk, r["text"][:80]); n_q += 1
    REPORT.append(f"**Q:** {q}  \n**Result:** {'PASS' if okk else 'FAIL'} · status `{r['status']}` · rows {r['rows']}  \n**Answer:** {preview(r)[:300]}\n")

# ------------------------------------------------------------------ D. follow-up questions
print("D. follow-ups"); REPORT.append("## D. Follow-up questions (context carried between turns)\n")
def convo(title, steps):
    st = None; REPORT.append(f"**{title}**\n")
    for q, test in steps:
        r = ask(q, state=st); st = r["state"]; okk = bool(test(r)); check(f"D:{title}:{q}", okk, r["text"][:100]); REPORT.append(f"- You: *{q}* → {'PASS' if okk else 'FAIL'} · {r['rows']} rows · {preview(r)[:170]}")
    REPORT.append("")
convo("City switch", [("Revenue by tier", lambda r: r["status"] == "ok"), ("What about Mumbai?", lambda r: "Mumbai" in r["sql"] and r["intent"].get("followup")), ("And Pune?", lambda r: "Pune" in r["sql"] and "Mumbai" not in r["sql"]), ("all cities", lambda r: "WHERE" not in r["sql"])])
convo("Top-N, more, reverse", [("Which outlets have the highest net margin?", lambda r: r["rows"] == 10), ("only top 3", lambda r: r["rows"] == 3), ("show more", lambda r: r["rows"] > 3), ("reverse it", lambda r: "ASC" in r["sql"])])
convo("Pronoun (it / that outlet)", [("Which outlets have the highest net margin?", lambda r: r["status"] == "ok"), ("why is it flagged?", lambda r: "flagged" in r["text"].lower() and r["intent"].get("followup"))])
convo("Threshold change", [("Which outlets have customer satisfaction below 3.5?", lambda r: "< 3.5" in r["sql"]), ("what about below 3.2?", lambda r: "< 3.2" in r["sql"])])
convo("Generic SQL follow-up", [("Top 5 outlets by revenue", lambda r: r["rows"] == 5), ("what about Kochi?", lambda r: "Kochi" in r["sql"])])

# ------------------------------------------------------------------ E. role-based restrictions
print("E. roles"); REPORT.append("## E. Role-based data restrictions\n")
SC = "OUT-007"
ROLE_CASES = [  # question, role, scope, expected status, extra check
 ("What is the average salary by role?", "Admin", None, "ok", None), ("What is the average salary by role?", "Franchise Owner", None, "ok", None), ("What is the average salary by role?", "HR Lead", None, "ok", None),
 ("What is the average salary by role?", "Store Manager", SC, "denied", None), ("What is the average salary by role?", "Staff", SC, "denied", None), ("What is the average salary by role?", "Customer", None, "denied", None),
 ("Which outlets have the highest net margin?", "Admin", None, "ok", None), ("Which outlets have the highest net margin?", "Store Manager", SC, "ok", lambda r: "OUT-007" in r["sql"]),
 ("Which outlets have the highest net margin?", "Staff", SC, "denied", None), ("Which outlets have the highest net margin?", "Customer", None, "denied", None), ("Which outlets have the highest net margin?", "HR Lead", None, "denied", None),
 ("Revenue by city", "Store Manager", SC, "denied", None), ("Revenue by city", "Customer", None, "denied", None), ("Revenue by city", "Marketing Manager", None, "ok", None),
 ("Which staff are at high risk of leaving?", "Store Manager", SC, "ok", lambda r: "name" not in r["text"].split("\n\n")[1].split("\n")[0] if "\n\n" in r["text"] else True),
 ("Which staff are at high risk of leaving?", "HR Lead", None, "ok", lambda r: "| name |" in r["text"]), ("Which staff are at high risk of leaving?", "Staff", SC, "denied", None), ("Which staff are at high risk of leaving?", "Customer", None, "denied", None),
 ("Which outlets have the best customer ratings?", "Customer", None, "ok", None), ("Which outlets have the best customer ratings?", "Staff", SC, "denied", None),
 ("Show pending purchase orders", "Staff", SC, None, lambda r: r["status"] in ("ok", "empty") and "OUT-007" in r["sql"]), ("Show pending purchase orders", "Customer", None, "denied", None),
 ("What is the marketing spend by channel?", "Marketing Manager", None, "ok", None), ("What is the marketing spend by channel?", "HR Lead", None, "denied", None), ("What is the marketing spend by channel?", "Customer", None, "denied", None),
 ("Show pending purchase orders", "Store Manager", None, "denied", lambda r: "not linked to an outlet" in r["text"]),
 ("Compare OUT-007 and OUT-012", "Store Manager", SC, "denied", None), ("Tell me about OUT-012", "Store Manager", SC, "denied", lambda r: "outside your outlet scope" in r["text"]),
 ("Give me an overview of the business", "Customer", None, "ok", lambda r: "Total revenue" not in r["text"] and "Operating costs" not in r["text"]),
 ("Give me an overview of the business", "Admin", None, "ok", lambda r: "Total revenue" in r["text"]),
 ("Top 5 outlets by revenue", "Customer", None, "denied", None), ("Top 5 outlets by revenue", "Admin", None, "ok", None), ("Average salary of staff", "Store Manager", SC, "denied", None),
 ("Which outlets have the most overtime?", "Customer", None, "denied", None), ("Which audits failed?", "Staff", SC, "denied", None),
]
REPORT.append("| # | Role | Scope | Question | Expected | Got | Rows | Result |\n|---|---|---|---|---|---|---|---|")
for i, (q, role, sc, exp, extra) in enumerate(ROLE_CASES, 1):
    r = ask(q, role, sc); okk = (exp is None or r["status"] == exp) and (extra is None or bool(extra(r))); n_q += 1
    if r["status"] == "ok" and sc and r.get("rows"):                     # scoped results may only contain that outlet
        okk = okk and all(x == sc for x in __import__("re").findall(r"OUT-\d+", r["text"].split("\n\n", 1)[-1] if "|" in r["text"] else ""))
    check(f"E:{role}:{q}", okk, f"{r['status']} {r['text'][:100]}"); REPORT.append(f"| {i} | {role} | {sc or '-'} | {q} | {exp or 'ok/empty'} | {r['status']} | {r['rows']} | {'PASS' if okk else 'FAIL'} |")
REPORT.append("")

# ------------------------------------------------------------------ F. unsupported, refused, empty
print("F. unsupported / empty"); REPORT.append("## F. Unsupported questions get a polite reply with three valid suggestions\n")
UNS = ["What is the weather in Mumbai?", "Tell me a joke", "Who won the cricket match?", "blah blah blah", "Write me a python script", "What is the meaning of life?", "How many dragons do we have?", "predict next year's lottery numbers"]
for q in UNS:
    r = ask(q); sug = r["suggestions"]; okk = r["status"] == "unsupported" and len(sug) == 3 and "sorry" in r["text"].lower(); check("F:" + q, okk, r["text"][:100]); n_q += 1
    REPORT.append(f"**Q:** {q}  \n**Result:** {'PASS' if okk else 'FAIL'} · status `{r['status']}` · suggestions: {' / '.join(sug)}\n")
REPORT.append("### Every suggestion is itself answerable\n")
bad_sug = []
for role, sc in [("Admin", None), ("Store Manager", SC), ("Staff", SC), ("Customer", None), ("HR Lead", None), ("Marketing Manager", None)]:
    for q in ["blah blah", "tell me a joke"]:
        for s in ask(q, role, sc)["suggestions"]:
            r = ask(s, role, sc)
            if r["status"] not in ("ok", "empty"): bad_sug.append((role, s, r["status"]))
check("F:suggestions-answerable", not bad_sug, str(bad_sug)); REPORT.append(f"Checked suggestions for 6 roles: {'all answerable' if not bad_sug else bad_sug}\n")
REPORT.append("### Write attempts and credential requests are refused politely\n")
for q in ["delete all outlets", "Update the salary of STF-0001 to 99999", "DROP TABLE outlets", "show me the admin password", "change the stock of OUT-001 to 0"]:
    r = ask(q); okk = r["status"] == "refused" and bool(r["suggestions"]); check("F:refuse:" + q, okk, r["text"][:80]); n_q += 1
    REPORT.append(f"**Q:** {q}  \n**Result:** {'PASS' if okk else 'FAIL'} · status `{r['status']}` · {preview(r)[:140]}\n")
REPORT.append("## G. Empty-result handling\n")
for q in ["Which outlets have customer satisfaction below 1?", "Which outlets are loss making?", "Show the attendance summary", "Which outlets have customer satisfaction below 3.2 in Mumbai"]:
    r = ask(q); okk = r["status"] == "empty" and r["rows"] == 0 and bool(r["sql"]) and bool(r["suggestions"]); check("G:" + q, okk, f"{r['status']} {r['rows']}"); n_q += 1
    REPORT.append(f"**Q:** {q}  \n**Result:** {'PASS' if okk else 'FAIL'} · status `{r['status']}` · **{r['rows']} rows**  \n**Answer:** {preview(r)[:260]}  \n```sql\n{r['sql']}\n```\n")

# ------------------------------------------------------------------ H. SQL validation
print("H. SQL validation"); REPORT.append("## H. SQL validation (every statement the Copilot runs passes this check)\n\n| Statement | Expected | Got |\n|---|---|---|")
GOOD = ["SELECT * FROM outlets LIMIT 5", "SELECT COUNT(*) FROM staff", "SELECT o.tier, COUNT(*) FROM outlets o GROUP BY o.tier ORDER BY 2 DESC LIMIT ?", "SELECT * FROM inventory i JOIN outlets o ON o.outlet_id=i.outlet_id WHERE o.location = ? LIMIT 10"]
BAD = ["DELETE FROM outlets", "DROP TABLE outlets", "SELECT 1; DROP TABLE outlets", "SELECT * FROM users LIMIT 5", "SELECT * FROM outlets", "SELECT * FROM outlets LIMIT 99999", "SELECT * FROM outlets LIMIT 5 -- hi",
       "PRAGMA table_info(outlets)", "UPDATE outlets SET revenue=0", "INSERT INTO outlets VALUES (1)", "ATTACH DATABASE 'x.db' AS x", "SELECT * FROM sqlite_master LIMIT 5", "", "SELECT * FROM outlets LIMIT 5; SELECT * FROM staff LIMIT 5",
       "WITH x AS (SELECT 1) DELETE FROM outlets", "SELECT * FROM outlets /* c */ LIMIT 5", "SELECT * FROM chat_history LIMIT 5"]
for s in GOOD:
    try: QA.validate_sql(s); okk = True
    except Exception: okk = False
    check("H:good:" + s, okk); REPORT.append(f"| `{s}` | accept | {'accept' if okk else 'REJECT'} |")
for s in BAD:
    try: QA.validate_sql(s); okk = False
    except QA.SQLRejected: okk = True
    check("H:bad:" + s, okk); REPORT.append(f"| `{s or '(empty)'}` | reject | {'reject' if okk else 'ACCEPTED'} |")
try: c = QA._ro_conn(); c.execute("UPDATE outlets SET revenue = revenue"); okk = False
except Exception: okk = True
check("H:read-only-connection", okk); REPORT.append(f"| *(write attempted on the Copilot's connection)* | blocked | {'blocked' if okk else 'ALLOWED'} |\n")
bad_state = IR.sanitize_state({"key": "x'; DROP TABLE outlets;--", "params": {"city": "x" * 500, "n": "99; DROP", "outlet": "OUT-1' OR 1=1"}, "entity": "OUT-1; --"})
check("H:state-sanitised", bad_state.get("key") is None and "city" not in bad_state.get("params", {}) and "outlet" not in bad_state.get("params", {}) and "entity" not in bad_state)

# ------------------------------------------------------------------ I. end to end over HTTP (server-sent events)
print("I. HTTP"); REPORT.append("## I. End-to-end over HTTP (`POST /api/copilot`)\n")
try:
    from fastapi.testclient import TestClient
    import server
    with TestClient(server.app) as cl:
        resp = cl.post("/api/copilot", json={"q": "Which marketing channel works best?"}); evs = [json.loads(x[6:]) for x in resp.text.split("\n\n") if x.startswith("data: ")]
        fin = next(e for e in evs if e["ev"] == "final")
        okk = resp.status_code == 200 and fin.get("rows") and fin.get("sql", "").startswith("SELECT") and fin.get("validated") and fin.get("state", {}).get("key") == "channel_performance"; check("I:http-final", bool(okk))
        r2 = cl.post("/api/copilot", json={"q": "what about Mumbai?", "state": fin["state"]}); f2 = next(json.loads(x[6:]) for x in r2.text.split("\n\n") if x.startswith("data: ") and '"final"' in x)
        check("I:http-followup", "Mumbai" in f2["sql"]); r3 = cl.post("/api/copilot", json={"q": "tell me a joke"}); f3 = next(json.loads(x[6:]) for x in r3.text.split("\n\n") if x.startswith("data: ") and '"final"' in x)
        check("I:http-unsupported", f3["status"] == "unsupported" and len(f3["suggestions"]) == 3)
        r4 = cl.post("/api/copilot", json={"q": "revenue by tier", "state": {"key": "evil", "params": {"n": "x"}}}); check("I:http-bad-state", r4.status_code == 200)
        REPORT.append(f"Final event fields: `{', '.join(sorted(fin.keys()))}`  \nRows: {fin['rows']} · validated: {fin['validated']} · state key: `{fin['state'].get('key')}`  \nFollow-up over HTTP produced SQL filtered to Mumbai: {'PASS' if 'Mumbai' in f2['sql'] else 'FAIL'}\n")
except Exception as e:
    check("I:http", False, repr(e)); REPORT.append(f"HTTP test failed: {e!r}\n")

total = len(PASS) + len(FAIL)
head = f"# AI Copilot Enhancement - Test Report\n\nGenerated {datetime.datetime.now():%Y-%m-%d %H:%M} against the seeded franchise database ({COUNTS['outlets']} outlets, {COUNTS['staff']:,} staff, {COUNTS['inventory']:,} inventory records).\n\n**{len(PASS)} of {total} checks passed.** {n_q} questions were asked through `intent_router.respond()`.\n\n" + \
       "| Section | What is tested |\n|---|---|\n| A | 30 new whitelisted questions x 3 phrasings (clean / alternative wording / misspelled) + 4 multi-query analyst questions |\n| B | Filters in the question: city, tier, outlet, top-N, thresholds, days |\n| C | Existing questions still work |\n| D | Follow-up questions |\n| E | Role-based data restrictions |\n| F | Unsupported questions, refusals, suggestions |\n| G | Empty-result handling |\n| H | SQL validation and read-only protection |\n| I | End-to-end over HTTP |\n\n"
open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "COPILOT_TEST_REPORT.md"), "w").write(head + "\n".join(REPORT) + "\n")
print(f"\n{len(PASS)}/{total} checks passed" + (f"; FAILED: {FAIL}" if FAIL else ""))
sys.exit(1 if FAIL else 0)


In [ ]:
import subprocess, sys
print("=== RUNNING COPILOT TEST SUITE ===")
res = subprocess.run([sys.executable, "/content/app/test_copilot.py"], capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("STDERR:", res.stderr)


In [ ]:
# RBAC verification. Run it AFTER re-running the %%writefile cells and starting the app once (so the live tables exist).
# Part 1: roles, menus, capabilities.  Part 2: outlet assignment on a throw-away database.
# Part 3: the real FastAPI routes, called as each role (no web server needed, nothing is written to your data).
import os, sys, re, json, sqlite3, tempfile, subprocess
os.environ.setdefault("APP_KIND", "Franchise"); os.environ["NO_WORKER"] = "1"; os.environ["RATE_LIMIT_OFF"] = "1"
sys.path.insert(0, "/content/app"); os.chdir("/content/app")
for m in ("server", "rbac", "db", "localuser", "views_core", "views_franchise"): sys.modules.pop(m, None)   # test the code just written
try: from fastapi.testclient import TestClient
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx"]); from fastapi.testclient import TestClient
import config, rbac, db

PASS, FAIL = [], []
def check(name, cond):
    (PASS if cond else FAIL).append(name)
    if not cond: print("FAIL:", name)
def is403(fn, *a):
    try: fn(*a)
    except Exception as e: return getattr(e, "status_code", None) == 403
    return False

ADMIN, OWNER, SM, STAFF, AUD = rbac.ROLES
U = lambda role, outlet=None: {"role": role, "outlet_id": outlet, "email": role.split()[0].lower() + "@test", "username": role}

# ---------------------------------------------------------------- 1. roles, menus, capabilities
check("exactly five roles", len(rbac.ROLES) == 5 and set(rbac.ROLES) == set(rbac.ROLE_MENU) == set(rbac.ROLE_CAPS))
check("menu_for: Admin sees Copilot", "copilot" in rbac.menu_for(U(ADMIN)))
check("menu_for: Staff sees Copilot", "copilot" in rbac.menu_for(U(STAFF)))
check("menu_for accepts a role string", rbac.menu_for(AUD) == sorted(rbac.ROLE_MENU[AUD]))
check("role names are normalised", rbac.normalize_role("store manager") == SM and rbac.normalize_role("owner") == OWNER and rbac.normalize_role("Customer") == STAFF)
ALLOWED = {ADMIN: {"run", "manage_users", "assign_outlet", "audit_write", "review"}, OWNER: {"run", "audit_write", "review"},
           SM: {"view", "review", "resolve"}, STAFF: {"view", "review"}, AUD: {"view", "export"}}
DENIED = {OWNER: {"manage_users", "assign_outlet", "admin"}, SM: {"run", "audit_write", "manage_users", "assign_outlet"},
          STAFF: {"run", "audit_write", "manage_users", "resolve"}, AUD: set(rbac.WRITE_CAPS)}
for role, caps in ALLOWED.items():
    for cap in caps:
        check(f"{role} may {cap}", rbac.has_capability(U(role), cap) and rbac.need({"user": U(role)}, cap) is True)
for role, caps in DENIED.items():
    for cap in caps:
        check(f"{role} may NOT {cap}", not rbac.has_capability(U(role), cap) and is403(rbac.need, {"user": U(role)}, cap))
check("Auditor is read-only (no write capability at all)", not (rbac.ROLE_CAPS[AUD] & rbac.WRITE_CAPS))
check("scope_for: Admin / Owner / Staff / Auditor are not outlet-scoped", all(rbac.scope_for(U(r)) is None for r in (ADMIN, OWNER, STAFF, AUD)))
check("scope_for: Store Manager is limited to the assigned outlet", rbac.scope_for(U(SM, "OUT-001")) == "OUT-001")
check("scope_for: unassigned Store Manager fails closed", rbac.scope_for(U(SM)) == rbac.NO_OUTLET)
ctx = {"user": U(SM, "OUT-001"), "scope": "OUT-001"}
check("Store Manager may use own outlet", rbac.enforce_outlet(ctx, "OUT-001") is True)
check("Store Manager is denied another outlet", is403(rbac.enforce_outlet, ctx, "OUT-002"))
check("unassigned Store Manager is denied every outlet", is403(rbac.enforce_outlet, {"user": U(SM), "scope": rbac.NO_OUTLET}, "OUT-001"))
check("Franchise Owner may use any outlet", rbac.enforce_outlet({"user": U(OWNER)}, "OUT-002") is True)

# ---------------------------------------------------------------- 2. outlet assignment (throw-away database)
real_path = db.DB_PATH
db.DB_PATH = os.path.join(tempfile.mkdtemp(), "rbac_test.db"); db.init_db()
with db.get_conn() as c: c.executemany("INSERT INTO outlets(outlet_id, outlet_name) VALUES (?, ?)", [("OUT-T1", "T1"), ("OUT-T2", "T2")])
db.seed_role_users(); db.seed_role_users()                                                  # second call must change nothing
users = {u["role"]: u for u in db.list_users()}
check("one demo account per role, created once", set(users) == set(rbac.ROLES) and len(db.list_users()) == 5)
check("demo Store Manager starts with an outlet", users[SM]["outlet_id"] == "OUT-T1")
db.assign_user_outlet("manager", "OUT-T2")
with db.get_conn() as c: rows = c.execute("SELECT outlet_id FROM user_outlet_assignments WHERE username='manager'").fetchall()
check("re-assigning replaces the old outlet (one row only)", rows == [("OUT-T2",)])
check("get_user_by_email returns the assigned outlet", db.get_user_by_email("MANAGER@infosys.com")["outlet_id"] == "OUT-T2")
for label, args in (("a non-manager", ("staff", "OUT-T1")), ("an unknown outlet", ("manager", "OUT-NOPE")), ("an unknown user", ("ghost", "OUT-T1"))):
    try: db.assign_user_outlet(*args); check(f"assignment rejected for {label}", False)
    except ValueError: check(f"assignment rejected for {label}", True)
db.DB_PATH = real_path

# ---------------------------------------------------------------- 3. the real routes, called as each role
import server
cl = TestClient(server.app, raise_server_exceptions=False)
def login(u): server.user_of = lambda request, u=u: {**u, "role": rbac.normalize_role(u["role"])}
def call(method, url, body=None): return cl.request(method, url, json=body).status_code
def ids_in(resp): return set(re.findall(r"OUT-\d{3}", json.dumps(resp.json())))

con = sqlite3.connect(config.DB_PATH)
tables = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
outs = [r[0] for r in con.execute("SELECT outlet_id FROM outlets ORDER BY outlet_id LIMIT 2")] if "outlets" in tables else []
live = "attendance" in tables and len(outs) == 2
if not live: print("NOTE: start the app once (launcher cell) so the live tables exist; page-level tests were skipped.")
O1, O2 = outs if live else ("OUT-001", "OUT-002")
PAGES = ["copilot", "workforce", "outlets", "inventory", "marketing", "sentiment", "audit"]

for role in rbac.ROLES:                                                                      # menu / page access
    login(U(role, O1 if role == SM else None)); me = cl.get("/api/me").json()
    check(f"{role}: /api/me lists exactly its capabilities", set(me["caps"]) == rbac.ROLE_CAPS[role])
    check(f"{role}: /api/me menu matches ROLE_MENU", {m["id"] for m in me["menu"]} == rbac.ROLE_MENU[role])
    if live:
        for pid in PAGES:
            want = 200 if pid in rbac.ROLE_MENU[role] else 403
            check(f"{role}: GET /api/page/{pid} -> {want}", call("GET", f"/api/page/{pid}") == want)

POSTS = [("/api/run_cycle", "run", {}), ("/api/run_agent", "run", {"agent": "nope"}), ("/api/weather/refresh", "run", {}),
         ("/api/admin/assign-outlet", "assign_outlet", {}), ("/api/admin/users/role", "manage_users", {}),
         ("/api/franchise/review", "review", {"outlet": O1, "comment": ""}), ("/api/franchise/audit", "audit_write", {"outlet": O1}),
         ("/api/freight/quote", "quote", {}), ("/api/freight/storm", "admin", {}), ("/api/freight/reroute", "approve", {}), ("/api/freight/attach", "attach_doc", {})]
for role in rbac.ROLES:
    login(U(role, O1 if role == SM else None))
    for url, cap, body in POSTS:
        held = cap in rbac.ROLE_CAPS[role]
        if held and url in ("/api/run_cycle", "/api/weather/refresh") or held and url.startswith("/api/freight"):
            continue                                                                         # allowed + would really execute: covered by has_capability above
        code = call("POST", url, body)
        check(f"{role}: POST {url} {'passes the permission gate' if held else '-> 403'}", (code != 403) if held else (code == 403))
login(U(ADMIN)); check("Admin: GET /api/admin/users -> 200", call("GET", "/api/admin/users") == 200)
for role in (OWNER, SM, STAFF, AUD):
    login(U(role, O1 if role == SM else None)); check(f"{role}: GET /api/admin/users -> 403", call("GET", "/api/admin/users") == 403)
login(U(ADMIN)); check("Admin: setting an unsupported role -> 400", call("POST", "/api/admin/users/role", {"username": "x", "role": "Root"}) == 400)

for role in (STAFF, AUD, SM):                                                                # Copilot is available to every role
    login(U(role, O1 if role == SM else None))
    check(f"{role}: POST /api/copilot passes the permission gate", call("POST", "/api/copilot", {"q": "hi"}) != 403)
    check(f"{role}: GET /api/chats passes the permission gate", call("GET", "/api/chats") != 403)
    check(f"{role}: POST /api/chats/save passes the permission gate", call("POST", "/api/chats/save", {"msgs": [{"role": "user", "text": "x"}]}) != 403)

if live:
    login(U(SM, O1))                                                                         # Store Manager of one outlet
    check("Store Manager: review for another outlet -> 403", call("POST", "/api/franchise/review", {"outlet": O2, "comment": "x", "rating": 4}) == 403)
    check("Store Manager: review for own outlet passes the gate", call("POST", "/api/franchise/review", {"outlet": O1, "comment": ""}) == 400)
    check("Store Manager: detail of own outlet -> 200", call("GET", f"/api/detail/{O1}") == 200)
    check("Store Manager: detail of another outlet -> 403", call("GET", f"/api/detail/{O2}") == 403)
    check("Store Manager: detail of a non-outlet entity -> 403", call("GET", "/api/detail/EMP-0001") == 403)
    for pid in ("workforce", "inventory", "sentiment", "outlets", "audit"):
        seen = ids_in(cl.get(f"/api/page/{pid}")); check(f"Store Manager: /{pid} shows only the assigned outlet ({sorted(seen)})", seen <= {O1})
    login(U(ADMIN)); admin_open = cl.get("/api/pulse").json()["open"]
    login(U(SM, O1)); p = cl.get("/api/pulse").json()
    check("Store Manager: header counters are outlet-scoped, no global health", p["open"] <= admin_open and p["health"] is None)
    login(U(SM))                                                                             # Store Manager with NO outlet assigned
    check("unassigned Store Manager: /api/me reports no outlet", cl.get("/api/me").json()["outlet_assigned"] is False)
    for pid in ("workforce", "inventory", "sentiment", "outlets", "audit"):
        seen = ids_in(cl.get(f"/api/page/{pid}")); check(f"unassigned Store Manager: /{pid} shows no outlet data ({len(seen)} outlets)", not seen)
    check("unassigned Store Manager: detail -> 403", call("GET", f"/api/detail/{O1}") == 403)
    check("unassigned Store Manager: review -> 403", call("POST", "/api/franchise/review", {"outlet": O1, "comment": "x", "rating": 4}) == 403)
    for role in (OWNER, AUD, STAFF):
        login(U(role)); check(f"{role}: detail of any outlet -> 200", call("GET", f"/api/detail/{O2}") == 200)
    for role in (OWNER, AUD):
        login(U(role)); check(f"{role}: /sentiment shows every outlet", len(ids_in(cl.get("/api/page/sentiment"))) > 1)
    login(U(AUD))
    body = json.dumps(cl.get("/api/page/sentiment").json()) + json.dumps(cl.get("/api/page/audit").json())
    check("Auditor: no 'Add a review' form, no 'Log an inspection' form, no 'Re-run' button", not any(s in body for s in ("/api/franchise/review", "/api/franchise/audit", "/api/run_cycle")))
    login(U(OWNER)); body = json.dumps(cl.get("/api/page/sentiment").json()) + json.dumps(cl.get("/api/page/audit").json())
    check("Franchise Owner: sees the review and inspection forms", "/api/franchise/review" in body and "/api/franchise/audit" in body)
    if "audit_log" in tables:                                                                # denials are recorded
        n0 = con.execute("SELECT COUNT(*) FROM audit_log WHERE event='access_denied'").fetchone()[0]
        login(U(AUD)); call("POST", "/api/run_cycle", {})
        n1 = sqlite3.connect(config.DB_PATH).execute("SELECT COUNT(*) FROM audit_log WHERE event='access_denied'").fetchone()[0]
        check("a denied request is written to the audit log", n1 == n0 + 1)

print(f"\nRBAC tests: {len(PASS)} passed, {len(FAIL)} failed")
assert not FAIL, FAIL
print("All RBAC checks passed: five roles, ROLE_MENU / ROLE_CAPS, menu_for / need, backend enforcement, outlet assignment, outlet isolation, read-only Auditor, and no unauthorised agent runs.")


In [ ]:
import os, subprocess, sys, time, urllib.request

# Forward Google Colab Secrets (key icon) to environment variables for the uvicorn subprocess
try:
    from google.colab import userdata
    for k in ["EMAIL_ID", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "JWT_SECRET"]:
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = str(v)
        except Exception:
            pass
    print("Colab Secrets successfully forwarded to server environment.")
except Exception as e:
    print(f"Colab secrets note: {e}")

os.chdir('/content/app')
os.environ['APP_KIND'] = 'Franchise'
os.environ['RATE_LIMIT_OFF'] = '1'
subprocess.run('fuser -k 8000/tcp', shell=True, capture_output=True)
log = open('/content/app/server.log', 'w')
proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'], stdout=log, stderr=log, cwd='/content/app')
for _ in range(90):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2); print('FranchiseOps AI is running'); break
    except Exception: time.sleep(2)
else:
    print(open('/content/app/server.log').read()[-3000:])


In [ ]:
from IPython.display import HTML, display
try:
    from google.colab import output
    url = output.eval_js("google.colab.kernel.proxyPort(8000, {'cache': false})")
    display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">Open the app</a><br><small>{url}</small>'))
except Exception as e:
    print('Colab proxy unavailable:', e)
    try:
        from google.colab import userdata
        from pyngrok import ngrok
        ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
        url = ngrok.connect(8000).public_url
        display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">Open the app</a><br><small>{url}</small>'))
    except Exception as e2:
        print('ngrok failed:', e2)
